In [1]:
pwd()

"/Users/josenino-mora/Documents/GitHub/paperPCLsensingErrors/notebooks"

In [2]:
Sys.CPU_THREADS

8

In [3]:
Threads.nthreads()

8

In [ ]:
using Base.Threads

println("Threads: ", Threads.nthreads())

@threads for i in 1:Threads.nthreads()
    @show i
end

In [4]:
using Pkg

@show VERSION
@show Base.active_project()
Pkg.status()

VERSION = v"1.12.6"
Base.active_project() = "/Users/josenino-mora/.julia/environments/v1.12/Project.toml"
Status `~/.julia/environments/v1.12/Project.toml`
  [336ed68f] CSV v0.10.16
  [a93c6f00] DataFrames v1.8.2
  [31c24e10] Distributions v0.25.127
  [7073ff75] IJulia v1.34.4
  [b964fa9f] LaTeXStrings v1.4.0
  [8314cec4] PGFPlotsX v1.6.2
  [2913bbd2] StatsBase v0.34.12


In [ ]:
import Pkg
Pkg.update() # remove it when freezing this
# Pkg.instantiate()

In [ ]:
using Pkg
Pkg.add(["PGFPlotsX", "LaTeXStrings", "StatsBase", "Distributions"])  # install once
Pkg.add("CSV")
Pkg.add("DataFrames")

# Optional, only if needed:
#Pkg.build("PGFPlotsX")

In [ ]:
using PGFPlotsX, LaTeXStrings
using CSV
using DataFrames
# Add calrsfs once, somewhere near the start of your session / script:
push!(PGFPlotsX.CUSTOM_PREAMBLE, raw"\usepackage{calrsfs}")

In [ ]:
using PGFPlotsX
using Printf

struct ChrNode
    u::String
    v::String
end

function christoffel_tree_levels(n::Integer)
    @assert n ≥ 0
    levels = Vector{Vector{ChrNode}}(undef, n + 1)
    levels[1] = [ChrNode("0", "1")]  # level 0
    for ℓ in 0:(n-1)
        prev = levels[ℓ+1]
        cur  = Vector{ChrNode}(undef, 2 * length(prev))
        for (i, node) in enumerate(prev)
            u, v = node.u, node.v
            uv   = u * v
            cur[2i - 1] = ChrNode(u,  uv)  # left child
            cur[2i]     = ChrNode(uv, v)   # right child
        end
        levels[ℓ+2] = cur
    end
    return levels
end

function compact_word(w::String; maxlen::Int=18)
    length(w) ≤ maxlen && return w
    k = max(1, maxlen ÷ 2)
    return first(w, k) * "…" * last(w, k)
end

function node_pos(ℓ::Int, j::Int, n::Int; xunit=1.4, yunit=1.2)
    m  = 2.0^ℓ
    x  = (j - (m - 1)/2) * 2.0^(n - ℓ) * xunit
    y  = -ℓ * yunit
    return x, y
end

function christoffel_tree_tikz(n::Integer;
        xunit=1.4, yunit=1.2,
        max_label_len=18,
        node_style="font=\\scriptsize, inner sep=1.2pt, rounded corners=1pt, draw",
        edge_style="line width=0.3pt",
    )

    n = Int(n)
    levels = christoffel_tree_levels(n)

    lines = String[]
    push!(lines, raw"\begin{tikzpicture}[>=Latex]")
    push!(lines, "\\tikzset{chrdnode/.style={$node_style}}")
    push!(lines, "\\tikzset{chredge/.style={$edge_style}}")

    # nodes
    for ℓ in 0:n
        nodes = levels[ℓ+1]
        for j in 0:(length(nodes)-1)
            node = nodes[j+1]
            x, y = node_pos(ℓ, j, n; xunit=xunit, yunit=yunit)
            name = "N$(ℓ)_$(j)"

            u = compact_word(node.u; maxlen=max_label_len)
            v = compact_word(node.v; maxlen=max_label_len)
            label = "\$({\\mathtt{$u}},{\\mathtt{$v}})\$"

            push!(lines, @sprintf("\\node[chrdnode] (%s) at (%.6f,%.6f) {%s};",
                                  name, x, y, label))
        end
    end

    # edges
    for ℓ in 0:(n-1)
        for j in 0:(2^ℓ - 1)
            parent = "N$(ℓ)_$(j)"
            left   = "N$(ℓ+1)_$(2j)"
            right  = "N$(ℓ+1)_$(2j+1)"
            push!(lines, "\\draw[chredge] ($parent) -- ($left);")
            push!(lines, "\\draw[chredge] ($parent) -- ($right);")
        end
    end

    push!(lines, raw"\end{tikzpicture}")

    # KEY FIX: no Raw()
    return TikzPicture(join(lines, "\n"))
end

function save_christoffel_tree_pdf(outpdf::AbstractString, n::Integer;
        latex_engine::PGFPlotsX.LaTeXEngine = PGFPlotsX.PDFLaTeX(),
        include_preamble::Bool=true,
        buildflags::AbstractString="",
        showing_ide::Bool=false,
        kwargs...
    )

    old = copy(PGFPlotsX.CUSTOM_PREAMBLE)
    try
        push!(PGFPlotsX.CUSTOM_PREAMBLE, raw"\usepackage{tikz}")
        push!(PGFPlotsX.CUSTOM_PREAMBLE, raw"\usetikzlibrary{arrows.meta}")

        pic = christoffel_tree_tikz(n; kwargs...)

        PGFPlotsX.save(outpdf, pic;
            include_preamble=include_preamble,
            latex_engine=latex_engine,
            buildflags=buildflags,
            showing_ide=showing_ide
        )
        return outpdf
    finally
        empty!(PGFPlotsX.CUSTOM_PREAMBLE)
        append!(PGFPlotsX.CUSTOM_PREAMBLE, old)
    end
end

# Example:
# save_christoffel_tree_pdf("christoffel_tree_L5.pdf", 5; max_label_len=14, xunit=1.2, yunit=1.0)

In [ ]:
save_christoffel_tree_pdf("christoffel_tree_L3.pdf", 3;
    latex_engine = PGFPlotsX.PDFLaTeX(),   # or PGFPlotsX.LuaLaTeX(), PGFPlotsX.XeLaTeX()
    buildflags = "-interaction=nonstopmode -halt-on-error",
    max_label_len = 14,
    xunit = 1.2,
    yunit = 1.0
)

In [ ]:
# --- belief updates (no z) ---

"""
    φ0(x; q, ρ)

Passive belief update: φ⁰(x)=q+ρx.
"""
φ0(x; q, ρ) = q + ρ*x

"""
    φ1(x; q, ρ, κ)

Active no-ACK belief update: φ¹(x)= q + ρ * ((1-κ)x)/(1-κx).
"""
φ1(x; q, ρ, κ) = q + ρ * ((1 - κ) * x) / (1 - κ * x)


# --- helpers for τ_{[0,z]}^1 ---

"""
    φ1_fixed_points(; q, ρ, κ) -> (x_lo, x_hi)
"""
function φ1_fixed_points(; q, ρ, κ)
    B  = 1 - ρ + κ * (q + ρ)
    Δ  = B^2 - 4 * κ * q
    @assert Δ > 0 "Discriminant must be positive (check parameters)."
    sΔ = sqrt(Δ)
    xlo = (B - sΔ) / (2 * κ)
    xhi = (B + sΔ) / (2 * κ)
    return xlo, xhi
end


# --- crossing times (depend on z) ---

"""
    τ0(x, z; q, ρ)

Passive threshold up-crossing time: min{t≥0 : φ_t⁰(x) > z}.
Returns Inf if no up-crossing occurs.
"""
function τ0(x, z; q, ρ)
    x0 = q / (1 - ρ)

    if x > z
        return 0
    elseif z >= x0
        return Inf
    else
        val = log((x0 - z) / (x0 - x)) / log(ρ)   # log(ρ) < 0
        return 1 + floor(Int, val)
    end
end

"""
    τ1_0z(x, z; q, ρ, κ)

Active no-ACK down-crossing time into [0,z]:
min{t≥0 : φ_t¹(x) ≤ z}.

Returns 0 if x≤z, Inf if z≤x¹<x, else the closed form.
"""
function τ1_0z(x, z; q, ρ, κ)
    if x <= z
        return 0
    end

    x1, xhi = φ1_fixed_points(; q, ρ, κ)          # x1 = x_lo^1
    if z <= x1
        return Inf
    end

    μ  = ρ * (1 - κ) / (1 - κ * x1)^2             # (φ¹)'(x¹) ∈ (0,1)
    Ψ(u) = (u - x1) / (u - xhi)

    rhs = Ψ(z) / Ψ(x)                              # > 0 when x>z>x1
    val = log(rhs) / log(μ)                        # log(μ) < 0
    t = ceil(Int, val - 1e-12)                     # numerical guard
    return max(t, 0)
end


# Optional ASCII aliases:
phi0(x; q, rho) = φ0(x; q=q, ρ=rho)
phi1(x; q, rho, kappa) = φ1(x; q=q, ρ=rho, κ=kappa)
tau0(x, z; q, rho) = τ0(x, z; q=q, ρ=rho)
tau1_0z(x, z; q, rho, kappa) = τ1_0z(x, z; q=q, ρ=rho, κ=kappa)

In [ ]:
# Passive fixed point: x0 = q/(1-ρ)
x0_fixed(; q, ρ) = q / (1 - ρ)

"""
    x1_fixed(; q, ρ, κ) -> (x1, xhi, Δ)

Active no-ACK fixed points of φ¹:
solves κ x^2 - (1-ρ + κ(q+ρ)) x + q = 0
returns the in-[0,1] fixed point x1 (= x_lo^1), the other root xhi (= x_hi^1), and the discriminant Δ.
"""
function x1_fixed(; q, ρ, κ)
    B = 1 - ρ + κ*(q + ρ)
    Δ = B^2 - 4*κ*q
    @assert Δ > 0 "Discriminant must be positive; check parameters."
    sΔ = sqrt(Δ)

    x1  = (B - sΔ) / (2*κ)   # x_lo^1 ∈ (q, q+ρ)
    xhi = (B + sΔ) / (2*κ)   # x_hi^1 ∈ (1, 1/κ)
    return x1, xhi, Δ
end

"""
    fixed_points(; q, ρ, κ) -> (x0, x1, xhi)

Convenience wrapper returning both passive and active (no-ACK) fixed points.
"""
function fixed_points(; q, ρ, κ)
    x0 = x0_fixed(; q=q, ρ=ρ)
    x1, xhi, _ = x1_fixed(; q=q, ρ=ρ, κ=κ)
    return x0, x1, xhi
end

In [ ]:
# Skeleton map-with-a-gap ϕ
ϕ(x, z; q, ρ, κ) = (x > z) ? φ1(x; q=q, ρ=ρ, κ=κ) : φ0(x; q=q, ρ=ρ)

"""
    varphi_t(x, z, t; q, ρ, κ)

Iterates of the map-with-a-gap ϕ:
    ϕ₀(x,z)=x,
    ϕ_t(x,z)=ϕ(ϕ_{t-1}(x,z), z),  t≥1.

Assumes ϕ(x,z; q,ρ,κ) is already defined.
"""
function varphi_t(x, z, t::Integer; q, ρ, κ)
    @assert t ≥ 0 "t must be nonnegative"
    y = x
    for _ in 1:t
        y = ϕ(y, z; q=q, ρ=ρ, κ=κ)
    end
    return y
end

"""
    skeleton_orbit(x, z, T; q, ρ, κ)

Returns the skeleton orbit X̃₀..X̃_T, where X̃_t = ϕ_t(x,z).
"""
function skeleton_orbit(x, z, T::Integer; q, ρ, κ)
    @assert T ≥ 0 "T must be nonnegative"
    X = Vector{Float64}(undef, T + 1)
    y = x
    for t in 0:T
        X[t+1] = y
        if t < T
            y = ϕ(y, z; q=q, ρ=ρ, κ=κ)
        end
    end
    return X
end

"""
    skeleton_itinerary(x, z, T; q, ρ, κ)

Returns the skeleton itinerary Ã₀..Ã_T as Ints, where Ã_t = 1{X̃_t > z}.
"""
function skeleton_itinerary(x, z, T::Integer; q, ρ, κ)
    @assert T ≥ 0 "T must be nonnegative"
    A = Vector{Int}(undef, T + 1)
    y = x
    for t in 0:T
        A[t+1] = (y > z) ? 1 : 0
        if t < T
            y = ϕ(y, z; q=q, ρ=ρ, κ=κ)
        end
    end
    return A
end

"""
    skeleton_orbit_itinerary(x, z, T; q, ρ, κ)

One-pass computation of both (often handy):
Returns (X, A) with X[t+1]=X̃_t and A[t+1]=Ã_t for t=0..T.
"""
function skeleton_orbit_itinerary(x, z, T::Integer; q, ρ, κ)
    @assert T ≥ 0 "T must be nonnegative"
    X = Vector{Float64}(undef, T + 1)
    A = Vector{Int}(undef, T + 1)

    y = x
    for t in 0:T
        X[t+1] = y
        A[t+1] = (y > z) ? 1 : 0
        if t < T
            y = ϕ(y, z; q=q, ρ=ρ, κ=κ)
        end
    end
    return X, A
end

"""
    skeleton_itinerary_string(x, z, T; q, ρ, κ)

String encoding of the skeleton itinerary (Ã₀..Ã_T), where Ã_t = 1{ϕ_t(x,z) > z}.
Returns a String of length T+1 over {'0','1'}.
"""
function skeleton_itinerary_string(x, z, T::Integer; q, ρ, κ)
    @assert T ≥ 0 "T must be nonnegative"

    chars = Vector{Char}(undef, T + 1)
    y = x

    for t in 0:T
        chars[t+1] = (y > z) ? '1' : '0'
        if t < T
            y = ϕ(y, z; q=q, ρ=ρ, κ=κ)
        end
    end

    return String(chars)
end

# Optional ASCII aliases
varphi_t(x, z, t; q, rho, kappa) = varphi_t(x, z, t; q=q, ρ=rho, κ=kappa)  # same name; skip if confusing
Xtilde(x, z, T; q, rho, kappa) = skeleton_orbit(x, z, T; q=q, ρ=rho, κ=kappa)
Atilde(x, z, T; q, rho, kappa) = skeleton_itinerary(x, z, T; q=q, ρ=rho, κ=kappa)

In [ ]:
"""
    Gamma_t(x, z, t; q, ρ, κ)

No-ACK survival probability:
    Γ_t(x,z) = P_x^z{T ≥ t},  t ≥ 0

Recursion/product (paper):
    Γ_0 = 1
    Γ_t = ∏_{j=0}^{t-1} (1 - κ * X̃_j(x,z) * Ã_j(x,z))
        = ∏_{0 ≤ j < t : X̃_j(x,z) > z} (1 - κ * X̃_j(x,z))

Computed in one pass using the map-with-a-gap ϕ (no allocations).
"""
function Gamma_t(x, z, t::Integer; q, ρ, κ)
    @assert t ≥ 0 "t must be nonnegative"
    Γ = 1.0
    y = x
    for j in 0:(t-1)
        if y > z
            Γ *= (1 - κ * y)
        end
        y = ϕ(y, z; q=q, ρ=ρ, κ=κ)
    end
    return Γ
end


"""
    Gamma_path(x, z, T; q, ρ, κ)

Returns Γ[1]=Γ_0, ..., Γ[T+1]=Γ_T as a Vector{Float64}.
Computed in one pass using the recursion.
"""
function Gamma_path(x, z, T::Integer; q, ρ, κ)
    @assert T ≥ 0 "T must be nonnegative"
    Γ = Vector{Float64}(undef, T + 1)
    Γ[1] = 1.0
    y = x
    for t in 1:T
        # update from Γ_{t-1} to Γ_t using X̃_{t-1}=y and Ã_{t-1}=1{y>z}
        Γ[t+1] = Γ[t] * ((y > z) ? (1 - κ * y) : 1.0)
        y = ϕ(y, z; q=q, ρ=ρ, κ=κ)
    end
    return Γ
end


"""
    Gamma_t_from_skeleton(x, z, t; q, ρ, κ)

Same Γ_t(x,z) but uses your (X̃, Ã) vectors for clarity/debugging.
This allocates X and A up to t-1.
"""
function Gamma_t_from_skeleton(x, z, t::Integer; q, ρ, κ)
    @assert t ≥ 0 "t must be nonnegative"
    t == 0 && return 1.0
    X, A = skeleton_orbit_itinerary(x, z, t-1; q=q, ρ=ρ, κ=κ)
    Γ = 1.0
    for j in 0:(t-1-0)  # j=0..t-1
        if A[j+1] == 1
            Γ *= (1 - κ * X[j+1])
        end
    end
    return Γ
end


"""
    Gamma_infty_approx(x, z; q, ρ, κ, maxiter=10^7, tol=1e-14)

Practical approximation to Γ_∞(x,z) = lim_{t→∞} Γ_t(x,z).
Stops when |Γ_{t}-Γ_{t-1}| ≤ tol (i.e., multiplicative update ~ 1),
or after maxiter steps.

Note: In regimes where activation occurs infinitely often and κ>0 and X̃_j stays
bounded away from 0 on those times, Γ_t will typically decay to 0.
"""
function Gamma_infty_approx(x, z; q, ρ, κ, maxiter::Int=10^7, tol::Float64=1e-14)
    Γ = 1.0
    y = x
    for _ in 1:maxiter
        Γnew = Γ * ((y > z) ? (1 - κ * y) : 1.0)
        if abs(Γnew - Γ) ≤ tol
            return Γnew
        end
        Γ = Γnew
        y = ϕ(y, z; q=q, ρ=ρ, κ=κ)
    end
    return Γ
end


# Optional ASCII aliases
Gamma_t(x, z, t; q, rho, kappa) = Gamma_t(x, z, t; q=q, ρ=rho, κ=kappa)
Gamma_path(x, z, T; q, rho, kappa) = Gamma_path(x, z, T; q=q, ρ=rho, κ=kappa)

In [ ]:
"""
    plot_Gamma_path_axis(x, z, T; q, ρ, κ, ...)

Axis for Γ_t(x,z) vs t=0..T with NO grid and NO title by default.
"""
function plot_Gamma_path_axis(x, z, T::Integer; q, ρ, κ,
                              xlabel=L"$t$",
                              ylabel=L"$\Gamma_t(x,z)$",
                              ymin=0, ymax=1, xmin=0,
                              width="11cm", height="7cm",
                              mark="*")
    @assert T ≥ 0 "T must be nonnegative"

    Γ = Gamma_path(x, z, T; q=q, ρ=ρ, κ=κ)
    coords = collect(zip(0:T, Γ))

    return @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,
            ymin   => ymin,
            ymax   => ymax,
            xmin => xmin,
            xmax   => T,
            "xtick" => "{0,5,...,$T}",
            "ytick" => "{0,1}",
            "tick align" => "outside",
            "scaled y ticks" => "false",
            # no grid, no title
        },
        Plot(
            {
                mark => mark,
                "mark size" => "1.6pt",
                "line width" => "0.5pt",
            },
            Coordinates(coords)
        )
    )
end

"""
    save_Gamma_path_plot_pdf(outpdf, x, z, T; q, ρ, κ, engine="pdflatex", quiet=true)

Writes standalone .tex and compiles to PDF.
If quiet=true, suppresses LaTeX console output; prints log tail only on failure.
"""
function save_Gamma_path_plot_pdf(outpdf::AbstractString, x, z, T::Integer;
                                  q, ρ, κ,
                                  axisfun::Function = plot_Gamma_path_axis,   # <--- add
                                  engine::AbstractString="pdflatex",
                                  quiet::Bool=true,
                                  keep_tex::Bool=true,
                                  kwargs...)
    ax  = axisfun(x, z, T; q=q, ρ=ρ, κ=κ, kwargs...)  # <--- change
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

In [ ]:
# parameters (example)
q  = 0.25
ρ  = 0.6
κ  = 0.8

x0, x1, xhi = fixed_points(; q=q, ρ=ρ, κ=κ)
z = 0.5 * (x1 + x0)
x  = z - 0.3
T  = 25

save_Gamma_path_plot_pdf("Gamma_path.pdf", x, z, T; q=q, ρ=ρ, κ=κ)  # quiet by default

In [ ]:
"""
    plot_Gamma_diff01_path_axis(x, z, T; q, ρ, κ, ...)

Axis for ΔΓ_t = Γ_t(φ^0(x), z) - Γ_t(φ^1(x), z) vs t=0..T.
"""
function plot_Gamma_diff01_path_axis(x, z, T::Integer; q, ρ, κ,
                                     xlabel=L"$t$",
                                     ylabel=L"$\Gamma_t(\phi^0(x),z)-\Gamma_t(\phi^1(x),z)$",
                                     ymin=-1, ymax=1, xmin=0,
                                     width="11cm", height="7cm",
                                     mark="*")
    @assert T ≥ 0 "T must be nonnegative"

    # Use ASCII aliases (they forward to the Greek versions)
    x0 = phi0(x; q=q, rho=ρ)
    x1 = phi1(x; q=q, rho=ρ, kappa=κ)

    # Gamma_path uses Greek keywords in your codebase
    Γ0 = Gamma_path(x0, z, T; q=q, ρ=ρ, κ=κ)
    Γ1 = Gamma_path(x1, z, T; q=q, ρ=ρ, κ=κ)

    ΔΓ = Γ0 .- Γ1
    coords = collect(zip(0:T, ΔΓ))

    return @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,
            ymin   => ymin,
            ymax   => ymax,
            xmin   => xmin,
            xmax   => T,
            "xtick" => "{0,5,...,$T}",
            "tick align" => "outside",
            "scaled y ticks" => "false",
            # no grid, no title
        },
        Plot(
            {
                mark => mark,
                "mark size" => "1.6pt",
                "line width" => "0.5pt",
            },
            Coordinates(coords)
        )
    )
end

In [ ]:
# parameters (example)
q  = 0.25
ρ  = 0.6
κ  = 0.8

x0, x1, xhi = fixed_points(; q=q, ρ=ρ, κ=κ)
# z = 0.5 * (x1 + x0)
z = 0.5*q
x  = z + 0.2
T  = 25
save_Gamma_path_plot_pdf("GammaDiff.pdf", x, z, T; q=q, ρ=ρ, κ=κ,
                         axisfun=plot_Gamma_diff01_path_axis)

In [ ]:
###############################################################################
# Pre-ACK metrics with guaranteed truncation error (X̃_t ∈ [0,1])
#
# Uses a common truncation horizon T based on the geometric tail bound for G̃:
#   tail_geom(T) = β^(T+1)/(1-β).
#
# Guarantees:
#   |G̃ - G̃_T| ≤ tail_geom(T).
#   |F̃ - F̃_T| ≤ rκ * tail_geom(T).
# In particular, if rκ ≤ 1 (e.g., κ<1 and you assume r≤1 / ignore r),
# then the same T ensures both errors ≤ ε.
###############################################################################

"""
    trunc_horizon_common(β, ε)

Smallest integer T ≥ 0 such that the geometric tail bound satisfies
    β^(T+1)/(1-β) ≤ ε.
"""
# --- Unicode versions (canonical) ---

function trunc_horizon_common(β::Real, ε::Real)
    @assert 0 < β < 1
    @assert ε > 0
    rhs = ε * (1 - β)
    rhs >= 1 && return 0
    T = ceil(Int, log(rhs) / log(β) - 1)
    return max(T, 0)
end

function tildeG(x, z; q, ρ, κ, β, ε::Real=1e-8)
    T = trunc_horizon_common(β, ε)
    y, Γ, βt = x, 1.0, 1.0
    sG = 0.0
    for t in 0:T
        if y > z
            sG += βt * Γ
            Γ *= (1 - κ * y)
        end
        y  = ϕ(y, z; q=q, ρ=ρ, κ=κ)
        βt *= β
    end
    tailG = β^(T+1) / (1 - β)
    return (sG, T, tailG)
end

function tildeF(x, z; q, ρ, κ, r, β, ε::Real=1e-8)
    T = trunc_horizon_common(β, ε)
    y, Γ, βt = x, 1.0, 1.0
    sF = 0.0
    for t in 0:T
        if y > z
            sF += βt * Γ * y
            Γ *= (1 - κ * y)
        end
        y  = ϕ(y, z; q=q, ρ=ρ, κ=κ)
        βt *= β
    end
    Fapprox = r * κ * sF
    tailF   = (r * κ) * (β^(T+1) / (1 - β))
    return (Fapprox, T, tailF)
end

function tildeFG(x, z; q, ρ, κ, r, β, ε::Real=1e-8)
    T = trunc_horizon_common(β, ε)
    y, Γ, βt = x, 1.0, 1.0
    sF, sG = 0.0, 0.0
    for t in 0:T
        if y > z
            sF += βt * Γ * y
            sG += βt * Γ
            Γ *= (1 - κ * y)
        end
        y  = ϕ(y, z; q=q, ρ=ρ, κ=κ)
        βt *= β
    end
    tailG = β^(T+1) / (1 - β)
    tailF = (r * κ) * tailG
    return (r * κ * sF, sG, T, tailF, tailG)
end

"""
    tildeTheta(x, z; q, ρ, κ, β, ε=1e-8)

Compute Θ̃(x,z) by evaluating F̃(x,z) at r=1, since F̃ = r Θ̃.

Returns (Θ̃_approx, T, tailΘ_bound), where
  |Θ̃ - Θ̃_approx| ≤ tailΘ_bound.

Note: tail bound is taken directly from tildeF(...; r=1).
"""
function tildeTheta(x, z; q, ρ, κ, β, ε::Real=1e-8)
    F̂1, T, tailF1 = tildeF(x, z; q=q, ρ=ρ, κ=κ, r=1.0, β=β, ε=ε)
    Θ̂   = F̂1
    tailΘ = tailF1
    return (Θ̂, T, tailΘ)
end

"""
    tildeTheta_from_Ftilde(F̃; r)

Utility: Θ̃ = F̃/r given a computed F̃ (requires r>0).
"""
tildeTheta_from_Ftilde(F̃; r) = ( @assert r > 0; F̃ / r )


# --- Optional ASCII wrappers (different names, so no collision) ---

tildeF_ascii(x, z; q, rho, kappa, r, beta, eps=1e-8) =
    tildeF(x, z; q=q, ρ=rho, κ=kappa, r=r, β=beta, ε=eps)

tildeG_ascii(x, z; q, rho, kappa, beta, eps=1e-8) =
    tildeG(x, z; q=q, ρ=rho, κ=kappa, β=beta, ε=eps)

tildeFG_ascii(x, z; q, rho, kappa, r, beta, eps=1e-8) =
    tildeFG(x, z; q=q, ρ=rho, κ=kappa, r=r, β=beta, ε=eps)

In [ ]:
q  = 0.25
ρ  = 0.6
κ  = 0.8
β  = 0.95
r  = 1.0

x0, x1, xhi = fixed_points(; q=q, ρ=ρ, κ=κ)
z = 0.5 * (x1 + x0)
x = z + 0.1
ε = 1e-10

# Compute (F̃, G̃) one-pass (recommended)
F̂, Ĝ, T, tailF, tailG = tildeFG(x, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)

# Compute Θ̃ and compare to F̃/r
Θ̂, TΘ, tailΘ = tildeTheta(x, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε)
Fr = F̂ / r

# Display nicely
println("Parameters: q=$q, ρ=$ρ, κ=$κ, β=$β, r=$r")
println("Point: x=$x, z=$z, ε=$ε")
println()

println("Horizon used:")
println("  T(F̃,G̃) = $T   (tailG ≤ $tailG, tailF ≤ $tailF)")
println("  T(Θ̃)    = $TΘ  (tailΘ ≤ $tailΘ)")
println()

println("Values (side-by-side):")
println("  F̃(x,z)        = $(F̂)")
println("  G̃(x,z)        = $(Ĝ)")
println("  Θ̃(x,z)        = $(Θ̂)")
println("  F̃(x,z)/r      = $(Fr)")
println()

# Sanity checks
@assert T == TΘ
@assert isapprox(Θ̂, Fr; rtol=0, atol=1e-12)

In [ ]:
using PGFPlotsX

""" make_xticks_and_labels(x; q, ρ, κ, tol=1e-12) 
Returns (xticks, xtick_str, xticklab_str) where: 
- xticks is sorted increasing
- xtick_str is "{...}" for pgfplots xtick
- xticklab_str is "{...}" for pgfplots xticklabels
Includes ticks at 0, q, x, x^1, x^0, q+ρ, 1 (unique up to tol).
Labels are typeset with uniform vphantom alignment. """

function make_xticks_and_labels(x; q, ρ, κ, tol=1e-12)
    x0_fp, x1_fp, _ = fixed_points(; q=q, ρ=ρ, κ=κ)
    # candidates: (location, label)
    items = [
        (0.0, "\\ensuremath{0}"),
        (float(q), "\\ensuremath{p_{01}}"),
        (float(x), "\\ensuremath{x}"),
        (float(x1_fp),"\\ensuremath{x^{1}}"),
        (float(x0_fp),"\\ensuremath{x^{0}}"),
        (float(q+ρ), "\\ensuremath{p_{11}}"),
        (1.0, "\\ensuremath{1}"),
    ]

    # sort by location sort!(items, by=first)
    # merge near-duplicates (keep the first label among merged points)
    merged = Tuple{Float64,String}[]
    for (z, lab) in items
        if isempty(merged) || abs(z - merged[end][1]) > tol
            push!(merged, (z, lab))
        else 
            # same tick (within tol): keep the existing label; or combine if you prefer
            # merged[end] = (merged[end][1], merged[end][2] * "," * lab) # optional
        end
    end

    xticks = [z for (z, _) in merged]
    labels = [lab for (_, lab) in merged]
    
    # strings for PGFPlots
    xtick_str = "{" * join(string.(xticks), ",") * "}"

    # uniform strut so all labels sit at same height/depth
    phantom = "\\vphantom{\\ensuremath{q+\\rho}}"
    xticklab_str = "{" * join([phantom * lab for lab in labels], ",") * "}"

    return xticks, xtick_str, xticklab_str
end

# --- helper: compress sampled step function to segments + jump markers (right-continuous) ---
function _stepdata_right_continuous(zgrid::AbstractVector{<:Real},
        ygrid::AbstractVector{<:Real};
        atol::Real=1e-12)
    @assert length(zgrid) == length(ygrid) ≥ 2
    
    breaks = Int[]
    for i in 1:length(ygrid)-1
        if !isapprox(ygrid[i+1], ygrid[i]; atol=atol, rtol=0)
            push!(breaks, i) # breakpoint at zgrid[i+1]
        end
    end

    line_coords = Tuple{Float64,Float64}[]
    vline_coords = Tuple{Float64,Float64}[]
    # NEW: vertical jump segments
    open_marks = Tuple{Float64,Float64}[]
    fill_marks = Tuple{Float64,Float64}[]
    
    push!(fill_marks, (float(zgrid[1]), float(ygrid[1])))

    seg_start = 1
    for i in breaks
        zL = float(zgrid[seg_start])
        zB = float(zgrid[i+1])
        yL = float(ygrid[i])
        yR = float(ygrid[i+1])
        
        # horizontal segment
        push!(line_coords, (zL, yL))
        push!(line_coords, (zB, yL))
        push!(line_coords, (NaN, NaN))

        # vertical jump segment at zB
        push!(vline_coords, (zB, yL))
        push!(vline_coords, (zB, yR))
        push!(vline_coords, (NaN, NaN))
        
        # endpoint markers
        push!(open_marks, (zB, yL))
        push!(fill_marks, (zB, yR))

        seg_start = i+1
    end
    
    # last horizontal segment to z=1
    zL = float(zgrid[seg_start])
    zE = float(zgrid[end])
    yE = float(ygrid[end])
    
    push!(line_coords, (zL, yE))
    push!(line_coords, (zE, yE))
    push!(line_coords, (NaN, NaN))
    
    push!(fill_marks, (zE, yE))
    
    return line_coords, vline_coords, open_marks, fill_marks
end

# Build a horizontal-only polyline for segments where F≈G (right-continuous steps),
# using the same break detection as _stepdata_right_continuous.
function _coincidence_hsegments(zgrid::AbstractVector{<:Real},
                                Fvals::AbstractVector{<:Real},
                                Gvals::AbstractVector{<:Real};
                                atol::Real=1e-12,
                                eqtol::Real=1e-12)
    @assert length(zgrid) == length(Fvals) == length(Gvals) ≥ 2

    # breakpoints where either curve changes (so we can segment consistently)
    breaks = Int[]
    for i in 1:length(zgrid)-1
        if !isapprox(Fvals[i+1], Fvals[i]; atol=atol, rtol=0) ||
           !isapprox(Gvals[i+1], Gvals[i]; atol=atol, rtol=0)
            push!(breaks, i)
        end
    end

    coords = Tuple{Float64,Float64}[]
    seg_start = 1

    for i in breaks
        zL = float(zgrid[seg_start])
        zB = float(zgrid[i+1])
        yF = float(Fvals[i])
        yG = float(Gvals[i])

        if isfinite(yF) && isfinite(yG) && isapprox(yF, yG; atol=eqtol, rtol=0)
            push!(coords, (zL, yF))
            push!(coords, (zB, yF))
            push!(coords, (NaN, NaN))
        end

        seg_start = i + 1
    end

    # last segment
    zL = float(zgrid[seg_start])
    zE = float(zgrid[end])
    yF = float(Fvals[end])
    yG = float(Gvals[end])

    if isfinite(yF) && isfinite(yG) && isapprox(yF, yG; atol=eqtol, rtol=0)
        push!(coords, (zL, yF))
        push!(coords, (zE, yF))
        push!(coords, (NaN, NaN))
    end

    return coords
end

function plot_tildeFG_vs_z_axis(x; q, ρ, κ, r, β, ε=1e-10,
                               zgrid=collect(range(0.0, 1.0; length=2001)),
                               xlabel=L"$z$",
                               ylabel=L"$\widetilde F(x,z),\ \widetilde G(x,z)$",
                               width="11cm", height="7cm",
                               xpad=0.03,
                               ypad=0.05,
                               atol=1e-12,
                               eqtol=1e-12)

    _, xtick_str, xticklab_str = make_xticks_and_labels(x; q=q, ρ=ρ, κ=κ)

    # sample
    Fvals = similar(zgrid, Float64)
    Gvals = similar(zgrid, Float64)
    for (i, z) in enumerate(zgrid)
        F̂, Ĝ, _, _, _ = tildeFG(x, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
        Fvals[i] = float(F̂)
        Gvals[i] = float(Ĝ)
    end

    # step data: keep ONLY horizontal polyline, but CAPTURE open_marks for jump locations
    F_line, _, F_open, _ = _stepdata_right_continuous(zgrid, Fvals; atol=atol)
    G_line, _, G_open, _ = _stepdata_right_continuous(zgrid, Gvals; atol=atol)

    # coincidence overlay segments (black)
    FG_eq_line = _coincidence_hsegments(zgrid, Fvals, Gvals; atol=atol, eqtol=eqtol)

    # y-limits with padding (as you had)
    ymin = min(minimum(Fvals), minimum(Gvals))
    ymax = max(maximum(Fvals), maximum(Gvals))
    if ymax > ymin
        pad = ypad * (ymax - ymin)
        ymin -= pad
        ymax += pad
    else
        ymin -= 1
        ymax += 1
    end

    # NEW: rug marks as EXTRA x-ticks (pointing UPWARDS into the plot)
    jump_zs = sort!(unique!(vcat(first.(F_open), first.(G_open))))
    extra_xtick_str    = "{" * join(string.(jump_zs), ",") * "}"
    extra_xticklab_str = "{" * join(fill("", length(jump_zs)), ",") * "}"  # blank labels

    return @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,

            xmin   => 0,
            xmax   => 1,
            ymin   => ymin,
            ymax   => ymax,

            "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
            "tick align" => "outside",
            "major tick length" => "2pt",
            "minor tick length" => "1pt",

            "xtick" => xtick_str,
            "xticklabels" => xticklab_str,

            # NEW: rug tick marks at jump locations (no labels), pointing upwards
            "extra x ticks" => extra_xtick_str,
            "extra x tick labels" => extra_xticklab_str,
            "extra x tick style" => "{tick align=inside, tick style={black!55}}",

            "ytick" => "{0}",
            "scaled y ticks" => "false",
            "unbounded coords" => "jump",
            "legend pos" => "north east",
        },

        # F̃: black
        Plot({ "line width" => "0.30pt", "solid" => nothing, "black" => nothing },
             Coordinates(F_line)),
        LegendEntry(L"$\widetilde F(x,z)$"),

        # G̃: gray everywhere
        Plot({ "line width" => "0.30pt", "solid" => nothing, "color" => "black!40" },
             Coordinates(G_line)),
        LegendEntry(L"$\widetilde G(x,z)$"),

        # Overlay where F̃≈G̃: black (no legend entry)
        Plot({ "line width" => "0.30pt", "solid" => nothing, "color" => "black!55", "forget plot" => nothing },
             Coordinates(FG_eq_line)),
    )
end

"""
    save_tildeFG_vs_z_plot_pdf(outpdf, x; q, ρ, κ, r, β,
                               engine="pdflatex", quiet=true, keep_tex=true, kwargs...)

Builds the axis via `plot_tildeFG_vs_z_axis(x; ...)`, writes a standalone .tex,
and compiles to PDF.

Assumes:
  - plot_tildeFG_vs_z_axis exists (as provided)
  - PGFPlotsX is available
"""
function save_tildeFG_vs_z_plot_pdf(outpdf::AbstractString, x;
                                    q, ρ, κ, r, β,
                                    engine::AbstractString="pdflatex",
                                    quiet::Bool=true,
                                    keep_tex::Bool=true,
                                    kwargs...)
    ax  = plot_tildeFG_vs_z_axis(x; q=q, ρ=ρ, κ=κ, r=r, β=β, kwargs...)
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    # Copy output to requested absolute path if needed
    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

In [ ]:
# parameters (example)
q  = 0.25
ρ  = 0.6
κ  = 0.8
β  = 0.95
r  = 1.0

x0, x1, xhi = fixed_points(; q=q, ρ=ρ, κ=κ)
x  = 0.5*(x0 + (q+ρ))

save_tildeFG_vs_z_plot_pdf("tildeFG_vs_z.pdf", x; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=1e-30,
                           zgrid=collect(range(0.0, 1.0; length=10001)),
                           atol=1e-15)

In [ ]:
"""
    F_metric(x, z; q, ρ, κ, r, β, ε=1e-10)

Uses: F(x,z) = F̃(x,z) + Θ̃(x,z) * F(q+ρ,z),
and F(q+ρ,z) = F̃(q+ρ,z) / (1 - Θ̃(q+ρ,z)).
"""
function F_metric(x, z; q, ρ, κ, r, β, ε=1e-10)
    F̃, _, _ = tildeF(x, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    Θ̃, _, _ = tildeTheta(x, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε)

    xR = q + ρ
    F̃R, _, _ = tildeF(xR, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    Θ̃R, _, _ = tildeTheta(xR, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε)

    FR = F̃R / (1.0 - β * Θ̃R)          # <-- β here
    return F̃ + β * Θ̃ * FR             # <-- β here
end

"""
    G_metric(x, z; q, ρ, κ, β, ε=1e-10)

Uses: G(x,z) = G̃(x,z) + Θ̃(x,z) * G(q+ρ,z),
and G(q+ρ,z) = G̃(q+ρ,z) / (1 - Θ̃(q+ρ,z)).
"""
function G_metric(x, z; q, ρ, κ, β, ε=1e-10)
    G̃, _, _ = tildeG(x, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε)
    Θ̃, _, _ = tildeTheta(x, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε)

    xR = q + ρ
    G̃R, _, _ = tildeG(xR, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε)
    Θ̃R, _, _ = tildeTheta(xR, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε)

    GR = G̃R / (1.0 - β * Θ̃R)          # <-- β here
    return G̃ + β * Θ̃ * GR             # <-- β here
end

In [ ]:
"""
Print F(x,z) and G(x,z) for given x,z using existing metric functions.

Assumes you already defined:
  - F_metric(x, z; q, ρ, κ, r, β, ε=1e-10)
  - G_metric(x, z; q, ρ, κ, β, ε=1e-10)
"""
function show_FG(x, z; q, ρ, κ, r, β, ε=1e-10, digits=12)
    Fxz = F_metric(x, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    Gxz = G_metric(x, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε)

    println("--------------------------------------------------")
    println("Inputs: x = $(round(x, digits=digits)), z = $(round(z, digits=digits))")
    println("Params: q = $q, ρ = $ρ, κ = $κ, r = $r, β = $β, ε = $ε")
    println("--------------------------------------------------")
    println("F(x,z) = $(round(Fxz, digits=digits))")
    println("G(x,z) = $(round(Gxz, digits=digits))")
    println("--------------------------------------------------")

    return Fxz, Gxz
end

# Example usage:
q  = 0.25
ρ  = 0.60
κ  = 0.80
r  = 1.0
β  = 0.95
ε  = 1e-10

x = 0.73
z = 0.55

Fxz, Gxz = show_FG(x, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)

In [ ]:
using PGFPlotsX

# ------------------------------------------------------------------
# FG vs z with the same styling logic as for tildeF/tildeG:
#   - draw both curves SOLID
#   - F in black, G in gray (black!55)
#   - if F and G coincide on a segment: show "gray shoulders" + black
#   - NO vertical jump lines (we use NaN breaks from _stepdata_right_continuous)
#   - requested line widths: base curves 0.25pt
# ------------------------------------------------------------------

# Build horizontal-only polyline for segments where F≈G (right-continuous steps).
# Uses common segmentation where either F or G changes.
function _coincidence_hsegments(zgrid::AbstractVector{<:Real},
                                Fvals::AbstractVector{<:Real},
                                Gvals::AbstractVector{<:Real};
                                atol::Real=1e-12,
                                eqtol::Real=1e-12)
    @assert length(zgrid) == length(Fvals) == length(Gvals) ≥ 2

    breaks = Int[]
    for i in 1:length(zgrid)-1
        if !isapprox(Fvals[i+1], Fvals[i]; atol=atol, rtol=0) ||
           !isapprox(Gvals[i+1], Gvals[i]; atol=atol, rtol=0)
            push!(breaks, i)
        end
    end

    coords = Tuple{Float64,Float64}[]
    seg_start = 1

    for i in breaks
        zL = float(zgrid[seg_start])
        zB = float(zgrid[i+1])
        yF = float(Fvals[i])
        yG = float(Gvals[i])

        if isfinite(yF) && isfinite(yG) && isapprox(yF, yG; atol=eqtol, rtol=0)
            push!(coords, (zL, yF))
            push!(coords, (zB, yF))
            push!(coords, (NaN, NaN))
        end

        seg_start = i + 1
    end

    # last segment
    zL = float(zgrid[seg_start])
    zE = float(zgrid[end])
    yF = float(Fvals[end])
    yG = float(Gvals[end])

    if isfinite(yF) && isfinite(yG) && isapprox(yF, yG; atol=eqtol, rtol=0)
        push!(coords, (zL, yF))
        push!(coords, (zE, yF))
        push!(coords, (NaN, NaN))
    end

    return coords
end

function plot_FG_vs_z_axis(x; q, ρ, κ, r, β, ε=1e-10,
                           zgrid=collect(range(0.0, 1.0; length=2001)),
                           xlabel=L"$z$",
                           ylabel=L"$F(x,z),\ G(x,z)$",
                           width="11cm", height="7cm",
                           xpad=0.03,
                           ypad=0.05,
                           atol=1e-12,
                           eqtol=1e-12)

    # ticks/labels
    _, xtick_str, xticklab_str = make_xticks_and_labels(x; q=q, ρ=ρ, κ=κ)

    # sample F and G
    Fvals = similar(zgrid, Float64)
    Gvals = similar(zgrid, Float64)
    for (i, z) in enumerate(zgrid)
        Fvals[i] = float(F_metric(x, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε))
        Gvals[i] = float(G_metric(x, z; q=q, ρ=ρ, κ=κ,      β=β, ε=ε))
    end

    # horizontal-only step polylines (NaN breaks); CAPTURE open_marks for jump locations
    F_line, _, F_open, _ = _stepdata_right_continuous(zgrid, Fvals; atol=atol)
    G_line, _, G_open, _ = _stepdata_right_continuous(zgrid, Gvals; atol=atol)

    # coincidence overlay segments (NaN breaks)
    FG_eq_line = _coincidence_hsegments(zgrid, Fvals, Gvals; atol=atol, eqtol=eqtol)

    # y-limits + padding
    ymin = min(minimum(Fvals), minimum(Gvals))
    ymax = max(maximum(Fvals), maximum(Gvals))
    if ymax > ymin
        pad = ypad * (ymax - ymin)
        ymin -= pad
        ymax += pad
    else
        ymin -= 1
        ymax += 1
    end

    # NEW: rug marks as EXTRA x-ticks (pointing UPWARDS into the plot)
    jump_zs = sort!(unique!(vcat(first.(F_open), first.(G_open))))
    extra_xtick_str    = "{" * join(string.(jump_zs), ",") * "}"
    extra_xticklab_str = "{" * join(fill("", length(jump_zs)), ",") * "}"  # blank labels

    # widths (requested base=0.25pt)
    lw_base   = "0.30pt"
    lw_shldr  = "0.30pt"  # gray shoulder on coincidences

    return @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,

            xmin   => 0,
            xmax   => 1,
            ymin   => ymin,
            ymax   => ymax,

            "xtick" => xtick_str,
            "xticklabels" => xticklab_str,
            "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
            "ytick" => "{0}",

            "tick align" => "outside",
            "major tick length" => "2pt",
            "minor tick length" => "1pt",
            "scaled y ticks" => "false",
            "unbounded coords" => "jump",
            "legend pos" => "north east",

            # NEW: rug tick marks at jump locations (no labels), pointing upwards
            "extra x ticks" => extra_xtick_str,
            "extra x tick labels" => extra_xticklab_str,
            "extra x tick style" => "{tick align=inside, tick style={black!55}}",
        },

        # F: solid black (base width)
        Plot({ "line width" => lw_base, "solid" => nothing, "black" => nothing },
             Coordinates(F_line)),
        LegendEntry(L"$F(x,z)$"),

        # G: solid gray (base width)
        Plot({ "line width" => lw_base, "solid" => nothing, "color" => "black!40" },
             Coordinates(G_line)),
        LegendEntry(L"$G(x,z)$"),

        # Coincidence styling: gray shoulders (no legend entry)
        Plot({ "line width" => lw_shldr, "solid" => nothing, "color" => "black!55",
               "forget plot" => nothing },
             Coordinates(FG_eq_line)),
        # Plot({ "line width" => lw_base, "solid" => nothing, "black" => nothing,
        #       "forget plot" => nothing },
        #     Coordinates(FG_eq_line)),
    )
end

function save_FG_vs_z_plot_pdf(outpdf::AbstractString, x;
                               q, ρ, κ, r, β,
                               engine::AbstractString="pdflatex",
                               quiet::Bool=true,
                               keep_tex::Bool=true,
                               kwargs...)
    ax  = plot_FG_vs_z_axis(x; q=q, ρ=ρ, κ=κ, r=r, β=β, kwargs...)
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(texfile, pic; include_preamble=true)
    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

In [ ]:
# parameters (example)
q  = 0.25
ρ  = 0.6
κ  = 0.8
β  = 0.95
r  = 1.0

x0, x1, xhi = fixed_points(; q=q, ρ=ρ, κ=κ)
# z = 0.5 * (x1 + x0)
x  = 0.5*(x0 + (q+ρ))

save_FG_vs_z_plot_pdf("FG_vs_z.pdf", x; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=1e-30,
                      zgrid=collect(range(0.0, 1.0; length=100001)), atol=1e-15)

In [ ]:
using PGFPlotsX

# ------------------------------------------------------------------
# FG vs x for fixed z:
#   - draw both curves SOLID
#   - F in black, G in gray (black!40)
#   - if F and G coincide on a segment: show gray shoulders + black
#   - optional NaN break at x = z to avoid a spurious vertical join
# ------------------------------------------------------------------

# Build a sampled polyline, inserting NaN breaks at prescribed x-locations.
function _curve_with_breaks(xgrid::AbstractVector{<:Real},
                            yvals::AbstractVector{<:Real};
                            breaks::AbstractVector{<:Real}=Float64[],
                            atol::Real=1e-12)
    @assert length(xgrid) == length(yvals)

    bks = sort(float.(collect(breaks)))
    coords = Tuple{Float64,Float64}[]

    for i in eachindex(xgrid)
        x = float(xgrid[i])
        y = float(yvals[i])

        push!(coords, isfinite(y) ? (x, y) : (NaN, NaN))

        if i < length(xgrid)
            xnext = float(xgrid[i+1])
            crossed = any(b -> ((x < b && b <= xnext) ||
                                isapprox(x, b; atol=atol, rtol=0)), bks)
            if crossed
                push!(coords, (NaN, NaN))
            end
        end
    end

    return coords
end

# Build curve pieces where Y1 ≈ Y2, again allowing NaN breaks at prescribed x-locations.
function _coincidence_segments(xgrid::AbstractVector{<:Real},
                               Y1::AbstractVector{<:Real},
                               Y2::AbstractVector{<:Real};
                               eqtol::Real=1e-12,
                               breaks::AbstractVector{<:Real}=Float64[],
                               atol::Real=1e-12)
    @assert length(xgrid) == length(Y1) == length(Y2)

    bks = sort(float.(collect(breaks)))
    coords = Tuple{Float64,Float64}[]
    inrun = false

    for i in eachindex(xgrid)
        same = isfinite(Y1[i]) && isfinite(Y2[i]) &&
               isapprox(Y1[i], Y2[i]; atol=eqtol, rtol=0)

        if same
            push!(coords, (float(xgrid[i]), float(Y1[i])))
            inrun = true
        elseif inrun
            push!(coords, (NaN, NaN))
            inrun = false
        end

        if i < length(xgrid)
            x     = float(xgrid[i])
            xnext = float(xgrid[i+1])
            crossed = any(b -> ((x < b && b <= xnext) ||
                                isapprox(x, b; atol=atol, rtol=0)), bks)
            if crossed && inrun
                push!(coords, (NaN, NaN))
                inrun = false
            end
        end
    end

    if inrun
        push!(coords, (NaN, NaN))
    end

    return coords
end

function plot_FG_vs_x_axis(z; q, ρ, κ, r, β, ε=1e-10,
                           xgrid=collect(range(0.0, 1.0; length=2001)),
                           xlabel=L"$x$",
                           ylabel=L"$F(x,z),\ G(x,z)$",
                           width="11cm", height="7cm",
                           xpad=0.03,
                           ypad=0.05,
                           atol=1e-12,
                           eqtol=1e-12)

    # same structural ticks as before, except the distinguished point should later be labeled z
    _, xtick_str, xticklab_str = make_xticks_and_labels(z; q=q, ρ=ρ, κ=κ)

    # sample F and G over x for fixed z
    Fvals = similar(xgrid, Float64)
    Gvals = similar(xgrid, Float64)
    for (i, x) in enumerate(xgrid)
        Fvals[i] = float(F_metric(x, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε))
        Gvals[i] = float(G_metric(x, z; q=q, ρ=ρ, κ=κ,      β=β, ε=ε))
    end

    # horizontal-only step polylines with NaN breaks: no vertical jump lines
    F_line, _, F_open, _ = _stepdata_right_continuous(xgrid, Fvals; atol=atol)
    G_line, _, G_open, _ = _stepdata_right_continuous(xgrid, Gvals; atol=atol)

    # coincidence overlay segments
    FG_eq_line = _coincidence_hsegments(xgrid, Fvals, Gvals; atol=atol, eqtol=eqtol)

    # y-limits + padding
    ymin = min(minimum(Fvals), minimum(Gvals))
    ymax = max(maximum(Fvals), maximum(Gvals))
    if ymax > ymin
        pad = ypad * (ymax - ymin)
        ymin -= pad
        ymax += pad
    else
        ymin -= 1
        ymax += 1
    end

    # rug marks at jump locations
    jump_xs = sort!(unique!(vcat(first.(F_open), first.(G_open))))
    extra_xtick_str    = isempty(jump_xs) ? "{}" : "{" * join(string.(jump_xs), ",") * "}"
    extra_xticklab_str = isempty(jump_xs) ? "{}" : "{" * join(fill("", length(jump_xs)), ",") * "}"

    lw_base  = "0.30pt"
    lw_shldr = "0.30pt"

    if isempty(FG_eq_line)
        return @pgf Axis(
            {
                width  => width,
                height => height,
                xlabel => xlabel,
                ylabel => ylabel,

                xmin   => 0,
                xmax   => 1,
                ymin   => ymin,
                ymax   => ymax,

                "xtick" => xtick_str,
                "xticklabels" => xticklab_str,
                "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
                "ytick" => "{0}",

                "tick align" => "outside",
                "major tick length" => "2pt",
                "minor tick length" => "1pt",
                "scaled y ticks" => "false",
                "unbounded coords" => "jump",
                "legend pos" => "north east",

                "extra x ticks" => extra_xtick_str,
                "extra x tick labels" => extra_xticklab_str,
                "extra x tick style" => "{tick align=inside, tick style={black!55}}",
            },

            Plot({ "line width" => lw_base, "solid" => nothing, "black" => nothing },
                 Coordinates(F_line)),
            LegendEntry(L"$F(x,z)$"),

            Plot({ "line width" => lw_base, "solid" => nothing, "color" => "black!40" },
                 Coordinates(G_line)),
            LegendEntry(L"$G(x,z)$"),
        )
    else
        return @pgf Axis(
            {
                width  => width,
                height => height,
                xlabel => xlabel,
                ylabel => ylabel,

                xmin   => 0,
                xmax   => 1,
                ymin   => ymin,
                ymax   => ymax,

                "xtick" => xtick_str,
                "xticklabels" => xticklab_str,
                "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
                "ytick" => "{0}",

                "tick align" => "outside",
                "major tick length" => "2pt",
                "minor tick length" => "1pt",
                "scaled y ticks" => "false",
                "unbounded coords" => "jump",
                "legend pos" => "north east",

                "extra x ticks" => extra_xtick_str,
                "extra x tick labels" => extra_xticklab_str,
                "extra x tick style" => "{tick align=inside, tick style={black!55}}",
            },

            Plot({ "line width" => lw_base, "solid" => nothing, "black" => nothing },
                 Coordinates(F_line)),
            LegendEntry(L"$F(x,z)$"),

            Plot({ "line width" => lw_base, "solid" => nothing, "color" => "black!40" },
                 Coordinates(G_line)),
            LegendEntry(L"$G(x,z)$"),

            Plot({ "line width" => lw_shldr, "solid" => nothing,
                   "color" => "black!55", "forget plot" => nothing },
                 Coordinates(FG_eq_line)),
        )
    end
end

function save_FG_vs_x_plot_pdf(outpdf::AbstractString, z;
                               q, ρ, κ, r, β,
                               engine::AbstractString="pdflatex",
                               quiet::Bool=true,
                               keep_tex::Bool=true,
                               kwargs...)
    ax  = plot_FG_vs_x_axis(z; q=q, ρ=ρ, κ=κ, r=r, β=β, kwargs...)
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(texfile, pic; include_preamble=true)
    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

In [ ]:
# parameters (example)
q  = 0.25
ρ  = 0.6
κ  = 0.8
β  = 0.95
r  = 1.0

x0, x1, xhi = fixed_points(; q=q, ρ=ρ, κ=κ)

# z = 0.5*q
# z = 0.5*q + 0.5*x1
#z = 0.5*x0 + 0.5*(q+ρ)
# z = 0.5*(q+ρ) + 0.5*1.0
z = 0.5*x1 + 0.5*x0

save_FG_vs_x_plot_pdf("FG_vs_x.pdf", z; q=q, ρ=ρ, κ=κ, r=r, β=β,
    ε=1e-20, xgrid=range(0, 1; length=1001))

In [ ]:
"""
    tildef_metric(x, z; q, ρ, κ, r, β, ε=1e-10)

Compute the pre-ACK marginal tilde reward metric:
    \tilde f(x,z) = r κ x + β[ (1-κx)\tildeF(φ¹(x),z) - \tildeF(φ⁰(x),z) ].
"""
function tildef_metric(x, z; q, ρ, κ, r, β, ε=1e-10)
    x0 = φ0(x; q=q, ρ=ρ)
    x1 = φ1(x; q=q, ρ=ρ, κ=κ)

    F̃1, _, _ = tildeF(x1, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    F̃0, _, _ = tildeF(x0, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)

    return r*κ*x + β*((1 - κ*x)*F̃1 - F̃0)
end


"""
    tildeg_metric(x, z; q, ρ, κ, β, ε=1e-10)

Compute the pre-ACK marginal tilde work metric:
    \tilde g(x,z) = 1 + β[ (1-κx)\tildeG(φ¹(x),z) - \tildeG(φ⁰(x),z) ].
Note: tildeG does not depend on r.
"""
function tildeg_metric(x, z; q, ρ, κ, β, ε=1e-10)
    x0 = φ0(x; q=q, ρ=ρ)
    x1 = φ1(x; q=q, ρ=ρ, κ=κ)

    G̃1, _, _ = tildeG(x1, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε)
    G̃0, _, _ = tildeG(x0, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε)

    return 1 + β*((1 - κ*x)*G̃1 - G̃0)
end

"""
    tildetheta_metric(x, z; q, ρ, κ, β, ε=1e-10)

Compute the pre-ACK marginal tilde discount/renewal factor:
    \tilde θ(x,z) = κx + β[ (1-κx)\tildeΘ(φ¹(x),z) - \tildeΘ(φ⁰(x),z) ].
Note: tildeTheta should NOT depend on r (you updated it to call tildeF with r=1).
"""
function tildetheta_metric(x, z; q, ρ, κ, β, ε=1e-10)
    x0 = φ0(x; q=q, ρ=ρ)
    x1 = φ1(x; q=q, ρ=ρ, κ=κ)

    Θ̃1, _, _ = tildeTheta(x1, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε)
    Θ̃0, _, _ = tildeTheta(x0, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε)

    return κ*x + β*((1 - κ*x)*Θ̃1 - Θ̃0)
end


"""
    f_metric2(x, z; q, ρ, κ, r, β, ε=1e-10)

Compute full marginal reward metric:
    f = \tilde f + β \tilde θ * F(q+ρ,z).
"""
function f_metric2(x, z; q, ρ, κ, r, β, ε=1e-10)
    tf  = tildef_metric(x, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    tθ  = tildetheta_metric(x, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε)
    FR  = F_metric(q + ρ, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    return tf + β * tθ * FR
end


"""
    g_metric2(x, z; q, ρ, κ, β, ε=1e-10)

Compute full marginal work metric:
    g = \tilde g + β \tilde θ * G(q+ρ,z).
"""
function g_metric2(x, z; q, ρ, κ, β, ε=1e-10)
    tg  = tildeg_metric(x, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε)
    tθ  = tildetheta_metric(x, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε)
    GR  = G_metric(q + ρ, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε)
    return tg + β * tθ * GR
end


"""
    tildef_from_tildetheta(x, z; q, ρ, κ, r, β, ε=1e-10)

Compute \tilde f via the identity  \tilde f = r \tilde θ.
"""
function tildef_from_tildetheta(x, z; q, ρ, κ, r, β, ε=1e-10)
    tθ = tildetheta_metric(x, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε)
    return r * tθ
end

In [ ]:
# parameters (example)
q  = 0.25
ρ  = 0.6
κ  = 0.8
β  = 0.95
r  = 1.0

x0, x1, xhi = fixed_points(; q=q, ρ=ρ, κ=κ)
@show x1 x0
# z = 0.5 * (x1 + x0)
#x  = 0.5*(x0 + (q+ρ))
x = 0.73
z = 0.55
tf1 = tildef_metric(x,z; q=q,ρ=ρ,κ=κ,r=r,β=β)
tg1 = tildeg_metric(x,z; q=q,ρ=ρ,κ=κ,β=β)
tf2 = tildef_from_tildetheta(x,z; q=q,ρ=ρ,κ=κ,r=r,β=β)
f1 = f_metric2(x,z; q=q,ρ=ρ,κ=κ,r=r,β=β)
g1 = g_metric2(x,z; q=q,ρ=ρ,κ=κ,β=β)
@show tf1 tf2 (tf1-tf2) tg1
@show f1 g1

In [ ]:
# Extract the scalar "value" from functions that may return (value, diagnostics...)
val(u) = u isa Tuple ? u[1] : u

function diagnose_zltq(x, z; q, ρ, κ, r, β, ε=1e-10)
    @assert 0.0 <= x <= 1.0
    @assert z < q   # here q = p_{01}

    println("\n==============================")
    println("Diagnose at x=$x, z=$z (z<q)")
    println("==============================")

    # ----------------------------------------------------------
    # Local one-step maps
    # ----------------------------------------------------------
    φ0loc(y) = q + ρ*y
    φ1loc(y) = q + ρ*(1-κ)*y/(1-κ*y)

    y0   = φ0loc(x)
    y1   = φ1loc(x)
    yR   = q + ρ   # = p11

    println("\n[Basic states]")
    println("x           = $x")
    println("z           = $z")
    println("phi0(x)     = $y0")
    println("phi1(x)     = $y1")
    println("q+rho       = $yR")
    println("x>z?        = $(x > z)")
    println("phi0(x)>z?  = $(y0 > z)")
    println("phi1(x)>z?  = $(y1 > z)")
    println("q+rho>z?    = $(yR > z)")

    # ----------------------------------------------------------
    # 1) Marginals as computed by the code
    # ----------------------------------------------------------
    tf_u = val(tildef_metric(x, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε))
    tg_u = val(tildeg_metric(x, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε))
    tt_u = val(tildetheta_metric(x, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε))

    f_u  = val(f_metric2(x, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε))
    g_u  = val(g_metric2(x, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε))

    println("\n[Computed marginal metrics]")
    println("tildef_metric      = $tf_u")
    println("tildeg_metric      = $tg_u")
    println("tildetheta_metric  = $tt_u")
    println("f_metric2          = $f_u")
    println("g_metric2          = $g_u")

    # In regime z<q, the paper predicts for the FULL marginals:
    #   f(x,z) = r κ x,   g(x,z) = 1
    println("\n[Full-marginal z<q predictions]")
    println("f_metric2 - r*κ*x = $(f_u - r*κ*x)")
    println("g_metric2 - 1     = $(g_u - 1.0)")

    # ----------------------------------------------------------
    # 2) Posterior-mean identity
    # ----------------------------------------------------------
    pm_res = y0 - (κ*x*yR + (1-κ*x)*y1)
    println("\n[Posterior-mean identity check]")
    println("phi0(x) - [κx(q+ρ)+(1-κx)phi1(x)] = $pm_res  (should be ~0)")

    # ----------------------------------------------------------
    # 3) Pre-ACK metrics and identities
    # ----------------------------------------------------------
    tF = val(tildeF(x, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε))
    tG = val(tildeG(x, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε))
    tT = val(tildeTheta(x, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε))

    println("\n[Pre-ACK metrics]")
    println("tildeF            = $tF")
    println("tildeG            = $tG")
    println("tildeTheta        = $tT")

    println("\n[Pre-ACK identity checks]")
    println("tildeTheta - tildeF/r = $(tT - tF/r)  (should be ~0)")
    println("tildef_metric - r*tildetheta_metric = $(tf_u - r*tt_u)  (should be ~0)")

    # ----------------------------------------------------------
    # 4) Renewal identities for full metrics
    # ----------------------------------------------------------
    Ffull = val(F_metric(x, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε))
    Gfull = val(G_metric(x, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε))

    F_R   = val(F_metric(yR, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε))
    G_R   = val(G_metric(yR, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε))

    F_renew_res = Ffull - (tF + β*tT*F_R)
    G_renew_res = Gfull - (tG + β*tT*G_R)

    println("\n[Renewal identity residuals]")
    println("F - (tildeF + β*tildeTheta*F(q+ρ)) = $F_renew_res  (should be ~0)")
    println("G - (tildeG + β*tildeTheta*G(q+ρ)) = $G_renew_res  (should be ~0)")

    # ----------------------------------------------------------
    # 5) z<q special constraint checks for FULL metrics
    # ----------------------------------------------------------
    G_y0 = val(G_metric(y0, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε))
    G_y1 = val(G_metric(y1, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε))

    F_y0 = val(F_metric(y0, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε))
    F_y1 = val(F_metric(y1, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε))

    println("\n[z<q constraint checks for full metrics]")
    println("G_metric(q+ρ,z)   = $G_R   ; should be ≈ $(1/(1-β))")
    println("G_metric(phi0(x),z) = $G_y0 ; should be ≈ $(1/(1-β))")
    println("G_metric(phi1(x),z) = $G_y1 ; should be ≈ $(1/(1-β))")

    # ----------------------------------------------------------
    # 6) z<q special constraint checks for PRE-ACK metrics
    #
    # If y>z, then:
    #   G(y,z) = tildeG(y,z) + β*tildeTheta(y,z)*G(q+ρ,z)
    # and G(q+ρ,z)=1/(1-β), so
    #   tildeG(y,z) = G(y,z) - β*tildeTheta(y,z)/(1-β)
    # In particular, when y>z and G(y,z)=1/(1-β),
    #   tildeG(y,z) = (1 - β*tildeTheta(y,z))/(1-β).
    # ----------------------------------------------------------
    tG_y0 = val(tildeG(y0, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε))
    tG_y1 = val(tildeG(y1, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε))
    tG_R  = val(tildeG(yR, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε))

    tT_y0 = val(tildeTheta(y0, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε))
    tT_y1 = val(tildeTheta(y1, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε))
    tT_R  = val(tildeTheta(yR, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε))

    println("\n[z<q constraint checks for pre-ACK metrics]")
    if x > z
        tG_pred_x = (1 - β*tT)/(1 - β)
        println("tildeG(x,z)          = $tG")
        println("tildeG_pred_x        = $tG_pred_x")
        println("tildeG(x,z) - pred   = $(tG - tG_pred_x)  (should be ~0 when x>z)")
    else
        println("Skipping x-level tildeG closed form because x<=z.")
    end

    tG_pred_y0 = (1 - β*tT_y0)/(1 - β)
    tG_pred_y1 = (1 - β*tT_y1)/(1 - β)
    tG_pred_R  = (1 - β*tT_R )/(1 - β)

    println("tildeG(phi0(x),z) - pred = $(tG_y0 - tG_pred_y0)  (should be ~0)")
    println("tildeG(phi1(x),z) - pred = $(tG_y1 - tG_pred_y1)  (should be ~0)")
    println("tildeG(q+ρ,z)      - pred = $(tG_R  - tG_pred_R )  (should be ~0)")

    # ----------------------------------------------------------
    # 7) Rebuild tildeg from its DEFINITION (paper eq. for tilde g)
    #   tilde g = 1 + β[(1-κx)tildeG(phi1(x),z) - tildeG(phi0(x),z)]
    # ----------------------------------------------------------
    Btilde = (1 - κ*x)*tG_y1 - tG_y0
    tg_rebuilt = 1 + β*Btilde

    println("\n[Rebuild tildeg from tildeG: correct formula]")
    println("Btilde                         = $Btilde")
    println("tildeg_rebuilt                 = $tg_rebuilt")
    println("tildeg_rebuilt - tildeg_metric = $(tg_rebuilt - tg_u)  (should be ~0)")

    # ----------------------------------------------------------
    # 8) Rebuild full g from its one-step formula
    #   g = 1 + β[κx G(q+ρ,z) + (1-κx)G(phi1(x),z) - G(phi0(x),z)]
    # ----------------------------------------------------------
    Bfull_g = κ*x*G_R + (1-κ*x)*G_y1 - G_y0
    g_rebuilt = 1 + β*Bfull_g

    println("\n[Rebuild full g from G]")
    println("Bfull_g                  = $Bfull_g")
    println("g_rebuilt                = $g_rebuilt")
    println("g_rebuilt - g_metric2    = $(g_rebuilt - g_u)  (should be ~0)")

    # ----------------------------------------------------------
    # 9) Rebuild full f from its one-step formula
    #   f = rκx + β[κx F(q+ρ,z) + (1-κx)F(phi1(x),z) - F(phi0(x),z)]
    # ----------------------------------------------------------
    Bfull_f = κ*x*F_R + (1-κ*x)*F_y1 - F_y0
    f_rebuilt = r*κ*x + β*Bfull_f

    println("\n[Rebuild full f from F]")
    println("Bfull_f                  = $Bfull_f")
    println("f_rebuilt                = $f_rebuilt")
    println("f_rebuilt - f_metric2    = $(f_rebuilt - f_u)  (should be ~0)")

    return nothing
end

In [ ]:
q  = 0.25
ρ  = 0.6
κ  = 0.8
β  = 0.95
r  = 1.0
x  = 0.5
z  = q/10

diagnose_zltq(x, z; q=q, ρ=ρ, κ=κ, r=r, β=β)

In [ ]:
function plot_tildeg_tildetheta_vs_z_axis(x; q, ρ, κ, β, ε=1e-10,
                                         zgrid=collect(range(0.0, 1.0; length=2001)),
                                         xlabel=L"$z$",
                                         ylabel=L"$\widetilde g(x,z),\ \widetilde\theta(x,z)$",
                                         width="11cm", height="7cm",
                                         xpad=0.03,
                                         ypad=0.05,
                                         atol=1e-12)

    # x-ticks/labels
    _, xtick_str, xticklab_str = make_xticks_and_labels(x; q=q, ρ=ρ, κ=κ)

    # sample
    gvals  = similar(zgrid, Float64)
    thvals = similar(zgrid, Float64)
    for (i, z) in enumerate(zgrid)
        gvals[i]  = float(tildeg_metric(x, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε))
        thvals[i] = float(tildetheta_metric(x, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε))
    end

    # step data: keep ONLY horizontal polyline, but CAPTURE open_marks for jump locations
    g_line,  _, g_open,  _ = _stepdata_right_continuous(zgrid, gvals;  atol=atol)
    th_line, _, th_open, _ = _stepdata_right_continuous(zgrid, thvals; atol=atol)

    # y-limits with padding (kept as you had)
    finite_vals = filter(isfinite, vcat(gvals, thvals))
    ymin = minimum(finite_vals)
    ymax = maximum(finite_vals)
    if ymax > ymin
        pad = ypad * (ymax - ymin)
        ymin -= pad
        ymax += pad
    else
        ymin -= 1
        ymax += 1
    end

    # NEW: rug marks as EXTRA x-ticks (pointing UPWARDS into the plot)
    jump_zs = sort!(unique!(vcat(first.(g_open), first.(th_open))))
    extra_xtick_str    = "{" * join(string.(jump_zs), ",") * "}"
    extra_xticklab_str = "{" * join(fill("", length(jump_zs)), ",") * "}"  # blank labels

    lw = "0.30pt"

    return @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,

            xmin   => 0,
            xmax   => 1,
            ymin   => ymin,
            ymax   => ymax,

            "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
            "tick align" => "outside",
            "major tick length" => "2pt",
            "minor tick length" => "1pt",

            "xtick" => xtick_str,
            "xticklabels" => xticklab_str,

            # NEW: rug tick marks at jump locations (no labels), pointing upwards
            "extra x ticks" => extra_xtick_str,
            "extra x tick labels" => extra_xticklab_str,
            "extra x tick style" => "{tick align=inside, tick style={black!55}}",

            # y ticks only at 0 and 1
            "ytick" => "{0,1}",

            "scaled y ticks" => "false",
            "unbounded coords" => "jump",

            # legend in north west
            "legend pos" => "north west",
        },

        # g̃: solid black
        Plot({ "line width" => lw, "solid" => nothing, "black" => nothing },
             Coordinates(g_line)),
        LegendEntry(L"$\widetilde g(x,z)$"),

        # θ̃: solid gray
        Plot({ "line width" => lw, "solid" => nothing, "color" => "black!35" },
             Coordinates(th_line)),
        LegendEntry(L"$\widetilde\theta(x,z)$"),
    )
end

"""
    save_tildeg_tildetheta_vs_z_plot_pdf(outpdf, x; q, ρ, κ, β, engine="pdflatex", quiet=true)

    Standalone .tex + compile to PDF (quiet by default).
    Matches your save_tildeFG_vs_z_plot_pdf wrapper.
"""
function save_tildeg_tildetheta_vs_z_plot_pdf(outpdf::AbstractString, x;
                                              q, ρ, κ, β,
                                              engine::AbstractString="pdflatex",
                                              quiet::Bool=true,
                                              keep_tex::Bool=true,
                                              kwargs...)
    ax  = plot_tildeg_tildetheta_vs_z_axis(x; q=q, ρ=ρ, κ=κ, β=β, kwargs...)
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

In [ ]:
# parameters (example)
q  = 0.25
ρ  = 0.6
κ  = 0.8
β  = 0.95
r  = 1.0

x0, x1, xhi = fixed_points(; q=q, ρ=ρ, κ=κ)
# z = 0.5 * (x1 + x0)
x  = 0.5*(x0 + (q+ρ))

save_tildeg_tildetheta_vs_z_plot_pdf("tilde_g_theta_vs_z.pdf", x; q=q, ρ=ρ, κ=κ, β=β, ε=1e-20, zgrid=collect(range(0.0, 1.0; length=100001)))

In [ ]:
using PGFPlotsX

function plot_tildef_tildeg_vs_z_axis(x; q, ρ, κ, r, β, ε=1e-10,
                                     zgrid=collect(range(0.0, 1.0; length=2001)),
                                     xlabel=L"$z$",
                                     ylabel=L"$\widetilde f(x,z),\ \widetilde g(x,z)$",
                                     width="11cm", height="7cm",
                                     xpad=0.03,
                                     ypad=0.03,
                                     atol=1e-12)

    _, xtick_str, xticklab_str = make_xticks_and_labels(x; q=q, ρ=ρ, κ=κ)

    # sample
    fvals = similar(zgrid, Float64)
    gvals = similar(zgrid, Float64)
    for (i, z) in enumerate(zgrid)
        fvals[i] = float(tildef_metric(x, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε))   # tilde f
        gvals[i] = float(tildeg_metric(x, z; q=q, ρ=ρ, κ=κ,      β=β, ε=ε))   # tilde g
    end

    # step data (right-continuous): keep ONLY horizontal polyline, but CAPTURE open_marks for jump locations
    f_line, _, f_open, _ = _stepdata_right_continuous(zgrid, fvals; atol=atol)
    g_line, _, g_open, _ = _stepdata_right_continuous(zgrid, gvals; atol=atol)

    # NEW: rug marks as EXTRA x-ticks (pointing UPWARDS into the plot)
    jump_zs = sort!(unique!(vcat(first.(f_open), first.(g_open))))
    extra_xtick_str    = "{" * join(string.(jump_zs), ",") * "}"
    extra_xticklab_str = "{" * join(fill("", length(jump_zs)), ",") * "}"  # blank labels

    # reference level y = 1 - β
    yref = 1 - β
    ref_tbl = PGFPlotsX.Table(x=[0.0, 1.0], y=[yref, yref])

    # y-limits with padding (include yref so it's visible)
    finite_vals = filter(isfinite, vcat(fvals, gvals, [yref]))
    ymin = minimum(finite_vals)
    ymax = maximum(finite_vals)
    if ymax > ymin
        pad = ypad * (ymax - ymin)
        ymin -= pad
        ymax += pad
    else
        ymin -= 1
        ymax += 1
    end

    lw = "0.30pt"

    return @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,

            xmin   => 0,
            xmax   => 1,
            ymin   => ymin,
            ymax   => ymax,

            "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
            "tick align" => "outside",
            "major tick length" => "2pt",
            "minor tick length" => "1pt",

            "xtick" => xtick_str,
            "xticklabels" => xticklab_str,

            # NEW: rug tick marks at jump locations (no labels), pointing upwards
            "extra x ticks" => extra_xtick_str,
            "extra x tick labels" => extra_xticklab_str,
            "extra x tick style" => "{tick align=inside, tick style={black!55}}",

            # y-ticks only at 0 and 1
            "ytick" => "{0,1}",

            "scaled y ticks" => "false",
            "unbounded coords" => "jump",

            # legend in north west
            "legend pos" => "north west",
        },

        # \tilde f: solid black
        Plot({ "line width" => lw, "solid" => nothing, "black" => nothing },
             Coordinates(f_line)),
        LegendEntry(L"$\widetilde f(x,z)$"),

        # \tilde g: solid gray
        Plot({ "line width" => lw, "solid" => nothing, "color" => "black!40" },
             Coordinates(g_line)),
        LegendEntry(L"$\widetilde g(x,z)$"),

        # dashed reference line y = 1 - β
        Plot({ "line width" => lw, "dash pattern" => "on 1pt off 1pt" },
             ref_tbl),
        LegendEntry(L"$1-\beta$"),
    )
end

# --- save wrapper for the tilde plot (mirrors your existing save_* style) ---
function save_tildef_tildeg_vs_z_plot_pdf(outpdf::AbstractString, x;
                                          q, ρ, κ, r, β,
                                          engine::AbstractString="pdflatex",
                                          quiet::Bool=true,
                                          keep_tex::Bool=true,
                                          kwargs...)
    ax  = plot_tildef_tildeg_vs_z_axis(x; q=q, ρ=ρ, κ=κ, r=r, β=β, kwargs...)
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

In [ ]:
# parameters (example)
q  = 0.25
ρ  = 0.6
κ  = 0.8
β  = 0.95
r  = 1.0

x0, x1, xhi = fixed_points(; q=q, ρ=ρ, κ=κ)
# z = 0.5 * (x1 + x0)
x  = 0.5*(x0 + (q+ρ))

# produce the PDF
save_tildef_tildeg_vs_z_plot_pdf(
    "tildef_tildeg_vs_z_plot.pdf", x;
    q=q, ρ=ρ, κ=κ, r=r, β=β,
    ε=1e-30,
    zgrid=collect(range(0.0, 1.0; length=10001))
)

In [ ]:
using PGFPlotsX

function plot_fg_metrics2_vs_z_axis(x; q, ρ, κ, r, β, ε=1e-10,
                                   zgrid=collect(range(0.0, 1.0; length=2001)),
                                   xlabel=L"$z$",
                                   ylabel=L"$f(x,z),\ g(x,z)$",
                                   width="11cm", height="7cm",
                                   xpad=0.03,
                                   ypad=0.03,
                                   atol=1e-12)

    _, xtick_str, xticklab_str = make_xticks_and_labels(x; q=q, ρ=ρ, κ=κ)

    # sample
    fvals = similar(zgrid, Float64)
    gvals = similar(zgrid, Float64)
    for (i, z) in enumerate(zgrid)
        fvals[i] = float(f_metric2(x, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε))
        gvals[i] = float(g_metric2(x, z; q=q, ρ=ρ, κ=κ,      β=β, ε=ε))
    end

    # step data (right-continuous): keep ONLY horizontal polyline, but CAPTURE open_marks for jump locations
    f_line, _, f_open, _ = _stepdata_right_continuous(zgrid, fvals; atol=atol)
    g_line, _, g_open, _ = _stepdata_right_continuous(zgrid, gvals; atol=atol)

    # NEW: rug marks as EXTRA x-ticks (pointing UPWARDS into the plot)
    jump_zs = sort!(unique!(vcat(first.(f_open), first.(g_open))))
    extra_xtick_str    = "{" * join(string.(jump_zs), ",") * "}"
    extra_xticklab_str = "{" * join(fill("", length(jump_zs)), ",") * "}"  # blank labels

    # reference level
    yref = 1 - β
    ref_tbl = PGFPlotsX.Table(x=[0.0, 1.0], y=[yref, yref])

    # y-limits with padding (include yref so it's always visible)
    finite_vals = filter(isfinite, vcat(fvals, gvals, [yref]))
    ymin = minimum(finite_vals)
    ymax = maximum(finite_vals)
    if ymax > ymin
        pad = ypad * (ymax - ymin)
        ymin -= pad
        ymax += pad
    else
        ymin -= 1
        ymax += 1
    end

    lw = "0.30pt"

    return @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,

            xmin   => 0,
            xmax   => 1,
            ymin   => ymin,
            ymax   => ymax,

            "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
            "tick align" => "outside",
            "major tick length" => "2pt",
            "minor tick length" => "1pt",

            "xtick" => xtick_str,
            "xticklabels" => xticklab_str,

            # NEW: rug tick marks at jump locations (no labels), pointing upwards
            "extra x ticks" => extra_xtick_str,
            "extra x tick labels" => extra_xticklab_str,
            "extra x tick style" => "{tick align=inside, tick style={black!55}}",

            # y-ticks only at 0 and 1
            "ytick" => "{0,1}",

            "scaled y ticks" => "false",
            "unbounded coords" => "jump",

            # legend in north west
            "legend pos" => "north west",
        },

        # f: solid black (no verticals)
        Plot({ "line width" => lw, "solid" => nothing, "black" => nothing },
             Coordinates(f_line)),
        LegendEntry(L"$f(x,z)$"),

        # g: solid gray (no verticals)
        Plot({ "line width" => lw, "solid" => nothing, "color" => "black!40" },
             Coordinates(g_line)),
        LegendEntry(L"$g(x,z)$"),

        # reference horizontal line y = 1-β (thin dashed) + legend
        Plot({ "line width" => "0.30pt", "dash pattern" => "on 1pt off 1pt" },
             ref_tbl),
        LegendEntry(L"$1-\beta$"),
    )
end

# --- save wrapper for the tilde plot (mirrors your existing save_* style) ---
function save_fg_metrics2_vs_z_plot_pdf(outpdf::AbstractString, x;
                                          q, ρ, κ, r, β,
                                          engine::AbstractString="pdflatex",
                                          quiet::Bool=true,
                                          keep_tex::Bool=true,
                                          kwargs...)
    ax  = plot_fg_metrics2_vs_z_axis(x; q=q, ρ=ρ, κ=κ, r=r, β=β, kwargs...)
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

In [ ]:
# parameters (example)
q  = 0.25
ρ  = 0.6
κ  = 0.8
β  = 0.95
r  = 1.0
x0, x1, xhi = fixed_points(; q=q, ρ=ρ, κ=κ)
z = 0.5 * (x1 + x0)
x  = 0.5*(x0 + (q+ρ))
tf1 = f_metric2(x,z; q=q,ρ=ρ,κ=κ,r=r,β=β)
tg1 = tildeg_metric(x,z; q=q,ρ=ρ,κ=κ,β=β)
tf2 = tildef_from_tildetheta(x,z; q=q,ρ=ρ,κ=κ,r=r,β=β)
@show tf1 tf2 (tf1-tf2) tg1

In [ ]:
# parameters (example)
q  = 0.25
ρ  = 0.6
κ  = 0.8
β  = 0.99
r  = 1.0

x0, x1, xhi = fixed_points(; q=q, ρ=ρ, κ=κ)
# z = 0.5 * (x1 + x0)
x  = 0.5*(x0 + (q+ρ))

save_fg_metrics2_vs_z_plot_pdf("fg_vs_z_plot.pdf", x; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=1e-20, zgrid=collect(range(0.0, 1.0; length=1001)))

In [ ]:
using PGFPlotsX

# ------------------------------------------------------------------
# f,g vs x for fixed z:
#   - draw both curves SOLID
#   - f in black, g in gray (black!40)
#   - if f and g coincide on a segment: show gray shoulders
#   - NO vertical jump lines (use NaN breaks from _stepdata_right_continuous)
#   - requested line widths: base curves 0.30pt
# ------------------------------------------------------------------

# Build horizontal-only polyline for segments where A≈B (right-continuous steps).
# Uses common segmentation where either A or B changes.
function _coincidence_hsegments(xgrid::AbstractVector{<:Real},
                                Avals::AbstractVector{<:Real},
                                Bvals::AbstractVector{<:Real};
                                atol::Real=1e-12,
                                eqtol::Real=1e-12)
    @assert length(xgrid) == length(Avals) == length(Bvals) ≥ 2

    breaks = Int[]
    for i in 1:length(xgrid)-1
        if !isapprox(Avals[i+1], Avals[i]; atol=atol, rtol=0) ||
           !isapprox(Bvals[i+1], Bvals[i]; atol=atol, rtol=0)
            push!(breaks, i)
        end
    end

    coords = Tuple{Float64,Float64}[]
    seg_start = 1

    for i in breaks
        xL = float(xgrid[seg_start])
        xB = float(xgrid[i+1])
        yA = float(Avals[i])
        yB = float(Bvals[i])

        if isfinite(yA) && isfinite(yB) && isapprox(yA, yB; atol=eqtol, rtol=0)
            push!(coords, (xL, yA))
            push!(coords, (xB, yA))
            push!(coords, (NaN, NaN))
        end

        seg_start = i + 1
    end

    # last segment
    xL = float(xgrid[seg_start])
    xE = float(xgrid[end])
    yA = float(Avals[end])
    yB = float(Bvals[end])

    if isfinite(yA) && isfinite(yB) && isapprox(yA, yB; atol=eqtol, rtol=0)
        push!(coords, (xL, yA))
        push!(coords, (xE, yA))
        push!(coords, (NaN, NaN))
    end

    return coords
end

function plot_fg_vs_x_axis(z; q, ρ, κ, r, β, ε=1e-10,
                           xgrid=collect(range(0.0, 1.0; length=2001)),
                           xlabel=L"$x$",
                           ylabel=L"$f(x,z),\ g(x,z)$",
                           width="11cm", height="7cm",
                           xpad=0.03,
                           ypad=0.05,
                           atol=1e-12,
                           eqtol=1e-12)

    # Same structural ticks as before, but with distinguished point at z.
    # If you have already modified make_xticks_and_labels to accept varlab, use:
    # _, xtick_str, xticklab_str = make_xticks_and_labels(z; q=q, ρ=ρ, κ=κ, varlab=L"z")
    #
    # Otherwise keep the old call temporarily:
    _, xtick_str, xticklab_str = make_xticks_and_labels(z; q=q, ρ=ρ, κ=κ)

    # sample f and g
    fvals = similar(xgrid, Float64)
    gvals = similar(xgrid, Float64)
    for (i, x) in enumerate(xgrid)
        fvals[i] = float(f_metric2(x, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε))
        gvals[i] = float(g_metric2(x, z; q=q, ρ=ρ, κ=κ,      β=β, ε=ε))
    end

    # horizontal-only step polylines (NaN breaks); capture jump locations
    f_line, _, f_open, _ = _stepdata_right_continuous(xgrid, fvals; atol=atol)
    g_line, _, g_open, _ = _stepdata_right_continuous(xgrid, gvals; atol=atol)

    # coincidence overlay segments
    fg_eq_line = _coincidence_hsegments(xgrid, fvals, gvals; atol=atol, eqtol=eqtol)

    # y-limits + padding
    ymin = min(minimum(fvals), minimum(gvals))
    ymax = max(maximum(fvals), maximum(gvals))
    if ymax > ymin
        pad = ypad * (ymax - ymin)
        ymin -= pad
        ymax += pad
    else
        ymin -= 1
        ymax += 1
    end

    # rug marks at jump locations
    jump_xs = sort!(unique!(vcat(first.(f_open), first.(g_open))))
    extra_xtick_str    = isempty(jump_xs) ? "{}" : "{" * join(string.(jump_xs), ",") * "}"
    extra_xticklab_str = isempty(jump_xs) ? "{}" : "{" * join(fill("", length(jump_xs)), ",") * "}"

    lw_base  = "0.30pt"
    lw_shldr = "0.30pt"

    if isempty(fg_eq_line)
        return @pgf Axis(
            {
                width  => width,
                height => height,
                xlabel => xlabel,
                ylabel => ylabel,

                xmin   => 0,
                xmax   => 1,
                ymin   => ymin,
                ymax   => ymax,

                "xtick" => xtick_str,
                "xticklabels" => xticklab_str,
                "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
                "ytick" => "{0}",

                "tick align" => "outside",
                "major tick length" => "2pt",
                "minor tick length" => "1pt",
                "scaled y ticks" => "false",
                "unbounded coords" => "jump",
                "legend pos" => "north east",

                # rug marks at jump locations
                "extra x ticks" => extra_xtick_str,
                "extra x tick labels" => extra_xticklab_str,
                "extra x tick style" => "{tick align=inside, tick style={black!55}}",
            },

            Plot({ "line width" => lw_base, "solid" => nothing, "black" => nothing },
                 Coordinates(f_line)),
            LegendEntry(L"$f(x,z)$"),

            Plot({ "line width" => lw_base, "solid" => nothing, "color" => "black!40" },
                 Coordinates(g_line)),
            LegendEntry(L"$g(x,z)$"),
        )
    else
        return @pgf Axis(
            {
                width  => width,
                height => height,
                xlabel => xlabel,
                ylabel => ylabel,

                xmin   => 0,
                xmax   => 1,
                ymin   => ymin,
                ymax   => ymax,

                "xtick" => xtick_str,
                "xticklabels" => xticklab_str,
                "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
                "ytick" => "{0}",

                "tick align" => "outside",
                "major tick length" => "2pt",
                "minor tick length" => "1pt",
                "scaled y ticks" => "false",
                "unbounded coords" => "jump",
                "legend pos" => "north east",

                # rug marks at jump locations
                "extra x ticks" => extra_xtick_str,
                "extra x tick labels" => extra_xticklab_str,
                "extra x tick style" => "{tick align=inside, tick style={black!55}}",
            },

            Plot({ "line width" => lw_base, "solid" => nothing, "black" => nothing },
                 Coordinates(f_line)),
            LegendEntry(L"$f(x,z)$"),

            Plot({ "line width" => lw_base, "solid" => nothing, "color" => "black!40" },
                 Coordinates(g_line)),
            LegendEntry(L"$g(x,z)$"),

            Plot({ "line width" => lw_shldr, "solid" => nothing,
                   "color" => "black!55", "forget plot" => nothing },
                 Coordinates(fg_eq_line)),
        )
    end
end

function save_fg_vs_x_plot_pdf(outpdf::AbstractString, z;
                               q, ρ, κ, r, β,
                               engine::AbstractString="pdflatex",
                               quiet::Bool=true,
                               keep_tex::Bool=true,
                               kwargs...)
    ax  = plot_fg_vs_x_axis(z; q=q, ρ=ρ, κ=κ, r=r, β=β, kwargs...)
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(texfile, pic; include_preamble=true)
    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

In [ ]:
# parameters (example)
q  = 0.25
ρ  = 0.6
κ  = 0.8
β  = 0.95
r  = 1.0

x0, x1, xhi = fixed_points(; q=q, ρ=ρ, κ=κ)

# z = 0.5*q
# z = 0.5*q + 0.5*x1
#z = 0.5*x0 + 0.5*(q+ρ)
# z = 0.5*(q+ρ) + 0.5*1.0
z = 0.5*x1 + 0.5*x0

save_fg_vs_x_plot_pdf("fg_vs_x.pdf", z;
    q=q, ρ=ρ, κ=κ, r=r, β=β,
    ε=1e-12,
    xgrid=collect(range(0.0, 1.0; length=1001)))

In [ ]:
using PGFPlotsX

# --- helper: snap consecutive values within tol to the same level ---
function _snap_consecutive(y::AbstractVector{<:Real}; tol::Real=1e-8)
    yy = Float64.(y)
    @inbounds for i in 2:length(yy)
        if isfinite(yy[i]) && isfinite(yy[i-1]) && abs(yy[i] - yy[i-1]) ≤ tol
            yy[i] = yy[i-1]
        end
    end
    return yy
end

# Return (coords, jump_locs) where coords has NaN breaks at detected jumps.
function _polyline_with_jump_breaks(xgrid::AbstractVector{<:Real},
                                    ygrid::AbstractVector{<:Real};
                                    jump_atol::Real=1e-6,
                                    jump_rtol::Real=0.0)
    @assert length(xgrid) == length(ygrid) ≥ 2
    coords = Tuple{Float64,Float64}[]
    jump_locs = Float64[]

    for i in 1:length(ygrid)-1
        x1 = float(xgrid[i]);   y1 = float(ygrid[i])
        x2 = float(xgrid[i+1]); y2 = float(ygrid[i+1])

        push!(coords, (x1, y1))

        if isfinite(y1) && isfinite(y2)
            if abs(y2 - y1) > max(jump_atol, jump_rtol * max(abs(y1), abs(y2)))
                # break the curve here so no connector is drawn across the jump
                push!(coords, (NaN, NaN))
                push!(jump_locs, x2)
            end
        end
    end

    push!(coords, (float(xgrid[end]), float(ygrid[end])))

    sort!(unique!(jump_locs))
    return coords, jump_locs
end

function plot_fg_metrics2_on_diagonal_vs_x_axis(; q, ρ, κ, r, β, ε=1e-10,
                                                xgrid=collect(range(0.0, 1.0; length=2001)),
                                                xlabel=L"$x$",
                                                ylabel=L"$f(x,x),\ g(x,x)$",
                                                width="11cm", height="7cm",
                                                snap_tol_f=0.0,
                                                snap_tol_g=1e-8,
                                                jump_atol=1e-6,
                                                jump_rtol=0.0,
                                                smooth_curves::Bool=true)

    # fixed points for ticks
    x0_fp, x1_fp, _ = fixed_points(; q=q, ρ=ρ, κ=κ)

    items = [
        (0.0, "\\ensuremath{0}"),
        (float(q), "\\ensuremath{p_{01}}"),
        (float(x), "\\ensuremath{x}"),
        (float(x1_fp),"\\ensuremath{x^{1}}"),
        (float(x0_fp),"\\ensuremath{x^{0}}"),
        (float(q+ρ), "\\ensuremath{p_{11}}"),
        (1.0, "\\ensuremath{1}"),
    ]
    
    sort!(items, by=first)

    merged = Tuple{Float64,String}[]
    for (u, lab) in items
        if isempty(merged) || abs(u - merged[end][1]) > 1e-12
            push!(merged, (u, lab))
        end
    end
    xticks  = [u for (u, _) in merged]
    labels  = [lab for (_, lab) in merged]
    xtick_str = "{" * join(string.(xticks), ",") * "}"
    phantom = "\\vphantom{\\ensuremath{q+\\rho}}"
    xticklab_str = "{" * join([phantom * lab for lab in labels], ",") * "}"

    # sample
    fvals = similar(xgrid, Float64)
    gvals = similar(xgrid, Float64)
    for (i, x) in enumerate(xgrid)
        fvals[i] = float(f_metric2(x, x; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε))
        gvals[i] = float(g_metric2(x, x; q=q, ρ=ρ, κ=κ,      β=β, ε=ε))
    end

    if snap_tol_f > 0
        fvals = _snap_consecutive(fvals; tol=snap_tol_f)
    end
    if snap_tol_g > 0
        gvals = _snap_consecutive(gvals; tol=snap_tol_g)
    end

    # sampled curves with NaN breaks at detected jumps
    f_line, f_jumps = _polyline_with_jump_breaks(xgrid, fvals;
                                                 jump_atol=jump_atol, jump_rtol=jump_rtol)
    g_line, g_jumps = _polyline_with_jump_breaks(xgrid, gvals;
                                                 jump_atol=jump_atol, jump_rtol=jump_rtol)

    # rug marks at union of detected jumps
    jump_xs = sort!(unique!(vcat(f_jumps, g_jumps)))
    extra_xtick_str    = isempty(jump_xs) ? "{}" : "{" * join(string.(jump_xs), ",") * "}"
    extra_xticklab_str = isempty(jump_xs) ? "{}" : "{" * join(fill("", length(jump_xs)), ",") * "}"

    # reference line
    yref = 1 - β
    ref_tbl = PGFPlotsX.Table(x=[0.0, 1.0], y=[yref, yref])

    # y-limits
    finite_vals = filter(isfinite, vcat(fvals, gvals, [yref]))
    ymax = isempty(finite_vals) ? 1.0 : maximum(finite_vals)
    ymax = max(ymax, 1e-12)

    # plot options for the sampled curves
    f_opts = Pair{Any,Any}[
        "line width" => "0.30pt",
        "black" => nothing,
    ]

    g_opts = Pair{Any,Any}[
        "line width" => "0.30pt",
        "color" => "black!40",
    ]

    if smooth_curves
        push!(f_opts, "smooth" => nothing)
        push!(g_opts, "smooth" => nothing)
    else
        push!(f_opts, "solid" => nothing)
        push!(g_opts, "solid" => nothing)
    end

    return @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,

            xmin   => 0,
            xmax   => 1,
            ymin   => 0,
            ymax   => ymax,

            "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
            "tick align" => "outside",
            "major tick length" => "2pt",
            "minor tick length" => "1pt",

            "xtick" => xtick_str,
            "xticklabels" => xticklab_str,

            # rug ticks (upwards), no labels
            "extra x ticks" => extra_xtick_str,
            "extra x tick labels" => extra_xticklab_str,
            "extra x tick style" => "{tick align=inside, tick style={black!55}}",

            "xtick pos" => "bottom",
            "ytick pos" => "left",

            "unbounded coords" => "jump",

            "ytick" => "{0,1}",
            "scaled y ticks" => "false",
            "legend pos" => "north west",
        },

        Plot(PGFPlotsX.Options(f_opts...), Coordinates(f_line)),
        LegendEntry(L"$f(x,x)$"),

        Plot(PGFPlotsX.Options(g_opts...), Coordinates(g_line)),
        LegendEntry(L"$g(x,x)$"),

        Plot({ "line width" => "0.30pt", "dash pattern" => "on 1pt off 1pt" },
             ref_tbl),
        LegendEntry(L"$1-\beta$"),
    )
end

function save_fg_metrics2_on_diagonal_vs_x_plot_pdf(outpdf::AbstractString;
                                                    q, ρ, κ, r, β,
                                                    engine::AbstractString="pdflatex",
                                                    quiet::Bool=true,
                                                    keep_tex::Bool=true,
                                                    kwargs...)
    ax  = plot_fg_metrics2_on_diagonal_vs_x_axis(; q=q, ρ=ρ, κ=κ, r=r, β=β, kwargs...)
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

In [ ]:
# parameters (example)
q  = 0.25
ρ  = 0.6
κ  = 0.8
β  = 0.95
r  = 1.0

x0, x1, xhi = fixed_points(; q=q, ρ=ρ, κ=κ)
# z = 0.5 * (x1 + x0)
x  = 0.5*(x0 + (q+ρ))

save_fg_metrics2_on_diagonal_vs_x_plot_pdf("fg_diag.pdf";
    q=q, ρ=ρ, κ=κ, r=r, β=β,
    ε=1e-20, xgrid=range(0,1; length=2001),
    jump_atol=0.5e-2)   # try 1e-4, then 1e-3 if still dense

In [ ]:
using PGFPlotsX

# --- helper: snap consecutive values within tol to the same level ---
function _snap_consecutive(y::AbstractVector{<:Real}; tol::Real=1e-8)
    yy = Float64.(y)
    @inbounds for i in 2:length(yy)
        if isfinite(yy[i]) && isfinite(yy[i-1]) && abs(yy[i] - yy[i-1]) ≤ tol
            yy[i] = yy[i-1]
        end
    end
    return yy
end

"""
    plot_g_metric2_on_diagonal_vs_x_axis(; q, ρ, κ, β, ε=1e-10, ...)

Step plot of g(x,x)=g_metric2(x,x) vs x∈[0,1], right-continuous,
with a horizontal reference line at y = 1-β.

Mimics your working PGFPlotsX style:
- x-ticks at {0, q, x^1, x^0, q+ρ, 1}
- y-ticks only at {0,1}
- legend in north west
- dotted curve for g
"""
function plot_g_metric2_on_diagonal_vs_x_axis(; q, ρ, κ, β, ε=1e-10,
                                             xgrid=collect(range(0.0, 1.0; length=2001)),
                                             xlabel=L"$x$",
                                             ylabel=L"$g(x,x)$",
                                             width="11cm", height="7cm",
                                             ypad=0.05,
                                             atol_g=1e-10,
                                             snap_tol_g=1e-8)

    # fixed points for tick positions
    x0_fp, x1_fp, _ = fixed_points(; q=q, ρ=ρ, κ=κ)

    # x-axis ticks/labels (static)
    items = [
        (0.0, "\\ensuremath{0}"),
        (float(q), "\\ensuremath{p_{01}}"),
        (float(x), "\\ensuremath{x}"),
        (float(x1_fp),"\\ensuremath{x^{1}}"),
        (float(x0_fp),"\\ensuremath{x^{0}}"),
        (float(q+ρ), "\\ensuremath{p_{11}}"),
        (1.0, "\\ensuremath{1}"),
    ]

    sort!(items, by=first)

    merged = Tuple{Float64,String}[]
    for (u, lab) in items
        if isempty(merged) || abs(u - merged[end][1]) > 1e-12
            push!(merged, (u, lab))
        end
    end
    xticks = [u for (u, _) in merged]
    labels = [lab for (_, lab) in merged]

    xtick_str = "{" * join(string.(xticks), ",") * "}"
    phantom = "\\vphantom{\\ensuremath{q+\\rho}}"
    xticklab_str = "{" * join([phantom * lab for lab in labels], ",") * "}"

    # sample g(x,x)
    gvals = similar(xgrid, Float64)
    for (i, x) in enumerate(xgrid)
        gvals[i] = float(g_metric2(x, x; q=q, ρ=ρ, κ=κ, β=β, ε=ε))
    end

    # snap jitter so piecewise-constant regions render as such
    if snap_tol_g > 0
        gvals = _snap_consecutive(gvals; tol=snap_tol_g)
    end

    # step data
    g_line, g_vline, _, _ = _stepdata_right_continuous(xgrid, gvals; atol=atol_g)

    # reference y = 1-β
    yref = 1 - β
    ref_tbl = PGFPlotsX.Table(x=[0.0, 1.0], y=[yref, yref])

    # y-limits with padding (include yref)
    finite_vals = filter(isfinite, vcat(gvals, [yref]))
    ymin = minimum(finite_vals)
    ymax = maximum(finite_vals)
    if ymax > ymin
        pad = ypad * (ymax - ymin)
        ymin -= pad
        ymax += pad
    else
        ymin -= 1
        ymax += 1
    end

    return @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,

            xmin   => 0,
            xmax   => 1,
            ymin   => ymin,
            ymax   => ymax,

            "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
            "tick align" => "outside",
            "major tick length" => "2pt",
            "minor tick length" => "1pt",

            "xtick" => xtick_str,
            "xticklabels" => xticklab_str,

            "ytick" => "{0,1}",
            "scaled y ticks" => "false",
            "unbounded coords" => "jump",

            "legend pos" => "north west",
        },

        # g(x,x) horizontal (densely dotted) + legend
        Plot({ "line width" => "0.30pt", "densely dotted" => nothing },
             Coordinates(g_line)),
        LegendEntry(L"$g(x,x)$"),

        # g(x,x) vertical jumps (densely dotted) - forget plot
        Plot({ "line width" => "0.30pt", "densely dotted" => nothing, "forget plot" => nothing },
             Coordinates(g_vline)),

        # reference line y = 1-β + legend
        Plot({ "line width" => "0.25pt", "dash pattern" => "on 1pt off 1pt" },
             ref_tbl),
        LegendEntry(L"$1-\beta$"),
    )
end


"""
    save_g_metric2_on_diagonal_vs_x_plot_pdf(outpdf; q, ρ, κ, β, engine="pdflatex", quiet=true)

Standalone .tex + compile to PDF (wrapper matches your pattern).
"""
function save_g_metric2_on_diagonal_vs_x_plot_pdf(outpdf::AbstractString;
                                                  q, ρ, κ, β,
                                                  engine::AbstractString="pdflatex",
                                                  quiet::Bool=true,
                                                  keep_tex::Bool=true,
                                                  kwargs...)
    ax  = plot_g_metric2_on_diagonal_vs_x_axis(; q=q, ρ=ρ, κ=κ, β=β, kwargs...)
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

# Example:
# q=0.25; ρ=0.6; κ=0.8; β=0.95
# save_g_metric2_on_diagonal_vs_x_plot_pdf("g_diag_vs_x.pdf"; q=q, ρ=ρ, κ=κ, β=β)


In [ ]:
# parameters (example)
q  = 0.25
ρ  = 0.6
κ  = 0.8
β  = 0.1
r  = 1.0

x0, x1, xhi = fixed_points(; q=q, ρ=ρ, κ=κ)
# z = 0.5 * (x1 + x0)
x  = 0.5*(x0 + (q+ρ))

save_g_metric2_on_diagonal_vs_x_plot_pdf("g_diag_vs_x.pdf"; q=q, ρ=ρ, κ=κ, β=β, ε=1e-20, xgrid=collect(range(0.0, 1.0; length=2001)))

In [ ]:
# MP index m(x) = f(x,x)/g(x,x)
function mp_index(x; q, ρ, κ, r, β, ε=1e-10)
    fxx = float(f_metric2(x, x; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε))
    gxx = float(g_metric2(x, x; q=q, ρ=ρ, κ=κ,      β=β, ε=ε))
    return (isfinite(fxx) && isfinite(gxx) && gxx != 0.0) ? (fxx / gxx) : NaN
end

"""
    zstar(λ; q, ρ, κ, r, β, ε=1e-10, xatol=1e-10, λatol=1e-10, maxiter=200)

Compute the optimal threshold z*(λ) by numerically inverting m(x).

Definition used:
- z*(λ) = 0  if λ < 0
- z*(λ) = 1  if λ > m(1)
- for 0 ≤ λ ≤ m(1), z*(λ) is such that m(z)=λ

Assumes m(x) is continuous and nondecreasing on [0,1].
"""
function zstar(λ; q, ρ, κ, r, β,
               ε=1e-10,
               xatol=1e-10,
               λatol=1e-10,
               maxiter=200)

    λ = float(λ)

    # boundary cases
    if λ < 0.0
        return 0.0
    end

    m1 = mp_index(1.0; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    if !isfinite(m1)
        error("m(1) is not finite.")
    end

    if λ > m1
        return 1.0
    end

    # use m(0)=0
    if abs(λ) <= λatol
        return 0.0
    end
    if abs(λ - m1) <= λatol
        return 1.0
    end

    # bisection on m(x)-λ
    a = 0.0
    b = 1.0
    fa = -λ                     # since m(0)=0
    fb = m1 - λ

    if fa > 0 || fb < 0
        error("Bisection bracket failed: λ is not bracketed by m(0)=0 and m(1)=$m1.")
    end

    for _ in 1:maxiter
        c = 0.5 * (a + b)
        mc = mp_index(c; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
        if !isfinite(mc)
            error("m($c) is not finite.")
        end

        fc = mc - λ

        if abs(fc) <= λatol || 0.5 * (b - a) <= xatol
            return c
        end

        if fc < 0
            a = c
            fa = fc
        else
            b = c
            fb = fc
        end
    end

    return 0.5 * (a + b)
end

In [ ]:
using PGFPlotsX

# --- helper: snap consecutive values within tol (optional) ---
function _snap_consecutive(y::AbstractVector{<:Real}; tol::Real=1e-8)
    yy = Float64.(y)
    @inbounds for i in 2:length(yy)
        if isfinite(yy[i]) && isfinite(yy[i-1]) && abs(yy[i] - yy[i-1]) ≤ tol
            yy[i] = yy[i-1]
        end
    end
    return yy
end

"""
Plot z*(λ) versus λ.

By default, plots on [0, m(1)] × [0,1].
If desired, one can extend the λ-range outside [0,m(1)] to show the clipped parts
z*(λ)=0 for λ<0 and z*(λ)=1 for λ>m(1).
"""
function plot_zstar_vs_lambda_axis(; q, ρ, κ, r, β, ε=1e-10,
                                   lambdagrid=nothing,
                                   lambda_min=nothing,
                                   lambda_max=nothing,
                                   nlambda=2001,
                                   xlabel=L"$\lambda$",
                                   ylabel=L"$z^\star(\lambda)$",
                                   width="8cm", height="8cm",
                                   snap_tol_z=0.0,
                                   smooth_curve::Bool=true)

    # fixed points
    x0_fp, x1_fp, _ = fixed_points(; q=q, ρ=ρ, κ=κ)

    # structural λ-values
    mp01 = mp_index(float(q);      q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    mx1  = mp_index(float(x1_fp);  q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    mx0  = mp_index(float(x0_fp);  q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    mp11 = mp_index(float(q+ρ);    q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    m1   = mp_index(1.0;           q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)

    if !isfinite(m1)
        error("m(1) is not finite.")
    end

    λmin = isnothing(lambda_min) ? 0.0       : float(lambda_min)
    λmax = isnothing(lambda_max) ? float(m1) : float(lambda_max)

    if isnothing(lambdagrid)
        λgrid = collect(range(λmin, λmax; length=nlambda))
    else
        λgrid = Float64.(collect(lambdagrid))
        λmin = minimum(λgrid)
        λmax = maximum(λgrid)
    end

    items = [
        (0.0,         "\\ensuremath{0}"),
        (float(mp01), "\\makebox[1.5pt][r]{\\ensuremath{m(p_{01})}\\hspace{0.5pt}}"),
        (float(mx1),  "\\makebox[1.5pt][l]{\\ensuremath{m(x^{1})}\\hspace{0.5pt}}"),
        (float(mx0),  "\\makebox[1.5pt][r]{\\ensuremath{m(x^{0})}\\hspace{0.5pt}}"),
        (float(mp11), "\\ensuremath{m(p_{11})}"),
        (float(m1),   "\\ensuremath{m(1)}"),
    ]
    sort!(items, by=first)

    merged = Tuple{Float64,String}[]
    for (u, lab) in items
        if isempty(merged) || abs(u - merged[end][1]) > 1e-12
            push!(merged, (u, lab))
        end
    end

    xticks = [u for (u, _) in merged]
    labels = [lab for (_, lab) in merged]

    xtick_str = "{" * join(string.(xticks), ",") * "}"
    phantom = "\\vphantom{\\ensuremath{m(p_{11})}}"
    xticklab_str = "{" * join([phantom * lab for lab in labels], ",") * "}"

    zvals = similar(λgrid, Float64)
    for (i, λ) in enumerate(λgrid)
        zvals[i] = float(zstar(λ; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε))
    end

    if snap_tol_z > 0
        zvals = _snap_consecutive(zvals; tol=snap_tol_z)
    end

    coords = Tuple{Float64,Float64}[
        (float(λ), isfinite(z) ? float(z) : NaN) for (λ, z) in zip(λgrid, zvals)
    ]

    curve_opts = Pair{Any,Any}[
        "line width" => "0.30pt",
    ]
    if smooth_curve
        push!(curve_opts, "smooth" => nothing)
    else
        push!(curve_opts, "solid" => nothing)
    end

    return @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,

            xmin   => λmin,
            xmax   => λmax,
            ymin   => 0,
            ymax   => 1,

            "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
            "tick align" => "outside",
            "major tick length" => "2pt",
            "minor tick length" => "1pt",

            "xtick" => xtick_str,
            "xticklabels" => xticklab_str,

            "ytick" => "{0,1}",
            "enlargelimits" => "false",

            "scaled x ticks" => "false",
            "scaled y ticks" => "false",
            "unbounded coords" => "jump",
        },

        Plot(PGFPlotsX.Options(curve_opts...), Coordinates(coords)),
    )
end

function save_zstar_vs_lambda_plot_pdf(outpdf::AbstractString;
                                       q, ρ, κ, r, β,
                                       engine::AbstractString="pdflatex",
                                       quiet::Bool=true,
                                       keep_tex::Bool=true,
                                       kwargs...)
    ax  = plot_zstar_vs_lambda_axis(; q=q, ρ=ρ, κ=κ, r=r, β=β, kwargs...)
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

In [ ]:
q=0.25; ρ=0.6; κ=0.8; r=1.0; β=0.95
save_zstar_vs_lambda_plot_pdf("zstar_vs_lambda.pdf"; q=q, ρ=ρ, κ=κ, r=r, β=β)

In [ ]:
function L_lambda(λ, x; q, ρ, κ, r, β, ε=1e-10)
    z = zstar(λ; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)

    Fx = F_metric(x, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    Gx = G_metric(x, z; q=q, ρ=ρ, κ=κ,      β=β, ε=ε)

    return Fx - λ * Gx
end

In [ ]:
using PGFPlotsX

# --- helper: snap consecutive values within tol (optional) ---
function _snap_consecutive(y::AbstractVector{<:Real}; tol::Real=1e-8)
    yy = Float64.(y)
    @inbounds for i in 2:length(yy)
        if isfinite(yy[i]) && isfinite(yy[i-1]) && abs(yy[i] - yy[i-1]) ≤ tol
            yy[i] = yy[i-1]
        end
    end
    return yy
end

"""
Plot L(x, λ) = F(x, z*(λ)) - λ G(x, z*(λ)) versus λ on [0,1], for fixed x.

The x-axis uses the structural λ-ticks
0, m(p01), m(x¹), m(x⁰), m(p11), m(1),
restricted to those that fall inside [0,1].
"""
function plot_L_lambda_vs_lambda_axis(x; q, ρ, κ, r, β, ε=1e-10,
                                      lambdagrid=collect(range(0.0, 1.0; length=2001)),
                                      xlabel=L"$\lambda$",
                                      ylabel=L"$L(x,\lambda)$",
                                      width="8cm", height="8cm",
                                      ypad=0.05,
                                      snap_tol_L=0.0)

    λgrid = Float64.(collect(lambdagrid))
    λmin = minimum(λgrid)
    λmax = maximum(λgrid)

    # fixed points
    x0_fp, x1_fp, _ = fixed_points(; q=q, ρ=ρ, κ=κ)

    # structural λ-values
    mp01 = mp_index(float(q);      q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    mx1  = mp_index(float(x1_fp);  q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    mx0  = mp_index(float(x0_fp);  q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    mp11 = mp_index(float(q+ρ);    q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    m1   = mp_index(1.0;           q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)

    items = [
        (0.0,        "\\ensuremath{0}"),
        (float(mp01),"\\makebox[1.5pt][r]{\\ensuremath{m(p_{01})}\\hspace{0.5pt}}"),
        (float(mx1), "\\makebox[1.5pt][l]{\\ensuremath{m(x^{1})}\\hspace{0.5pt}}"),
        (float(mx0), "\\makebox[0.5pt][r]{\\ensuremath{m(x^{0})}\\hspace{1.5pt}}"),
        (float(mp11),"\\makebox[20.0pt][l]{\\ensuremath{m(p_{11})}\\hspace{4.0pt}}"),
        (float(m1),  "\\makebox[1.5pt][l]{\\ensuremath{m(1)}\\hspace{0.5pt}}"),
        (1.0,        "\\ensuremath{1}"),
    ]
    sort!(items, by=first)

    # keep only ticks inside the plotting λ-range
    items = [(u, lab) for (u, lab) in items if λmin - 1e-12 <= u <= λmax + 1e-12]
    sort!(items, by=first)

    merged = Tuple{Float64,String}[]
    for (u, lab) in items
        if isempty(merged) || abs(u - merged[end][1]) > 1e-12
            push!(merged, (u, lab))
        end
    end

    xticks = [u for (u, _) in merged]
    labels = [lab for (_, lab) in merged]

    xtick_str = isempty(xticks) ? "{}" : "{" * join(string.(xticks), ",") * "}"
    phantom = "\\vphantom{\\ensuremath{m(p_{11})}}"
    xticklab_str = isempty(labels) ? "{}" : "{" * join([phantom * lab for lab in labels], ",") * "}"

    # sample L(x, λ)
    Lvals = similar(λgrid, Float64)
    for (i, λ) in enumerate(λgrid)
        Lvals[i] = float(L_lambda(λ, x; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε))
    end

    if snap_tol_L > 0
        Lvals = _snap_consecutive(Lvals; tol=snap_tol_L)
    end

    finitevals = [v for v in Lvals if isfinite(v)]
    if isempty(finitevals)
        error("No finite values found for L(x, λ) on the supplied lambda grid.")
    end

    ymin = minimum(finitevals)
    ymax = maximum(finitevals)
    if ymax > ymin
        pad = ypad * (ymax - ymin)
        ymin -= pad
        ymax += pad
    else
        ymin -= 1
        ymax += 1
    end

    coords = Tuple{Float64,Float64}[
        (float(λ), isfinite(L) ? float(L) : NaN) for (λ, L) in zip(λgrid, Lvals)
    ]

    return @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,

            xmin   => λmin,
            xmax   => λmax,
            ymin   => ymin,
            ymax   => ymax,

            "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
            "tick align" => "outside",
            "major tick length" => "2pt",
            "minor tick length" => "1pt",

            "xtick" => xtick_str,
            "xticklabels" => xticklab_str,

            "ytick" => "{0}",
            "enlargelimits" => "false",

            "scaled x ticks" => "false",
            "scaled y ticks" => "false",
            "unbounded coords" => "jump",
        },

        Plot({ "line width" => "0.30pt", "solid" => nothing },
             Coordinates(coords)),
    )
end

function save_L_lambda_vs_lambda_plot_pdf(outpdf::AbstractString, x;
                                          q, ρ, κ, r, β,
                                          engine::AbstractString="pdflatex",
                                          quiet::Bool=true,
                                          keep_tex::Bool=true,
                                          kwargs...)
    ax  = plot_L_lambda_vs_lambda_axis(x; q=q, ρ=ρ, κ=κ, r=r, β=β, kwargs...)
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

In [ ]:
q=0.25; ρ=0.6; κ=0.8; r=1.0; β=0.95; x=0.4
x0, x1, xhi = fixed_points(; q=q, ρ=ρ, κ=κ)
x = 0.5 * (x1 + x0)
save_L_lambda_vs_lambda_plot_pdf("L_lambda_vs_lambda.pdf", x;
    q=q, ρ=ρ, κ=κ, r=r, β=β)

In [ ]:
function Lvector(λ, xvector; qvector, ρvector, κvector, rvector, β, M, ε=1e-10)
    xvec = Float64.(collect(xvector))
    qvec = Float64.(collect(qvector))
    ρvec = Float64.(collect(ρvector))
    κvec = Float64.(collect(κvector))
    rvec = Float64.(collect(rvector))

    N = length(xvec)
    length(qvec) == N || throw(ArgumentError("qvector must have length $N"))
    length(ρvec) == N || throw(ArgumentError("ρvector must have length $N"))
    length(κvec) == N || throw(ArgumentError("κvector must have length $N"))
    length(rvec) == N || throw(ArgumentError("rvector must have length $N"))

    β < 1 || throw(ArgumentError("β must satisfy β < 1"))
    M ≥ 0 || throw(ArgumentError("M must satisfy M ≥ 0"))

    total = 0.0
    @inbounds for n in 1:N
        total += L_lambda(λ, xvec[n];
                          q=qvec[n], ρ=ρvec[n], κ=κvec[n], r=rvec[n], β=β, ε=ε)
    end

    return total + M * λ / (1 - β)
end

In [ ]:
# Example with 3 projects
q=0.25; ρ=0.6; κ=0.8; r=1.0; β=0.95

qvector =   [0.25, 0.1, 0.3]
ρvector = [0.6, 0.8, 0.65]
κvector = [0.8, 0.6, 0.7]
rvector = [1.0, 0.8,  1.2]
beta  = 0.95
M     = 2
xvector = [0.5, 0.5, 0.5]

λ = 0.8

L_total = Lvector(λ, xvector; qvector, ρvector, κvector, rvector, β, M, ε=1e-10)
println("L_vec($λ) = ", L_total)

In [ ]:
using PGFPlotsX

function plotLvector(xvector; qvector, ρvector, κvector, rvector, β, M, ε=1e-10,
                     lambdagrid=collect(range(0.0, 1.0; length=101)),
                     xlabel=L"$\lambda$",
                     ylabel=L"$L(\mathbf{x};\lambda)$",
                     width="8cm", height="8cm",
                     ypad=0.05,
                     snap_tol_L=0.0)

    λgrid = Float64.(collect(lambdagrid))
    λmin = minimum(λgrid)
    λmax = maximum(λgrid)

    # sample Lvector(λ)
    Lvals = similar(λgrid, Float64)
    for (i, λ) in enumerate(λgrid)
        Lvals[i] = float(Lvector(λ, xvector;
                                 qvector=qvector, ρvector=ρvector,
                                 κvector=κvector, rvector=rvector,
                                 β=β, M=M, ε=ε))
    end

    if snap_tol_L > 0
        Lvals = _snap_consecutive(Lvals; tol=snap_tol_L)
    end

    finitevals = [v for v in Lvals if isfinite(v)]
    isempty(finitevals) && error("No finite values found for Lvector on the supplied lambda grid.")

    ymin = minimum(finitevals)
    ymax = maximum(finitevals)
    if ymax > ymin
        pad = ypad * (ymax - ymin)
        ymin -= pad
        ymax += pad
    else
        ymin -= 1
        ymax += 1
    end

    coords = Tuple{Float64,Float64}[
        (float(λ), isfinite(L) ? float(L) : NaN) for (λ, L) in zip(λgrid, Lvals)
    ]

    return @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,

            xmin   => λmin,
            xmax   => λmax,
            ymin   => ymin,
            ymax   => ymax,

            "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
            "tick align" => "outside",
            "major tick length" => "2pt",
            "minor tick length" => "1pt",

            # only ticks 0 and 1 on the lambda axis
            "xtick" => "{0,1}",
            "xticklabels" => "{0,1}",

            "ytick" => "{0}",
            "enlargelimits" => "false",

            "scaled x ticks" => "false",
            "scaled y ticks" => "false",
            "unbounded coords" => "jump",
        },

        Plot({ "line width" => "0.30pt", "solid" => nothing },
             Coordinates(coords)),
    )
end

function save_Lvector_plot_pdf(outpdf::AbstractString, xvector;
                               qvector, ρvector, κvector, rvector, β, M,
                               engine::AbstractString="pdflatex",
                               quiet::Bool=true,
                               keep_tex::Bool=true,
                               kwargs...)
    ax  = plotLvector(xvector;
                      qvector=qvector, ρvector=ρvector,
                      κvector=κvector, rvector=rvector,
                      β=β, M=M, kwargs...)
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

In [ ]:
# Example with 3 projects
q=0.25; ρ=0.6; κ=0.8; r=1.0; β=0.95

qvector =   [0.25, 0.1, 0.3]
ρvector = [0.6, 0.8, 0.65]
κvector = [0.8, 0.6, 0.7]
rvector = [1.0, 0.8,  1.2]
beta  = 0.95
M     = 2
xvector = [0.5, 0.5, 0.5]

save_Lvector_plot_pdf("Lvector_vs_lambda.pdf", xvector;
    qvector=qvector, ρvector=ρvector, κvector=κvector, rvector=rvector,
    β=β, M=M)

In [ ]:
using CSV
using DataFrames

# -----------------------------------------------------------
# 1. Load the timed experiment results (or use your existing df)
# -----------------------------------------------------------

df_timed = CSV.read("typed_scaling_experiment_binned_timed.csv", DataFrame)

# -----------------------------------------------------------
# 2. Pick the pathological instance
#    A_mild_B_fragile, prop_type = 0.8,0.2, alphaM_target = 0.1, N = 100
# -----------------------------------------------------------

row = only(filter(r ->
    r.scenario == "A_mild_B_fragile" &&
    r.prop_type == "0.8,0.2" &&
    isapprox(r.alphaM_target, 0.10; atol=1e-12) &&
    r.N == 100,
    df_timed
))

@show row.scenario row.prop_type row.alphaM_target row.capacity_ratio row.N row.M row.seed row.t_bound_case

# -----------------------------------------------------------
# 3. Rebuild the exact realized instance used in that experiment
# -----------------------------------------------------------

scenario = (
    name = "A_mild_B_fragile",
    q_types = [0.03, 0.10],
    rho_types = [0.80, 0.10],
    kappa_types = [0.70, 0.95],
    r_types = [1.0, 1.0]
)

inst = build_typed_instance(
    Int(row.N);
    prop_type = [0.8, 0.2],
    q_types = scenario.q_types,
    rho_types = scenario.rho_types,
    kappa_types = scenario.kappa_types,
    r_types = scenario.r_types,
    x_init_scheme = :linspace,
    xlo = 0.0,
    xhi = 1.0,
    shuffle_types = true,
    shuffle_x = true,
    seed = Int(row.seed)
)

# -----------------------------------------------------------
# 4. Plot Lvector_vs_lambda for that exact instance
# -----------------------------------------------------------

save_Lvector_plot_pdf(
    "Lvector_vs_lambda_pathological_A_mild_B_fragile_prop_0p8_0p2_alpha_0p1_N100.pdf",
    inst.x_vec;
    qvector = inst.q_vec,
    ρvector = inst.rho_vec,
    κvector = inst.kappa_vec,
    rvector = inst.r_vec,
    β = row.beta,
    M = Int(row.M)
)

In [ ]:
"""
    lambda_star_vector(xvector; qvector, ρvector, κvector, rvector, β, M,
                       ε=1e-10, λlo=0.0, λhi=1.0, tol=1e-8, maxiter=200)

Compute a minimizer λ* of the convex function

    Lvector(λ, xvector; qvector=qvector, ρvector=ρvector,
            κvector=κvector, rvector=rvector, β=β, M=M, ε=ε)

over λ in [λlo, λhi] (default [0,1]), together with the minimum value.

Returns `(λstar, Lstar)`.

Method:
- golden-section search on [λlo, λhi]
- final comparison with endpoints for robustness
"""
function lambda_star_vector(xvector; qvector, ρvector, κvector, rvector, β, M,
                            ε=1e-10,
                            λlo=0.0,
                            λhi=1.0,
                            tol=1e-8,
                            maxiter=200)

    λlo = float(λlo)
    λhi = float(λhi)

    0.0 <= λlo <= λhi <= 1.0 || throw(ArgumentError("Require 0 ≤ λlo ≤ λhi ≤ 1."))

    f(λ) = Lvector(λ, xvector;
                   qvector=qvector, ρvector=ρvector,
                   κvector=κvector, rvector=rvector,
                   β=β, M=M, ε=ε)

    # golden-section search
    ϕ = (sqrt(5.0) - 1.0) / 2.0   # 0.618...

    a = λlo
    b = λhi
    c = b - ϕ * (b - a)
    d = a + ϕ * (b - a)

    fc = f(c)
    fd = f(d)

    for _ in 1:maxiter
        if (b - a) <= tol
            break
        end

        if fc <= fd
            b  = d
            d  = c
            fd = fc
            c  = b - ϕ * (b - a)
            fc = f(c)
        else
            a  = c
            c  = d
            fc = fd
            d  = a + ϕ * (b - a)
            fd = f(d)
        end
    end

    # candidate from final interval midpoint
    λmid = 0.5 * (a + b)
    fmid = f(λmid)

    # compare with endpoints too, for robustness when the minimum is at 0 or 1
    fa = f(λlo)
    fb = f(λhi)

    if fa <= fmid && fa <= fb
        return λlo, fa
    elseif fb <= fmid && fb <= fa
        return λhi, fb
    else
        return λmid, fmid
    end
end

In [ ]:
# Example with 3 projects
q=0.25; ρ=0.6; κ=0.8; r=1.0; β=0.95

qvector =   [0.25, 0.1, 0.3]
ρvector = [0.6, 0.8, 0.65]
κvector = [0.8, 0.6, 0.7]
rvector = [1.0, 0.8,  1.2]
beta  = 0.95
M     = 2
xvector = [0.5, 0.5, 0.5]

λstar, Lmin = lambda_star_vector(xvector;
    qvector=qvector, ρvector=ρvector,
    κvector=κvector, rvector=rvector,
    β=β, M=M)

In [ ]:
using LinearAlgebra
using Printf

# ----------------------------------------------------------
# Helper: extract scalar value if metric returns a tuple
# ----------------------------------------------------------
val(u) = u isa Tuple ? u[1] : u

# ----------------------------------------------------------
# Compressed representation of repeated project types
# ----------------------------------------------------------
struct CompressedTypeGroup
    q::Float64
    ρ::Float64
    κ::Float64
    r::Float64
    xs::Vector{Float64}   # distinct initial beliefs for this type
    mult::Vector{Int}     # multiplicities of those beliefs
end

"""
    compress_project_vectors(xvector; qvector, ρvector, κvector, rvector)

Compress projects by identical parameter tuples (q,ρ,κ,r), and within each type
compress repeated x-values by multiplicity.

This lets us:
- compute zstar(λ) once per type per λ
- evaluate F/G once per distinct x-value per type, multiplying by multiplicity
"""
function compress_project_vectors(xvector;
                                  qvector,
                                  ρvector,
                                  κvector,
                                  rvector)

    N = length(xvector)
    @assert length(qvector) == N
    @assert length(ρvector) == N
    @assert length(κvector) == N
    @assert length(rvector) == N

    groups_dict = Dict{NTuple{4,Float64}, Dict{Float64,Int}}()

    for i in 1:N
        key = (
            Float64(qvector[i]),
            Float64(ρvector[i]),
            Float64(κvector[i]),
            Float64(rvector[i])
        )
        x = Float64(xvector[i])

        xdict = get!(groups_dict, key, Dict{Float64,Int}())
        xdict[x] = get(xdict, x, 0) + 1
    end

    groups = CompressedTypeGroup[]
    for (key, xdict) in groups_dict
        xs = collect(keys(xdict))
        sort!(xs)
        mult = [xdict[x] for x in xs]
        push!(groups, CompressedTypeGroup(key[1], key[2], key[3], key[4], xs, mult))
    end

    sort!(groups; by = g -> (g.q, g.ρ, g.κ, g.r))
    return groups
end

# ----------------------------------------------------------
# Evaluate Lvector(λ) and one valid subgradient value
# ----------------------------------------------------------
"""
    eval_Lvector_grouped(λ, groups; β, M, ε=1e-10, xatol=1e-10, λatol=1e-10, maxiter=200)

Evaluate
    L(λ) = M*λ/(1-β) + Σ_n [F_n(x_n,z*_n(λ)) - λ G_n(x_n,z*_n(λ))]
and one valid subgradient
    s(λ) = M/(1-β) - Σ_n G_n(x_n,z*_n(λ))

where z*_n(λ) is common within each compressed type group.
"""
function eval_Lvector_grouped(λ::Real,
                              groups::Vector{CompressedTypeGroup};
                              β::Float64,
                              M::Int,
                              ε::Float64 = 1e-10,
                              xatol::Float64 = 1e-10,
                              λatol::Float64 = 1e-10,
                              maxiter::Int = 200)

    λf = Float64(λ)

    sumL = 0.0
    sumG = 0.0
    zstars = Float64[]

    for g in groups
        z = zstar(λf;
                  q = g.q, ρ = g.ρ, κ = g.κ, r = g.r, β = β,
                  ε = ε, xatol = xatol, λatol = λatol, maxiter = maxiter)
        push!(zstars, z)

        for (x, m) in zip(g.xs, g.mult)
            F = val(F_metric(x, z; q = g.q, ρ = g.ρ, κ = g.κ, r = g.r, β = β, ε = ε))
            G = val(G_metric(x, z; q = g.q, ρ = g.ρ, κ = g.κ,      β = β, ε = ε))

            sumL += m * (F - λf * G)
            sumG += m * G
        end
    end

    L = M * λf / (1 - β) + sumL
    s = M / (1 - β) - sumG

    return (L = L, subgrad = s, zstars = zstars)
end

# ----------------------------------------------------------
# Bracket the minimizer using the monotone subgradient
# ----------------------------------------------------------
"""
    bracket_lambda_grouped(groups; β, M, λ0=0.0, step0=1.0, max_expand=60, ...)

Find λlo <= λhi such that
    s(λlo) <= 0 <= s(λhi),
where s(λ) is a subgradient value.

If s(λ0) < 0 we expand to the right.
If s(λ0) > 0 we expand to the left.
"""
function bracket_lambda_grouped(groups::Vector{CompressedTypeGroup};
                                β::Float64,
                                M::Int,
                                λ0::Float64 = 0.0,
                                step0::Float64 = 1.0,
                                max_expand::Int = 60,
                                ε::Float64 = 1e-10,
                                xatol::Float64 = 1e-10,
                                λatol::Float64 = 1e-10,
                                maxiter::Int = 200)

    f0 = eval_Lvector_grouped(λ0, groups;
                              β = β, M = M,
                              ε = ε, xatol = xatol, λatol = λatol, maxiter = maxiter)
    s0 = f0.subgrad

    if s0 == 0.0
        return (λlo = λ0, λhi = λ0, slo = s0, shi = s0)
    elseif s0 < 0.0
        λlo = λ0
        slo = s0
        λhi = λ0 + step0
        for _ in 1:max_expand
            fhi = eval_Lvector_grouped(λhi, groups;
                                       β = β, M = M,
                                       ε = ε, xatol = xatol, λatol = λatol, maxiter = maxiter)
            shi = fhi.subgrad
            if shi >= 0.0
                return (λlo = λlo, λhi = λhi, slo = slo, shi = shi)
            end
            λhi = λ0 + 2.0 * (λhi - λ0)
        end
        error("Failed to bracket λ*: subgradient stayed negative while expanding to the right.")
    else
        λhi = λ0
        shi = s0
        λlo = λ0 - step0
        for _ in 1:max_expand
            flo = eval_Lvector_grouped(λlo, groups;
                                       β = β, M = M,
                                       ε = ε, xatol = xatol, λatol = λatol, maxiter = maxiter)
            slo = flo.subgrad
            if slo <= 0.0
                return (λlo = λlo, λhi = λhi, slo = slo, shi = shi)
            end
            λlo = λ0 + 2.0 * (λlo - λ0)
        end
        error("Failed to bracket λ*: subgradient stayed positive while expanding to the left.")
    end
end

# ----------------------------------------------------------
# Convexity-aware minimization by bisection on the subgradient
# ----------------------------------------------------------
"""
    lambda_star_vector_grouped_bisect(xvector; qvector, ρvector, κvector, rvector, β, M, ...)

Convexity-aware replacement for `lambda_star_vector`.

Method:
- compress repeated project types
- bracket λ* using the sign of a subgradient
- bisection on the monotone subgradient
- final comparison of λlo, λmid, λhi for robustness

Returns
    λstar, Lmin, info
"""
function lambda_star_vector_grouped_bisect(xvector,
                                           qvector,
                                           ρvector,
                                           κvector,
                                           rvector,
                                           β::Float64,
                                           M::Int;
                                           λ0::Float64 = 0.0,
                                           step0::Float64 = 1.0,
                                           λtol::Float64 = 1e-6,
                                           max_bisect_iter::Int = 80,
                                           ε::Float64 = 1e-10,
                                           xatol::Float64 = 1e-10,
                                           λatol::Float64 = 1e-10,
                                           maxiter::Int = 200,
                                           verbose::Bool = false)

    @assert 0.0 < β < 1.0
    groups = compress_project_vectors(xvector;
                                      qvector = qvector,
                                      ρvector = ρvector,
                                      κvector = κvector,
                                      rvector = rvector)

    if verbose
        println("Compressed instance into ", length(groups), " type groups.")
        println("Total projects = ", length(xvector))
    end

    # Cache evaluations at λ values actually used by the outer algorithm
    cache = Dict{Float64, NamedTuple}()

    function eval_cached(λ::Float64)
        get!(cache, λ) do
            eval_Lvector_grouped(λ, groups;
                                 β = β, M = M,
                                 ε = ε, xatol = xatol, λatol = λatol, maxiter = maxiter)
        end
    end

    # Bracket λ*
    bracket = let
        # local wrapper so bracket search uses cache too
        function eval_for_bracket(λ)
            eval_cached(Float64(λ))
        end

        f0 = eval_for_bracket(λ0)
        s0 = f0.subgrad

        if s0 == 0.0
            (λlo = λ0, λhi = λ0, slo = s0, shi = s0)
        elseif s0 < 0.0
            λlo = λ0
            slo = s0
            λhi = λ0 + step0
            found = false
            shi = NaN
            for _ in 1:60
                shi = eval_for_bracket(λhi).subgrad
                if shi >= 0.0
                    found = true
                    break
                end
                λhi = λ0 + 2.0 * (λhi - λ0)
            end
            found || error("Failed to bracket λ*: subgradient stayed negative while expanding right.")
            (λlo = λlo, λhi = λhi, slo = slo, shi = shi)
        else
            λhi = λ0
            shi = s0
            λlo = λ0 - step0
            found = false
            slo = NaN
            for _ in 1:60
                slo = eval_for_bracket(λlo).subgrad
                if slo <= 0.0
                    found = true
                    break
                end
                λlo = λ0 + 2.0 * (λlo - λ0)
            end
            found || error("Failed to bracket λ*: subgradient stayed positive while expanding left.")
            (λlo = λlo, λhi = λhi, slo = slo, shi = shi)
        end
    end

    λlo = bracket.λlo
    λhi = bracket.λhi

    if verbose
        println("Initial bracket: [", λlo, ", ", λhi, "]")
        println("Initial subgradient signs: slo=", bracket.slo, ", shi=", bracket.shi)
    end

    # Bisection on subgradient
    nit = 0
    while (λhi - λlo > λtol) && (nit < max_bisect_iter)
        nit += 1
        λmid = (λlo + λhi) / 2
        smid = eval_cached(λmid).subgrad

        if verbose
            println(@sprintf("iter %3d | λlo = %.8g | λhi = %.8g | λmid = %.8g | s(mid) = %.8g",
                             nit, λlo, λhi, λmid, smid))
        end

        if smid <= 0.0
            λlo = λmid
        else
            λhi = λmid
        end
    end

    λmid = (λlo + λhi) / 2

    # Final robust comparison of candidates
    candidates = unique(sort([λlo, λmid, λhi]))
    vals = [eval_cached(λ).L for λ in candidates]
    jstar = argmin(vals)

    λstar = candidates[jstar]
    Lmin  = vals[jstar]

    info = (
        nit = nit,
        λlo = λlo,
        λhi = λhi,
        candidates = candidates,
        cache_size = length(cache),
        ngroups = length(groups),
        nprojects = length(xvector),
        bracket_subgrad_lo = eval_cached(λlo).subgrad,
        bracket_subgrad_hi = eval_cached(λhi).subgrad
    )

    return λstar, Lmin, info
end

In [ ]:
# Example with 3 projects
q = 0.25; ρ = 0.6; κ = 0.8; r = 1.0; β = 0.95

qvector = [0.25, 0.10, 0.30]
ρvector = [0.60, 0.80, 0.65]
κvector = [0.80, 0.60, 0.70]
rvector = [1.00, 0.80, 1.20]
beta    = 0.95
M       = 2
xvector = [0.5, 0.5, 0.5]

λstar_old, Lmin_old = lambda_star_vector(
    xvector;
    qvector = qvector,
    ρvector = ρvector,
    κvector = κvector,
    rvector = rvector,
    β = β,
    M = M
)

λstar_new, Lmin_new, info_new = lambda_star_vector_grouped_bisect(
    xvector,
    qvector,
    ρvector,
    κvector,
    rvector,
    β,
    M;
    verbose = false
)

@show λstar_old Lmin_old
@show λstar_new Lmin_new
@show info_new

In [ ]:
using CSV
using DataFrames

df_timed = CSV.read("typed_scaling_experiment_binned_timed.csv", DataFrame)

row = only(filter(r ->
    r.scenario == "A_mild_B_fragile" &&
    r.prop_type == "0.8,0.2" &&
    isapprox(r.alphaM_target, 0.10; atol=1e-12) &&
    r.N == 100,
    df_timed
))

@show row.scenario row.prop_type row.alphaM_target row.capacity_ratio row.N row.M row.seed row.t_bound_case

scenario = (
    name = "A_mild_B_fragile",
    q_types = [0.03, 0.10],
    rho_types = [0.80, 0.10],
    kappa_types = [0.70, 0.95],
    r_types = [1.0, 1.0]
)

inst = build_typed_instance(
    Int(row.N);
    prop_type = [0.8, 0.2],
    q_types = scenario.q_types,
    rho_types = scenario.rho_types,
    kappa_types = scenario.kappa_types,
    r_types = scenario.r_types,
    x_init_scheme = :linspace,
    xlo = 0.0,
    xhi = 1.0,
    shuffle_types = true,
    shuffle_x = true,
    seed = Int(row.seed)
)

# exact realized vectors
xvector = inst.x_vec
qvector = inst.q_vec
ρvector = inst.rho_vec
κvector = inst.kappa_vec
rvector = inst.r_vec
β = row.beta
M = Int(row.M)

@time λstar_old, Lmin_old = lambda_star_vector(
    xvector;
    qvector = qvector,
    ρvector = ρvector,
    κvector = κvector,
    rvector = rvector,
    β = β,
    M = M
)

@time λstar_new, Lmin_new, info_new = lambda_star_vector_grouped_bisect(
    xvector,
    qvector,
    ρvector,
    κvector,
    rvector,
    β,
    M;
    λtol = 1e-6,
    verbose = false
)

@show λstar_old Lmin_old
@show λstar_new Lmin_new
@show info_new

In [ ]:
using PGFPlotsX

# --- helper: snap consecutive values within tol to the same level (optional) ---
function _snap_consecutive(y::AbstractVector{<:Real}; tol::Real=1e-8)
    yy = Float64.(y)
    @inbounds for i in 2:length(yy)
        if isfinite(yy[i]) && isfinite(yy[i-1]) && abs(yy[i] - yy[i-1]) ≤ tol
            yy[i] = yy[i-1]
        end
    end
    return yy
end

"""
Square axis plot on [0,1]×[0,1] with y-ticks only at 0 and 1, and no padding.

Unlike the step-plot version, this one plots the sampled curve directly.
If `smooth_curve=true`, PGFPlots is asked to draw a smooth interpolating curve.
"""
function plot_mp_index_vs_x_axis(; q, ρ, κ, r, β, ε=1e-10,
                                 xgrid=collect(range(0.0, 1.0; length=2001)),
                                 xlabel=L"$x$",
                                 ylabel=L"$m(x)$",
                                 width="8cm", height="8cm",
                                 snap_tol_m=1e-8,
                                 smooth_curve::Bool=true)

    # fixed points for tick positions
    x0_fp, x1_fp, _ = fixed_points(; q=q, ρ=ρ, κ=κ)

    items = [
        (0.0,          "\\ensuremath{0}"),
        (float(q),     "\\ensuremath{p_{01}}"),
        (float(x1_fp), "\\makebox[0pt][l]{\\hspace{0.00pt}\\ensuremath{x^{1}}}"),
        (float(x0_fp), "\\ensuremath{x^{0}}"),
        (float(q+ρ),   "\\ensuremath{p_{11}}"),
        (1.0,          "\\ensuremath{1}"),
    ]
    sort!(items, by=first)

    merged = Tuple{Float64,String}[]
    for (u, lab) in items
        if isempty(merged) || abs(u - merged[end][1]) > 1e-12
            push!(merged, (u, lab))
        end
    end
    xticks = [u for (u, _) in merged]
    labels = [lab for (_, lab) in merged]

    xtick_str = "{" * join(string.(xticks), ",") * "}"
    phantom = "\\vphantom{\\ensuremath{q+\\rho}}"
    xticklab_str = "{" * join([phantom * lab for lab in labels], ",") * "}"

    # sample m(x) = f(x,x)/g(x,x)
    mvals = similar(xgrid, Float64)
    for (i, x) in enumerate(xgrid)
        fxx = float(f_metric2(x, x; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε))
        gxx = float(g_metric2(x, x; q=q, ρ=ρ, κ=κ,      β=β, ε=ε))
        mvals[i] = (isfinite(fxx) && isfinite(gxx) && gxx != 0.0) ? (fxx / gxx) : NaN
    end

    if snap_tol_m > 0
        mvals = _snap_consecutive(mvals; tol=snap_tol_m)
    end

    coords = Tuple{Float64,Float64}[
        (float(x), isfinite(y) ? float(y) : NaN) for (x, y) in zip(xgrid, mvals)
    ]

    curve_opts = Pair{Any,Any}[
        "line width" => "0.30pt",
    ]
    if smooth_curve
        push!(curve_opts, "smooth" => nothing)
    else
        push!(curve_opts, "solid" => nothing)
    end

    return @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,

            xmin   => 0,
            xmax   => 1,
            ymin   => 0,
            ymax   => 1,
            "axis equal image" => nothing,

            "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
            "tick align" => "outside",
            "major tick length" => "2pt",
            "minor tick length" => "1pt",

            "xtick" => xtick_str,
            "xticklabels" => xticklab_str,

            "ytick" => "{0,1}",
            "enlargelimits" => "false",

            "scaled y ticks" => "false",
            "unbounded coords" => "jump",
            "legend pos" => "north west",
        },

        Plot(PGFPlotsX.Options(curve_opts...), Coordinates(coords)),
    )
end

function save_mp_index_vs_x_plot_pdf(outpdf::AbstractString;
                                     q, ρ, κ, r, β,
                                     engine::AbstractString="pdflatex",
                                     quiet::Bool=true,
                                     keep_tex::Bool=true,
                                     kwargs...)
    ax  = plot_mp_index_vs_x_axis(; q=q, ρ=ρ, κ=κ, r=r, β=β, kwargs...)
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end
# Example:
# q=0.25; ρ=0.6; κ=0.8; r=1.0; β=0.95
# save_mp_index_vs_x_plot_pdf("mp_index_vs_x.pdf"; q=q, ρ=ρ, κ=κ, r=r, β=β)

In [ ]:
# parameters (example)
q  = 0.25
ρ  = 0.6
κ  = 0.8
β  = 0.95
r  = 1.0

x0, x1, xhi = fixed_points(; q=q, ρ=ρ, κ=κ)
# z = 0.5 * (x1 + x0)
# x  = 0.5*(x0 + (q+ρ))

save_mp_index_vs_x_plot_pdf("mp_index_vs_x.pdf"; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=1e-30, xgrid=collect(range(0.0, 1.0; length=4001)))

In [ ]:
using PGFPlotsX
using Printf

function _snap_consecutive(y::AbstractVector{<:Real}; tol::Real=1e-8)
    yy = Float64.(y)
    @inbounds for i in 2:length(yy)
        if isfinite(yy[i]) && isfinite(yy[i-1]) && abs(yy[i] - yy[i-1]) ≤ tol
            yy[i] = yy[i-1]
        end
    end
    return yy
end

_fmt_beta(β; digits::Int=3) = Printf.format(Printf.Format("%.$(digits)f"), β)

function _sorted_betas_and_shades(betas::AbstractVector{<:Real};
                                  shade_min::Int=30,
                                  shade_max::Int=100)
    βs = sort!(unique!(Float64.(collect(betas))))
    n = length(βs)
    shades = n == 1 ? [shade_max] :
             round.(Int, collect(range(shade_min, shade_max; length=n)))
    return βs, shades
end

function plot_mp_index_vs_x_axis_betas(; q, ρ, κ, r, betas,
                                       ε=1e-10,
                                       xgrid=collect(range(0.0, 1.0; length=2001)),
                                       xlabel=L"$x$",
                                       ylabel=L"$m(x)$",
                                       width="8cm", height="8cm",
                                       snap_tol_m=0.0,
                                       shade_min=30,
                                       shade_max=100,
                                       line_width="0.30pt",
                                       beta_digits=3,
                                       legend_extremes_only::Bool=true,
                                       smooth_curves::Bool=true)

    x0_fp, x1_fp, _ = fixed_points(; q=q, ρ=ρ, κ=κ)

    items = [
        (0.0,          "\\ensuremath{0}"),
        (float(q),     "\\ensuremath{p_{01}}"),
        (float(x1_fp), "\\makebox[0pt][l]{\\hspace{0.00pt}\\ensuremath{x^{1}}}"),
        (float(x0_fp), "\\ensuremath{x^{0}}"),
        (float(q+ρ),   "\\ensuremath{p_{11}}"),
        (1.0,          "\\ensuremath{1}"),
    ]
    sort!(items, by=first)

    merged = Tuple{Float64,String}[]
    for (u, lab) in items
        if isempty(merged) || abs(u - merged[end][1]) > 1e-12
            push!(merged, (u, lab))
        end
    end
    xticks = [u for (u, _) in merged]
    labels = [lab for (_, lab) in merged]

    xtick_str = "{" * join(string.(xticks), ",") * "}"
    phantom = "\\vphantom{\\ensuremath{q+\\rho}}"
    xticklab_str = "{" * join([phantom * lab for lab in labels], ",") * "}"

    βs, shades = _sorted_betas_and_shades(betas;
                                          shade_min=shade_min,
                                          shade_max=shade_max)
    nβ = length(βs)

    elems = Any[]

    for (j, (βj, shade)) in enumerate(zip(βs, shades))
        mvals = similar(xgrid, Float64)

        for (i, x) in enumerate(xgrid)
            fxx = float(f_metric2(x, x; q=q, ρ=ρ, κ=κ, r=r, β=βj, ε=ε))
            gxx = float(g_metric2(x, x; q=q, ρ=ρ, κ=κ,      β=βj, ε=ε))
            mvals[i] = (isfinite(fxx) && isfinite(gxx) && gxx != 0.0) ? (fxx / gxx) : NaN
        end

        if snap_tol_m > 0
            mvals = _snap_consecutive(mvals; tol=snap_tol_m)
        end

        coords = Tuple{Float64,Float64}[
            (float(x), isfinite(y) ? float(y) : NaN) for (x, y) in zip(xgrid, mvals)
        ]

        show_in_legend = legend_extremes_only ? (nβ == 1 || j == 1 || j == nβ) : true

        plot_pairs = Pair{Any,Any}[
            "line width" => line_width,
            "color" => "black!$(shade)",
        ]

        if smooth_curves
            push!(plot_pairs, "smooth" => nothing)
        else
            push!(plot_pairs, "solid" => nothing)
        end

        if !show_in_legend
            push!(plot_pairs, "forget plot" => nothing)
        end

        push!(elems, Plot(PGFPlotsX.Options(plot_pairs...), Coordinates(coords)))

        if show_in_legend
            push!(elems, LegendEntry("\$\\beta = $(_fmt_beta(βj; digits=beta_digits))\$"))
        end
    end

    axis_opts = PGFPlotsX.Options(
        "width"  => width,
        "height" => height,
        "xlabel" => xlabel,
        "ylabel" => ylabel,

        "xmin"   => 0,
        "xmax"   => 1,
        "ymin"   => 0,
        "ymax"   => 1,
        "axis equal image" => nothing,

        "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
        "tick align" => "outside",
        "major tick length" => "2pt",
        "minor tick length" => "1pt",

        "xtick" => xtick_str,
        "xticklabels" => xticklab_str,

        "ytick" => "{0,1}",
        "enlargelimits" => "false",

        "scaled y ticks" => "false",
        "unbounded coords" => "jump",
        "legend pos" => "north west",
    )

    return Axis(axis_opts, elems...)
end

function save_mp_index_vs_x_betas_plot_pdf(outpdf::AbstractString;
                                           q, ρ, κ, r, betas,
                                           engine::AbstractString="pdflatex",
                                           quiet::Bool=true,
                                           keep_tex::Bool=true,
                                           kwargs...)
    ax  = plot_mp_index_vs_x_axis_betas(; q=q, ρ=ρ, κ=κ, r=r, betas=betas, kwargs...)
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

In [ ]:
# parameters (example)
q  = 0.25
ρ  = 0.6
κ  = 0.8
β  = 0.95
r  = 1.0

betas = [0.1, 0.3, 0.5, 0.70, 0.90, 0.95, 0.999]

save_mp_index_vs_x_betas_plot_pdf("mp_family_beta.pdf";
    q=q, ρ=ρ, κ=κ, r=r,
    betas=betas,
    xgrid=collect(range(0.0, 1.0; length=1001)),
    shade_min=35,
    shade_max=100,
    legend_extremes_only=true)

In [ ]:
using PGFPlotsX
using Printf

function _snap_consecutive(y::AbstractVector{<:Real}; tol::Real=1e-8)
    yy = Float64.(y)
    @inbounds for i in 2:length(yy)
        if isfinite(yy[i]) && isfinite(yy[i-1]) && abs(yy[i] - yy[i-1]) ≤ tol
            yy[i] = yy[i-1]
        end
    end
    return yy
end

_fmt_kappa(κ; digits::Int=3) = Printf.format(Printf.Format("%.$(digits)f"), κ)

function _sorted_kappas_and_shades(kappas::AbstractVector{<:Real};
                                   shade_min::Int=30,
                                   shade_max::Int=100)
    κs = sort!(unique!(Float64.(collect(kappas))))
    n = length(κs)
    shades = n == 1 ? [shade_max] :
             round.(Int, collect(range(shade_min, shade_max; length=n)))
    return κs, shades
end

function plot_mp_index_vs_x_axis_kappas(; q, ρ, β, r, kappas,
                                        ε=1e-10,
                                        xgrid=collect(range(0.0, 1.0; length=2001)),
                                        xlabel=L"$x$",
                                        ylabel=L"$m(x)$",
                                        width="8cm", height="8cm",
                                        snap_tol_m=0.0,
                                        shade_min=30,
                                        shade_max=100,
                                        line_width="0.30pt",
                                        kappa_digits=3,
                                        legend_extremes_only::Bool=true,
                                        kappa_ref=nothing,
                                        show_ref_fixed_points::Bool=false,
                                        smooth_curves::Bool=true)

    # x-axis ticks: only invariant locations by default
    items = Tuple{Float64,String}[
        (0.0,        "\\ensuremath{0}"),
        (float(q),   "\\ensuremath{p_{01}}"),
        (float(q+ρ), "\\ensuremath{p_{11}}"),
        (1.0,        "\\ensuremath{1}"),
    ]

    # optionally add x¹ and x⁰ for a chosen reference κ
    if show_ref_fixed_points
        κref = isnothing(kappa_ref) ? first(sort(Float64.(collect(kappas)))) : float(kappa_ref)
        x0_fp, x1_fp, _ = fixed_points(; q=q, ρ=ρ, κ=κref)
        push!(items, (float(x1_fp), "\\ensuremath{x^{1}}"))
        push!(items, (float(x0_fp), "\\ensuremath{x^{0}}"))
    end

    sort!(items, by=first)

    merged = Tuple{Float64,String}[]
    for (u, lab) in items
        if isempty(merged) || abs(u - merged[end][1]) > 1e-12
            push!(merged, (u, lab))
        end
    end

    xticks  = [u for (u, _) in merged]
    labels  = [lab for (_, lab) in merged]
    xtick_str = "{" * join(string.(xticks), ",") * "}"
    phantom = "\\vphantom{\\ensuremath{q+\\rho}}"
    xticklab_str = "{" * join([phantom * lab for lab in labels], ",") * "}"

    κs, shades = _sorted_kappas_and_shades(kappas;
                                           shade_min=shade_min,
                                           shade_max=shade_max)
    nκ = length(κs)

    elems = Any[]

    # draw lowest κ first, highest κ last
    for (j, (κj, shade)) in enumerate(zip(κs, shades))
        mvals = similar(xgrid, Float64)

        for (i, x) in enumerate(xgrid)
            fxx = float(f_metric2(x, x; q=q, ρ=ρ, κ=κj, r=r, β=β, ε=ε))
            gxx = float(g_metric2(x, x; q=q, ρ=ρ, κ=κj,      β=β, ε=ε))
            mvals[i] = (isfinite(fxx) && isfinite(gxx) && gxx != 0.0) ? (fxx / gxx) : NaN
        end

        if snap_tol_m > 0
            mvals = _snap_consecutive(mvals; tol=snap_tol_m)
        end

        coords = Tuple{Float64,Float64}[
            (float(x), isfinite(y) ? float(y) : NaN) for (x, y) in zip(xgrid, mvals)
        ]

        show_in_legend = legend_extremes_only ? (nκ == 1 || j == 1 || j == nκ) : true

        # slightly emphasize the largest κ curve
        lw = (j == nκ && nκ > 1) ? "0.45pt" : line_width

        plot_pairs = Pair{Any,Any}[
            "line width" => lw,
            "color" => "black!$(shade)",
        ]

        if smooth_curves
            push!(plot_pairs, "smooth" => nothing)
        else
            push!(plot_pairs, "solid" => nothing)
        end

        if !show_in_legend
            push!(plot_pairs, "forget plot" => nothing)
        end

        push!(elems, Plot(PGFPlotsX.Options(plot_pairs...), Coordinates(coords)))

        if show_in_legend
            push!(elems, LegendEntry("\$\\kappa = $(_fmt_kappa(κj; digits=kappa_digits))\$"))
        end
    end

    axis_opts = PGFPlotsX.Options(
        "width"  => width,
        "height" => height,
        "xlabel" => xlabel,
        "ylabel" => ylabel,

        "xmin"   => 0,
        "xmax"   => 1,
        "ymin"   => 0,
        "ymax"   => 1,
        "axis equal image" => nothing,

        "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
        "tick align" => "outside",
        "major tick length" => "2pt",
        "minor tick length" => "1pt",

        "xtick" => xtick_str,
        "xticklabels" => xticklab_str,

        "ytick" => "{0,1}",
        "enlargelimits" => "false",

        "scaled y ticks" => "false",
        "unbounded coords" => "jump",
        "legend pos" => "north west",
    )

    return Axis(axis_opts, elems...)
end

function save_mp_index_vs_x_kappas_plot_pdf(outpdf::AbstractString;
                                            q, ρ, β, r, kappas,
                                            engine::AbstractString="pdflatex",
                                            quiet::Bool=true,
                                            keep_tex::Bool=true,
                                            kwargs...)
    ax  = plot_mp_index_vs_x_axis_kappas(; q=q, ρ=ρ, β=β, r=r, kappas=kappas, kwargs...)
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

In [ ]:
# parameters (example)
q  = 0.25
ρ  = 0.6
κ  = 0.8
β  = 0.95
r  = 1.0

kappas = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.70, 0.8, 0.90, 0.95, 0.999]

save_mp_index_vs_x_kappas_plot_pdf("mp_family_kappa.pdf";
    q=q, ρ=ρ, β=β, r=r,
    kappas=kappas,
    xgrid=collect(range(0.0, 1.0; length=1001)),
    shade_min=35,
    shade_max=100,
    legend_extremes_only=true)

In [ ]:
using PGFPlotsX
using Printf

function _snap_consecutive(y::AbstractVector{<:Real}; tol::Real=1e-8)
    yy = Float64.(y)
    @inbounds for i in 2:length(yy)
        if isfinite(yy[i]) && isfinite(yy[i-1]) && abs(yy[i] - yy[i-1]) ≤ tol
            yy[i] = yy[i-1]
        end
    end
    return yy
end

_fmt_rho(ρ; digits::Int=3) = Printf.format(Printf.Format("%.$(digits)f"), ρ)

function _sorted_rhos_and_shades(rhos::AbstractVector{<:Real};
                                 shade_min::Int=30,
                                 shade_max::Int=100)
    ρs = sort!(unique!(Float64.(collect(rhos))))
    n = length(ρs)
    shades = n == 1 ? [shade_max] :
             round.(Int, collect(range(shade_min, shade_max; length=n)))
    return ρs, shades
end

function plot_mp_index_vs_x_axis_rhos(; q, κ, β, r, rhos,
                                      ε=1e-10,
                                      xgrid=collect(range(0.0, 1.0; length=2001)),
                                      xlabel=L"$x$",
                                      ylabel=L"$m(x)$",
                                      width="8cm", height="8cm",
                                      snap_tol_m=0.0,
                                      shade_min=30,
                                      shade_max=100,
                                      line_width="0.30pt",
                                      rho_digits=3,
                                      legend_extremes_only::Bool=true,
                                      rho_ref=nothing,
                                      show_ref_p11::Bool=false,
                                      show_ref_fixed_points::Bool=false,
                                      smooth_curves::Bool=true)

    ρs, shades = _sorted_rhos_and_shades(rhos;
                                         shade_min=shade_min,
                                         shade_max=shade_max)

    # admissibility checks: 0 < ρ < 1-q
    for ρj in ρs
        if !(0.0 < ρj < 1.0 - q)
            error("Each rho must satisfy 0 < rho < 1-q. Got rho = $(ρj), q = $(q).")
        end
    end

    # invariant x-axis ticks by default
    items = Tuple{Float64,String}[
        (0.0,      "\\ensuremath{0}"),
        (float(q), "\\ensuremath{p_{01}}"),
        (1.0,      "\\ensuremath{1}"),
    ]

    # optionally add p11 for a chosen reference rho
    ρref = isnothing(rho_ref) ? first(ρs) : float(rho_ref)
    if !(0.0 < ρref < 1.0 - q)
        error("rho_ref must satisfy 0 < rho_ref < 1-q. Got rho_ref = $(ρref), q = $(q).")
    end

    if show_ref_p11
        push!(items, (float(q + ρref), "\\ensuremath{p_{11}}"))
    end

    if show_ref_fixed_points
        x0_fp, x1_fp, _ = fixed_points(; q=q, ρ=ρref, κ=κ)
        push!(items, (float(x1_fp), "\\ensuremath{x^{1}}"))
        push!(items, (float(x0_fp), "\\ensuremath{x^{0}}"))
    end

    sort!(items, by=first)

    merged = Tuple{Float64,String}[]
    for (u, lab) in items
        if isempty(merged) || abs(u - merged[end][1]) > 1e-12
            push!(merged, (u, lab))
        end
    end

    xticks = [u for (u, _) in merged]
    labels = [lab for (_, lab) in merged]

    xtick_str = "{" * join(string.(xticks), ",") * "}"
    phantom = "\\vphantom{\\ensuremath{q+\\rho}}"
    xticklab_str = "{" * join([phantom * lab for lab in labels], ",") * "}"

    nρ = length(ρs)
    elems = Any[]

    # draw smallest rho first, largest rho last
    for (j, (ρj, shade)) in enumerate(zip(ρs, shades))
        mvals = similar(xgrid, Float64)

        for (i, x) in enumerate(xgrid)
            fxx = float(f_metric2(x, x; q=q, ρ=ρj, κ=κ, r=r, β=β, ε=ε))
            gxx = float(g_metric2(x, x; q=q, ρ=ρj, κ=κ,      β=β, ε=ε))
            mvals[i] = (isfinite(fxx) && isfinite(gxx) && gxx != 0.0) ? (fxx / gxx) : NaN
        end

        if snap_tol_m > 0
            mvals = _snap_consecutive(mvals; tol=snap_tol_m)
        end

        coords = Tuple{Float64,Float64}[
            (float(x), isfinite(y) ? float(y) : NaN) for (x, y) in zip(xgrid, mvals)
        ]

        show_in_legend = legend_extremes_only ? (nρ == 1 || j == 1 || j == nρ) : true

        # slightly emphasize the largest-rho curve
        lw = (j == nρ && nρ > 1) ? "0.45pt" : line_width

        plot_pairs = Pair{Any,Any}[
            "line width" => lw,
            "color" => "black!$(shade)",
        ]

        if smooth_curves
            push!(plot_pairs, "smooth" => nothing)
        else
            push!(plot_pairs, "solid" => nothing)
        end

        if !show_in_legend
            push!(plot_pairs, "forget plot" => nothing)
        end

        push!(elems, Plot(PGFPlotsX.Options(plot_pairs...), Coordinates(coords)))

        if show_in_legend
            push!(elems, LegendEntry("\$\\rho = $(_fmt_rho(ρj; digits=rho_digits))\$"))
        end
    end

    axis_opts = PGFPlotsX.Options(
        "width"  => width,
        "height" => height,
        "xlabel" => xlabel,
        "ylabel" => ylabel,

        "xmin"   => 0,
        "xmax"   => 1,
        "ymin"   => 0,
        "ymax"   => 1,
        "axis equal image" => nothing,

        "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
        "tick align" => "outside",
        "major tick length" => "2pt",
        "minor tick length" => "1pt",

        "xtick" => xtick_str,
        "xticklabels" => xticklab_str,

        "ytick" => "{0,1}",
        "enlargelimits" => "false",

        "scaled y ticks" => "false",
        "unbounded coords" => "jump",
        "legend pos" => "north west",
    )

    return Axis(axis_opts, elems...)
end

function save_mp_index_vs_x_rhos_plot_pdf(outpdf::AbstractString;
                                          q, κ, β, r, rhos,
                                          engine::AbstractString="pdflatex",
                                          quiet::Bool=true,
                                          keep_tex::Bool=true,
                                          kwargs...)
    ax  = plot_mp_index_vs_x_axis_rhos(; q=q, κ=κ, β=β, r=r, rhos=rhos, kwargs...)
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

In [ ]:
# parameters (example)
q  = 0.25
ρ  = 0.6
κ  = 0.8
β  = 0.95
r  = 1.0

rhos = [0.001, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.70, 0.749]   # must satisfy 0 < rho < 1-q

save_mp_index_vs_x_rhos_plot_pdf("mp_family_rho.pdf";
    q=q, κ=κ, β=β, r=r,
    rhos=rhos,
    xgrid=collect(range(0.0, 1.0; length=2001)),
    shade_min=35,
    shade_max=100,
    legend_extremes_only=true)

In [ ]:
using PGFPlotsX

# --- helper: snap consecutive values within tol (optional) ---
function _snap_consecutive(y::AbstractVector{<:Real}; tol::Real=1e-8)
    yy = Float64.(y)
    @inbounds for i in 2:length(yy)
        if isfinite(yy[i]) && isfinite(yy[i-1]) && abs(yy[i] - yy[i-1]) ≤ tol
            yy[i] = yy[i-1]
        end
    end
    return yy
end

"""
Plot the MP index m(x) versus rho in (0, 1-q), for fixed x.

By default:
- rho is sampled on (rhoeps, 1-q-rhoeps)
- the horizontal axis is [0, 1-q]
- the curve is drawn smoothly unless smooth_curve=false
"""
function plot_mp_index_vs_rho_axis(x; q, κ, r, β, ε=1e-10,
                                   rhogrid=nothing,
                                   rhoeps=1e-6,
                                   nrho=2001,
                                   xlabel=L"$\rho$",
                                   ylabel=L"$m(x)$",
                                   width="8cm", height="7cm",
                                   snap_tol_m=0.0,
                                   smooth_curve::Bool=true)

    0.0 < q < 1.0 || error("Need 0 < q < 1. Got q = $q.")

    ρmin_axis = 0.0
    ρmax_axis = 1.0 - q
    ρmax_axis > 0 || error("Need 1-q > 0. Got q = $q.")

    if isnothing(rhogrid)
        ρlo = rhoeps
        ρhi = ρmax_axis - rhoeps
        ρlo < ρhi || error("rhoeps is too large relative to 1-q.")
        rhogrid = collect(range(ρlo, ρhi; length=nrho))
    else
        rhogrid = Float64.(collect(rhogrid))
        if any(ρ -> !(0.0 < ρ < 1.0 - q), rhogrid)
            error("All rho values in rhogrid must satisfy 0 < rho < 1-q.")
        end
    end

    # x-axis ticks
    items = [
        (0.0,          "\\ensuremath{0}"),
        (float(1.0-q), "\\ensuremath{p_{00}}"),
    ]
    xtick_str = "{" * join(string.(first.(items)), ",") * "}"
    phantom = "\\vphantom{\\ensuremath{1-q}}"
    xticklab_str = "{" * join([phantom * lab for (_, lab) in items], ",") * "}"

    # sample m(x) as a function of rho
    mvals = similar(rhogrid, Float64)
    for (i, ρ) in enumerate(rhogrid)
        fxx = float(f_metric2(x, x; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε))
        gxx = float(g_metric2(x, x; q=q, ρ=ρ, κ=κ,      β=β, ε=ε))
        mvals[i] = (isfinite(fxx) && isfinite(gxx) && gxx != 0.0) ? (fxx / gxx) : NaN
    end

    if snap_tol_m > 0
        mvals = _snap_consecutive(mvals; tol=snap_tol_m)
    end

    coords = Tuple{Float64,Float64}[
        (float(ρ), isfinite(y) ? float(y) : NaN) for (ρ, y) in zip(rhogrid, mvals)
    ]

    curve_opts = Pair{Any,Any}[
        "line width" => "0.40pt",
    ]
    if smooth_curve
        push!(curve_opts, "smooth" => nothing)
    else
        push!(curve_opts, "solid" => nothing)
    end

    return @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,

            xmin   => ρmin_axis,
            xmax   => ρmax_axis,
            ymin   => 0,
            ymax   => 1,

            "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
            "tick align" => "outside",
            "major tick length" => "2pt",
            "minor tick length" => "1pt",

            "xtick" => xtick_str,
            "xticklabels" => xticklab_str,

            "ytick" => "{0,1}",
            "enlargelimits" => "false",

            "scaled x ticks" => "false",
            "scaled y ticks" => "false",
            "unbounded coords" => "jump",
        },

        Plot(PGFPlotsX.Options(curve_opts...), Coordinates(coords)),
    )
end

function save_mp_index_vs_rho_plot_pdf(outpdf::AbstractString, x;
                                       q, κ, r, β,
                                       engine::AbstractString="pdflatex",
                                       quiet::Bool=true,
                                       keep_tex::Bool=true,
                                       kwargs...)
    ax  = plot_mp_index_vs_rho_axis(x; q=q, κ=κ, r=r, β=β, kwargs...)
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

In [ ]:
x = 0.4
q = 0.25
κ = 0.8
r = 1.0
β = 0.95

save_mp_index_vs_rho_plot_pdf("mp_index_vs_rho.pdf", x;
    q=q, κ=κ, r=r, β=β)

In [ ]:
using PGFPlotsX
using Printf

function _snap_consecutive(y::AbstractVector{<:Real}; tol::Real=1e-8)
    yy = Float64.(y)
    @inbounds for i in 2:length(yy)
        if isfinite(yy[i]) && isfinite(yy[i-1]) && abs(yy[i] - yy[i-1]) ≤ tol
            yy[i] = yy[i-1]
        end
    end
    return yy
end

_fmt_q(q; digits::Int=3) = Printf.format(Printf.Format("%.$(digits)f"), q)

function _sorted_qs_and_shades(qs::AbstractVector{<:Real};
                               shade_min::Int=30,
                               shade_max::Int=100)
    qvals = sort!(unique!(Float64.(collect(qs))))
    n = length(qvals)
    shades = n == 1 ? [shade_max] :
             round.(Int, collect(range(shade_min, shade_max; length=n)))
    return qvals, shades
end

function plot_mp_index_vs_x_axis_qs(; qs, ρ, κ, β, r,
                                    ε=1e-10,
                                    xgrid=collect(range(0.0, 1.0; length=2001)),
                                    xlabel=L"$x$",
                                    ylabel=L"$m(x)$",
                                    width="8cm", height="8cm",
                                    snap_tol_m=0.0,
                                    shade_min=30,
                                    shade_max=100,
                                    line_width="0.30pt",
                                    q_digits=3,
                                    legend_extremes_only::Bool=true,
                                    q_ref=nothing,
                                    show_ref_p01::Bool=false,
                                    show_ref_p11::Bool=false,
                                    show_ref_fixed_points::Bool=false,
                                    smooth_curves::Bool=true)

    qvals, shades = _sorted_qs_and_shades(qs;
                                          shade_min=shade_min,
                                          shade_max=shade_max)

    # admissibility checks: 0 < q < 1-rho
    for qj in qvals
        if !(0.0 < qj < 1.0 - ρ)
            error("Each q must satisfy 0 < q < 1-rho. Got q = $(qj), rho = $(ρ).")
        end
    end

    # invariant ticks by default
    items = Tuple{Float64,String}[
        (0.0, "\\ensuremath{0}"),
        (1.0, "\\ensuremath{1}"),
    ]

    qref = isnothing(q_ref) ? first(qvals) : float(q_ref)
    if !(0.0 < qref < 1.0 - ρ)
        error("q_ref must satisfy 0 < q_ref < 1-rho. Got q_ref = $(qref), rho = $(ρ).")
    end

    if show_ref_p01
        push!(items, (float(qref), "\\ensuremath{p_{01}}"))
    end

    if show_ref_p11
        push!(items, (float(qref + ρ), "\\ensuremath{p_{11}}"))
    end

    if show_ref_fixed_points
        x0_fp, x1_fp, _ = fixed_points(; q=qref, ρ=ρ, κ=κ)
        push!(items, (float(x1_fp), "\\ensuremath{x^{1}}"))
        push!(items, (float(x0_fp), "\\ensuremath{x^{0}}"))
    end

    sort!(items, by=first)

    merged = Tuple{Float64,String}[]
    for (u, lab) in items
        if isempty(merged) || abs(u - merged[end][1]) > 1e-12
            push!(merged, (u, lab))
        end
    end

    xticks = [u for (u, _) in merged]
    labels = [lab for (_, lab) in merged]

    xtick_str = "{" * join(string.(xticks), ",") * "}"
    phantom = "\\vphantom{\\ensuremath{q+\\rho}}"
    xticklab_str = "{" * join([phantom * lab for lab in labels], ",") * "}"

    nq = length(qvals)
    elems = Any[]

    # draw smallest q first, largest q last
    for (j, (qj, shade)) in enumerate(zip(qvals, shades))
        mvals = similar(xgrid, Float64)

        for (i, x) in enumerate(xgrid)
            fxx = float(f_metric2(x, x; q=qj, ρ=ρ, κ=κ, r=r, β=β, ε=ε))
            gxx = float(g_metric2(x, x; q=qj, ρ=ρ, κ=κ,      β=β, ε=ε))
            mvals[i] = (isfinite(fxx) && isfinite(gxx) && gxx != 0.0) ? (fxx / gxx) : NaN
        end

        if snap_tol_m > 0
            mvals = _snap_consecutive(mvals; tol=snap_tol_m)
        end

        coords = Tuple{Float64,Float64}[
            (float(x), isfinite(y) ? float(y) : NaN) for (x, y) in zip(xgrid, mvals)
        ]

        show_in_legend = legend_extremes_only ? (nq == 1 || j == 1 || j == nq) : true

        lw = (j == nq && nq > 1) ? "0.45pt" : line_width

        plot_pairs = Pair{Any,Any}[
            "line width" => lw,
            "color" => "black!$(shade)",
        ]

        if smooth_curves
            push!(plot_pairs, "smooth" => nothing)
        else
            push!(plot_pairs, "solid" => nothing)
        end

        if !show_in_legend
            push!(plot_pairs, "forget plot" => nothing)
        end

        push!(elems, Plot(PGFPlotsX.Options(plot_pairs...), Coordinates(coords)))

        if show_in_legend
            push!(elems, LegendEntry("\$q = $(_fmt_q(qj; digits=q_digits))\$"))
        end
    end

    axis_opts = PGFPlotsX.Options(
        "width"  => width,
        "height" => height,
        "xlabel" => xlabel,
        "ylabel" => ylabel,

        "xmin"   => 0,
        "xmax"   => 1,
        "ymin"   => 0,
        "ymax"   => 1,
        "axis equal image" => nothing,

        "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
        "tick align" => "outside",
        "major tick length" => "2pt",
        "minor tick length" => "1pt",

        "xtick" => xtick_str,
        "xticklabels" => xticklab_str,

        "ytick" => "{0,1}",
        "enlargelimits" => "false",

        "scaled y ticks" => "false",
        "unbounded coords" => "jump",
        "legend pos" => "north west",
    )

    return Axis(axis_opts, elems...)
end

function save_mp_index_vs_x_qs_plot_pdf(outpdf::AbstractString;
                                        qs, ρ, κ, β, r,
                                        engine::AbstractString="pdflatex",
                                        quiet::Bool=true,
                                        keep_tex::Bool=true,
                                        kwargs...)
    ax  = plot_mp_index_vs_x_axis_qs(; qs=qs, ρ=ρ, κ=κ, β=β, r=r, kwargs...)
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

In [ ]:
# parameters (example)
q  = 0.25
ρ  = 0.6
κ  = 0.8
β  = 0.95
r  = 1.0

qs = [0.001, 0.10, 0.15, 0.20, 0.3, 0.35, 0.399]   # each must satisfy 0 < q < 1-rho

save_mp_index_vs_x_qs_plot_pdf("mp_family_q.pdf";
    qs=qs, ρ=ρ, κ=κ, β=β, r=r,
    xgrid=collect(range(0.0, 1.0; length=2001)),
    shade_min=35,
    shade_max=100,
    legend_extremes_only=true)

In [ ]:
using PGFPlotsX

# --- helper: snap consecutive values within tol (optional) ---
function _snap_consecutive(y::AbstractVector{<:Real}; tol::Real=1e-8)
    yy = Float64.(y)
    @inbounds for i in 2:length(yy)
        if isfinite(yy[i]) && isfinite(yy[i-1]) && abs(yy[i] - yy[i-1]) ≤ tol
            yy[i] = yy[i-1]
        end
    end
    return yy
end

"""
Plot the MP index m(x) versus q in (0, 1-rho), for fixed x.

By default:
- q is sampled on (qeps, 1-rho-qeps)
- the horizontal axis is [0, 1-rho]
- the curve is drawn smoothly unless smooth_curve=false
"""
function plot_mp_index_vs_q_axis(x; ρ, κ, r, β, ε=1e-10,
                                 qgrid=nothing,
                                 qeps=1e-6,
                                 nq=2001,
                                 xlabel=L"$p_{01}$",
                                 ylabel=L"$m(x)$",
                                 width="8cm", height="7cm",
                                 snap_tol_m=0.0,
                                 smooth_curve::Bool=true)

    0.0 < ρ < 1.0 || error("Need 0 < ρ < 1. Got ρ = $ρ.")

    qmin_axis = 0.0
    qmax_axis = 1.0 - ρ
    qmax_axis > 0 || error("Need 1-ρ > 0. Got ρ = $ρ.")

    if isnothing(qgrid)
        qlo = qeps
        qhi = qmax_axis - qeps
        qlo < qhi || error("qeps is too large relative to 1-ρ.")
        qgrid = collect(range(qlo, qhi; length=nq))
    else
        qgrid = Float64.(collect(qgrid))
        if any(q -> !(0.0 < q < 1.0 - ρ), qgrid)
            error("All q values in qgrid must satisfy 0 < q < 1-ρ.")
        end
    end

    # x-axis ticks
    items = [
        (0.0,           "\\ensuremath{0}"),
        (float(1.0-ρ),  "\\ensuremath{1-\\rho}"),
    ]
    xtick_str = "{" * join(string.(first.(items)), ",") * "}"
    phantom = "\\vphantom{\\ensuremath{1-\\rho}}"
    xticklab_str = "{" * join([phantom * lab for (_, lab) in items], ",") * "}"

    # sample m(x) as a function of q
    mvals = similar(qgrid, Float64)
    for (i, q) in enumerate(qgrid)
        fxx = float(f_metric2(x, x; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε))
        gxx = float(g_metric2(x, x; q=q, ρ=ρ, κ=κ,      β=β, ε=ε))
        mvals[i] = (isfinite(fxx) && isfinite(gxx) && gxx != 0.0) ? (fxx / gxx) : NaN
    end

    if snap_tol_m > 0
        mvals = _snap_consecutive(mvals; tol=snap_tol_m)
    end

    coords = Tuple{Float64,Float64}[
        (float(q), isfinite(y) ? float(y) : NaN) for (q, y) in zip(qgrid, mvals)
    ]

    curve_opts = Pair{Any,Any}[
        "line width" => "0.40pt",
    ]
    if smooth_curve
        push!(curve_opts, "smooth" => nothing)
    else
        push!(curve_opts, "solid" => nothing)
    end

    return @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,

            xmin   => qmin_axis,
            xmax   => qmax_axis,
            ymin   => 0,
            ymax   => 1,

            "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
            "tick align" => "outside",
            "major tick length" => "2pt",
            "minor tick length" => "1pt",

            "xtick" => xtick_str,
            "xticklabels" => xticklab_str,

            "ytick" => "{0,1}",
            "enlargelimits" => "false",

            "scaled x ticks" => "false",
            "scaled y ticks" => "false",
            "unbounded coords" => "jump",
        },

        Plot(PGFPlotsX.Options(curve_opts...), Coordinates(coords)),
    )
end

function save_mp_index_vs_q_plot_pdf(outpdf::AbstractString, x;
                                     ρ, κ, r, β,
                                     engine::AbstractString="pdflatex",
                                     quiet::Bool=true,
                                     keep_tex::Bool=true,
                                     kwargs...)
    ax  = plot_mp_index_vs_q_axis(x; ρ=ρ, κ=κ, r=r, β=β, kwargs...)
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

In [ ]:
ρ = 0.6
κ = 0.8
r = 1.0
β = 0.95

x0, x1, xhi = fixed_points(; q=q, ρ=ρ, κ=κ)

# x = 0.5*q
# x = 0.5*q + 0.5*x1
x = 0.5*x1 + 0.5*x0
# x  = 0.5*x0 + 0.5*(q+ρ)
# x = 0.5*(q+ρ) + 0.5*1.0


save_mp_index_vs_q_plot_pdf("mp_index_vs_q.pdf", x;
    ρ=ρ, κ=κ, r=r, β=β)

In [ ]:
"""
    theta_diff_at_x(x, z; q, ρ, κ, r, β, ε=1e-10)

Compute ΔΘ̃(x,z) = Θ̃(φ⁰(x), z) - Θ̃(φ¹(x), z),
using your previously-defined `φ0`, `φ1`, and `tildeTheta`.

Returns:
  (ΔΘ̂, Θ̂1, Θ̂0, x1, x0, info)

where:
  x1 = φ¹(x), x0 = φ⁰(x),
  ΔΘ̂ = Θ̂0 - Θ̂1,
  info is a NamedTuple with truncation horizons and tail bounds.
"""
function theta_diff_at_x(x, z; q, ρ, κ, r, β, ε::Real=1e-10)
    x1 = φ1(x; q=q, ρ=ρ, κ=κ)
    x0 = φ0(x; q=q, ρ=ρ)

    Θ1, T1, tailΘ1 = tildeTheta(x1, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    Θ0, T0, tailΘ0 = tildeTheta(x0, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)

    ΔΘ = Θ0 - Θ1 

    info = (T1=T1, T0=T0,
            tailΘ1=tailΘ1, tailΘ0=tailΘ0,
            # guaranteed bound on the truncation error of the difference:
            tailΔ = tailΘ1 + tailΘ0)

    return (ΔΘ, Θ1, Θ0, x1, x0, info)
end

# Example usage:
q  = 0.25
ρ  = 0.6
κ  = 0.8
β  = 0.95
r  = 1.0

x0, x1, xhi = fixed_points(; q=q, ρ=ρ, κ=κ)
z = 0.5 * (x1 + x0)
x = z + 0.1
ε = 1e-10
ΔΘ, Θ1, Θ0, phi1x, phi0x, info = theta_diff_at_x(x, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=1e-10)
@show phi0x phi1x Θ0 Θ1 ΔΘ info.tailΔ

In [ ]:
using PGFPlotsX
using LaTeXStrings

# --- assumes you already have: φ0, φ1, ϕ, tildeTheta, theta_diff_at_x ---

"""
    plot_theta_diff_vs_z(x; q, ρ, κ, r, β, ε=1e-10, nz=201, ...)

Return a PGFPlotsX.Axis plotting
    ΔΘ̃(x,z) = Θ̃(φ¹(x),z) - Θ̃(φ⁰(x),z)
versus z ∈ [0,1].

No grid, no title by default.
"""
function plot_theta_diff_vs_z(x; q, ρ, κ, r, β, ε::Real=1e-10, nz::Int=201,
                              width::AbstractString="12cm",
                              height::AbstractString="7cm",
                              mark::AbstractString="none",
                              xlabel=L"$z$",
                              ylabel=L"$\widetilde{\Theta}(\phi^0(x),z)-\widetilde{\Theta}(\phi^1(x),z)$")
    @assert nz ≥ 2 "nz must be ≥ 2"

    zgrid = collect(range(0.0, 1.0; length=nz))
    ΔΘ = Vector{Float64}(undef, nz)

    for (i, z) in enumerate(zgrid)
        d, _, _, _, _, _ = theta_diff_at_x(x, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
        ΔΘ[i] = d
    end

    coords = collect(zip(zgrid, ΔΘ))

    return @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,
            xmin   => 0,
            xmax   => 1,
            "tick align" => "outside",
            "scaled y ticks" => "false",
        },
        Plot(
            {
                mark => mark,
                "line width" => "0.9pt",
            },
            Coordinates(coords)
        )
    )
end


"""
    save_theta_diff_vs_z_pdf(outpdf, x; q, ρ, κ, r, β, ε=1e-10, nz=201,
                             engine="pdflatex", quiet=true, keep_tex=true, ...)

Create a PDF plot of ΔΘ̃(x,z) vs z on [0,1], writing a standalone .tex next to
the PDF and compiling it.

- quiet=true suppresses LaTeX console spam; on failure prints the log tail.
- keep_tex=true keeps .tex/.log for inspection.

Returns the absolute path of the produced PDF.
"""
function save_theta_diff_vs_z_pdf(outpdf::AbstractString, x;
                                  q, ρ, κ, r, β,
                                  ε::Real=1e-10, nz::Int=201,
                                  engine::AbstractString="pdflatex",
                                  quiet::Bool=true,
                                  keep_tex::Bool=true,
                                  kwargs...)
    ax  = plot_theta_diff_vs_z(x; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε, nz=nz, kwargs...)
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    # Write standalone TeX
    PGFPlotsX.save(texfile, pic; include_preamble=true)

    # Compile next to it
    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`
    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    # Ensure requested output name exists
    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    # Optionally clean up aux files
    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end


# ----------------
# Example usage
# ----------------
q=0.25; ρ=0.6; κ=0.8; β=0.25; r=1.0;
x = 0.6
save_theta_diff_vs_z_pdf("theta_diff_vs_z.pdf", x; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=1e-10, nz=3001)

In [ ]:
using PGFPlotsX
using Printf

# ---- Helpers: Julia vectors -> TeX list strings (avoid PGFPlotsX tick bugs) ----
tex_num_list(xs::AbstractVector{<:Real}) =
    isempty(xs) ? "{}" : "{" * join((@sprintf("%.15g", x) for x in xs), ",") * "}"

tex_label_list(ls::AbstractVector{<:AbstractString}) =
    isempty(ls) ? "{}" : "{" * join(ls, ",") * "}"

# ---- Main plotter ----
"""
    plot_varphi_t_pgf(t, z; q, ρ, κ, nx=2001, refine_iters=60,
                      width="10cm", height="10cm",
                      curve_lw="0.35pt", jump_mark_size="0.9pt",
                      diag_lw="0.25pt")

Plot x ↦ varphi_t(x,z) over x∈[0,1] (left-continuous, no vertical jump lines),
with open/filled circles at jump points, square axes [0,1]×[0,1],
diagonal dashed y=x, and ticks/labels only at {0,q,x1,x0,q+ρ,1} plus {z}
on BOTH axes.

Requires:
- varphi_t(x, z, t; q, ρ, κ)   (your iterate function)
- fixed_points(; q, ρ, κ)      returning (x0, x1, xhi)
- φ0(x; q, ρ) and φ1(x; q, ρ, κ) defined (for internal step updates)
"""
function plot_varphi_t_pgf(t::Integer, z;
        q, ρ, κ,
        nx::Int = 2001,
        refine_iters::Int = 60,
        width::AbstractString = "10cm",
        height::AbstractString = "10cm",
        curve_lw::AbstractString = "0.35pt",
        jump_mark_size::AbstractString = "0.9pt",
        diag_lw::AbstractString = "0.25pt")

    @assert 0 ≤ t
    @assert 0.0 ≤ z ≤ 1.0
    @assert nx ≥ 3

    # step update (avoid dependence on ϕ name)
    @inline step_map(y) = (y > z) ? φ1(y; q=q, ρ=ρ, κ=κ) : φ0(y; q=q, ρ=ρ)

    # Find earliest step k where actions differ between xL and xR (0-based)
    function first_diff_step(xL, xR)
        yL, yR = xL, xR
        for k in 0:(t-1)
            aL = (yL > z)
            aR = (yR > z)
            if aL != aR
                return k
            end
            yL = step_map(yL)
            yR = step_map(yR)
        end
        return nothing
    end

    # Locate breakpoint x* where varphi_k(x*,z) = z (monotone bisection)
    function locate_break(xa, xb, k::Int)
        f(x) = varphi_t(x, z, k; q=q, ρ=ρ, κ=κ) - z
        fa, fb = f(xa), f(xb)

        # We want fa ≤ 0 < fb; if reversed, swap.
        if fa > 0 && fb ≤ 0
            xa, xb = xb, xa
            fa, fb = fb, fa
        end
        # If still not bracketing (rare numeric edge), just return midpoint
        if !(fa ≤ 0 && fb > 0)
            return (xa + xb) / 2
        end

        for _ in 1:refine_iters
            xm = (xa + xb) / 2
            fm = f(xm)
            if fm ≤ 0
                xa = xm
            else
                xb = xm
            end
        end
        return (xa + xb) / 2
    end

    # Sample grid
    xs = collect(range(0.0, 1.0; length=nx))
    ys = [varphi_t(x, z, t; q=q, ρ=ρ, κ=κ) for x in xs]

    # Detect jump intervals via itinerary change and refine breakpoints
    # Store breaks keyed by left index i (between xs[i] and xs[i+1])
    breaks = Dict{Int, Tuple{Float64,Float64,Float64}}()  # i => (x*, y_left, y_right)

    if t ≥ 1
        for i in 1:(nx-1)
            k = first_diff_step(xs[i], xs[i+1])
            if k !== nothing
                xstar = locate_break(xs[i], xs[i+1], k)
                xl = clamp(prevfloat(xstar), 0.0, 1.0)
                xr = clamp(nextfloat(xstar), 0.0, 1.0)
                yL = varphi_t(xl, z, t; q=q, ρ=ρ, κ=κ)   # left limit (= value at x* by left-cont)
                yR = varphi_t(xr, z, t; q=q, ρ=ρ, κ=κ)   # right limit
                breaks[i] = (xstar, yL, yR)
            end
        end
    end

    # Build polyline coordinates with NaN breaks (no vertical lines)
    xline = Float64[]
    yline = Float64[]
    xclosed = Float64[]; yclosed = Float64[]   # filled circles (left-continuous value)
    xopen   = Float64[]; yopen   = Float64[]   # open circles (right limit)

    push!(xline, xs[1]); push!(yline, ys[1])
    for i in 1:(nx-1)
        if haskey(breaks, i)
            xstar, yL, yR = breaks[i]

            # end left piece
            push!(xline, xstar); push!(yline, yL)

            # mark jump (filled at left value, open at right value)
            push!(xclosed, xstar); push!(yclosed, yL)
            push!(xopen,   xstar); push!(yopen,   yR)

            # break (prevents vertical line)
            push!(xline, NaN); push!(yline, NaN)

            # start right piece
            push!(xline, xstar); push!(yline, yR)
        end

        push!(xline, xs[i+1]); push!(yline, ys[i+1])
    end

    # --- ticks and labels on BOTH axes: 0,q,x1,x0,q+ρ,1 plus z ---
    x0, x1, _ = fixed_points(; q=q, ρ=ρ, κ=κ)
    pts = [(0.0, raw"$0$"),
           (q,   raw"$q$"),
           (x1,  raw"$x^1$"),
           (x0,  raw"$x^0$"),
           (q+ρ, raw"$q+\rho$"),
           (1.0, raw"$1$")]

    # add z if not already essentially present
    tol = 1e-12
    if all(abs(z - p[1]) > tol for p in pts)
        push!(pts, (Float64(z), raw"$z$"))
    end
    sort!(pts, by = first)

    tick_pos = [p[1] for p in pts]
    tick_lab = [String(p[2]) for p in pts]

    tick_pos_tex = tex_num_list(tick_pos)
    tick_lab_tex = tex_label_list(tick_lab)

    # --- plots ---
    diag = @pgf Plot(
        { "no marks", color="gray", style="dashed", "line width"=diag_lw },
        Coordinates([(0.0, 0.0), (1.0, 1.0)])
    )

    curve = @pgf Plot(
        { "no marks", "unbounded coords"="jump", color="black", "line width"=curve_lw },
        Table(x=xline, y=yline)
    )

    closed_marks = @pgf Plot(
        { "only marks", mark="*", "mark size"=jump_mark_size, color="black" },
        Table(x=xclosed, y=yclosed)
    )

    open_marks = @pgf Plot(
        { "only marks", mark="o", "mark size"=jump_mark_size, color="black",
          "mark options"="{fill=white}" },
        Table(x=xopen, y=yopen)
    )

    ax = @pgf Axis(
        {
            width = width,
            height = height,
            xmin = 0, xmax = 1,
            ymin = 0, ymax = 1,

            "axis equal image",
            "axis on top" = "false",
            grid = "none",

            xlabel = raw"$x$",
            ylabel = raw"$\varphi_t(x,z)$",

            xtick = tick_pos_tex,
            xticklabels = tick_lab_tex,

            ytick = tick_pos_tex,
            yticklabels = tick_lab_tex,
        },
        diag, curve, closed_marks, open_marks
    )

    return TikzPicture(ax)
end

In [ ]:
q  = 0.25
ρ  = 0.6
κ  = 0.8

x0, x1, xhi = fixed_points(; q=q, ρ=ρ, κ=κ)
z = 0.5 * (x1 + x0)

t = 100
# z = 0.625
fig = plot_varphi_t_pgf(t, z; q=q, ρ=ρ, κ=κ, nx=2501)
display(fig)

In [ ]:
using PGFPlotsX
using Printf

# --- helpers: Julia vectors -> TeX list strings (robust for PGFPlotsX) ---
tex_num_list(xs::AbstractVector{<:Real}) =
    isempty(xs) ? "{}" : "{" * join((@sprintf("%.15g", x) for x in xs), ",") * "}"

tex_label_list(ls::AbstractVector{<:AbstractString}) =
    isempty(ls) ? "{}" : "{" * join(ls, ",") * "}"

function _unique_sorted_points!(pts::Vector{Tuple{Float64,String}}; tol=1e-12)
    sort!(pts, by=first)
    out = Tuple{Float64,String}[]
    for (v, lab) in pts
        if isempty(out) || abs(v - out[end][1]) > tol
            push!(out, (v, lab))
        end
    end
    return out
end

"""
    plot_varphi_t_vs_z_pgf(t, x; q, ρ, κ, nz=1201, zgrid=nothing,
                           refine_breaks=true, refine_iters=50,
                           width="10cm", height="10cm",
                           seg_lw="0.50pt", diag_lw="0.25pt",
                           mark_size="1.3pt")

Plot z ↦ varphi_t(x,z) over z∈[0,1] for fixed x.

- Horizontal segments (no vertical jump lines).
- Right-continuous in z (filled circle = value at breakpoint, open/white circle = left limit).
- Square axes [0,1]×[0,1], dashed diagonal y=x.
- Ticks/labels on BOTH axes only at {0,q,x^1,x^0,q+ρ,1} plus {x}.
- Open circles are *white-filled* and drawn AFTER the line to hide it underneath.
- Tick label heights are normalized (no vertical wobble for x^1, x^0, x, etc.).

REQUIRES (already defined in your notebook):
  - varphi_t(x, z, t; q, ρ, κ)
  - skeleton_itinerary_string(x, z, T; q, ρ, κ)
  - fixed_points(; q, ρ, κ) returning (x0, x1, xhi)
"""
function plot_varphi_t_vs_z_pgf(t::Integer, x;
        q, ρ, κ,
        nz::Int = 1201,
        zgrid = nothing,
        refine_breaks::Bool = true,
        refine_iters::Int = 50,
        width::AbstractString = "10cm",
        height::AbstractString = "10cm",
        seg_lw::AbstractString = "0.50pt",
        diag_lw::AbstractString = "0.25pt",
        mark_size::AbstractString = "1.3pt")

    @assert t ≥ 0
    @assert 0.0 ≤ x ≤ 1.0

    zs = zgrid === nothing ? collect(range(0.0, 1.0; length=nz)) : collect(zgrid)
    @assert issorted(zs) && zs[1] ≥ 0 && zs[end] ≤ 1
    n = length(zs)
    @assert n ≥ 2

    # --- signature word and value at a given z ---
    # IMPORTANT: to determine varphi_t, we only need A_0..A_{t-1} (word length t).
    function word_and_value(z)
        if t == 0
            return "", Float64(x)
        else
            w = skeleton_itinerary_string(x, z, t-1; q=q, ρ=ρ, κ=κ)   # length t
            y = varphi_t(x, z, t; q=q, ρ=ρ, κ=κ)
            return w, y
        end
    end

    # precompute on the grid
    words = Vector{String}(undef, n)
    vals  = Vector{Float64}(undef, n)
    for i in 1:n
        w, y = word_and_value(zs[i])
        words[i] = w
        vals[i]  = y
    end

    # refine breakpoint between zL and zR where word changes (binary search on word)
    function refine_boundary(zL, zR, wL)
        lo, hi = zL, zR
        for _ in 1:refine_iters
            mid = (lo + hi) / 2
            wM, _ = word_and_value(mid)
            if wM == wL
                lo = mid
            else
                hi = mid
            end
        end
        return hi
    end

    segments = NamedTuple[]  # (zL,zR, word, value)

    # polyline for horizontal segments (with NaN breaks)
    xline = Float64[]
    yline = Float64[]

    # jump markers: open = left limit, filled = value at breakpoint (right-continuity)
    z_open = Float64[]; y_open = Float64[]
    z_fill = Float64[]; y_fill = Float64[]

    zL = zs[1]
    w_cur = words[1]
    y_cur = vals[1]

    for i in 2:n
        if words[i] != w_cur
            zb = refine_breaks ? refine_boundary(zs[i-1], zs[i], w_cur) : zs[i]

            # close previous segment at zb
            push!(segments, (zL=zL, zR=zb, word=w_cur, value=y_cur))
            push!(xline, zL); push!(yline, y_cur)
            push!(xline, zb); push!(yline, y_cur)
            push!(xline, NaN); push!(yline, NaN)

            # left limit uses z just below zb; right value uses z=zb
            zl = clamp(prevfloat(zb), 0.0, 1.0)
            zr = clamp(zb, 0.0, 1.0)
            _, y_left  = word_and_value(zl)
            _, y_right = word_and_value(zr)

            push!(z_open, zb); push!(y_open, y_left)   # left limit (white-filled marker)
            push!(z_fill, zb); push!(y_fill, y_right)  # value at zb (filled marker)

            # start new segment
            zL = zb
            w_cur = words[i]
            y_cur = vals[i]
        end
    end

    # last segment
    push!(segments, (zL=zL, zR=zs[end], word=w_cur, value=y_cur))
    push!(xline, zL); push!(yline, y_cur)
    push!(xline, zs[end]); push!(yline, y_cur)

    # ticks: {0, q, x^1, x^0, q+ρ, 1} plus {x} on BOTH axes
    x0, x1, _ = fixed_points(; q=q, ρ=ρ, κ=κ)
    pts = Tuple{Float64,String}[
        (0.0,          "\\ensuremath{0}"),
        (float(q),     "\\ensuremath{p_{01}}"),
        (float(x1_fp), "\\makebox[0pt][l]{\\hspace{0.00pt}\\ensuremath{x^{1}}}"),
        (float(x0_fp), "\\ensuremath{x^{0}}"),
        (float(q+ρ),   "\\ensuremath{p_{11}}"),
        (1.0,          "\\ensuremath{1}"),
        (Float64(x), raw"$x$")
    ]

    pts = _unique_sorted_points!(pts)

    tick_pos = [p[1] for p in pts]
    tick_lab = [String(p[2]) for p in pts]
    tick_pos_tex = tex_num_list(tick_pos)
    tick_lab_tex = tex_label_list(tick_lab)

    # plots
    diag = @pgf Plot(
        { "no marks", color="gray", style="dashed", "line width"=diag_lw },
        Coordinates([(0.0, 0.0), (1.0, 1.0)])
    )

    stepplot = @pgf Plot(
        { "no marks", "unbounded coords"="jump", color="black", "line width"=seg_lw },
        Table(x=xline, y=yline)
    )

    filled_marks = @pgf Plot(
        { "only marks", mark="*", "mark size"=mark_size, color="black" },
        Table(x=z_fill, y=y_fill)
    )

    # "Open" markers: actually a white-filled dot, drawn LAST to cover the line underneath
    open_marks = @pgf Plot(
        {
            "only marks",
            mark="*",
            "mark size"=mark_size,
            color="black",
            "mark options"="{fill=white, draw=black, line width=0.25pt}",
        },
        Table(x=z_open, y=y_open)
    )

    ax = @pgf Axis(
        {
            width = width,
            height = height,
            xmin = 0, xmax = 1,
            ymin = 0, ymax = 1,
            "axis equal image",
            grid = "none",
            "axis on top" = "false",

            xlabel = raw"$z$",
            ylabel = raw"$\varphi_t(x,z)$",

            xtick = tick_pos_tex,
            xticklabels = tick_lab_tex,
            ytick = tick_pos_tex,
            yticklabels = tick_lab_tex,

            # normalize tick-label box heights (fixes vertical wobble of x^1, x^0, x, etc.)
            "xticklabel style" = "{text height=1.6ex, text depth=0.3ex}",
            "yticklabel style" = "{text height=1.6ex, text depth=0.3ex}",
        },
        diag, stepplot, filled_marks, open_marks
    )

    return TikzPicture(ax), segments
end

In [ ]:
q  = 0.25
ρ  = 0.6
κ  = 0.8

x0, x1, xhi = fixed_points(; q=q, ρ=ρ, κ=κ)

t = 4
x = 0.5 * (x0 + x1)
fig, segs = plot_varphi_t_vs_z_pgf(t, x; q=q, ρ=ρ, κ=κ, nz=1201, mark_size="1.2pt")
display(fig)

In [ ]:
using PGFPlotsX
using Printf

# --- helpers: Julia vectors -> TeX list strings (robust for PGFPlotsX) ---
tex_num_list(xs::AbstractVector{<:Real}) =
    isempty(xs) ? "{}" : "{" * join((@sprintf("%.15g", x) for x in xs), ",") * "}"

tex_label_list(ls::AbstractVector{<:AbstractString}) =
    isempty(ls) ? "{}" : "{" * join(ls, ",") * "}"

function _unique_sorted_points!(pts::Vector{Tuple{Float64,String}}; tol=1e-12)
    sort!(pts, by=first)
    out = Tuple{Float64,String}[]
    for (v, lab) in pts
        if isempty(out) || abs(v - out[end][1]) > tol
            push!(out, (v, lab))
        end
    end
    return out
end

# ------------------------------------------------------------
# Build ONE Axis for z ↦ varphi_t(x,z), with title "t = ...".
# Requires your existing:
#   varphi_t(x,z,t; q,ρ,κ)
#   skeleton_itinerary_string(x,z,T; q,ρ,κ)
#   fixed_points(; q,ρ,κ) -> (x0,x1,xhi)
# ------------------------------------------------------------
function axis_varphi_t_vs_z_pgf(t::Integer, x;
        q, ρ, κ,
        nz::Int = 1201,
        zgrid = nothing,
        refine_breaks::Bool = true,
        refine_iters::Int = 50,
        width::AbstractString = "8cm",
        height::AbstractString = "8cm",
        seg_lw::AbstractString = "0.50pt",
        diag_lw::AbstractString = "0.25pt",
        mark_size::AbstractString = "1.3pt",
        show_ylabel::Bool = true,
        hide_right_yticklabels::Bool = true)

    @assert t ≥ 0
    @assert 0.0 ≤ x ≤ 1.0

    zs = zgrid === nothing ? collect(range(0.0, 1.0; length=nz)) : collect(zgrid)
    @assert issorted(zs) && zs[1] ≥ 0 && zs[end] ≤ 1
    n = length(zs)
    @assert n ≥ 2

    # word length must be t (symbols A_0..A_{t-1})
    function word_and_value(z)
        if t == 0
            return "", Float64(x)
        else
            w = skeleton_itinerary_string(x, z, t-1; q=q, ρ=ρ, κ=κ)   # length t
            y = varphi_t(x, z, t; q=q, ρ=ρ, κ=κ)
            return w, y
        end
    end

    # precompute on grid
    words = Vector{String}(undef, n)
    vals  = Vector{Float64}(undef, n)
    for i in 1:n
        w, y = word_and_value(zs[i])
        words[i] = w
        vals[i]  = y
    end

    # refine breakpoint between zL and zR where word changes (binary search on word)
    function refine_boundary(zL, zR, wL)
        lo, hi = zL, zR
        for _ in 1:refine_iters
            mid = (lo + hi) / 2
            wM, _ = word_and_value(mid)
            if wM == wL
                lo = mid
            else
                hi = mid
            end
        end
        return hi
    end

    # polyline (horizontal segments) with NaN breaks; jump markers
    xline = Float64[]; yline = Float64[]
    z_open = Float64[]; y_open = Float64[]   # left limit (white-filled marker)
    z_fill = Float64[]; y_fill = Float64[]   # value at breakpoint (filled marker)

    zL = zs[1]
    w_cur = words[1]
    y_cur = vals[1]

    for i in 2:n
        if words[i] != w_cur
            zb = refine_breaks ? refine_boundary(zs[i-1], zs[i], w_cur) : zs[i]

            # close previous segment at zb
            push!(xline, zL); push!(yline, y_cur)
            push!(xline, zb); push!(yline, y_cur)
            push!(xline, NaN); push!(yline, NaN)

            # left limit uses z just below zb; right value uses z=zb (right-continuous)
            zl = clamp(prevfloat(zb), 0.0, 1.0)
            zr = clamp(zb, 0.0, 1.0)
            _, y_left  = word_and_value(zl)
            _, y_right = word_and_value(zr)

            push!(z_open, zb); push!(y_open, y_left)
            push!(z_fill, zb); push!(y_fill, y_right)

            # start new segment
            zL = zb
            w_cur = words[i]
            y_cur = vals[i]
        end
    end

    # last segment
    push!(xline, zL);      push!(yline, y_cur)
    push!(xline, zs[end]); push!(yline, y_cur)

    # ticks: {0,q,x^1,x^0,q+ρ,1} plus {x} on BOTH axes
    x0, x1, _ = fixed_points(; q=q, ρ=ρ, κ=κ)
    pts = Tuple{Float64,String}[
        (0.0,          "\\ensuremath{0}"),
        (float(q),     "\\ensuremath{p_{01}}"),
        (float(x1_fp), "\\makebox[0pt][l]{\\hspace{0.00pt}\\ensuremath{x^{1}}}"),
        (float(x0_fp), "\\ensuremath{x^{0}}"),
        (float(q+ρ),   "\\ensuremath{p_{11}}"),
        (1.0,          "\\ensuremath{1}"),
        (Float64(x), raw"$x$")
    ]

    pts = _unique_sorted_points!(pts)

    tick_pos = [p[1] for p in pts]
    tick_lab = [String(p[2]) for p in pts]
    tick_pos_tex = tex_num_list(tick_pos)
    tick_lab_tex = tex_label_list(tick_lab)

    title_tex = "\$t = $(t)\$"   # TeX math title

    # plots
    diag = @pgf Plot(
        { "no marks", color="gray", style="dashed", "line width"=diag_lw },
        Coordinates([(0.0, 0.0), (1.0, 1.0)])
    )

    stepplot = @pgf Plot(
        { "no marks", "unbounded coords"="jump", color="black", "line width"=seg_lw },
        Table(x=xline, y=yline)
    )

    filled_marks = @pgf Plot(
        { "only marks", mark="*", "mark size"=mark_size, color="black" },
        Table(x=z_fill, y=y_fill)
    )

    # white-filled markers drawn LAST to cover line underneath
    open_marks = @pgf Plot(
        {
            "only marks",
            mark="*",
            "mark size"=mark_size,
            color="black",
            "mark options"="{fill=white, draw=black, line width=0.25pt}",
        },
        Table(x=z_open, y=y_open)
    )

    # axis options (no Dict!)
    yticklabels_val = tick_lab_tex
    if !show_ylabel && hide_right_yticklabels
        yticklabels_val = "{}"
    end

    ax = @pgf Axis(
        {
            title = title_tex,
            "title style" = "{at={(0.5,1.02)}, anchor=south}",

            width = width,
            height = height,
            xmin = 0, xmax = 1,
            ymin = 0, ymax = 1,
            "axis equal image",
            grid = "none",
            "axis on top" = "false",

            xlabel = raw"$z$",
            ylabel = show_ylabel ? raw"$\varphi_t(x,z)$" : "",

            xtick = tick_pos_tex,
            xticklabels = tick_lab_tex,
            ytick = tick_pos_tex,
            yticklabels = yticklabels_val,

            # normalize tick-label box heights (fixes wobble)
            "xticklabel style" = "{text height=1.6ex, text depth=0.3ex}",
            "yticklabel style" = "{text height=1.6ex, text depth=0.3ex}",
        },
        diag, stepplot, filled_marks, open_marks
    )

    return ax
end

"""
    plot_varphi_t_vs_z_1x2_pgf(t1, t2, x; q, ρ, κ, ...)

Return a TikzPicture with a 1×2 layout (GroupPlot) for two t values.
"""
function plot_varphi_t_vs_z_1x2_pgf(t1::Integer, t2::Integer, x;
        q, ρ, κ,
        nz::Int = 1201,
        zgrid = nothing,
        width_each::AbstractString = "8cm",
        height_each::AbstractString = "8cm",
        hsep::AbstractString = "1.2cm",
        kwargs...)

    ax1 = axis_varphi_t_vs_z_pgf(t1, x; q=q, ρ=ρ, κ=κ,
                                 nz=nz, zgrid=zgrid,
                                 width=width_each, height=height_each,
                                 show_ylabel=true, kwargs...)
    ax2 = axis_varphi_t_vs_z_pgf(t2, x; q=q, ρ=ρ, κ=κ,
                                 nz=nz, zgrid=zgrid,
                                 width=width_each, height=height_each,
                                 show_ylabel=false, kwargs...)

    gp = @pgf GroupPlot(
        { "group style" = "{group size=2 by 1, horizontal sep=$(hsep)}" },
        ax1, ax2
    )

    return TikzPicture(gp)
end

In [ ]:
q  = 0.25
ρ  = 0.6
κ  = 0.8

x0, x1, xhi = fixed_points(; q=q, ρ=ρ, κ=κ)

t = 2
x = 0.5 * (x0 + x1)
fig = plot_varphi_t_vs_z_1x2_pgf(1, 2, x; q=q, ρ=ρ, κ=κ, nz=1501)
display(fig)

In [ ]:
using PGFPlotsX

# Enable the PGFPlots groupplots library (needed for GroupPlot)
push!(PGFPlotsX.CUSTOM_PREAMBLE, raw"\usepgfplotslibrary{groupplots}")

# Avoid trying to open an external viewer
PGFPlotsX.enable_interactive(false)

In [ ]:
fig = plot_varphi_t_vs_z_1x2_pgf(3, 4, x; q=q, ρ=ρ, κ=κ, nz=3000, 
                                mark_size="0.5pt", seg_lw="0.55pt")
PGFPlotsX.pgfsave("tmp_groupplot.pdf", fig)
display("application/pdf", read("tmp_groupplot.pdf"))

In [ ]:
#### Convert itinerary vector -> "0101..." word
itinerary_word(A::AbstractVector{<:Integer}) =
    String([a == 1 ? '1' : '0' for a in A])

# Check if bytes `b` are consistent with period p (allowing final partial block)
# i.e., b == prefix of (b[1:p])^∞
function _is_prefix_of_periodic(b::Vector{UInt8}, p::Int)
    @assert 1 ≤ p ≤ length(b)
    @inbounds for k in 1:length(b)
        if b[k] != b[((k - 1) % p) + 1]
            return false
        end
    end
    return true
end

"""
    eventual_periodic_decomp(w; min_full_repeats=2)

Find earliest decomposition of a finite word w as
    w = u ⋅ s^m ⋅ r
where r is a prefix of s (possibly empty). Equivalently, the suffix after u is a
prefix of the infinite periodic word s^∞.

Input:
- w: String like "001010..." or a 0/1 itinerary vector.

Option:
- min_full_repeats: require at least this many full copies of s in the periodic part
  (default 2, to avoid trivial fits).

Returns NamedTuple:
(u, s, m, r, start_index, period_length)

- start_index is 1-based index where periodicity begins (i.e., u = w[1:start_index-1]).
"""
function eventual_periodic_decomp(w; min_full_repeats::Int = 2)
    ws = w isa AbstractString ? String(w) : itinerary_word(w)
    b  = collect(codeunits(ws))
    n  = length(b)

    # Try earliest start position of the periodic regime
    for i in 1:n
        L = n - i + 1
        # Need at least min_full_repeats full blocks => p ≤ L ÷ min_full_repeats
        maxp = L ÷ min_full_repeats
        maxp < 1 && continue

        suf = @view b[i:n]

        # Choose smallest period length p that works
        for p in 1:maxp
            # Base block s is the first p symbols of the suffix
            if _is_prefix_of_periodic(collect(suf), p)
                m = L ÷ p
                rlen = L % p

                u = (i == 1) ? "" : String(b[1:i-1])
                s = String(b[i:i+p-1])
                r = (rlen == 0) ? "" : String(b[n - rlen + 1:n])  # last rlen chars

                return (u=u, s=s, m=m, r=r,
                        start_index=i, period_length=p)
            end
        end
    end

    # No nontrivial eventual periodicity found under the min_full_repeats criterion
    return (u=ws, s="", m=0, r="", start_index=n+1, period_length=0)
end

"""
    describe_eventual_periodic(w; min_full_repeats=2)

Human-readable description.
"""
function describe_eventual_periodic(w; min_full_repeats::Int=2)
    d = eventual_periodic_decomp(w; min_full_repeats=min_full_repeats)
    if d.m == 0
        return "No repeating tail found with ≥$(min_full_repeats) full repeats."
    end
    return "u=\"$(d.u)\", tail ≈ s^m r with s=\"$(d.s)\", m=$(d.m), r=\"$(d.r)\" (starts at index $(d.start_index))"
end

"""
    format_eventual_periodic(w; min_full_repeats=2)

Returns a single string formatted as:
    u (s)^m r
where w = u ⋅ s^m ⋅ r and r is a prefix of s (possibly empty).
If no repeating tail is found (under the min_full_repeats rule), returns w itself.
"""
function format_eventual_periodic(w; min_full_repeats::Int=2)
    d = eventual_periodic_decomp(w; min_full_repeats=min_full_repeats)
    ws = w isa AbstractString ? String(w) : itinerary_word(w)

    if d.m == 0
        return ws
    end

    u = d.u
    s = d.s
    m = d.m
    r = d.r

    # Build: u (s)^m r, skipping empty parts neatly.
    parts = String[]
    if !isempty(u); push!(parts, u); end
    push!(parts, "($(s))^$(m)")
    if !isempty(r); push!(parts, r); end

    return join(parts, " ")
end

In [ ]:
# Assumes φ0(x; q,ρ) and φ1(x; q,ρ,κ) are already defined.

# Apply one symbol ('0' or '1') to a belief
@inline function apply_symbol(x, sym::UInt8; q, ρ, κ)
    sym == UInt8('1') ? φ1(x; q=q, ρ=ρ, κ=κ) : φ0(x; q=q, ρ=ρ)
end

"""
    apply_word(x, w; q, ρ, κ)

Apply the sequence of maps dictated by the word w (String over '0','1'):
x ↦   ↦ φ^{w[2]}(·) ↦ ... ↦ φ^{w[end]}(·).
"""
function apply_word(x, w::AbstractString; q, ρ, κ)
    y = x
    for sym in codeunits(w)
        y = apply_symbol(y, sym; q=q, ρ=ρ, κ=κ)
    end
    return y
end

"""
    fixed_point_of_word(w, xinit; q, ρ, κ, tol=1e-12, maxiter=10_000)

Find y such that y = apply_word(y, w), by fixed-point iteration.
Returns (y, iters).
"""
function fixed_point_of_word(w::AbstractString, xinit;
                            q, ρ, κ, tol=1e-12, maxiter=10_000)
    y = xinit
    for it in 1:maxiter
        ynew = apply_word(y, w; q=q, ρ=ρ, κ=κ)
        if abs(ynew - y) ≤ tol
            return ynew, it
        end
        y = ynew
    end
    error("fixed_point_of_word did not converge (increase maxiter / check params).")
end

"""
    limiting_cycle_points(xinit, z, w; q, ρ, κ, tol=1e-12, maxiter=10_000, check=true)

Given a repeating itinerary block w (e.g. "10"), compute the limiting cycle points
(y₀, y₁, ..., y_{k-1}) associated with w, where k = length(w).

- y₀ is the fixed point of Φ_w (composition dictated by w).
- y_{i+1} = φ^{w[i]}(y_i) for i=0..k-2.
Also checks (optionally) that y_i > z iff w[i] == '1'.
"""
function limiting_cycle_points(xinit, z, w::AbstractString;
                              q, ρ, κ, tol=1e-12, maxiter=10_000, check::Bool=true)

    @assert all(c == '0' || c == '1' for c in w) "w must be a binary word over '0'/'1'"
    k = length(w)
    @assert k ≥ 1 "s must be nonempty"

    y0, _ = fixed_point_of_word(w, xinit; q=q, ρ=ρ, κ=κ, tol=tol, maxiter=maxiter)

    # Build the cycle points in phase order
    cyc = Vector{Float64}(undef, k)
    cyc[1] = y0
    y = y0
    bytes = codeunits(w)

    for i in 1:(k-1)
        y = apply_symbol(y, bytes[i]; q=q, ρ=ρ, κ=κ)
        cyc[i+1] = y
    end

    # Closure check: after the last symbol we should return to y0
    yclose = apply_symbol(y, bytes[end]; q=q, ρ=ρ, κ=κ)

    if check
        if !isapprox(yclose, y0; atol=1e-9, rtol=0.0)
            @warn "Cycle did not close numerically (increase tol/maxiter?)" y0 yclose
        end
        for i in 1:k
            sym1 = (bytes[i] == UInt8('1'))
            reg1 = (cyc[i] > z)
            if sym1 != reg1
                @warn "Threshold consistency failed at phase $i" symbol=Char(bytes[i]) value=cyc[i] z=z
            end
        end
    end

    return cyc
end

In [ ]:
##############################
# tests_words_cycles.jl
##############################
using Printf
using Test

# -------------------------------------------------------------------
# OPTIONAL: stand-alone toy maps (uncomment ONLY if you don't already
# have φ0, φ1 defined in your session).
# These keep beliefs in [0,1] and are contractions for ρ,κ ∈ (0,1).
# -------------------------------------------------------------------
# clamp01(x) = max(0.0, min(1.0, x))
# function φ0(x; q, ρ)
#     # affine contraction towards q
#     return clamp01(q + ρ*(x - q))
# end
# function φ1(x; q, ρ, κ)
#     # "more aggressive" contraction towards 1-q (just for testing)
#     return clamp01((1-κ)*(q + ρ*(x - q)) + κ*(1-q))
# end

# -------------------------------------------------------------------
# Paste/`include` your apply_symbol/apply_word/fixed_point_of_word/
# limiting_cycle_points definitions above this line, OR include the file.
# -------------------------------------------------------------------

# --- parameters / initial conditions --------------------------------
q  = 0.25
ρ  = 0.60
κ  = 0.80

x  = 0.329269626091      # point to test apply_word
xinit = 0.10   # seed for fixed-point iteration

@printf("Params: q=%.3f  ρ=%.3f  κ=%.3f\n\n", q, ρ, κ)

# --- 1) smoke tests for apply_word + order checks -------------------
words = ["0", "1", "01", "10", "001", "100", "010", "011", "101", "110"]

@printf("=== apply_word smoke tests (x=%.6f) ===\n", x)
for w in words
    y = apply_word(x, w; q=q, ρ=ρ, κ=κ)
    @printf("apply_word(x, \"%s\") = %.12f\n", w, y)
end
println()

@testset "apply_word basics" begin
    @test isapprox(apply_word(x, "0"; q=q, ρ=ρ, κ=κ), φ0(x; q=q, ρ=ρ); atol=0.0, rtol=1e-14)
    @test isapprox(apply_word(x, "1"; q=q, ρ=ρ, κ=κ), φ1(x; q=q, ρ=ρ, κ=κ); atol=0.0, rtol=1e-14)

    # order check: "10" means first φ1 then φ0
    y10 = apply_word(x, "10"; q=q, ρ=ρ, κ=κ)
    y10_manual = φ0(φ1(x; q=q, ρ=ρ, κ=κ); q=q, ρ=ρ)
    @test isapprox(y10, y10_manual; atol=1e-12, rtol=0.0)

    # order check: "01" means first φ0 then φ1
    y01 = apply_word(x, "01"; q=q, ρ=ρ, κ=κ)
    y01_manual = φ1(φ0(x; q=q, ρ=ρ); q=q, ρ=ρ, κ=κ)
    @test isapprox(y01, y01_manual; atol=1e-12, rtol=0.0)
end

# --- 2) fixed-point iteration tests ---------------------------------
@printf("\n=== fixed_point_of_word tests (xinit=%.6f) ===\n", xinit)

function fp_residual(y, w; q, ρ, κ)
    return apply_word(y, w; q=q, ρ=ρ, κ=κ) - y
end

@testset "fixed_point_of_word residuals" begin
    for w in ["0", "1", "01", "10", "001", "100", "010", "011", "101", "110"]
        y, it = fixed_point_of_word(w, xinit; q=q, ρ=ρ, κ=κ, tol=1e-12, maxiter=10_000)
        res = fp_residual(y, w; q=q, ρ=ρ, κ=κ)
        @printf("w=\"%s\": y=%.12f  it=%d  residual=%.3e\n", w, y, it, res)
        @test abs(res) ≤ 1e-9
    end
end

# --- 3) limiting cycles + automatic z that matches the itinerary -----
# Your limiting_cycle_points checks: (cyc[i] > z) iff w[i]=='1'.
# Given computed cycle points, we can pick a z (if possible) that satisfies this.
function choose_consistent_z(cyc::AbstractVector{<:Real}, w::AbstractString; margin=1e-10)
    bytes = codeunits(w)
    @assert length(cyc) == length(w)

    max0 = -Inf
    min1 =  Inf
    any0 = false
    any1 = false

    for i in eachindex(cyc)
        if bytes[i] == UInt8('0')
            any0 = true
            max0 = max(max0, cyc[i])
        else
            any1 = true
            min1 = min(min1, cyc[i])
        end
    end

    # If both symbols appear, need max0 < z < min1 (strict, because condition uses > z).
    if any0 && any1
        if max0 < min1 - margin
            return (max0 + min1)/2
        else
            return nothing
        end
    elseif any0
        # all zeros: need cyc[i] > z always false -> z >= max(cyc)
        z = min(1.0, max0 + 0.5*(1.0 - max0))
        return z
    else
        # all ones: need cyc[i] > z always true -> z < min(cyc)
        z = max(0.0, 0.5*min1)
        return z
    end
end

@printf("\n=== limiting_cycle_points tests (auto z) ===\n")

@testset "limiting_cycle_points closure + threshold consistency" begin
    for w in ["10", "01", "011", "001"]
        # compute cycle without checks first (z dummy)
        cyc = limiting_cycle_points(xinit, 0.5, w; q=q, ρ=ρ, κ=κ, check=false)

        z = choose_consistent_z(cyc, w)
        if z === nothing
            @printf("w=\"%s\": NO consistent z found for strict rule (cyc[i] > z iff '1')\n", w)
            @printf("         cycle points = %s\n\n", repr(cyc))
            continue
        end

        @printf("w=\"%s\": z=%.12f\n", w, z)
        @printf("         cycle points = %s\n", repr(cyc))

        # recompute with checks enabled (should produce no warnings if z works)
        cyc2 = limiting_cycle_points(xinit, z, w; q=q, ρ=ρ, κ=κ, check=true)

        # closure test: applying the whole word to y0 returns y0
        y0 = cyc2[1]
        yclose = apply_word(y0, w; q=q, ρ=ρ, κ=κ)
        @test isapprox(yclose, y0; atol=1e-9, rtol=0.0)

        # phase relation test: cyc[i+1] = φ^{w[i]}(cyc[i])
        bytes = codeunits(w)
        for i in 1:(length(w)-1)
            ystep = apply_symbol(cyc2[i], bytes[i]; q=q, ρ=ρ, κ=κ)
            @test isapprox(ystep, cyc2[i+1]; atol=1e-9, rtol=0.0)
        end

        println()
    end
end

println("All done.")

In [ ]:
"""
    limiting_cycle_from_itinerary(w, z; q, ρ, κ,
                                 min_full_repeats=2,
                                 xinit=nothing,
                                 tol=1e-12, maxiter=10_000,
                                 check=true)

Given an itinerary word `w` (String "0101..." or Vector{0,1}),
- decomposes it as w = u ⋅ s^m ⋅ r (r is a prefix of s),
- computes the limiting cycle points associated with the repeating block s.

Returns a NamedTuple with:
(found, formatted, u, s, m, r, start_index, period_length, cycle)

If no periodic tail is found (under min_full_repeats), found=false and cycle=[].
"""
function limiting_cycle_from_itinerary(w, z; q, ρ, κ,
                                      min_full_repeats::Int=2,
                                      xinit=nothing,
                                      tol=1e-12, maxiter=10_000,
                                      check::Bool=true)

    ws = w isa AbstractString ? String(w) : itinerary_word(w)

    d = eventual_periodic_decomp(ws; min_full_repeats=min_full_repeats)
    formatted = format_eventual_periodic(ws; min_full_repeats=min_full_repeats)

    if d.m == 0
        return (found=false, formatted=formatted,
                u=d.u, s="", m=0, r="",
                start_index=d.start_index, period_length=0,
                cycle=Float64[])
    end

    # Pick an initial seed in the invariant interval if none is provided
    xseed = isnothing(xinit) ? (q + ρ) : xinit

    cyc = limiting_cycle_points(xseed, z, d.s;
                                q=q, ρ=ρ, κ=κ,
                                tol=tol, maxiter=maxiter,
                                check=check)

    return (found=true, formatted=formatted,
            u=d.u, s=d.s, m=d.m, r=d.r,
            start_index=d.start_index, period_length=d.period_length,
            cycle=cyc)
end

In [ ]:
using Printf

"""
    print_cycle(cycle, s, z)

Print cycle points y_i with the corresponding symbol s[i] and whether y_i > z.
"""
function print_cycle(cycle::AbstractVector{<:Real}, s::AbstractString, z)
    bytes = codeunits(s)
    k = length(bytes)
    @assert length(cycle) == k "cycle length must equal length(s)"
    println("Cycle for s=\"$(s)\":")
    println(" i  sym     y_i       region")
    println("-------------------------------")
    for i in 1:k
        sym = Char(bytes[i])
        yi = cycle[i]
        reg = (yi > z) ? "> z (active)" : "≤ z (passive)"
        @printf("%2d   %c   %8.5f   %s\n", i, sym, yi, reg)
    end
end

In [ ]:
"""
    cycle_one_fraction(out) -> (a, b, frac_str)

Given `out` = limiting_cycle_from_itinerary(...), return the fraction of 1's
in the limiting symbolic cycle (repeating block `s`) as:

- a = number of '1' symbols in the cycle
- b = cycle length = length(s)
- frac_str = "a/b" (reduced)

If `out.found == false`, returns (0, 0, "0/0").
"""
function cycle_one_fraction(out)
    if !getproperty(out, :found)
        return (0, 0, "0/0")
    end

    s = getproperty(out, :s)
    b = length(s)
    b == 0 && return (0, 0, "0/0")

    a = count(==(UInt8('1')), codeunits(s))

    g = gcd(a, b)
    a2, b2 = a ÷ g, b ÷ g
    return (a2, b2, string(a2, "/", b2))
end

In [ ]:
# Example parameters (pick any valid ones)
q  = 0.25
ρ  = 0.6          
κ  = 0.8

x0, x1, xhi = fixed_points(; q=q, ρ=ρ, κ=κ)

using Printf

@printf("q=%.5f, x1=%.5f, x0=%.5f, q+ρ=%.5f\n", q, x1, x0, q + ρ)

# Threshold and initial belief
# z = 0.5 * q
# z = q + 0.5* (x1 - q)
# z  = x1 + 0.9(x0-x1)
# z = x0 + 0.5*((q + ρ) - x0)
# z = q + ρ + 0.5*(1 - (q + ρ))
# x = 0.5*(x0 + x1)

x01, it = fixed_point_of_word("01", xinit; q=q, ρ=ρ, κ=κ, tol=1e-12, maxiter=10_000)
x10, it = fixed_point_of_word("10", xinit; q=q, ρ=ρ, κ=κ, tol=1e-12, maxiter=10_000)

x011, it = fixed_point_of_word("011", xinit; q=q, ρ=ρ, κ=κ, tol=1e-12, maxiter=10_000)
x101, it = fixed_point_of_word("101", xinit; q=q, ρ=ρ, κ=κ, tol=1e-12, maxiter=10_000)

x010, it = fixed_point_of_word("010", xinit; q=q, ρ=ρ, κ=κ, tol=1e-12, maxiter=10_000)
x100, it = fixed_point_of_word("100", xinit; q=q, ρ=ρ, κ=κ, tol=1e-12, maxiter=10_000)

# z = 0.5 * x01 + 0.5 * x10

#x = 0.5 * q + 0.5 * x1

# z = 0.5 * x011 + 0.5 * x101

z = 0.5 * x010 + 0.5 * x100

# φ1(x; q=q, ρ=ρ, κ=κ) : φ0(x; q=q, ρ=ρ)

#x = φ1(z; q, ρ, κ)
# x = φ0(z; q, ρ)
# x = z

x = 0.01 * φ1(z; q, ρ, κ) + 0.99 * z

# x = z

# x = 0.

@printf("x011=%.5f, x101=%.5f, x01=%.5f, x10=%.5f,  x010=%.5f, x100=%.5f, z=%.5f, x=%.5f\n", 
    x011, x101, x01, x10, x010, x100, z, x)

# Horizon
T  = 20

# --- iterates / orbit / itinerary ---
Xt = varphi_t(x, z, 5; q=q, ρ=ρ, κ=κ)                 # X̃_5 = ϕ_5(x0,z)
X  = skeleton_orbit(x, z, T; q=q, ρ=ρ, κ=κ)           # [X̃_0,...,X̃_T]
A  = skeleton_itinerary(x, z, T; q=q, ρ=ρ, κ=κ)       # [Ã_0,...,Ã_T]
X2, A2 = skeleton_orbit_itinerary(x, z, T; q=q, ρ=ρ, κ=κ)
# If you also added the string and RLE helpers from earlier:
w   = skeleton_itinerary_string(x, z, T; q=q, ρ=ρ, κ=κ)
# rle = skeleton_itinerary_rle(x, z, T; q=q, ρ=ρ, κ=κ)
println("Skeleton itinerary string: ", w)
println("Pattern: ",format_eventual_periodic(w))
# println("Itinerary RLE (symbol,length): ", rle)

out = limiting_cycle_from_itinerary(w, z; q=q, ρ=ρ, κ=κ, min_full_repeats=2)

println(out.formatted)         # u (s)^m r
println("s = ", out.s)

if out.found
    print_cycle(out.cycle, out.s, z)
    # For s="10", out.cycle will be [y1, y2] giving (y1 y2)^m in the limit.
end

a, b, frac = cycle_one_fraction(out)
println("fraction of ones in cycle = ", frac)   # e.g., "1/2"

# Check they match
@assert X == X2
@assert A == A2

# println("X̃_5 = ", Xt)
# println("Skeleton itinerary as 0/1 vector: ", A)

# Assuming you already computed:
# X = skeleton_orbit(x0, z, T; q=q, ρ=ρ, κ=κ)

using Printf

println(" t    X̃_t      Ã_t")
println("----------------------")
for t in 0:T
    @printf("%2d  %8.5f   %d\n", t, X[t+1], A[t+1])
end

# (Optional) compact one-line view:
# println("\nCompact orbit vector:")
# println(X)


In [ ]:
using PGFPlotsX
using Printf

# Helper: convert Julia vectors to TeX list strings
tex_num_list(xs::AbstractVector{<:Real}) =
    isempty(xs) ? "{}" : "{" * join((@sprintf("%.15g", x) for x in xs), ",") * "}"

tex_label_list(ls::AbstractVector{<:AbstractString}) =
    isempty(ls) ? "{}" : "{" * join(ls, ",") * "}"

"""
    bifurcation_segments_pgf(zgrid; q, ρ, κ, ...)

Segment-aware bifurcation diagram over a user-provided (zoom) z-grid.

- Detects intervals [zL,zR) where the *repeating block* `s` (from limiting_cycle_from_itinerary) is constant.
- Plots horizontal line segments for each limiting cycle point over that interval.
- Major x-ticks (and vertical major grid lines) only at {0,q,x^1,x^0,q+ρ,1} that fall in the zoom window.
- Adds *extra x ticks* (unlabeled) at all detected interval endpoints.

REQUIRES (already defined in your notebook):
  - skeleton_itinerary_string(x, z, T; q, ρ, κ)
  - limiting_cycle_from_itinerary(w, z; q, ρ, κ, min_full_repeats=..., check=...)

Returns (TikzPicture, segments).
Each element of `segments` is a NamedTuple: (zL,zR,s,cycle,formatted).
"""
function bifurcation_segments_pgf(zgrid;
        q, ρ, κ,
        xinit = q + ρ,
        T::Int = 600,
        min_full_repeats::Int = 2,
        width::AbstractString = "14cm",
        height::AbstractString = "9cm",
        seg_line_width::AbstractString = "0.45pt",
        grid_line_width::AbstractString = "0.10pt",
        endpoint_tick_width::AbstractString = "0.20pt",
        show_break_lines::Bool = false,
        # NEW:
        mode::Symbol = :limitcycle,          # :limitcycle (old) or :iterate (new)
        square::Bool = true,                  # force square figure if true
        mark_size::AbstractString = "1.2pt"   # markers at discontinuities (iterate mode)
    )

    zs = collect(zgrid)
    @assert issorted(zs) "zgrid must be sorted increasing"
    @assert length(zs) ≥ 2 "zgrid must have at least 2 points"
    zmin, zmax = zs[1], zs[end]

    # square figure if requested
    if square
        height = width
    end

    # fixed points for labeled ticks
    x0 = q / (1 - ρ)
    B  = 1 - ρ + κ*(q + ρ)
    Δ  = B^2 - 4*κ*q
    @assert Δ > 0 "Bad parameters: discriminant ≤ 0"
    x1 = (B - sqrt(Δ)) / (2κ)  # x_lo^1

    # --- major x-ticks/labels: {0,q,x1,x0,q+ρ,1} within zoom window ---
    tick_candidates = [
        (0.0,  raw"$0$"),
        (q,    raw"$q$"),
        (x1,   raw"$x^1$"),
        (x0,   raw"$x^0$"),
        (q+ρ,  raw"$q+\rho$"),
        (1.0,  raw"$1$")
    ]
    
    xticks = Float64[]
    xtlabs = String[]
    for (tv, tl) in tick_candidates
        if zmin - 1e-12 ≤ tv ≤ zmax + 1e-12
            push!(xticks, Float64(tv))
            push!(xtlabs, String(tl))
        end
    end

    # Also include xinit as a labeled tick (very useful when T=1)
    if zmin - 1e-12 ≤ xinit ≤ zmax + 1e-12
        push!(xticks, Float64(xinit))
        push!(xtlabs, String(raw"$x$"))
    end

    # sort/unique ticks (tolerance)
    p = sortperm(xticks)
    xticks = xticks[p]; xtlabs = xtlabs[p]
    tol = 1e-12
    xticks_u = Float64[]; xtlabs_u = String[]
    for i in eachindex(xticks)
        if isempty(xticks_u) || abs(xticks[i] - xticks_u[end]) > tol
            push!(xticks_u, xticks[i])
            push!(xtlabs_u, xtlabs[i])
        end
    end
    xticks = xticks_u; xtlabs = xtlabs_u

    xtick_tex       = tex_num_list(xticks)
    xticklabels_tex = tex_label_list(xtlabs)

    # -----------------------------
    # MODE 1: original limit-cycle segmentation (your current logic)
    # -----------------------------
    if mode == :limitcycle
        # helper: repeating block + limiting cycle at z
        function sig_and_cycle(z)
            w = skeleton_itinerary_string(xinit, z, T; q=q, ρ=ρ, κ=κ)

            out = try
                limiting_cycle_from_itinerary(w, z;
                    q=q, ρ=ρ, κ=κ,
                    min_full_repeats=min_full_repeats,
                    check=false
                )
            catch e
                if e isa MethodError || e isa UndefKeywordError
                    limiting_cycle_from_itinerary(w, z;
                        q=q, ρ=ρ, κ=κ,
                        min_full_repeats=min_full_repeats
                    )
                else
                    rethrow()
                end
            end

            s   = out.found ? out.s : ""
            cyc = out.found ? out.cycle : Float64[]
            return s, cyc, out.formatted
        end

        segments = NamedTuple[]  # (zL,zR,s,cycle,formatted)

        i = 1
        s_cur, cyc_cur, fmt_cur = sig_and_cycle(zs[i])
        zL = zs[i]

        while i < length(zs)
            i += 1
            s_new, cyc_new, fmt_new = sig_and_cycle(zs[i])
            if s_new != s_cur
                zR = zs[i]
                push!(segments, (zL=zL, zR=zR, s=s_cur, cycle=cyc_cur, formatted=fmt_cur))
                zL = zs[i]
                s_cur, cyc_cur, fmt_cur = s_new, cyc_new, fmt_new
            end
        end
        push!(segments, (zL=zL, zR=zs[end], s=s_cur, cycle=cyc_cur, formatted=fmt_cur))

        xsline = Float64[]
        ysline = Float64[]
        for seg in segments
            for y in seg.cycle
                push!(xsline, seg.zL); push!(ysline, y)
                push!(xsline, seg.zR); push!(ysline, y)
                push!(xsline, NaN);    push!(ysline, NaN)
            end
        end

        endpoints = sort(unique(vcat([seg.zL for seg in segments], [seg.zR for seg in segments])))
        extra_xticks = [a for a in endpoints if all(abs(a - t) > tol for t in xticks)]
        extra_xlabs  = fill("\\empty", length(extra_xticks))
        extra_xtick_tex     = tex_num_list(extra_xticks)
        extra_xtlabels_tex  = tex_label_list(extra_xlabs)

        diag = @pgf Plot(
            { "no marks", color="gray", style="dashed", "line width"="0.25pt" },
            Coordinates([(zmin, zmin), (zmax, zmax)])
        )

        mainplot = @pgf Plot(
            { "no marks", "unbounded coords"="jump", "line width"=seg_line_width, color="black" },
            Table(x=xsline, y=ysline)
        )

        ax = @pgf Axis(
            {
                xlabel = raw"$z$",
                ylabel = raw"limiting orbit points",
                width  = width,
                height = height,
                xmin = zmin, xmax = zmax,
                ymin = 0, ymax = 1,

                ytick = "{0,1}",
                yticklabels = raw"{$0$,$1$}",

                xtick = xtick_tex,
                xticklabels = xticklabels_tex,
                "extra x ticks" = extra_xtick_tex,
                "extra x tick labels" = extra_xtlabels_tex,
                "extra x tick style" = "{tick style={black, line width=" * endpoint_tick_width * "}}",

                grid = "none",
                xmajorgrids = "true",
                ymajorgrids = "true",
                xminorgrids = "false",
                yminorgrids = "false",
                "major grid style" = "{draw=gray!40, line width=" * grid_line_width * "}",

                "axis on top" = "false",
                "axis equal image",
            },
            diag, mainplot
        )

        return TikzPicture(ax), segments
    end

    # -----------------------------
    # MODE 2: NEW iterate segmentation for z ↦ varphi_T(xinit,z)
    # -----------------------------
    @assert mode == :iterate "mode must be :limitcycle or :iterate"

    # Signature word that determines varphi_T:
    # need only A_0..A_{T-1}, so call skeleton_itinerary_string with horizon T-1.
    function word_at(z)
        if T == 0
            return ""
        else
            return skeleton_itinerary_string(xinit, z, T-1; q=q, ρ=ρ, κ=κ)  # length T
        end
    end
    value_at(z) = varphi_t(xinit, z, T; q=q, ρ=ρ, κ=κ)

    # precompute on grid
    words = [word_at(z) for z in zs]
    vals  = [value_at(z) for z in zs]

    # refine breakpoint between zL and zR where word changes
    refine_iters = 50
    function refine_boundary(zL, zR, wL)
        lo, hi = zL, zR
        for _ in 1:refine_iters
            mid = (lo + hi)/2
            if word_at(mid) == wL
                lo = mid
            else
                hi = mid
            end
        end
        return hi
    end

    # build segments + polyline + right-continuity markers
    segments = NamedTuple[]  # (zL,zR,word,value)

    xsline = Float64[]
    ysline = Float64[]
    z_open = Float64[]; y_open = Float64[]  # left limit (white)
    z_fill = Float64[]; y_fill = Float64[]  # value at breakpoint (black)

    zL = zs[1]
    w_cur = words[1]
    y_cur = vals[1]

    for i in 2:length(zs)
        if words[i] != w_cur
            zb = refine_boundary(zs[i-1], zs[i], w_cur)

            # close old segment
            push!(segments, (zL=zL, zR=zb, word=w_cur, value=y_cur))
            push!(xsline, zL); push!(ysline, y_cur)
            push!(xsline, zb); push!(ysline, y_cur)
            push!(xsline, NaN); push!(ysline, NaN)

            # markers at breakpoint (right-continuous in z)
            zl = clamp(prevfloat(zb), 0.0, 1.0)
            zr = clamp(zb, 0.0, 1.0)
            yL = value_at(zl)     # left limit
            yR = value_at(zr)     # value at zb (right)
            push!(z_open, zb); push!(y_open, yL)
            push!(z_fill, zb); push!(y_fill, yR)

            # start new segment
            zL = zb
            w_cur = words[i]
            y_cur = vals[i]
        end
    end

    # last segment
    push!(segments, (zL=zL, zR=zs[end], word=w_cur, value=y_cur))
    push!(xsline, zL);      push!(ysline, y_cur)
    push!(xsline, zs[end]); push!(ysline, y_cur)

    # extra ticks at endpoints (unlabeled)
    endpoints = sort(unique(vcat([seg.zL for seg in segments], [seg.zR for seg in segments])))
    extra_xticks = [a for a in endpoints if all(abs(a - t) > tol for t in xticks)]
    extra_xlabs  = fill("\\empty", length(extra_xticks))
    extra_xtick_tex    = tex_num_list(extra_xticks)
    extra_xtlabels_tex = tex_label_list(extra_xlabs)

    diag = @pgf Plot(
        { "no marks", color="gray", style="dashed", "line width"="0.25pt" },
        Coordinates([(zmin, zmin), (zmax, zmax)])
    )

    mainplot = @pgf Plot(
        { "no marks", "unbounded coords"="jump", "line width"=seg_line_width, color="black" },
        Table(x=xsline, y=ysline)
    )

    filled_marks = @pgf Plot(
        { "only marks", mark="*", "mark size"=mark_size, color="black" },
        Table(x=z_fill, y=y_fill)
    )

    open_marks = @pgf Plot(
        {
            "only marks",
            mark="*",
            "mark size"=mark_size,
            color="black",
            "mark options"="{fill=white, draw=black, line width=0.25pt}",
        },
        Table(x=z_open, y=y_open)
    )

    ax = @pgf Axis(
        {
            xlabel = raw"$z$",
            ylabel = raw"$\varphi_T(x,z)$",
            width  = width,
            height = height,

            xmin = zmin, xmax = zmax,
            ymin = 0,    ymax = 1,

            # y ticks: keep it simple here
            ytick = "{0,1}",
            yticklabels = raw"{$0$,$1$}",

            xtick = xtick_tex,
            xticklabels = xticklabels_tex,
            "extra x ticks" = extra_xtick_tex,
            "extra x tick labels" = extra_xtlabels_tex,
            "extra x tick style" = "{tick style={black, line width=" * endpoint_tick_width * "}}",

            grid = "none",
            xmajorgrids = "true",
            ymajorgrids = "true",
            xminorgrids = "false",
            yminorgrids = "false",
            "major grid style" = "{draw=gray!40, line width=" * grid_line_width * "}",

            "axis on top" = "false",
            "axis equal image",
        },
        diag, mainplot, filled_marks, open_marks
    )

    return TikzPicture(ax), segments
end

In [ ]:
# Example parameters (pick any valid ones)
q  = 0.25
ρ  = 0.6          
κ  = 0.8

# choose a zoom interval and grid resolution
zgrid = range(0.0, 1.0; length=1001)

x = 0.5 * (x0 + x1)

println("x = ", x)

# fig, segments = bifurcation_segments_pgf(zgrid; q=q, ρ=ρ, κ=κ, xinit=x, T=1)
# fig, segments = bifurcation_segments_pgf(zgrid; q=q, ρ=ρ, κ=κ, T=1)

fig, segments = bifurcation_segments_pgf(zgrid; q=q, ρ=ρ, κ=κ, xinit=x, T=1, mode=:iterate);

PGFPlotsX.enable_interactive(false)
PGFPlotsX.pgfsave("tmp.pdf", fig)
display("application/pdf", read("tmp.pdf"))

for seg in segments
    if hasproperty(seg, :s)
        # mode = :limitcycle
        outseg = (found = !isempty(seg.s), s = seg.s)
        _, _, frac = cycle_one_fraction(outseg)
        println("[$(seg.zL), $(seg.zR))  s=\"$(seg.s)\"  ones=$(frac)  pattern: $(seg.formatted)")

    elseif hasproperty(seg, :word)
        # mode = :iterate
        zmid = (seg.zL + seg.zR)/2
        zmid = (zmid == seg.zR) ? prevfloat(zmid) : zmid

        # Full itinerary of length T+1:
        aT = (seg.value > zmid) ? '1' : '0'          # right-continuous convention
        wfull = string(seg.word, aT)                 # length (T + 1)

        b = length(wfull)
        a = count(==(UInt8('1')), codeunits(wfull))
        g = gcd(a, b)
        frac = string(a ÷ g, "/", b ÷ g)

        println("[$(seg.zL), $(seg.zR))  itinerary=\"$(wfull)\"  ones=$(frac)  value=$(seg.value)")
    else
        error("Unknown segment type: fields = $(propertynames(seg))")
    end
end

In [ ]:
using PGFPlotsX
using Printf

# Helpers: convert Julia vectors to TeX list strings
tex_num_list(xs::AbstractVector{<:Real}) =
    isempty(xs) ? "{}" : "{" * join((@sprintf("%.15g", x) for x in xs), ",") * "}"

tex_label_list(ls::AbstractVector{<:AbstractString}) =
    isempty(ls) ? "{}" : "{" * join(ls, ",") * "}"

# rate of 1's in a binary word (reduced string and numeric)
function ones_rate_from_word(w::AbstractString)
    T = length(w)
    T == 0 && return ("0/0", NaN)
    a = count(==(UInt8('1')), codeunits(w))
    g = gcd(a, T)
    return (string(a ÷ g, "/", T ÷ g), a / T)
end

"""
    bifurcation_ones_segments_pgf(zgrid; q, ρ, κ, xinit=q+ρ, T=1, ...)

Piecewise-constant plot of the *rate of 1's in the finite skeleton itinerary* w(z)
as a function of z.

IMPORTANT: here `T` is the **itinerary length**:
  w(z) = Ã₀ … Ã_{T-1} (length T),
implemented by calling `skeleton_itinerary_string(xinit,z,T-1; ...)`.

For T=1: w(z) is exactly "1" for z<xinit and "0" for z≥xinit.

Adds right-continuity markers at discontinuities:
- black filled dot = value at breakpoint (right-continuous),
- white filled dot = left limit,
with no vertical jump lines.

Returns (TikzPicture, segments) where each segment is:
  (zL,zR,word,ones_str,ones_val).
"""
function bifurcation_ones_segments_pgf(zgrid;
        q, ρ, κ,
        xinit = q + ρ,
        T::Int = 1,                       # itinerary length
        width::AbstractString = "10cm",
        height::AbstractString = "10cm",
        seg_line_width::AbstractString = "0.70pt",
        grid_line_width::AbstractString = "0.08pt",
        endpoint_tick_width::AbstractString = "0.20pt",
        mark_size::AbstractString = "1.8pt",
        square::Bool = true,
        refine_breaks::Bool = true,
        refine_iters::Int = 60)

    @assert T ≥ 1 "Here T is the itinerary length; require T ≥ 1."

    zs = collect(zgrid)
    @assert issorted(zs) "zgrid must be sorted increasing"
    @assert length(zs) ≥ 2 "zgrid must have at least 2 points"
    zmin, zmax = zs[1], zs[end]

    if square
        height = width
    end

    # itinerary word of length T (uses horizon T-1)
    word_at(z) = skeleton_itinerary_string(xinit, z, T-1; q=q, ρ=ρ, κ=κ)

    # precompute words on grid
    words = [word_at(z) for z in zs]

    # refine breakpoint between zL and zR where word changes
    function refine_boundary(zL, zR, wL)
        lo, hi = zL, zR
        for _ in 1:refine_iters
            mid = (lo + hi)/2
            if word_at(mid) == wL
                lo = mid
            else
                hi = mid
            end
        end
        return hi
    end

    # Build segments and polyline with NaN breaks (no vertical lines)
    segments = NamedTuple[]  # (zL,zR,word,ones_str,ones_val)

    xsline = Float64[]
    ysline = Float64[]

    # discontinuity markers (right-continuity convention)
    z_open = Float64[]; y_open = Float64[]  # left limit (white)
    z_fill = Float64[]; y_fill = Float64[]  # right value at breakpoint (black)

    zL = zs[1]
    w_cur = words[1]
    ones_str_cur, ones_val_cur = ones_rate_from_word(w_cur)

    for i in 2:length(zs)
        if words[i] != w_cur
            zb = refine_breaks ? refine_boundary(zs[i-1], zs[i], w_cur) : zs[i]

            # close old segment at zb
            push!(segments, (zL=zL, zR=zb, word=w_cur, ones_str=ones_str_cur, ones_val=ones_val_cur))
            push!(xsline, zL); push!(ysline, ones_val_cur)
            push!(xsline, zb); push!(ysline, ones_val_cur)
            push!(xsline, NaN); push!(ysline, NaN)

            # markers at breakpoint: left limit at z<zb, right value at z=zb
            zl = clamp(prevfloat(zb), 0.0, 1.0)
            w_left = word_at(zl);  _, yL = ones_rate_from_word(w_left)
            w_right = word_at(zb); _, yR = ones_rate_from_word(w_right)

            push!(z_open, zb); push!(y_open, yL)
            push!(z_fill, zb); push!(y_fill, yR)

            # start new segment
            zL = zb
            w_cur = words[i]
            ones_str_cur, ones_val_cur = ones_rate_from_word(w_cur)
        end
    end

    # last segment
    push!(segments, (zL=zL, zR=zs[end], word=w_cur, ones_str=ones_str_cur, ones_val=ones_val_cur))
    push!(xsline, zL);      push!(ysline, ones_val_cur)
    push!(xsline, zs[end]); push!(ysline, ones_val_cur)

    # major x-ticks/labels: include xinit for visibility (especially for T=1)
    # (0,q,x^1,x^0,q+ρ,1) are optional; keep them if you like
    x0 = q / (1 - ρ)
    B  = 1 - ρ + κ*(q + ρ)
    x1 = (B - sqrt(B^2 - 4*κ*q)) / (2κ)

    tick_candidates = [
        (0.0,  raw"$0$"),
        (q,    raw"$q$"),
        (x1,   raw"$x^1$"),
        (x0,   raw"$x^0$"),
        (q+ρ,  raw"$q+\rho$"),
        (1.0,  raw"$1$"),
        (xinit, raw"$x$")
    ]

    xticks = Float64[]
    xtlabs = String[]
    for (tv, tl) in tick_candidates
        if zmin - 1e-12 ≤ tv ≤ zmax + 1e-12
            push!(xticks, Float64(tv))
            push!(xtlabs, String(tl))
        end
    end

    # sort/unique ticks (tolerance)
    p = sortperm(xticks)
    xticks = xticks[p]; xtlabs = xtlabs[p]
    tol = 1e-12
    xt_u = Float64[]; xl_u = String[]
    for i in eachindex(xticks)
        if isempty(xt_u) || abs(xticks[i] - xt_u[end]) > tol
            push!(xt_u, xticks[i]); push!(xl_u, xtlabs[i])
        end
    end
    xticks, xtlabs = xt_u, xl_u

    xtick_tex       = tex_num_list(xticks)
    xticklabels_tex = tex_label_list(xtlabs)

    # extra x ticks at all segment endpoints (unlabeled)
    endpoints = sort(unique(vcat([seg.zL for seg in segments], [seg.zR for seg in segments])))
    extra_xticks = [a for a in endpoints if all(abs(a - t) > tol for t in xticks)]
    extra_xlabs  = fill("\\empty", length(extra_xticks))
    extra_xtick_tex    = tex_num_list(extra_xticks)
    extra_xtlabs_tex   = tex_label_list(extra_xlabs)

    # plots
    stepplot = @pgf Plot(
        { "no marks", "unbounded coords"="jump", "line width"=seg_line_width, color="black" },
        Table(x = xsline, y = ysline)
    )

    filled_marks = @pgf Plot(
        { "only marks", mark="*", "mark size"=mark_size, color="black" },
        Table(x=z_fill, y=y_fill)
    )

    open_marks = @pgf Plot(
        {
            "only marks",
            mark="*",
            "mark size"=mark_size,
            color="black",
            "mark options"="{fill=white, draw=black, line width=0.25pt}",
        },
        Table(x=z_open, y=y_open)
    )

    ax = @pgf Axis(
        {
            xlabel = raw"$z$",
            ylabel = raw"rate of $1$'s in itinerary",
            width  = width,
            height = height,

            xmin = zmin, xmax = zmax,
            ymin = 0, ymax = 1,

            "axis equal image",
            "axis on top" = "false",

            # y-axis: only 0 and 1
            ytick = "{0,1}",
            yticklabels = raw"{$0$,$1$}",

            # x ticks: major labeled + extra unlabeled endpoints
            xtick = xtick_tex,
            xticklabels = xticklabels_tex,
            "extra x ticks" = extra_xtick_tex,
            "extra x tick labels" = extra_xtlabs_tex,
            "extra x tick style" = "{tick style={black, line width=" * endpoint_tick_width * "}}",

            # grid: only major
            grid = "none",
            xmajorgrids = "true",
            ymajorgrids = "true",
            xminorgrids = "false",
            yminorgrids = "false",
            "major grid style" = "{draw=gray!40, line width=" * grid_line_width * "}",
        },
        stepplot, filled_marks, open_marks
    )

    return TikzPicture(ax), segments
end

In [ ]:
zgrid = range(0.0, 1.0; length=2001)
fig, segs = bifurcation_ones_segments_pgf(zgrid; q=q, ρ=ρ, κ=κ, xinit=x, T=4)

PGFPlotsX.enable_interactive(false)
PGFPlotsX.pgfsave("tmp_rate.pdf", fig)
display("application/pdf", read("tmp_rate.pdf"))

for s in segs
    println("[$(s.zL), $(s.zR))  word=\"$(s.word)\"  ones=$(s.ones_str)")
end

In [ ]:
"""
    mword(α; tol=1e-12)

Return the **shortest (primitive) periodic binary word** `w` whose infinite repetition `w^∞`
satisfies, for all j ≥ 0,
    (w^∞)_j = ⌊α (j+1)⌋ - ⌊α j⌋.

This is the (lower) mechanical word with intercept 0, but returned as a *finite* word.
A finite `w` exists iff α is rational (or α ∈ {0,1}); for Float64 input we first
`rationalize(α; tol=tol)`.

Output is a String over {'0','1'}.
"""
function mword(α; tol=1e-12)
    # --- coerce α to a rational p//q when possible ---
    a = α isa Rational ? α : rationalize(Float64(α); tol=tol)
    p = numerator(a)
    q = denominator(a)

    # clamp/validate for safety
    if p < 0 || p > q
        throw(ArgumentError("α must lie in [0,1]; got α = $a"))
    end

    # edge cases
    if p == 0
        return "0"
    elseif p == q
        return "1"
    end

    # --- build the length-q mechanical word for p/q ---
    # w_j = floor(p(j+1)/q) - floor(pj/q), j=0..q-1
    bytes = Vector{UInt8}(undef, q)
    @inbounds for j in 0:(q-1)
        bit = fld(p*(j+1), q) - fld(p*j, q)  # should be 0 or 1
        bytes[j+1] = (bit == 1) ? UInt8('1') : UInt8('0')
    end

    w = String(bytes)
    return _primitive_period(w)  # “shortest word w such that w^∞ = …”
end

# ---------- internal: reduce a word to its shortest repeating block ----------
function _prefix_function(b::Vector{UInt8})
    n = length(b)
    pi = zeros(Int, n)
    j = 0
    @inbounds for i in 2:n
        while j > 0 && b[i] != b[j+1]
            j = pi[j]
        end
        if b[i] == b[j+1]
            j += 1
        end
        pi[i] = j
    end
    return pi
end

function _primitive_period(s::AbstractString)
    b = collect(codeunits(s))
    n = length(b)
    n == 0 && return ""

    pi = _prefix_function(b)
    p  = n - pi[end]
    if p < n && n % p == 0
        return String(b[1:p])
    else
        return String(b)
    end
end

In [ ]:
println(mword(3//5))
println(mword((3//5 + 2//3)/2))

In [ ]:
"""
    christoffel_conjugates_lstep(w::AbstractString)

Given a (binary) Christoffel word w = w₀…w_{n-1}, let:
- n = length(w)
- m = |w|₁ (# of '1' in w)
- l be the modular inverse of m mod n, i.e. (l*m) mod n = 1

Return the n conjugates
    u(0), u(l), u(2l), …, u((n-1)l)
where u(i) is the rotation (conjugate)
    u(i) = w[i:n-1] w[0:i-1]
with indices taken mod n (0-based), and u(0)=w.

Returns:
- vec::Vector{String} of length n
- l::Int, m::Int, n::Int
"""
function christoffel_conjugates_lstep(w::AbstractString)
    b = collect(codeunits(w))  # safe for ASCII '0','1'
    n = length(b)
    n == 0 && throw(ArgumentError("w must be nonempty"))

    # count ones
    m = count(==(UInt8('1')), b)

    # modular inverse l of m modulo n
    l = invmod(m, n)  # throws if gcd(m,n) != 1

    # rotate by k positions to the left (0 ≤ k < n): w[k+1:end] * w[1:k]
    rotate_left(k::Int) = (k == 0) ? w : String(vcat(b[k+1:end], b[1:k]))

    out = Vector{String}(undef, n)
    for j in 0:(n-1)
        k = mod(j*l, n)
        out[j+1] = rotate_left(k)
    end

    return out, l, m, n
end

In [ ]:
# w = "0111"               
w = mword(2//4)
println(mword(1//2))
conjs, l, m, n = christoffel_conjugates_lstep(w)
println("n=$n, m=$m, l=$l")
println.(conjs)

In [ ]:
using Base.Threads
using LinearAlgebra
BLAS.set_num_threads(1)

cheb01(n) = (1 .- cos.(range(0, π; length=n))) ./ 2

function beta_grid(; nmid=9, nnear=10, δmin=1e-3, δmax=0.1)
    βmid  = collect(range(0.1, 0.9; length=nmid))
    δnear = exp.(range(log(δmax), log(δmin); length=nnear))
    βnear = 1 .- δnear
    unique(sort(vcat(βmid, βnear)))
end

function make_params(qgrid, αgrid, κgrid, βgrid)
    [(q=q, ρ=α*(1-q), κ=κ, β=β)
     for q in qgrid, α in αgrid, κ in κgrid, β in βgrid] |> vec
end

function scan_one_tuple(; q, ρ, κ, β, Nx=51, Nu=51, ε=1e-10, endpoint_dense=true)
    x0v, x1v, _ = fixed_points(; q=q, ρ=ρ, κ=κ)

    xgrid = endpoint_dense ? cheb01(Nx) : collect(range(0.0, 1.0; length=Nx))
    ugrid = endpoint_dense ? cheb01(Nu) : collect(range(0.0, 1.0; length=Nu))

    δ = 1 - β
    min_slack = Inf
    xstar = NaN
    zstar = NaN

    for x in xgrid
        for u in ugrid
            z = x1v + u*(x0v - x1v)
            g = g_metric2(x, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε)
            s = g - δ
            if s < min_slack
                min_slack = s
                xstar = x
                zstar = z
            end
        end
    end

    return (
        min_slack = min_slack,
        rel_slack = min_slack / δ,
        xstar = xstar,
        zstar = zstar,
        x0 = x0v,
        x1 = x1v,
    )
end

function threaded_scan(params; Nx=51, Nu=51, ε=1e-10, endpoint_dense=true)
    results = Vector{NamedTuple}(undef, length(params))
    Threads.@threads :dynamic for i in eachindex(params)
        p = params[i]
        s = scan_one_tuple(; p..., Nx=Nx, Nu=Nu, ε=ε, endpoint_dense=endpoint_dense)
        results[i] = merge(p, s)
    end
    return results
end

In [ ]:
using Base.Threads
using LinearAlgebra
using Dates
using Printf
using DataFrames
using CSV

BLAS.set_num_threads(1)

# -----------------------------------------------------------
# Helper grids
# -----------------------------------------------------------

cheb01(n::Integer) = (1 .- cos.(range(0, π; length=n))) ./ 2

function beta_grid_11()
    return [
        0.10, 0.20, 0.30, 0.40, 0.50,
        0.60, 0.70, 0.80, 0.90, 0.95, 0.99
    ]
end

function make_params(qgrid, αgrid, κgrid, βgrid)
    params = NamedTuple[]
    for q in qgrid, α in αgrid, κ in κgrid, β in βgrid
        ρ = α * (1 - q)
        push!(params, (q=q, ρ=ρ, κ=κ, β=β))
    end
    return params
end

# -----------------------------------------------------------
# One parameter tuple:
# collect minima over x in [0,1], z in [x1,x0]
# -----------------------------------------------------------

function scan_one_tuple_fg(; q, ρ, κ, β, r=1.0,
                           Nx=121, Nu=121,
                           ε=1e-8,
                           endpoint_dense=true)

    x0v, x1v, _ = fixed_points(; q=q, ρ=ρ, κ=κ)

    xgrid = endpoint_dense ? cheb01(Nx) : collect(range(0.0, 1.0; length=Nx))
    ugrid = endpoint_dense ? cheb01(Nu) : collect(range(0.0, 1.0; length=Nu))

    δ = 1 - β

    # min of f - (1-beta)
    min_slack_f = Inf
    xstar_f = NaN
    zstar_f = NaN

    # min of f
    min_f = Inf
    xstar_f0 = NaN
    zstar_f0 = NaN

    # min of f/(1-beta)
    min_ratio_f = Inf
    xstar_ratio_f = NaN
    zstar_ratio_f = NaN

    # min of g - f
    min_gap_g_minus_f = Inf
    xstar_gf = NaN
    zstar_gf = NaN

    # min of g - (1-beta)
    min_slack_g = Inf
    xstar_g = NaN
    zstar_g = NaN

    for x in xgrid
        for u in ugrid
            z = x1v + u * (x0v - x1v)

            f = f_metric2(x, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
            g = g_metric2(x, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε)

            sf  = f - δ
            s0  = f
            rf  = f / δ
            sgf = g - f
            sg  = g - δ

            if sf < min_slack_f
                min_slack_f = sf
                xstar_f = x
                zstar_f = z
            end

            if s0 < min_f
                min_f = s0
                xstar_f0 = x
                zstar_f0 = z
            end

            if rf < min_ratio_f
                min_ratio_f = rf
                xstar_ratio_f = x
                zstar_ratio_f = z
            end

            if sgf < min_gap_g_minus_f
                min_gap_g_minus_f = sgf
                xstar_gf = x
                zstar_gf = z
            end

            if sg < min_slack_g
                min_slack_g = sg
                xstar_g = x
                zstar_g = z
            end
        end
    end

    return (
        x0 = x0v,
        x1 = x1v,

        min_slack_f = min_slack_f,
        rel_slack_f = min_slack_f / δ,
        xstar_f = xstar_f,
        zstar_f = zstar_f,

        min_f = min_f,
        xstar_f0 = xstar_f0,
        zstar_f0 = zstar_f0,

        min_ratio_f = min_ratio_f,
        xstar_ratio_f = xstar_ratio_f,
        zstar_ratio_f = zstar_ratio_f,

        min_gap_g_minus_f = min_gap_g_minus_f,
        xstar_gf = xstar_gf,
        zstar_gf = zstar_gf,

        min_slack_g = min_slack_g,
        rel_slack_g = min_slack_g / δ,
        xstar_g = xstar_g,
        zstar_g = zstar_g,
    )
end

# -----------------------------------------------------------
# Threaded outer scan
# -----------------------------------------------------------

function threaded_scan_fg(params; r=1.0,
                          Nx=121, Nu=121,
                          ε=1e-8,
                          endpoint_dense=true)

    results = Vector{NamedTuple}(undef, length(params))

    Threads.@threads :dynamic for i in eachindex(params)
        p = params[i]
        s = scan_one_tuple_fg(; p..., r=r, Nx=Nx, Nu=Nu, ε=ε, endpoint_dense=endpoint_dense)
        results[i] = merge(p, s)
    end

    return results
end

# -----------------------------------------------------------
# Chunked run
# -----------------------------------------------------------

function serious_run_fg_chunked(;
    r=1.0,
    Nq=14, Nα=14, Nκ=14,
    Nx=121, Nu=121,
    ε=1e-8,
    endpoint_dense=true,
    chunk_size=512,
    qmin=0.05, qmax=0.95,
    αmin=0.10, αmax=0.90,
    κmin=0.05, κmax=0.95,
    outdir="serious_run_fg_14x14x14x11_121x121_eps1e8"
)

    println("Threads.nthreads() = ", Threads.nthreads())

    qgrid = collect(range(qmin, qmax; length=Nq))
    αgrid = collect(range(αmin, αmax; length=Nα))
    κgrid = collect(range(κmin, κmax; length=Nκ))
    βgrid = beta_grid_11()

    params = make_params(qgrid, αgrid, κgrid, βgrid)

    nparams = length(params)
    nevals_total = nparams * Nx * Nu
    nchunks = cld(nparams, chunk_size)

    mkpath(outdir)

    CSV.write(joinpath(outdir, "beta_grid.csv"), DataFrame(β=βgrid))
    CSV.write(joinpath(outdir, "q_grid.csv"), DataFrame(q=qgrid))
    CSV.write(joinpath(outdir, "alpha_grid.csv"), DataFrame(α=αgrid))
    CSV.write(joinpath(outdir, "kappa_grid.csv"), DataFrame(κ=κgrid))

    open(joinpath(outdir, "run_info.txt"), "w") do io
        println(io, "Threads.nthreads() = ", Threads.nthreads())
        println(io, "r = ", r)
        println(io, "Nq = ", Nq)
        println(io, "Nα = ", Nα)
        println(io, "Nκ = ", Nκ)
        println(io, "Nβ = ", length(βgrid))
        println(io, "Nx = ", Nx)
        println(io, "Nu = ", Nu)
        println(io, "ε = ", ε)
        println(io, "endpoint_dense = ", endpoint_dense)
        println(io, "chunk_size = ", chunk_size)
        println(io, "q range = [", qmin, ", ", qmax, "]")
        println(io, "α range = [", αmin, ", ", αmax, "]")
        println(io, "κ range = [", κmin, ", ", κmax, "]")
        println(io, "parameter tuples = ", nparams)
        println(io, "total (x,z) evaluations = ", nevals_total)
        println(io, "checks: g(x,z)>f(x,z), f(x,z)>=0, min f/(1-beta)")
    end

    println("parameter tuples = ", nparams)
    println("Nx = ", Nx, ", Nu = ", Nu)
    println("total (x,z) evaluations = ", nevals_total)
    println("number of chunks = ", nchunks)
    println("output directory = ", outdir)
    println("β grid = ", βgrid)

    all_results = NamedTuple[]

    global_min_slack_f = Inf
    global_min_f = Inf
    global_min_ratio_f = Inf
    global_min_gap_gf = Inf
    global_min_slack_g = Inf

    t0_total = time()

    for c in 1:nchunks
        lo = (c - 1) * chunk_size + 1
        hi = min(c * chunk_size, nparams)
        params_chunk = params[lo:hi]

        GC.gc()
        t0_chunk = time()
        res = threaded_scan_fg(params_chunk;
                               r=r,
                               Nx=Nx, Nu=Nu,
                               ε=ε,
                               endpoint_dense=endpoint_dense)
        t_chunk = time() - t0_chunk

        df_chunk = DataFrame(res)
        CSV.write(joinpath(outdir, @sprintf("chunk_%03d.csv", c)), df_chunk)

        append!(all_results, res)

        global_min_slack_f = min(global_min_slack_f, minimum(df_chunk.min_slack_f))
        global_min_f       = min(global_min_f,       minimum(df_chunk.min_f))
        global_min_ratio_f = min(global_min_ratio_f, minimum(df_chunk.min_ratio_f))
        global_min_gap_gf  = min(global_min_gap_gf,  minimum(df_chunk.min_gap_g_minus_f))
        global_min_slack_g = min(global_min_slack_g, minimum(df_chunk.min_slack_g))

        elapsed = time() - t0_total
        frac_done = hi / nparams
        eta = elapsed * (1 - frac_done) / frac_done

        println(@sprintf(
            "[%s] chunk %3d/%3d | tuples %6d:%6d | chunk %.1f s | elapsed %.2f h | ETA %.2f h | min(f-(1-β)) = %.6e | min(f) = %.6e | min(f/(1-β)) = %.6e | min(g-f) = %.6e | min(g-(1-β)) = %.6e",
            Dates.format(now(), "HH:MM:SS"), c, nchunks, lo, hi, t_chunk,
            elapsed / 3600, eta / 3600,
            global_min_slack_f, global_min_f, global_min_ratio_f,
            global_min_gap_gf, global_min_slack_g
        ))
    end

    df_all = DataFrame(all_results)

    df_fslack = sort(df_all, :min_slack_f)
    CSV.write(joinpath(outdir, "all_results_sorted_by_min_slack_f.csv"), df_fslack)

    df_f0 = sort(df_all, :min_f)
    CSV.write(joinpath(outdir, "all_results_sorted_by_min_f.csv"), df_f0)
    CSV.write(joinpath(outdir, "worst_200_f_ge_0.csv"),
              first(df_f0, min(200, nrow(df_f0))))

    df_ratiof = sort(df_all, :min_ratio_f)
    CSV.write(joinpath(outdir, "all_results_sorted_by_min_ratio_f.csv"), df_ratiof)
    CSV.write(joinpath(outdir, "worst_200_min_f_over_1minusbeta.csv"),
              first(df_ratiof, min(200, nrow(df_ratiof))))

    df_gf = sort(df_all, :min_gap_g_minus_f)
    CSV.write(joinpath(outdir, "all_results_sorted_by_min_gap_g_minus_f.csv"), df_gf)
    CSV.write(joinpath(outdir, "worst_200_g_gt_f.csv"),
              first(df_gf, min(200, nrow(df_gf))))

    df_g = sort(df_all, :min_slack_g)
    CSV.write(joinpath(outdir, "all_results_sorted_by_min_slack_g.csv"), df_g)

    total_time = time() - t0_total

    println("\nFinished.")
    println("Total runtime = ", round(total_time, digits=1), " s = ",
            round(total_time / 3600, digits=3), " h")

    println("\nWorst case for f(x,z) > 1-β:")
    println(df_fslack[1, [:q, :ρ, :κ, :β, :xstar_f, :zstar_f, :min_slack_f, :rel_slack_f]])

    println("\nWorst case for f(x,z) >= 0:")
    println(df_f0[1, [:q, :ρ, :κ, :β, :xstar_f0, :zstar_f0, :min_f]])

    println("\nWorst case for min f(x,z)/(1-β):")
    println(df_ratiof[1, [:q, :ρ, :κ, :β, :xstar_ratio_f, :zstar_ratio_f, :min_ratio_f]])

    println("\nWorst case for g(x,z) > f(x,z):")
    println(df_gf[1, [:q, :ρ, :κ, :β, :xstar_gf, :zstar_gf, :min_gap_g_minus_f]])

    println("\nWorst case for g(x,z) > 1-β:")
    println(df_g[1, [:q, :ρ, :κ, :β, :xstar_g, :zstar_g, :min_slack_g, :rel_slack_g]])

    return df_all
end

In [ ]:
df_fg = serious_run_fg_chunked(
    r=1.0,
    Nq=14, Nα=14, Nκ=14,
    Nx=121, Nu=121,
    ε=1e-8,
    endpoint_dense=true,
    chunk_size=512,
    outdir="serious_run_fg_14x14x14x11_121x121_eps1e8"
)

In [ ]:
qgrid0 = range(0.05, 0.95; length=3)
αgrid0 = range(0.10, 0.90; length=3)
κgrid0 = range(0.05, 0.95; length=3)
βgrid0 = beta_grid(nmid=3, nnear=3, δmin=1e-2, δmax=0.1)

params0 = make_params(qgrid0, αgrid0, κgrid0, βgrid0)

Nx0, Nu0 = 21, 21
nevals0 = length(params0) * Nx0 * Nu0

t0 = @elapsed res0 = threaded_scan(params0; Nx=Nx0, Nu=Nu0, ε=1e-8)

println("threads = ", Threads.nthreads())
println("pilot time = ", t0, " seconds")
println("eval rate = ", nevals0 / t0, " evaluations / second")

In [ ]:
include("pcli1_pilot.jl")

df_jup, stats_jup = pilot_benchmark(
    Nq=3, Nα=3, Nκ=3,
    Nx=21, Nu=21,
    ε=1e-8,
    repeats=3,
    nβmid=3, nβnear=4,
    δmin=1e-2, δmax=0.1,
    csvfile="pilot_jupyter_threads$(Threads.nthreads()).csv",
)

In [ ]:
include("pcli1_pilot.jl")

using Dates
using Printf
using DataFrames
using CSV

# Keep the same function name as before, so the rest of the code changes minimally.
function beta_grid_24()
    β = [
        0.10, 0.20, 0.30, 0.40, 0.50,
        0.60, 0.70, 0.80, 0.90, 0.95, 0.99
    ]
    @assert length(β) == 11
    return β
end

function serious_run_chunked(;
    Nq=14, Nα=14, Nκ=14,
    Nx=121, Nu=121,
    ε=1e-8,
    endpoint_dense=true,
    chunk_size=512,
    qmin=0.05, qmax=0.95,
    αmin=0.10, αmax=0.90,
    κmin=0.05, κmax=0.95,
    outdir="serious_run_14x14x14x11_121x121_eps1e8",
)

    println("Threads.nthreads() = ", Threads.nthreads())

    qgrid = collect(range(qmin, qmax; length=Nq))
    αgrid = collect(range(αmin, αmax; length=Nα))
    κgrid = collect(range(κmin, κmax; length=Nκ))
    βgrid = beta_grid_24()

    params = make_params(qgrid, αgrid, κgrid, βgrid)

    nparams = length(params)
    nevals_total = nparams * Nx * Nu
    nchunks = cld(nparams, chunk_size)

    mkpath(outdir)

    CSV.write(joinpath(outdir, "beta_grid.csv"), DataFrame(β=βgrid))
    CSV.write(joinpath(outdir, "q_grid.csv"), DataFrame(q=qgrid))
    CSV.write(joinpath(outdir, "alpha_grid.csv"), DataFrame(α=αgrid))
    CSV.write(joinpath(outdir, "kappa_grid.csv"), DataFrame(κ=κgrid))

    open(joinpath(outdir, "run_info.txt"), "w") do io
        println(io, "Threads.nthreads() = ", Threads.nthreads())
        println(io, "Nq = ", Nq)
        println(io, "Nα = ", Nα)
        println(io, "Nκ = ", Nκ)
        println(io, "Nβ = ", length(βgrid))
        println(io, "Nx = ", Nx)
        println(io, "Nu = ", Nu)
        println(io, "ε = ", ε)
        println(io, "endpoint_dense = ", endpoint_dense)
        println(io, "chunk_size = ", chunk_size)
        println(io, "q range = [", qmin, ", ", qmax, "]")
        println(io, "α range = [", αmin, ", ", αmax, "]")
        println(io, "κ range = [", κmin, ", ", κmax, "]")
        println(io, "parameter tuples = ", nparams)
        println(io, "total g_metric2 evaluations = ", nevals_total)
    end

    println("parameter tuples = ", nparams)
    println("Nx = ", Nx, ", Nu = ", Nu)
    println("total g_metric2 evaluations = ", nevals_total)
    println("number of chunks = ", nchunks)
    println("output directory = ", outdir)
    println("β grid = ", βgrid)

    all_results = NamedTuple[]
    global_worst = Inf

    t0_total = time()

    for c in 1:nchunks
        lo = (c - 1) * chunk_size + 1
        hi = min(c * chunk_size, nparams)
        params_chunk = params[lo:hi]

        GC.gc()
        t0_chunk = time()
        res = threaded_scan(params_chunk; Nx=Nx, Nu=Nu, ε=ε, endpoint_dense=endpoint_dense)
        t_chunk = time() - t0_chunk

        df_chunk = DataFrame(res)
        sort!(df_chunk, :min_slack)

        chunkfile = joinpath(outdir, @sprintf("chunk_%03d.csv", c))
        CSV.write(chunkfile, df_chunk)

        append!(all_results, res)

        if df_chunk.min_slack[1] < global_worst
            global_worst = df_chunk.min_slack[1]
        end

        elapsed = time() - t0_total
        frac_done = hi / nparams
        eta = elapsed * (1 - frac_done) / frac_done

        println(@sprintf(
            "[%s] chunk %3d/%3d | tuples %6d:%6d | chunk %.1f s | elapsed %.2f h | ETA %.2f h | current worst min_slack = %.6e",
            Dates.format(now(), "HH:MM:SS"), c, nchunks, lo, hi, t_chunk, elapsed / 3600, eta / 3600, global_worst
        ))
    end

    df_all = DataFrame(all_results)
    sort!(df_all, :min_slack)

    CSV.write(joinpath(outdir, "all_results_sorted.csv"), df_all)
    CSV.write(joinpath(outdir, "worst_200.csv"), first(df_all, min(200, nrow(df_all))))

    println("\nFinished.")
    println("Final worst min_slack = ", df_all.min_slack[1])
    println("Worst tuple:")
    println(df_all[1, [:q, :ρ, :κ, :β, :xstar, :zstar, :min_slack, :rel_slack]])

    return df_all
end

# Run the modified serious experiment
df_serious = serious_run_chunked(
    Nq=14, Nα=14, Nκ=14,
    Nx=121, Nu=121,
    ε=1e-8,
    endpoint_dense=true,
    chunk_size=512,
    outdir="serious_run_14x14x14x11_121x121_eps1e8"
)

In [ ]:
using Base.Threads
using LinearAlgebra
using Dates
using Printf
using CSV
using DataFrames

BLAS.set_num_threads(1)

function beta_grid_11()
    return [
        0.10, 0.20, 0.30, 0.40, 0.50,
        0.60, 0.70, 0.80, 0.90, 0.95, 0.99
    ]
end

function make_params(qgrid, αgrid, κgrid, βgrid)
    params = NamedTuple[]
    for q in qgrid, α in αgrid, κ in κgrid, β in βgrid
        ρ = α * (1 - q)
        push!(params, (q=q, ρ=ρ, κ=κ, β=β))
    end
    return params
end

function uniform_x_grid_with_padding(x1::Float64, x0::Float64;
                                     Nmid::Int=2001,
                                     Nside::Int=201,
                                     pad_frac::Float64=0.05)

    @assert 0.0 <= x1 <= x0 <= 1.0
    @assert Nmid >= 2
    @assert Nside >= 0

    gap = x0 - x1
    pad = pad_frac * gap

    xL = max(0.0, x1 - pad)
    xR = min(1.0, x0 + pad)

    left = if Nside > 0 && xL < x1
        collect(range(xL, x1; length=Nside + 1))[1:end-1]
    else
        Float64[]
    end

    mid = collect(range(x1, x0; length=Nmid))

    right = if Nside > 0 && x0 < xR
        collect(range(x0, xR; length=Nside + 1))[2:end]
    else
        Float64[]
    end

    xgrid = vcat(left, mid, right)

    i1 = length(left) + 1
    i0 = length(left) + length(mid)

    return xgrid, i1, i0, xL, xR
end

function finite_min_with_index(v::AbstractVector{<:Real})
    best = Inf
    ibest = 0
    @inbounds for i in eachindex(v)
        vi = v[i]
        if isfinite(vi) && vi < best
            best = vi
            ibest = i
        end
    end
    return best, ibest
end

function finite_maxabs_with_index(v::AbstractVector{<:Real})
    best = -Inf
    ibest = 0
    @inbounds for i in eachindex(v)
        vi = v[i]
        if isfinite(vi)
            avi = abs(vi)
            if avi > best
                best = avi
                ibest = i
            end
        end
    end
    return best, ibest
end

function scan_one_tuple_pcli2(; q, ρ, κ, β, r,
                              Nmid=2001,
                              Nside=201,
                              pad_frac=0.05,
                              ε=1e-8,
                              mono_tol=1e-12)

    x0v, x1v, _ = fixed_points(; q=q, ρ=ρ, κ=κ)

    xgrid, i1, i0, xL, xR = uniform_x_grid_with_padding(
        x1v, x0v; Nmid=Nmid, Nside=Nside, pad_frac=pad_frac
    )

    n = length(xgrid)
    fvals = Vector{Float64}(undef, n)
    gvals = Vector{Float64}(undef, n)
    mvals = Vector{Float64}(undef, n)

    @inbounds for i in eachindex(xgrid)
        x = xgrid[i]
        f = f_metric2(x, x; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
        g = g_metric2(x, x; q=q, ρ=ρ, κ=κ, β=β, ε=ε)
        fvals[i] = f
        gvals[i] = g
        mvals[i] = f / g
    end

    n_nonfinite_f = count(x -> !isfinite(x), fvals)
    n_nonfinite_g = count(x -> !isfinite(x), gvals)
    n_nonpositive_g = count(x -> !(isfinite(x) && x > 0.0), gvals)
    n_nonfinite_m = count(x -> !isfinite(x), mvals)

    diffs = Vector{Float64}(undef, n - 1)
    @inbounds for i in 1:n-1
        a = mvals[i]
        b = mvals[i + 1]
        diffs[i] = (isfinite(a) && isfinite(b)) ? (b - a) : NaN
    end

    min_diff_all, j_all = finite_min_with_index(diffs)

    core_start = i1
    core_stop  = i0 - 1
    if core_start <= core_stop
        diffs_core = @view diffs[core_start:core_stop]
    else
        diffs_core = Float64[]
    end

    min_diff_core, j_core_local = isempty(diffs_core) ? (NaN, 0) : finite_min_with_index(diffs_core)
    j_core = j_core_local == 0 ? 0 : (core_start + j_core_local - 1)

    xdiff_all_left  = j_all == 0 ? NaN : xgrid[j_all]
    xdiff_all_right = j_all == 0 ? NaN : xgrid[j_all + 1]

    xdiff_core_left  = j_core == 0 ? NaN : xgrid[j_core]
    xdiff_core_right = j_core == 0 ? NaN : xgrid[j_core + 1]

    Δx1_left  = i1 > 1 ? mvals[i1]   - mvals[i1 - 1] : NaN
    Δx1_right = i1 < n ? mvals[i1+1] - mvals[i1]     : NaN
    Δx0_left  = i0 > 1 ? mvals[i0]   - mvals[i0 - 1] : NaN
    Δx0_right = i0 < n ? mvals[i0+1] - mvals[i0]     : NaN

    cont_x1_left  = i1 > 1 ? abs(mvals[i1]   - mvals[i1 - 1]) : NaN
    cont_x1_right = i1 < n ? abs(mvals[i1+1] - mvals[i1])     : NaN
    cont_x0_left  = i0 > 1 ? abs(mvals[i0]   - mvals[i0 - 1]) : NaN
    cont_x0_right = i0 < n ? abs(mvals[i0+1] - mvals[i0])     : NaN

    cont_vals = [cont_x1_left, cont_x1_right, cont_x0_left, cont_x0_right]
    cont_vals_finite = filter(isfinite, cont_vals)
    endpoint_cont_proxy = isempty(cont_vals_finite) ? NaN : maximum(cont_vals_finite)

    max_adj_jump, j_jump = finite_maxabs_with_index(diffs)
    xjump_left  = j_jump == 0 ? NaN : xgrid[j_jump]
    xjump_right = j_jump == 0 ? NaN : xgrid[j_jump + 1]

    mono_pass_all = isfinite(min_diff_all)  && min_diff_all  >= -mono_tol
    mono_pass_core = isfinite(min_diff_core) && min_diff_core >= -mono_tol

    endpoint_diffs = filter(isfinite, [Δx1_left, Δx1_right, Δx0_left, Δx0_right])
    mono_pass_endpoints = !isempty(endpoint_diffs) && minimum(endpoint_diffs) >= -mono_tol

    return (
        x1 = x1v,
        x0 = x0v,
        xL = xL,
        xR = xR,
        n_x = n,

        min_g_diag = minimum(gvals),
        n_nonfinite_f = n_nonfinite_f,
        n_nonfinite_g = n_nonfinite_g,
        n_nonpositive_g = n_nonpositive_g,
        n_nonfinite_m = n_nonfinite_m,

        min_diff_all = min_diff_all,
        xdiff_all_left = xdiff_all_left,
        xdiff_all_right = xdiff_all_right,

        min_diff_core = min_diff_core,
        xdiff_core_left = xdiff_core_left,
        xdiff_core_right = xdiff_core_right,

        Δx1_left = Δx1_left,
        Δx1_right = Δx1_right,
        Δx0_left = Δx0_left,
        Δx0_right = Δx0_right,

        cont_x1_left = cont_x1_left,
        cont_x1_right = cont_x1_right,
        cont_x0_left = cont_x0_left,
        cont_x0_right = cont_x0_right,
        endpoint_cont_proxy = endpoint_cont_proxy,

        max_adj_jump = max_adj_jump,
        xjump_left = xjump_left,
        xjump_right = xjump_right,

        mono_pass_all = mono_pass_all,
        mono_pass_core = mono_pass_core,
        mono_pass_endpoints = mono_pass_endpoints,
    )
end

function threaded_scan_pcli2(params; r, Nmid=2001, Nside=201, pad_frac=0.05,
                             ε=1e-8, mono_tol=1e-12)

    results = Vector{NamedTuple}(undef, length(params))

    Threads.@threads :dynamic for i in eachindex(params)
        p = params[i]
        s = scan_one_tuple_pcli2(; p..., r=r,
                                 Nmid=Nmid, Nside=Nside, pad_frac=pad_frac,
                                 ε=ε, mono_tol=mono_tol)
        results[i] = merge(p, s)
    end

    return results
end

function serious_run_pcli2_chunked(;
    r=1.0,
    Nq=14, Nα=14, Nκ=14,
    Nmid=2001,
    Nside=201,
    pad_frac=0.05,
    ε=1e-8,
    mono_tol=1e-12,
    chunk_size=512,
    qmin=0.05, qmax=0.95,
    αmin=0.10, αmax=0.90,
    κmin=0.05, κmax=0.95,
    outdir="serious_run_pcli2_14x14x14x11_uniformx"
)

    println("Threads.nthreads() = ", Threads.nthreads())

    qgrid = collect(range(qmin, qmax; length=Nq))
    αgrid = collect(range(αmin, αmax; length=Nα))
    κgrid = collect(range(κmin, κmax; length=Nκ))
    βgrid = beta_grid_11()

    params = make_params(qgrid, αgrid, κgrid, βgrid)

    nparams = length(params)
    n_x_total_per_tuple = Nmid + 2*Nside
    nevals_total = nparams * n_x_total_per_tuple
    nchunks = cld(nparams, chunk_size)

    mkpath(outdir)

    CSV.write(joinpath(outdir, "beta_grid.csv"), DataFrame(β=βgrid))
    CSV.write(joinpath(outdir, "q_grid.csv"), DataFrame(q=qgrid))
    CSV.write(joinpath(outdir, "alpha_grid.csv"), DataFrame(α=αgrid))
    CSV.write(joinpath(outdir, "kappa_grid.csv"), DataFrame(κ=κgrid))

    open(joinpath(outdir, "run_info.txt"), "w") do io
        println(io, "Threads.nthreads() = ", Threads.nthreads())
        println(io, "r = ", r)
        println(io, "Nq = ", Nq)
        println(io, "Nα = ", Nα)
        println(io, "Nκ = ", Nκ)
        println(io, "Nβ = ", length(βgrid))
        println(io, "Nmid = ", Nmid)
        println(io, "Nside = ", Nside)
        println(io, "pad_frac = ", pad_frac)
        println(io, "ε = ", ε)
        println(io, "mono_tol = ", mono_tol)
        println(io, "chunk_size = ", chunk_size)
        println(io, "parameter tuples = ", nparams)
        println(io, "x evaluations per tuple (nominal) = ", n_x_total_per_tuple)
        println(io, "total m(x) evaluations (nominal) = ", nevals_total)
    end

    println("parameter tuples = ", nparams)
    println("Nmid = ", Nmid, ", Nside = ", Nside, ", pad_frac = ", pad_frac)
    println("nominal x evaluations per tuple = ", n_x_total_per_tuple)
    println("nominal total evaluations = ", nevals_total)
    println("number of chunks = ", nchunks)
    println("output directory = ", outdir)
    println("β grid = ", βgrid)

    all_results = NamedTuple[]
    global_min_diff = Inf
    global_max_endpoint_cont = -Inf

    t0_total = time()

    for c in 1:nchunks
        lo = (c - 1) * chunk_size + 1
        hi = min(c * chunk_size, nparams)
        params_chunk = params[lo:hi]

        GC.gc()
        t0_chunk = time()
        res = threaded_scan_pcli2(params_chunk;
                                  r=r,
                                  Nmid=Nmid, Nside=Nside, pad_frac=pad_frac,
                                  ε=ε, mono_tol=mono_tol)
        t_chunk = time() - t0_chunk

        df_chunk = DataFrame(res)
        sort!(df_chunk, :min_diff_all)

        chunkfile = joinpath(outdir, @sprintf("chunk_%03d.csv", c))
        CSV.write(chunkfile, df_chunk)

        append!(all_results, res)

        if df_chunk.min_diff_all[1] < global_min_diff
            global_min_diff = df_chunk.min_diff_all[1]
        end

        epvals = filter(isfinite, collect(df_chunk.endpoint_cont_proxy))
        chunk_max_endpoint_cont = isempty(epvals) ? NaN : maximum(epvals)

        if isfinite(chunk_max_endpoint_cont) && chunk_max_endpoint_cont > global_max_endpoint_cont
            global_max_endpoint_cont = chunk_max_endpoint_cont
        end

        elapsed = time() - t0_total
        frac_done = hi / nparams
        eta = elapsed * (1 - frac_done) / frac_done

        println(@sprintf(
            "[%s] chunk %3d/%3d | tuples %6d:%6d | chunk %.1f s | elapsed %.2f h | ETA %.2f h | global min_diff_all = %.6e | global max endpoint proxy = %.6e",
            Dates.format(now(), "HH:MM:SS"), c, nchunks, lo, hi, t_chunk,
            elapsed / 3600, eta / 3600, global_min_diff, global_max_endpoint_cont
        ))
    end

    df_all = DataFrame(all_results)

    df_sorted_mono = sort(df_all, :min_diff_all)
    CSV.write(joinpath(outdir, "all_results_sorted_by_min_diff_all.csv"), df_sorted_mono)
    CSV.write(joinpath(outdir, "worst_200_monotonicity.csv"),
              first(df_sorted_mono, min(200, nrow(df_sorted_mono))))

    df_sorted_core = sort(df_all, :min_diff_core)
    CSV.write(joinpath(outdir, "all_results_sorted_by_min_diff_core.csv"), df_sorted_core)
    CSV.write(joinpath(outdir, "worst_200_core_monotonicity.csv"),
              first(df_sorted_core, min(200, nrow(df_sorted_core))))

    df_sorted_cont = sort(df_all, :endpoint_cont_proxy, rev=true)
    CSV.write(joinpath(outdir, "all_results_sorted_by_endpoint_cont_proxy_desc.csv"), df_sorted_cont)
    CSV.write(joinpath(outdir, "worst_200_endpoint_continuity_proxy.csv"),
              first(df_sorted_cont, min(200, nrow(df_sorted_cont))))

    println("\nFinished.")
    println("Worst monotonicity case over extended interval:")
    println(df_sorted_mono[1, [:q, :ρ, :κ, :β, :min_diff_all, :xdiff_all_left, :xdiff_all_right]])

    println("\nWorst monotonicity case on [x1, x0]:")
    println(df_sorted_core[1, [:q, :ρ, :κ, :β, :min_diff_core, :xdiff_core_left, :xdiff_core_right]])

    println("\nLargest endpoint continuity proxy:")
    println(df_sorted_cont[1, [:q, :ρ, :κ, :β, :endpoint_cont_proxy,
                               :cont_x1_left, :cont_x1_right, :cont_x0_left, :cont_x0_right]])

    return df_all
end

df_pcli2 = serious_run_pcli2_chunked(
    r=1.0,
    Nq=14, Nα=14, Nκ=14,
    Nmid=2001,
    Nside=201,
    pad_frac=0.05,
    ε=1e-8,
    mono_tol=1e-12,
    chunk_size=512,
    outdir="serious_run_pcli2_14x14x14x11_uniformx_Nmid2001_Nside201"
)

In [ ]:
using Base.Threads
using LinearAlgebra
using Dates
using Printf
using CSV
using DataFrames

BLAS.set_num_threads(1)

# ============================================================
# Same outer parameter grid as before
# ============================================================

function beta_grid_11()
    return [
        0.10, 0.20, 0.30, 0.40, 0.50,
        0.60, 0.70, 0.80, 0.90, 0.95, 0.99
    ]
end

function make_params(qgrid, αgrid, κgrid, βgrid)
    params = NamedTuple[]
    for q in qgrid, α in αgrid, κ in κgrid, β in βgrid
        ρ = α * (1 - q)
        push!(params, (q=q, ρ=ρ, κ=κ, β=β))
    end
    return params
end

# ============================================================
# Uniform x-grid on [0,1]
# Use Nx=2403 by default to match previous PCLI2 count
# ============================================================

uniform01_grid(Nx::Int=2403) = collect(range(0.0, 1.0; length=Nx))

# ============================================================
# Diagonal MP index on a grid
# ============================================================

function diag_index_grid(xgrid; q, ρ, κ, r, β, ε=1e-8)
    mvals = Vector{Float64}(undef, length(xgrid))
    @inbounds for i in eachindex(xgrid)
        x = xgrid[i]
        f = f_metric2(x, x; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
        g = g_metric2(x, x; q=q, ρ=ρ, κ=κ, β=β, ε=ε)
        mvals[i] = f / g
    end
    return mvals
end

# ============================================================
# Per-parameter lambda grid
#
# Built from sampled diagonal-index range on [0,1], padded a bit.
# ============================================================

function lambda_grid_from_diag(; q, ρ, κ, r, β,
                               Nλ::Int=41,
                               Nxλ::Int=401,
                               λpad_frac::Float64=0.05,
                               ε::Float64=1e-8)

    xgridλ = uniform01_grid(Nxλ)
    mvals = diag_index_grid(xgridλ; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)

    finite_m = filter(isfinite, mvals)
    @assert !isempty(finite_m) "No finite diagonal index values found."

    λmin0 = minimum(finite_m)
    λmax0 = maximum(finite_m)

    span = λmax0 - λmin0
    scale = max(abs(λmin0), abs(λmax0), 1.0)
    pad = span > 0 ? λpad_frac * span : λpad_frac * scale

    λmin = λmin0 - pad
    λmax = λmax0 + pad

    return collect(range(λmin, λmax; length=Nλ)), λmin0, λmax0
end

# ============================================================
# Delta L*(x, lambda)
# ============================================================

function DeltaLstar(x, λ; q, ρ, κ, r, β,
                    ε=1e-8,
                    xatol=1e-10,
                    λatol=1e-10,
                    maxiter=200)

    z = zstar(λ; q=q, ρ=ρ, κ=κ, r=r, β=β,
              ε=ε, xatol=xatol, λatol=λatol, maxiter=maxiter)

    f = f_metric2(x, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    g = g_metric2(x, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε)

    return f - λ * g
end

# ============================================================
# One parameter tuple:
# test monotonicity in x for a grid of lambdas
# ============================================================

function scan_one_tuple_DeltaLstar(; q, ρ, κ, β, r,
                                   Nx::Int=2403,
                                   Nλ::Int=41,
                                   Nxλ::Int=401,
                                   λpad_frac::Float64=0.05,
                                   ε::Float64=1e-8,
                                   xatol::Float64=1e-10,
                                   λatol::Float64=1e-10,
                                   maxiter::Int=200,
                                   mono_tol::Float64=1e-12)

    xgrid = uniform01_grid(Nx)

    λgrid, λmin_diag, λmax_diag = lambda_grid_from_diag(
        q=q, ρ=ρ, κ=κ, r=r, β=β,
        Nλ=Nλ, Nxλ=Nxλ, λpad_frac=λpad_frac, ε=ε
    )

    global_min_diff = Inf
    global_λstar = NaN
    global_zstar = NaN
    global_xleft = NaN
    global_xright = NaN

    n_nonfinite = 0
    n_viol_λ = 0

    for λ in λgrid
        z = zstar(λ; q=q, ρ=ρ, κ=κ, r=r, β=β,
                  ε=ε, xatol=xatol, λatol=λatol, maxiter=maxiter)

        yprev = begin
            f0 = f_metric2(xgrid[1], z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
            g0 = g_metric2(xgrid[1], z; q=q, ρ=ρ, κ=κ, β=β, ε=ε)
            f0 - λ * g0
        end

        λ_min_diff = Inf
        λ_has_violation = false

        @inbounds for i in 1:length(xgrid)-1
            xL = xgrid[i]
            xR = xgrid[i + 1]

            fR = f_metric2(xR, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
            gR = g_metric2(xR, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε)
            yR = fR - λ * gR

            d = yR - yprev

            if !isfinite(d)
                n_nonfinite += 1
            else
                if d < λ_min_diff
                    λ_min_diff = d
                end
                if d < global_min_diff
                    global_min_diff = d
                    global_λstar = λ
                    global_zstar = z
                    global_xleft = xL
                    global_xright = xR
                end
                if d < -mono_tol
                    λ_has_violation = true
                end
            end

            yprev = yR
        end

        if λ_has_violation
            n_viol_λ += 1
        end
    end

    mono_pass = isfinite(global_min_diff) && global_min_diff >= -mono_tol

    return (
        λmin_diag = λmin_diag,
        λmax_diag = λmax_diag,
        λgrid_min = first(λgrid),
        λgrid_max = last(λgrid),
        Nλ = length(λgrid),

        min_diff = global_min_diff,
        λstar = global_λstar,
        zstar_at_λstar = global_zstar,
        xleft = global_xleft,
        xright = global_xright,

        n_nonfinite = n_nonfinite,
        n_viol_λ = n_viol_λ,
        mono_pass = mono_pass,
    )
end

# ============================================================
# Threaded outer scan
# ============================================================

function threaded_scan_DeltaLstar(params; r,
                                  Nx::Int=2403,
                                  Nλ::Int=41,
                                  Nxλ::Int=401,
                                  λpad_frac::Float64=0.05,
                                  ε::Float64=1e-8,
                                  xatol::Float64=1e-10,
                                  λatol::Float64=1e-10,
                                  maxiter::Int=200,
                                  mono_tol::Float64=1e-12)

    results = Vector{NamedTuple}(undef, length(params))

    Threads.@threads :dynamic for i in eachindex(params)
        p = params[i]
        s = scan_one_tuple_DeltaLstar(; p..., r=r,
                                      Nx=Nx, Nλ=Nλ, Nxλ=Nxλ,
                                      λpad_frac=λpad_frac,
                                      ε=ε, xatol=xatol, λatol=λatol,
                                      maxiter=maxiter, mono_tol=mono_tol)
        results[i] = merge(p, s)
    end

    return results
end

# ============================================================
# Chunked run over the same parameter grid as before
# ============================================================

function serious_run_DeltaLstar_chunked(;
    r=1.0,
    Nq=14, Nα=14, Nκ=14,
    Nx=2403,              # same number of x-points as previous PCLI2 test
    Nλ=41,                # start moderate; increase later if needed
    Nxλ=401,              # grid size used to build the lambda range
    λpad_frac=0.05,
    ε=1e-8,
    xatol=1e-10,
    λatol=1e-10,
    maxiter=200,
    mono_tol=1e-12,
    chunk_size=256,       # smaller chunk_size is safer here
    qmin=0.05, qmax=0.95,
    αmin=0.10, αmax=0.90,
    κmin=0.05, κmax=0.95,
    outdir="serious_run_DeltaLstar_14x14x14x11_Nx2403"
)

    println("Threads.nthreads() = ", Threads.nthreads())

    qgrid = collect(range(qmin, qmax; length=Nq))
    αgrid = collect(range(αmin, αmax; length=Nα))
    κgrid = collect(range(κmin, κmax; length=Nκ))
    βgrid = beta_grid_11()

    params = make_params(qgrid, αgrid, κgrid, βgrid)

    nparams = length(params)
    nominal_total_evals = nparams * Nx * Nλ
    nchunks = cld(nparams, chunk_size)

    mkpath(outdir)

    CSV.write(joinpath(outdir, "beta_grid.csv"), DataFrame(β=βgrid))
    CSV.write(joinpath(outdir, "q_grid.csv"), DataFrame(q=qgrid))
    CSV.write(joinpath(outdir, "alpha_grid.csv"), DataFrame(α=αgrid))
    CSV.write(joinpath(outdir, "kappa_grid.csv"), DataFrame(κ=κgrid))

    open(joinpath(outdir, "run_info.txt"), "w") do io
        println(io, "Threads.nthreads() = ", Threads.nthreads())
        println(io, "r = ", r)
        println(io, "Nq = ", Nq)
        println(io, "Nα = ", Nα)
        println(io, "Nκ = ", Nκ)
        println(io, "Nβ = ", length(βgrid))
        println(io, "Nx = ", Nx)
        println(io, "Nλ = ", Nλ)
        println(io, "Nxλ = ", Nxλ)
        println(io, "λpad_frac = ", λpad_frac)
        println(io, "ε = ", ε)
        println(io, "xatol = ", xatol)
        println(io, "λatol = ", λatol)
        println(io, "maxiter = ", maxiter)
        println(io, "mono_tol = ", mono_tol)
        println(io, "chunk_size = ", chunk_size)
        println(io, "parameter tuples = ", nparams)
        println(io, "nominal total ΔL* evaluations = ", nominal_total_evals)
    end

    println("parameter tuples = ", nparams)
    println("Nx = ", Nx, ", Nλ = ", Nλ, ", Nxλ = ", Nxλ)
    println("nominal total ΔL* evaluations = ", nominal_total_evals)
    println("number of chunks = ", nchunks)
    println("output directory = ", outdir)
    println("β grid = ", βgrid)

    all_results = NamedTuple[]
    global_min_diff = Inf

    t0_total = time()

    for c in 1:nchunks
        lo = (c - 1) * chunk_size + 1
        hi = min(c * chunk_size, nparams)
        params_chunk = params[lo:hi]

        GC.gc()
        t0_chunk = time()
        res = threaded_scan_DeltaLstar(params_chunk;
                                       r=r,
                                       Nx=Nx, Nλ=Nλ, Nxλ=Nxλ,
                                       λpad_frac=λpad_frac,
                                       ε=ε, xatol=xatol, λatol=λatol,
                                       maxiter=maxiter, mono_tol=mono_tol)
        t_chunk = time() - t0_chunk

        df_chunk = DataFrame(res)
        sort!(df_chunk, :min_diff)

        chunkfile = joinpath(outdir, @sprintf("chunk_%03d.csv", c))
        CSV.write(chunkfile, df_chunk)

        append!(all_results, res)

        if df_chunk.min_diff[1] < global_min_diff
            global_min_diff = df_chunk.min_diff[1]
        end

        elapsed = time() - t0_total
        frac_done = hi / nparams
        eta = elapsed * (1 - frac_done) / frac_done

        println(@sprintf(
            "[%s] chunk %3d/%3d | tuples %6d:%6d | chunk %.1f s | elapsed %.2f h | ETA %.2f h | global min_diff = %.6e",
            Dates.format(now(), "HH:MM:SS"), c, nchunks, lo, hi, t_chunk,
            elapsed / 3600, eta / 3600, global_min_diff
        ))
    end

    df_all = DataFrame(all_results)
    sort!(df_all, :min_diff)

    CSV.write(joinpath(outdir, "all_results_sorted_by_min_diff.csv"), df_all)
    CSV.write(joinpath(outdir, "worst_200.csv"),
              first(df_all, min(200, nrow(df_all))))

    println("\nFinished.")
    println("Worst case:")
    println(df_all[1, [:q, :ρ, :κ, :β, :min_diff, :λstar, :zstar_at_λstar, :xleft, :xright, :n_viol_λ]])

    return df_all
end

In [ ]:
df_DL = serious_run_DeltaLstar_chunked(
    r=1.0,
    Nq=14, Nα=14, Nκ=14,
    Nx=2403,
    Nλ=41,
    Nxλ=401,
    λpad_frac=0.05,
    ε=1e-8,
    xatol=1e-10,
    λatol=1e-10,
    maxiter=200,
    mono_tol=1e-12,
    chunk_size=256,
    outdir="serious_run_DeltaLstar_14x14x14x11_Nx2403_Nλ41"
)

In [ ]:
using Random
using Statistics
using Distributions
using Base.Threads

"""
    simulate_multi_project_policies(x_vec, q_vec, rho_vec, kappa_vec, r_vec, beta, M;
                                    T=100, nruns=1000,
                                    conf_level=0.95,
                                    seed=nothing, ε=1e-10)

Monte Carlo estimation of the approximate long-run average discounted reward
per project for the current ACK/NACK sensing model under four policies:

  1. Random policy,
  2. Myopic policy,
  3. Round-robin policy,
  4. MP-index policy,

together with the Lagrangian upper bound obtained from

    λstar, Lmin = lambda_star_vector(...)

The initial beliefs are fixed at X_i(0)=x_vec[i] for all runs.
"""
function simulate_multi_project_policies(x_vec::AbstractVector{<:Real},
                                         q_vec::AbstractVector{<:Real},
                                         rho_vec::AbstractVector{<:Real},
                                         kappa_vec::AbstractVector{<:Real},
                                         r_vec::AbstractVector{<:Real},
                                         beta::Float64,
                                         M::Integer;
                                         T::Int = 100,
                                         nruns::Int = 1_000,
                                         conf_level::Float64 = 0.95,
                                         seed::Union{Int,Nothing} = nothing,
                                         ε::Float64 = 1e-10)

    N = length(x_vec)
    @assert length(q_vec)     == N "q_vec must have the same length as x_vec"
    @assert length(rho_vec)   == N "rho_vec must have the same length as x_vec"
    @assert length(kappa_vec) == N "kappa_vec must have the same length as x_vec"
    @assert length(r_vec)     == N "r_vec must have the same length as x_vec"
    @assert 0.0 < beta < 1.0  "beta must be in (0,1)"
    @assert 1 <= M <= N       "M must satisfy 1 ≤ M ≤ N"
    @assert T > 0             "T must be positive"
    @assert nruns > 0         "nruns must be positive"

    x_vec_f     = Float64.(x_vec)
    q_vec_f     = Float64.(q_vec)
    rho_vec_f   = Float64.(rho_vec)
    kappa_vec_f = Float64.(kappa_vec)
    r_vec_f     = Float64.(r_vec)

    for i in 1:N
        @assert 0.0 <= x_vec_f[i] <= 1.0 "Each x_i must satisfy 0 ≤ x_i ≤ 1"
        @assert 0.0 <= q_vec_f[i] < 1.0 "Each q_i must satisfy 0 ≤ q_i < 1"
        @assert 0.0 <= rho_vec_f[i] < 1.0 - q_vec_f[i] "Each rho_i must satisfy 0 ≤ rho_i < 1-q_i"
        @assert 0.0 <= kappa_vec_f[i] <= 1.0 "Each kappa_i must satisfy 0 ≤ kappa_i ≤ 1"
        @assert r_vec_f[i] >= 0.0 "Each r_i must satisfy r_i ≥ 0"
    end

    # ----------------------------------------------------------
    # Lagrangian upper bound (computed once)
    # ----------------------------------------------------------
    λstar, Lmin = lambda_star_vector(
        x_vec_f;
        qvector = q_vec_f,
        ρvector = rho_vec_f,
        κvector = kappa_vec_f,
        rvector = r_vec_f,
        β = beta,
        M = M
    )

    β_powers = [beta^t for t in 0:(T-1)]

    total_random     = zeros(Float64, nruns)
    total_myopic     = zeros(Float64, nruns)
    total_roundrobin = zeros(Float64, nruns)
    total_mpindex    = zeros(Float64, nruns)

    seedgen = seed === nothing ? Random.default_rng() : MersenneTwister(seed)
    run_seeds = rand(seedgen, UInt64, nruns)

    @inline function phi0(x, q, rho)
        q + rho * x
    end

    @inline function phi1(x, q, rho, kappa)
        denom = 1.0 - kappa * x
        if denom <= 0.0
            return q + rho
        end
        post_nack = (1.0 - kappa) * x / denom
        return q + rho * post_nack
    end

    @inline function mp_index_current(x, q, rho, kappa, r, beta, ε)
        fxx = float(f_metric2(x, x; q=q, ρ=rho, κ=kappa, r=r, β=beta, ε=ε))
        gxx = float(g_metric2(x, x; q=q, ρ=rho, κ=kappa,      β=beta, ε=ε))
        return (isfinite(fxx) && isfinite(gxx) && gxx != 0.0) ? (fxx / gxx) : -Inf
    end

    function simulate_one_run!(run_idx::Int)
        rng = MersenneTwister(run_seeds[run_idx])

        # Same initial beliefs in every run
        X0 = copy(x_vec_f)

        X_rand = copy(X0)
        X_myop = copy(X0)
        X_rr   = copy(X0)
        X_mp   = copy(X0)

        rr_start = 1

        R_rand = 0.0
        R_myop = 0.0
        R_rr   = 0.0
        R_mp   = 0.0

        for t in 1:T
            disc = β_powers[t]

            # ----------------------------------------------------------
            # 1. Random policy
            # ----------------------------------------------------------
            active_rand = randperm(rng, N)[1:M]
            active_rand_mask = falses(N)
            active_rand_mask[active_rand] .= true

            reward_t = 0.0
            for i in 1:N
                q  = q_vec_f[i]
                ρ  = rho_vec_f[i]
                κ  = kappa_vec_f[i]
                r  = r_vec_f[i]
                xi = X_rand[i]

                if active_rand_mask[i]
                    p_ack = κ * xi
                    if rand(rng) < p_ack
                        reward_t += r
                        X_rand[i] = q + ρ
                    else
                        X_rand[i] = phi1(xi, q, ρ, κ)
                    end
                else
                    X_rand[i] = phi0(xi, q, ρ)
                end
            end
            R_rand += disc * reward_t

            # ----------------------------------------------------------
            # 2. Myopic policy
            # ----------------------------------------------------------
            myopic_value = similar(X_myop)
            for i in 1:N
                myopic_value[i] = r_vec_f[i] * kappa_vec_f[i] * X_myop[i]
            end

            idx_candidates = findall(i -> myopic_value[i] > 0.0, 1:N)

            active_myop_mask = falses(N)
            if !isempty(idx_candidates)
                if length(idx_candidates) <= M
                    active_myop_mask[idx_candidates] .= true
                else
                    vals = myopic_value[idx_candidates]
                    local_idx = partialsortperm(vals, rev=true, 1:M)
                    chosen = idx_candidates[local_idx]
                    active_myop_mask[chosen] .= true
                end
            end

            reward_t = 0.0
            for i in 1:N
                q  = q_vec_f[i]
                ρ  = rho_vec_f[i]
                κ  = kappa_vec_f[i]
                r  = r_vec_f[i]
                xi = X_myop[i]

                if active_myop_mask[i]
                    p_ack = κ * xi
                    if rand(rng) < p_ack
                        reward_t += r
                        X_myop[i] = q + ρ
                    else
                        X_myop[i] = phi1(xi, q, ρ, κ)
                    end
                else
                    X_myop[i] = phi0(xi, q, ρ)
                end
            end
            R_myop += disc * reward_t

            # ----------------------------------------------------------
            # 3. Round-robin policy
            # ----------------------------------------------------------
            active_rr_mask = falses(N)
            for k in 0:(M-1)
                i = ((rr_start - 1 + k) % N) + 1
                active_rr_mask[i] = true
            end
            rr_start = ((rr_start - 1 + M) % N) + 1

            reward_t = 0.0
            for i in 1:N
                q  = q_vec_f[i]
                ρ  = rho_vec_f[i]
                κ  = kappa_vec_f[i]
                r  = r_vec_f[i]
                xi = X_rr[i]

                if active_rr_mask[i]
                    p_ack = κ * xi
                    if rand(rng) < p_ack
                        reward_t += r
                        X_rr[i] = q + ρ
                    else
                        X_rr[i] = phi1(xi, q, ρ, κ)
                    end
                else
                    X_rr[i] = phi0(xi, q, ρ)
                end
            end
            R_rr += disc * reward_t

            # ----------------------------------------------------------
            # 4. MP-index policy
            # ----------------------------------------------------------
            mpvals = similar(X_mp)
            for i in 1:N
                mpvals[i] = mp_index_current(X_mp[i],
                                             q_vec_f[i], rho_vec_f[i], kappa_vec_f[i],
                                             r_vec_f[i], beta, ε)
            end

            idx_candidates = findall(i -> mpvals[i] >= 0.0, 1:N)

            active_mp_mask = falses(N)
            if !isempty(idx_candidates)
                if length(idx_candidates) <= M
                    active_mp_mask[idx_candidates] .= true
                else
                    vals = mpvals[idx_candidates]
                    local_idx = partialsortperm(vals, rev=true, 1:M)
                    chosen = idx_candidates[local_idx]
                    active_mp_mask[chosen] .= true
                end
            end

            reward_t = 0.0
            for i in 1:N
                q  = q_vec_f[i]
                ρ  = rho_vec_f[i]
                κ  = kappa_vec_f[i]
                r  = r_vec_f[i]
                xi = X_mp[i]

                if active_mp_mask[i]
                    p_ack = κ * xi
                    if rand(rng) < p_ack
                        reward_t += r
                        X_mp[i] = q + ρ
                    else
                        X_mp[i] = phi1(xi, q, ρ, κ)
                    end
                else
                    X_mp[i] = phi0(xi, q, ρ)
                end
            end
            R_mp += disc * reward_t
        end

        total_random[run_idx]     = R_rand
        total_myopic[run_idx]     = R_myop
        total_roundrobin[run_idx] = R_rr
        total_mpindex[run_idx]    = R_mp
        return nothing
    end

    if Threads.nthreads() > 1
        @threads for run_idx in 1:nruns
            simulate_one_run!(run_idx)
        end
    else
        for run_idx in 1:nruns
            simulate_one_run!(run_idx)
        end
    end

    # Normalize to approximate long-run average discounted reward per project
    norm_factor = (1.0 - beta) / N

    function summarize(raw_samples::Vector{Float64})
        samples = norm_factor .* raw_samples
        μ  = mean(samples)
        s  = std(samples)
        se = s / sqrt(length(samples))
        z  = quantile(Normal(), 0.5 + conf_level / 2)
        ci_low  = μ - z * se
        ci_high = μ + z * se
        return (mean = μ,
                stderr = se,
                ci_low = ci_low,
                ci_high = ci_high,
                nruns = length(samples))
    end

    results = Dict{String,NamedTuple}()

    results["random"]      = summarize(total_random)
    results["myopic"]      = summarize(total_myopic)
    results["round_robin"] = summarize(total_roundrobin)
    results["mp_index"]    = summarize(total_mpindex)

    # Deterministic Lagrangian upper bound, normalized the same way
    lagrangian_bound = norm_factor * Lmin
    results["lagrangian_bound"] = (
        lambda_star = λstar,
        raw_Lmin = Lmin,
        mean = lagrangian_bound,
        stderr = 0.0,
        ci_low = lagrangian_bound,
        ci_high = lagrangian_bound,
        nruns = 1
    )

    return results
end

In [ ]:
using Random
using Statistics
using Distributions
using Base.Threads
using LinearAlgebra

BLAS.set_num_threads(1)

struct MPIndexLookupUniform
    xlo::Float64
    xhi::Float64
    h::Float64
    vals::Vector{Float64}
    mode::Symbol   # :leftbin, :nearest, :linear
end

"""
    build_mp_index_lookup(; q, ρ, κ, r, β, ε=1e-10,
                             ngrid=2001, xlo=0.0, xhi=1.0,
                             mode=:linear)

Precompute m(x)=f(x,x)/g(x,x) on a uniform grid of x-values.
"""
function build_mp_index_lookup(; q, ρ, κ, r, β,
                                 ε::Float64 = 1e-10,
                                 ngrid::Int = 2001,
                                 xlo::Float64 = 0.0,
                                 xhi::Float64 = 1.0,
                                 mode::Symbol = :linear)

    @assert ngrid >= 2
    @assert xlo < xhi
    @assert mode in (:leftbin, :nearest, :linear)

    xgrid = collect(range(xlo, xhi; length=ngrid))
    vals = Vector{Float64}(undef, ngrid)

    for i in eachindex(xgrid)
        x = xgrid[i]
        fxx = float(f_metric2(x, x; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε))
        gxx = float(g_metric2(x, x; q=q, ρ=ρ, κ=κ,      β=β, ε=ε))
        vals[i] = (isfinite(fxx) && isfinite(gxx) && gxx != 0.0) ? (fxx / gxx) : -Inf
    end

    return MPIndexLookupUniform(xlo, xhi, (xhi - xlo) / (ngrid - 1), vals, mode)
end

"""
    mp_index_lookup(L, x)

Approximate m(x) from a precomputed uniform lookup table.
"""
@inline function mp_index_lookup(L::MPIndexLookupUniform, x::Real)
    xf = clamp(Float64(x), L.xlo, L.xhi)
    vals = L.vals
    n = length(vals)

    if xf <= L.xlo
        return vals[1]
    elseif xf >= L.xhi
        return vals[end]
    end

    t = (xf - L.xlo) / L.h
    i = floor(Int, t) + 1
    i = clamp(i, 1, n - 1)

    if L.mode === :leftbin
        return vals[i]
    elseif L.mode === :nearest
        α = t - floor(t)
        return α < 0.5 ? vals[i] : vals[i + 1]
    else
        α = t - floor(t)
        return (1 - α) * vals[i] + α * vals[i + 1]
    end
end

In [ ]:
using Random
using Statistics
using Distributions
using Base.Threads
using Printf
using LinearAlgebra
BLAS.set_num_threads(1)

# ============================================================
# Fast simulator for identical projects
#
# Policies simulated exactly:
#   1. random
#   2. myopic
#   3. MP-index
#
# plus the deterministic Lagrangian upper bound
# ============================================================

"""
    simulate_identical_project_policies_fast(
        N, M, x0, q, rho, kappa, r, beta;
        T=100, nruns=1000, conf_level=0.95,
        seed=nothing, ε=1e-10
    )

Fast Monte Carlo simulator for the identical-project setting.

All N projects have:
- common initial belief x0,
- common parameters (q, rho, kappa, r).

The state of each policy is represented by a dictionary
    counts[x] = number of projects currently with belief x.

This exploits exchangeability and caches MP index values by belief x.
"""


using Random
using Statistics
using Distributions
using Base.Threads
using LinearAlgebra

BLAS.set_num_threads(1)

function simulate_identical_project_policies_fast(
    N::Integer,
    M::Integer,
    x0::Real,
    q::Real,
    rho::Real,
    kappa::Real,
    r::Real,
    beta::Float64;
    T::Int = 100,
    nruns::Int = 1_000,
    conf_level::Float64 = 0.95,
    seed::Union{Int,Nothing} = nothing,
    ε::Float64 = 1e-10
)

    @assert N >= 1               "N must be positive"
    @assert 1 <= M <= N          "M must satisfy 1 ≤ M ≤ N"
    @assert 0.0 <= x0 <= 1.0     "x0 must satisfy 0 ≤ x0 ≤ 1"
    @assert 0.0 <= q < 1.0       "q must satisfy 0 ≤ q < 1"
    @assert 0.0 <= rho < 1.0 - q "rho must satisfy 0 ≤ rho < 1-q"
    @assert 0.0 <= kappa <= 1.0  "kappa must satisfy 0 ≤ kappa ≤ 1"
    @assert r >= 0.0             "r must satisfy r ≥ 0"
    @assert 0.0 < beta < 1.0     "beta must be in (0,1)"
    @assert T > 0                "T must be positive"
    @assert nruns > 0            "nruns must be positive"

    x0f    = Float64(x0)
    qf     = Float64(q)
    rhof   = Float64(rho)
    kappaf = Float64(kappa)
    rf     = Float64(r)

    β_powers = [beta^t for t in 0:(T-1)]
    x_ack = qf + rhof

    # Lagrangian upper bound (computed once)
    x_vec     = fill(x0f, N)
    q_vec     = fill(qf, N)
    rho_vec   = fill(rhof, N)
    kappa_vec = fill(kappaf, N)
    r_vec     = fill(rf, N)

    λstar, Lmin = lambda_star_vector(
        x_vec;
        qvector = q_vec,
        ρvector = rho_vec,
        κvector = kappa_vec,
        rvector = r_vec,
        β = beta,
        M = M
    )

    total_random  = zeros(Float64, nruns)
    total_myopic  = zeros(Float64, nruns)
    total_mpindex = zeros(Float64, nruns)

    seedgen = seed === nothing ? Random.default_rng() : MersenneTwister(seed)
    run_seeds = rand(seedgen, UInt64, nruns)

    @inline function addcount!(d::Dict{Float64,Int}, x::Float64, n::Int)
        n <= 0 && return nothing
        d[x] = get(d, x, 0) + n
        return nothing
    end

    @inline function get_transitions!(cache::Dict{Float64,NTuple{3,Float64}}, x::Float64)
        get!(cache, x) do
            p_ack = kappaf * x
            x_pass = qf + rhof * x

            denom = 1.0 - kappaf * x
            x_nack = if denom <= 0.0
                qf + rhof
            else
                qf + rhof * ((1.0 - kappaf) * x / denom)
            end

            (p_ack, x_pass, x_nack)
        end
    end

    @inline function get_mpindex!(cache::Dict{Float64,Float64}, x::Float64)
        get!(cache, x) do
            fxx = float(f_metric2(x, x; q=qf, ρ=rhof, κ=kappaf, r=rf, β=beta, ε=ε))
            gxx = float(g_metric2(x, x; q=qf, ρ=rhof, κ=kappaf,      β=beta, ε=ε))
            (isfinite(fxx) && isfinite(gxx) && gxx != 0.0) ? (fxx / gxx) : -Inf
        end
    end

    function apply_active_counts!(
        rng::AbstractRNG,
        pairs::Vector{Pair{Float64,Int}},
        active_counts::Vector{Int},
        trans_cache::Dict{Float64,NTuple{3,Float64}}
    )
        new_counts = Dict{Float64,Int}()
        total_acks = 0

        @inbounds for j in eachindex(pairs)
            x = first(pairs[j])
            n = last(pairs[j])
            a = active_counts[j]
            p = n - a

            p_ack, x_pass, x_nack = get_transitions!(trans_cache, x)

            if a > 0
                n_ack = rand(rng, Binomial(a, p_ack))
                n_nack = a - n_ack
                total_acks += n_ack

                addcount!(new_counts, x_ack, n_ack)
                addcount!(new_counts, x_nack, n_nack)
            end

            if p > 0
                addcount!(new_counts, x_pass, p)
            end
        end

        return new_counts, rf * total_acks
    end

    function step_random!(
        rng::AbstractRNG,
        counts::Dict{Float64,Int},
        trans_cache::Dict{Float64,NTuple{3,Float64}}
    )
        pairs = collect(counts)
        J = length(pairs)
        active_counts = zeros(Int, J)

        rem_pop = N
        rem_draw = M

        if J == 1
            active_counts[1] = M
        else
            for j in 1:(J-1)
                n = last(pairs[j])

                a = if rem_draw == 0
                    0
                elseif rem_draw == rem_pop
                    n
                else
                    rand(rng, Hypergeometric(n, rem_pop - n, rem_draw))
                end

                active_counts[j] = a
                rem_draw -= a
                rem_pop  -= n
            end
            active_counts[J] = rem_draw
        end

        return apply_active_counts!(rng, pairs, active_counts, trans_cache)
    end

    function step_myopic!(
        rng::AbstractRNG,
        counts::Dict{Float64,Int},
        trans_cache::Dict{Float64,NTuple{3,Float64}}
    )
        pairs = collect(counts)
        sort!(pairs; by = p -> first(p), rev = true)

        active_counts = zeros(Int, length(pairs))
        rem = M

        @inbounds for j in eachindex(pairs)
            rem == 0 && break
            x = first(pairs[j])
            n = last(pairs[j])

            if x > 0.0
                a = min(n, rem)
                active_counts[j] = a
                rem -= a
            else
                break
            end
        end

        return apply_active_counts!(rng, pairs, active_counts, trans_cache)
    end

    function step_mpindex!(
        rng::AbstractRNG,
        counts::Dict{Float64,Int},
        trans_cache::Dict{Float64,NTuple{3,Float64}},
        mp_cache::Dict{Float64,Float64}
    )
        pairs = collect(counts)
        J = length(pairs)

        mpvals = Vector{Float64}(undef, J)
        @inbounds for j in 1:J
            mpvals[j] = get_mpindex!(mp_cache, first(pairs[j]))
        end

        order = sortperm(mpvals; rev = true)

        active_counts = zeros(Int, J)
        rem = M

        @inbounds for idx in order
            rem == 0 && break
            v = mpvals[idx]
            if v >= 0.0
                n = last(pairs[idx])
                a = min(n, rem)
                active_counts[idx] = a
                rem -= a
            else
                break
            end
        end

        return apply_active_counts!(rng, pairs, active_counts, trans_cache)
    end

    function simulate_one_run!(run_idx::Int)
        rng = MersenneTwister(run_seeds[run_idx])

        # Local caches for this run
        trans_cache = Dict{Float64,NTuple{3,Float64}}()
        mp_cache    = Dict{Float64,Float64}()

        counts_rand = Dict{Float64,Int}(x0f => N)
        counts_myop = Dict{Float64,Int}(x0f => N)
        counts_mp   = Dict{Float64,Int}(x0f => N)

        R_rand = 0.0
        R_myop = 0.0
        R_mp   = 0.0

        for t in 1:T
            disc = β_powers[t]

            counts_rand, rew = step_random!(rng, counts_rand, trans_cache)
            R_rand += disc * rew

            counts_myop, rew = step_myopic!(rng, counts_myop, trans_cache)
            R_myop += disc * rew

            counts_mp, rew = step_mpindex!(rng, counts_mp, trans_cache, mp_cache)
            R_mp += disc * rew
        end

        total_random[run_idx]  = R_rand
        total_myopic[run_idx]  = R_myop
        total_mpindex[run_idx] = R_mp
        return nothing
    end

    if Threads.nthreads() > 1
        Threads.@threads for run_idx in 1:nruns
            simulate_one_run!(run_idx)
        end
    else
        for run_idx in 1:nruns
            simulate_one_run!(run_idx)
        end
    end

    norm_factor = (1.0 - beta) / N

    function summarize(raw_samples::Vector{Float64})
        samples = norm_factor .* raw_samples
        μ  = mean(samples)
        s  = std(samples)
        se = s / sqrt(length(samples))
        z  = quantile(Normal(), 0.5 + conf_level / 2)
        ci_low  = μ - z * se
        ci_high = μ + z * se
        return (
            mean = μ,
            stderr = se,
            ci_low = ci_low,
            ci_high = ci_high,
            nruns = length(samples)
        )
    end

    results = Dict{String,NamedTuple}()

    results["random"]   = summarize(total_random)
    results["myopic"]   = summarize(total_myopic)
    results["mp_index"] = summarize(total_mpindex)

    lagrangian_bound = norm_factor * Lmin
    results["lagrangian_bound"] = (
        lambda_star = λstar,
        raw_Lmin = Lmin,
        mean = lagrangian_bound,
        stderr = 0.0,
        ci_low = lagrangian_bound,
        ci_high = lagrangian_bound,
        nruns = 1
    )

    return results
end

# ============================================================
# Scaling experiment driver
# ============================================================

"""
    run_identical_scaling_experiment(; ...)

Runs the identical-project asymptotic-optimality experiment for a list of N values,
with M scaled in fixed proportion to N.
"""
function run_identical_scaling_experiment(;
    N_values = [50, 100, 200, 400, 800],
    αcap = 0.10,
    beta = 0.99,
    x0 = 0.50,
    q = 0.01,
    rho = 0.90,
    kappa = 0.70,
    r = 1.0,
    T_policy = 200,
    nruns_policy = 1000,
    conf_level = 0.95,
    seed = 1234,
    ε = 1e-10,
    mp_lookup_ngrid = 2001,
    mp_lookup_mode = :linear
)
    # Build once and reuse
    mp_lookup = build_mp_index_lookup(
        q = q, ρ = rho, κ = kappa, r = r, β = beta,
        ε = ε,
        ngrid = mp_lookup_ngrid,
        mode = mp_lookup_mode
    )

    rows = NamedTuple[]

    for (j, N) in enumerate(N_values)
        M = max(1, min(N, round(Int, αcap * N)))

        println("Running N = $N, M = $M ...")

        results = simulate_identical_project_policies_fast(
            N, M, x0, q, rho, kappa, r, beta;
            T = T_policy,
            nruns = nruns_policy,
            conf_level = conf_level,
            seed = seed + j,
            ε = ε,
            mp_lookup = mp_lookup
        )

        rand_stats = results["random"]
        myop_stats = results["myopic"]
        mp_stats   = results["mp_index"]
        ub_stats   = results["lagrangian_bound"]

        ub = ub_stats.mean
        mp = mp_stats.mean

        abs_gap = ub - mp
        rel_gap = ub != 0.0 ? abs_gap / ub : NaN

        push!(rows, (
            N = N,
            M = M,
            capacity_ratio = M / N,
            lambda_star = ub_stats.lambda_star,
            raw_Lmin = ub_stats.raw_Lmin,
            lagrangian_bound = ub,
            random_mean = rand_stats.mean,
            myopic_mean = myop_stats.mean,
            mp_index_mean = mp_stats.mean,
            mp_index_ci_low = mp_stats.ci_low,
            mp_index_ci_high = mp_stats.ci_high,
            abs_gap_mp_to_bound = abs_gap,
            rel_gap_mp_to_bound = rel_gap
        ))
    end

    return rows
end

In [ ]:
using DataFrames
using CSV
using Printf
using PGFPlotsX

function scaling_rows_to_dataframe(rows)
    df = DataFrame(rows)
    sort!(df, :N)

    df.mp_index_ci_halfwidth =
        (df.mp_index_ci_high .- df.mp_index_ci_low) ./ 2

    df.abs_gap_ci_low  =
        df.lagrangian_bound .- df.mp_index_ci_high
    df.abs_gap_ci_high =
        df.lagrangian_bound .- df.mp_index_ci_low

    df.rel_gap_ci_low  =
        df.abs_gap_ci_low ./ df.lagrangian_bound
    df.rel_gap_ci_high =
        df.abs_gap_ci_high ./ df.lagrangian_bound

    # New scaled-gap columns
    df.N_abs_gap_mp_to_bound      = df.N .* df.abs_gap_mp_to_bound
    df.sqrtN_abs_gap_mp_to_bound  = sqrt.(Float64.(df.N)) .* df.abs_gap_mp_to_bound

    df.N_abs_gap_ci_low           = df.N .* df.abs_gap_ci_low
    df.N_abs_gap_ci_high          = df.N .* df.abs_gap_ci_high

    df.sqrtN_abs_gap_ci_low       = sqrt.(Float64.(df.N)) .* df.abs_gap_ci_low
    df.sqrtN_abs_gap_ci_high      = sqrt.(Float64.(df.N)) .* df.abs_gap_ci_high

    return df
end

function print_scaling_summary(df::DataFrame; conf_level::Float64=0.95)
    ci_pct = Int(round(conf_level * 100))

    println("Identical-project scaling experiment")
    println()

    for row in eachrow(df)
        println("--------------------------------------------------")
        println("N = ", row.N, ", M = ", row.M,
                " (M/N = ", @sprintf("%.4f", row.capacity_ratio), ")")
        println("lambda_star        = ", row.lambda_star)
        println("lagrangian bound   = ", row.lagrangian_bound)
        println("random mean        = ", row.random_mean)
        println("myopic mean        = ", row.myopic_mean)
        println("MP-index mean      = ", row.mp_index_mean)
        println("MP-index ", ci_pct, "% CI = [",
                row.mp_index_ci_low, ", ", row.mp_index_ci_high, "]")
        println("abs gap to bound   = ", row.abs_gap_mp_to_bound)
        println("rel gap to bound   = ", row.rel_gap_mp_to_bound)
        println()
    end
end

function save_scaling_results(df::DataFrame; csvfile::AbstractString="identical_scaling_results.csv")
    CSV.write(csvfile, df)
    println("Saved results to: ", csvfile)
    return csvfile
end

# -----------------------------------------------------------
# PGFPlotsX helpers
# -----------------------------------------------------------

_pgf_xmode(xscale) = xscale == :log10 ? "log" : "linear"

function _legend_style(; pos="north east")
    return (
        "legend pos" => pos,
        "legend cell align" => "left",
        "font" => raw"\small",
    )
end

function scaling_performance_axis(df::DataFrame;
                                  xscale=:identity,
                                  width="11cm",
                                  height="7cm",
                                  legend_pos="south east")

    N = Float64.(df.N)

    ax = @pgf Axis(
        {
            width = width,
            height = height,
            xlabel = raw"$N$",
            ylabel = raw"normalized reward per project",
            xmode = _pgf_xmode(xscale),
            ymajorgrids,
            xmajorgrids,
            "legend pos" = legend_pos,
            "legend cell align" = "left",
            "font" = raw"\small",
            "mark size" = "2.2pt"
        },

        Plot(
            {
                color = "black",
                mark = "square*",
                "thick"
            },
            Table(x = N, y = df.lagrangian_bound)
        ),
        LegendEntry("Lagrangian bound"),

        Plot(
            {
                color = "blue",
                mark = "*",
                "thick"
            },
            Table(x = N, y = df.mp_index_mean)
        ),
        LegendEntry("MP-index"),
        Plot(
            {
                color = "blue",
                style = "dashed",
                "thin"
            },
            Table(x = N, y = df.mp_index_ci_low)
        ),
        LegendEntry("MP-index CI bounds"),

        Plot(
            {
                color = "blue",
                style = "dashed",
                "thin",
                "forget plot"
            },
            Table(x = N, y = df.mp_index_ci_high)
        ),

        Plot(
            {
                color = "red",
                mark = "triangle*",
                "thick"
            },
            Table(x = N, y = df.myopic_mean)
        ),
        LegendEntry("Myopic"),

        Plot(
            {
                color = "gray",
                mark = "o",
                "thick"
            },
            Table(x = N, y = df.random_mean)
        ),
        LegendEntry("Random")
    )

    return ax
end

function scaling_abs_gap_axis(df::DataFrame;
                              xscale=:identity,
                              width="11cm",
                              height="7cm",
                              legend_pos="north east")

    N = Float64.(df.N)

    ax = @pgf Axis(
        {
            width = width,
            height = height,
            xlabel = raw"$N$",
            ylabel = raw"absolute gap to bound",
            xmode = _pgf_xmode(xscale),
            ymajorgrids,
            xmajorgrids,
            "legend pos" = legend_pos,
            "legend cell align" = "left",
            "font" = raw"\small",
            "mark size" = "2.2pt"
        },

        Plot(
            {
                color = "blue",
                mark = "*",
                "thick"
            },
            Table(x = N, y = df.abs_gap_mp_to_bound)
        ),
        LegendEntry("Absolute gap"),

        Plot(
            {
                color = "blue",
                style = "dashed",
                "thin"
            },
            Table(x = N, y = df.abs_gap_ci_low)
        ),
        LegendEntry("Gap CI bounds"),

        Plot(
            {
                color = "blue",
                style = "dashed",
                "thin",
                "forget plot"
            },
            Table(x = N, y = df.abs_gap_ci_high)
        )
    )

    return ax
end

function scaling_rel_gap_axis(df::DataFrame;
                              xscale=:identity,
                              width="11cm",
                              height="7cm",
                              legend_pos="north east")

    N = Float64.(df.N)

    ax = @pgf Axis(
        {
            width = width,
            height = height,
            xlabel = raw"$N$",
            ylabel = raw"relative gap to bound",
            xmode = _pgf_xmode(xscale),
            ymajorgrids,
            xmajorgrids,
            "legend pos" = legend_pos,
            "legend cell align" = "left",
            "font" = raw"\small",
            "mark size" = "2.2pt"
        },

        Plot(
            {
                color = "blue",
                mark = "*",
                "thick"
            },
            Table(x = N, y = df.rel_gap_mp_to_bound)
        ),
        LegendEntry("Relative gap"),

        Plot(
            {
                color = "blue",
                style = "dashed",
                "thin"
            },
            Table(x = N, y = df.rel_gap_ci_low)
        ),
        LegendEntry("Gap CI bounds"),
        Plot(
            {
                color = "blue",
                style = "dashed",
                "thin",
                "forget plot"
            },
            Table(x = N, y = df.rel_gap_ci_high)
        )
    )

    return ax
end


function save_scaling_pgfplots(df::DataFrame;
                               prefix::AbstractString="identical_scaling",
                               xscale=:identity,
                               width="11cm",
                               height="7cm")

    ax_perf  = scaling_performance_axis(df; xscale=xscale, width=width, height=height)
    ax_abs   = scaling_abs_gap_axis(df; xscale=xscale, width=width, height=height)
    ax_rel   = scaling_rel_gap_axis(df; xscale=xscale, width=width, height=height)
    ax_Ngap  = scaling_N_gap_axis(df; xscale=xscale, width=width, height=height)
    ax_sNgap = scaling_sqrtN_gap_axis(df; xscale=xscale, width=width, height=height)

    PGFPlotsX.pgfsave(prefix * "_performance.tex", ax_perf)
    PGFPlotsX.pgfsave(prefix * "_abs_gap.tex", ax_abs)
    PGFPlotsX.pgfsave(prefix * "_rel_gap.tex", ax_rel)
    PGFPlotsX.pgfsave(prefix * "_N_gap.tex", ax_Ngap)
    PGFPlotsX.pgfsave(prefix * "_sqrtN_gap.tex", ax_sNgap)

    return (
        performance = ax_perf,
        abs_gap = ax_abs,
        rel_gap = ax_rel,
        N_gap = ax_Ngap,
        sqrtN_gap = ax_sNgap
    )
end

function scaling_N_gap_axis(df::DataFrame;
                            xscale=:identity,
                            width="11cm",
                            height="7cm",
                            legend_pos="north east")

    Nvals = Float64.(df.N)

    ax = @pgf Axis(
        {
            width = width,
            height = height,
            xlabel = raw"$N$",
            ylabel = raw"$N\,(\mathrm{bound}-\mathrm{MP})$",
            xmode = _pgf_xmode(xscale),
            ymajorgrids,
            xmajorgrids,
            "legend pos" = legend_pos,
            "legend cell align" = "left",
            "font" = raw"\small",
            "mark size" = "2.2pt"
        },

        Plot(
            {
                color = "blue",
                mark = "*",
                "thick"
            },
            Table(x = Nvals, y = df.N_abs_gap_mp_to_bound)
        ),
        LegendEntry(raw"$N(\mathrm{bound}-\mathrm{MP})$"),

        Plot(
            {
                color = "blue",
                style = "dashed",
                "thin"
            },
            Table(x = Nvals, y = df.N_abs_gap_ci_low)
        ),
        LegendEntry("CI bounds"),

        Plot(
            {
                color = "blue",
                style = "dashed",
                "thin",
                "forget plot"
            },
            Table(x = Nvals, y = df.N_abs_gap_ci_high)
        )
    )

    return ax
end

function scaling_sqrtN_gap_axis(df::DataFrame;
                                xscale=:identity,
                                width="11cm",
                                height="7cm",
                                legend_pos="north east")

    Nvals = Float64.(df.N)

    ax = @pgf Axis(
        {
            width = width,
            height = height,
            xlabel = raw"$N$",
            ylabel = raw"$\sqrt{N}\,(\mathrm{bound}-\mathrm{MP})$",
            xmode = _pgf_xmode(xscale),
            ymajorgrids,
            xmajorgrids,
            "legend pos" = legend_pos,
            "legend cell align" = "left",
            "font" = raw"\small",
            "mark size" = "2.2pt"
        },

        Plot(
            {
                color = "blue",
                mark = "*",
                "thick"
            },
            Table(x = Nvals, y = df.sqrtN_abs_gap_mp_to_bound)
        ),
        LegendEntry(raw"$\sqrt{N}(\mathrm{bound}-\mathrm{MP})$"),

        Plot(
            {
                color = "blue",
                style = "dashed",
                "thin"
            },
            Table(x = Nvals, y = df.sqrtN_abs_gap_ci_low)
        ),
        LegendEntry("CI bounds"),
        Plot(
            {
                color = "blue",
                style = "dashed",
                "thin",
                "forget plot"
            },
            Table(x = Nvals, y = df.sqrtN_abs_gap_ci_high)
        )
    )

    return ax
end

In [ ]:
rows = run_identical_scaling_experiment(
    N_values = [50, 100, 200, 400],
    αcap = 0.10,
    beta = 0.99,
    x0 = 0.50,
    q = 0.01,
    rho = 0.90,
    kappa = 0.70,
    r = 1.0,
    T_policy = 200,
    nruns_policy = 1000,
    seed = 1234,
    mp_lookup_ngrid = 1001,   # start here
    mp_lookup_mode = :linear
)

df = scaling_rows_to_dataframe(rows)

print_scaling_summary(df)
save_scaling_results(df; csvfile="identical_scaling_results.csv")

axs = save_scaling_pgfplots(
    df;
    prefix = "identical_scaling",
    xscale = :log10,
    width = "12cm",
    height = "7cm"
)

# Optional: inspect the most relevant columns in tabular form
select(df,
    :N, :M, :capacity_ratio,
    :lagrangian_bound, :mp_index_mean,
    :abs_gap_mp_to_bound, :rel_gap_mp_to_bound,
    :N_abs_gap_mp_to_bound, :sqrtN_abs_gap_mp_to_bound
)

# In Jupyter, display the PGFPlotsX axes directly:
axs.performance
axs.abs_gap
axs.rel_gap
axs.N_gap
axs.sqrtN_gap

In [ ]:
println("Threads.nthreads()      = ", Threads.nthreads())
println("Threads.nthreads(:default) = ", Threads.nthreads(:default))
println("Threads.nthreads(:interactive) = ", Threads.nthreads(:interactive))
println("Threads.maxthreadid()   = ", Threads.maxthreadid())
println("Threads.threadid()      = ", Threads.threadid())

In [ ]:
using Random
using Statistics
using Distributions
using Base.Threads
using LinearAlgebra
using DataFrames
using CSV
using Printf

BLAS.set_num_threads(1)

# -----------------------------------------------------------
# Uniform MP-index lookup table
# -----------------------------------------------------------

struct MPIndexLookupUniform
    xlo::Float64
    xhi::Float64
    h::Float64
    vals::Vector{Float64}
    mode::Symbol   # :leftbin, :nearest, :linear
end

"""
    build_mp_index_lookup(; q, ρ, κ, r, β, ε=1e-10,
                             ngrid=2001, xlo=0.0, xhi=1.0,
                             mode=:linear)

Precompute m(x)=f(x,x)/g(x,x) on a uniform x-grid.
"""
function build_mp_index_lookup(; q, ρ, κ, r, β,
                                 ε::Float64 = 1e-10,
                                 ngrid::Int = 2001,
                                 xlo::Float64 = 0.0,
                                 xhi::Float64 = 1.0,
                                 mode::Symbol = :linear)

    @assert ngrid >= 2
    @assert xlo < xhi
    @assert mode in (:leftbin, :nearest, :linear)

    xgrid = collect(range(xlo, xhi; length=ngrid))
    vals = Vector{Float64}(undef, ngrid)

    for i in eachindex(xgrid)
        x = xgrid[i]
        fxx = float(f_metric2(x, x; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε))
        gxx = float(g_metric2(x, x; q=q, ρ=ρ, κ=κ,      β=β, ε=ε))
        vals[i] = (isfinite(fxx) && isfinite(gxx) && gxx != 0.0) ? (fxx / gxx) : -Inf
    end

    return MPIndexLookupUniform(xlo, xhi, (xhi - xlo) / (ngrid - 1), vals, mode)
end

"""
    mp_index_lookup(L, x)

Approximate m(x) from a precomputed uniform lookup table.
"""
@inline function mp_index_lookup(L::MPIndexLookupUniform, x::Real)
    xf = clamp(Float64(x), L.xlo, L.xhi)
    vals = L.vals
    n = length(vals)

    if xf <= L.xlo
        return vals[1]
    elseif xf >= L.xhi
        return vals[end]
    end

    t = (xf - L.xlo) / L.h
    i = floor(Int, t) + 1
    i = clamp(i, 1, n - 1)

    if L.mode === :leftbin
        return vals[i]
    elseif L.mode === :nearest
        α = t - floor(t)
        return α < 0.5 ? vals[i] : vals[i + 1]
    else
        α = t - floor(t)
        return (1 - α) * vals[i] + α * vals[i + 1]
    end
end

"""
    build_type_mp_lookups(q_types, rho_types, kappa_types, r_types, beta; ...)

Build one MP-index lookup table per project type.
"""
function build_type_mp_lookups(q_types::AbstractVector,
                               rho_types::AbstractVector,
                               kappa_types::AbstractVector,
                               r_types::AbstractVector,
                               beta::Real;
                               ε::Float64 = 1e-10,
                               ngrid::Int = 2001,
                               xlo::Float64 = 0.0,
                               xhi::Float64 = 1.0,
                               mode::Symbol = :linear)

    K = length(q_types)
    @assert length(rho_types)   == K
    @assert length(kappa_types) == K
    @assert length(r_types)     == K

    return [
        build_mp_index_lookup(
            q = Float64(q_types[k]),
            ρ = Float64(rho_types[k]),
            κ = Float64(kappa_types[k]),
            r = Float64(r_types[k]),
            β = Float64(beta),
            ε = ε,
            ngrid = ngrid,
            xlo = xlo,
            xhi = xhi,
            mode = mode
        )
        for k in 1:K
    ]
end

In [ ]:
# -----------------------------------------------------------
# Structured K-type instance builder
# -----------------------------------------------------------

"""
    build_typed_instance(N; prop_type, q_types, rho_types, kappa_types, r_types,
                            x_init_scheme=:linspace, x0=0.5,
                            xlo=0.0, xhi=1.0,
                            shuffle_types=true, shuffle_x=false, seed=nothing)

Build one structured instance with fixed type proportions.

Returns a NamedTuple with:
- x_vec
- type_id_vec
- q_vec, rho_vec, kappa_vec, r_vec
- n_types
"""
function build_typed_instance(N::Integer;
                              prop_type::AbstractVector,
                              q_types::AbstractVector,
                              rho_types::AbstractVector,
                              kappa_types::AbstractVector,
                              r_types::AbstractVector,
                              x_init_scheme::Symbol = :linspace,
                              x0::Real = 0.5,
                              xlo::Real = 0.0,
                              xhi::Real = 1.0,
                              shuffle_types::Bool = true,
                              shuffle_x::Bool = false,
                              seed::Union{Int,Nothing} = nothing)

    @assert N >= 1
    K = length(prop_type)
    @assert length(q_types)     == K
    @assert length(rho_types)   == K
    @assert length(kappa_types) == K
    @assert length(r_types)     == K
    @assert isapprox(sum(prop_type), 1.0; atol=1e-12)

    n_types = round.(Int, N .* Float64.(prop_type))
    n_types[end] += N - sum(n_types)

    type_id_vec = Vector{Int}(undef, N)
    idx = 1
    for k in 1:K
        for _ in 1:n_types[k]
            type_id_vec[idx] = k
            idx += 1
        end
    end

    rng = seed === nothing ? Random.default_rng() : MersenneTwister(seed)

    if shuffle_types
        shuffle!(rng, type_id_vec)
    end

    x_vec = if x_init_scheme === :linspace
        collect(range(Float64(xlo), Float64(xhi); length=N))
    elseif x_init_scheme === :constant
        fill(Float64(x0), N)
    else
        error("Unsupported x_init_scheme = $x_init_scheme. Use :linspace or :constant.")
    end

    if shuffle_x
        shuffle!(rng, x_vec)
    end

    q_vec     = [Float64(q_types[type_id_vec[i]])     for i in 1:N]
    rho_vec   = [Float64(rho_types[type_id_vec[i]])   for i in 1:N]
    kappa_vec = [Float64(kappa_types[type_id_vec[i]]) for i in 1:N]
    r_vec     = [Float64(r_types[type_id_vec[i]])     for i in 1:N]

    return (
        x_vec = x_vec,
        type_id_vec = type_id_vec,
        q_vec = q_vec,
        rho_vec = rho_vec,
        kappa_vec = kappa_vec,
        r_vec = r_vec,
        n_types = n_types
    )
end

In [ ]:
# -----------------------------------------------------------
# Helper for top-M selection
# -----------------------------------------------------------

@inline function choose_topM_mask!(mask::BitVector,
                                   vals::AbstractVector{<:Real},
                                   M::Int;
                                   positive::Bool = false,
                                   nonnegative::Bool = false)
    fill!(mask, false)
    m = min(M, length(vals))
    ord = partialsortperm(vals, 1:m; rev=true)

    for idx in ord
        v = vals[idx]
        if positive
            v > 0.0 || break
        elseif nonnegative
            v >= 0.0 || break
        end
        mask[idx] = true
    end
    return nothing
end

# -----------------------------------------------------------
# Multi-project simulator with optional lookup MP-index
# -----------------------------------------------------------

"""
    simulate_multi_project_policies_lookup(
        x_vec, type_id_vec, q_types, rho_types, kappa_types, r_types, beta, M;
        T=100, nruns=1000, conf_level=0.95,
        seed=nothing, ε=1e-10, mp_lookups=nothing
    )

Monte Carlo estimation of approximate long-run average discounted reward per project
under four policies:

  1. Random
  2. Myopic
  3. Round-robin
  4. MP-index

with the MP-index policy using precomputed lookup tables if `mp_lookups !== nothing`.

Also computes the normalized Lagrangian upper bound via
    λstar, Lmin = lambda_star_vector(...)
"""
function simulate_multi_project_policies_lookup(
    x_vec::AbstractVector{<:Real},
    type_id_vec::AbstractVector{<:Integer},
    q_types::AbstractVector{<:Real},
    rho_types::AbstractVector{<:Real},
    kappa_types::AbstractVector{<:Real},
    r_types::AbstractVector{<:Real},
    beta::Float64,
    M::Integer;
    T::Int = 100,
    nruns::Int = 1_000,
    conf_level::Float64 = 0.95,
    seed::Union{Int,Nothing} = nothing,
    ε::Float64 = 1e-10,
    mp_lookups::Union{Nothing,AbstractVector{<:MPIndexLookupUniform}} = nothing
)

    N = length(x_vec)
    @assert length(type_id_vec) == N
    K = length(q_types)
    @assert length(rho_types)   == K
    @assert length(kappa_types) == K
    @assert length(r_types)     == K
    @assert 0.0 < beta < 1.0
    @assert 1 <= M <= N
    @assert T > 0
    @assert nruns > 0

    if mp_lookups !== nothing
        @assert length(mp_lookups) == K
    end

    x_vec_f = Float64.(x_vec)

    q_types_f     = Float64.(q_types)
    rho_types_f   = Float64.(rho_types)
    kappa_types_f = Float64.(kappa_types)
    r_types_f     = Float64.(r_types)

    type_id = Int.(type_id_vec)

    for i in 1:N
        @assert 1 <= type_id[i] <= K
        @assert 0.0 <= x_vec_f[i] <= 1.0
    end

    for k in 1:K
        @assert 0.0 <= q_types_f[k] < 1.0
        @assert 0.0 <= rho_types_f[k] < 1.0 - q_types_f[k]
        @assert 0.0 <= kappa_types_f[k] <= 1.0
        @assert r_types_f[k] >= 0.0
    end

    q_vec     = [q_types_f[type_id[i]]     for i in 1:N]
    rho_vec   = [rho_types_f[type_id[i]]   for i in 1:N]
    kappa_vec = [kappa_types_f[type_id[i]] for i in 1:N]
    r_vec     = [r_types_f[type_id[i]]     for i in 1:N]

    λstar, Lmin = lambda_star_vector(
        x_vec_f;
        qvector = q_vec,
        ρvector = rho_vec,
        κvector = kappa_vec,
        rvector = r_vec,
        β = beta,
        M = M
    )

    β_powers = [beta^t for t in 0:(T-1)]

    total_random     = zeros(Float64, nruns)
    total_myopic     = zeros(Float64, nruns)
    total_roundrobin = zeros(Float64, nruns)
    total_mpindex    = zeros(Float64, nruns)

    seedgen = seed === nothing ? Random.default_rng() : MersenneTwister(seed)
    run_seeds = rand(seedgen, UInt64, nruns)

    @inline function phi0(x, q, rho)
        q + rho * x
    end

    @inline function phi1(x, q, rho, kappa)
        denom = 1.0 - kappa * x
        if denom <= 0.0
            return q + rho
        end
        post_nack = (1.0 - kappa) * x / denom
        return q + rho * post_nack
    end

    @inline function mp_index_current_exact(x, q, rho, kappa, r, beta, ε)
        fxx = float(f_metric2(x, x; q=q, ρ=rho, κ=kappa, r=r, β=beta, ε=ε))
        gxx = float(g_metric2(x, x; q=q, ρ=rho, κ=kappa,      β=beta, ε=ε))
        return (isfinite(fxx) && isfinite(gxx) && gxx != 0.0) ? (fxx / gxx) : -Inf
    end

    function simulate_one_run!(run_idx::Int)
        rng = MersenneTwister(run_seeds[run_idx])

        X_rand = copy(x_vec_f)
        X_myop = copy(x_vec_f)
        X_rr   = copy(x_vec_f)
        X_mp   = copy(x_vec_f)

        rr_start = 1

        R_rand = 0.0
        R_myop = 0.0
        R_rr   = 0.0
        R_mp   = 0.0

        # reusable work arrays
        myopic_value = zeros(Float64, N)
        mpvals       = zeros(Float64, N)

        active_rand_mask = falses(N)
        active_myop_mask = falses(N)
        active_rr_mask   = falses(N)
        active_mp_mask   = falses(N)

        perm = collect(1:N)

        for t in 1:T
            disc = β_powers[t]

            # ------------------------------------------------------
            # 1. Random policy
            # ------------------------------------------------------
            fill!(active_rand_mask, false)
            shuffle!(rng, perm)
            active_rand_mask[perm[1:M]] .= true

            reward_t = 0.0
            for i in 1:N
                q  = q_vec[i]
                rho = rho_vec[i]
                kappa = kappa_vec[i]
                r  = r_vec[i]
                xi = X_rand[i]

                if active_rand_mask[i]
                    p_ack = kappa * xi
                    if rand(rng) < p_ack
                        reward_t += r
                        X_rand[i] = q + rho
                    else
                        X_rand[i] = phi1(xi, q, rho, kappa)
                    end
                else
                    X_rand[i] = phi0(xi, q, rho)
                end
            end
            R_rand += disc * reward_t

            # ------------------------------------------------------
            # 2. Myopic policy
            # ------------------------------------------------------
            for i in 1:N
                myopic_value[i] = r_vec[i] * kappa_vec[i] * X_myop[i]
            end
            choose_topM_mask!(active_myop_mask, myopic_value, M; positive=true)

            reward_t = 0.0
            for i in 1:N
                q  = q_vec[i]
                rho = rho_vec[i]
                kappa = kappa_vec[i]
                r  = r_vec[i]
                xi = X_myop[i]

                if active_myop_mask[i]
                    p_ack = kappa * xi
                    if rand(rng) < p_ack
                        reward_t += r
                        X_myop[i] = q + rho
                    else
                        X_myop[i] = phi1(xi, q, rho, kappa)
                    end
                else
                    X_myop[i] = phi0(xi, q, rho)
                end
            end
            R_myop += disc * reward_t

            # ------------------------------------------------------
            # 3. Round-robin policy
            # ------------------------------------------------------
            fill!(active_rr_mask, false)
            for k in 0:(M-1)
                i = ((rr_start - 1 + k) % N) + 1
                active_rr_mask[i] = true
            end
            rr_start = ((rr_start - 1 + M) % N) + 1

            reward_t = 0.0
            for i in 1:N
                q  = q_vec[i]
                rho = rho_vec[i]
                kappa = kappa_vec[i]
                r  = r_vec[i]
                xi = X_rr[i]

                if active_rr_mask[i]
                    p_ack = kappa * xi
                    if rand(rng) < p_ack
                        reward_t += r
                        X_rr[i] = q + rho
                    else
                        X_rr[i] = phi1(xi, q, rho, kappa)
                    end
                else
                    X_rr[i] = phi0(xi, q, rho)
                end
            end
            R_rr += disc * reward_t

            # ------------------------------------------------------
            # 4. MP-index policy (lookup-based if provided)
            # ------------------------------------------------------
            for i in 1:N
                k = type_id[i]
                xi = X_mp[i]
                mpvals[i] = mp_lookups === nothing ?
                    mp_index_current_exact(xi,
                                           q_types_f[k], rho_types_f[k], kappa_types_f[k],
                                           r_types_f[k], beta, ε) :
                    mp_index_lookup(mp_lookups[k], xi)
            end
            choose_topM_mask!(active_mp_mask, mpvals, M; nonnegative=true)

            reward_t = 0.0
            for i in 1:N
                q  = q_vec[i]
                rho = rho_vec[i]
                kappa = kappa_vec[i]
                r  = r_vec[i]
                xi = X_mp[i]

                if active_mp_mask[i]
                    p_ack = kappa * xi
                    if rand(rng) < p_ack
                        reward_t += r
                        X_mp[i] = q + rho
                    else
                        X_mp[i] = phi1(xi, q, rho, kappa)
                    end
                else
                    X_mp[i] = phi0(xi, q, rho)
                end
            end
            R_mp += disc * reward_t
        end

        total_random[run_idx]     = R_rand
        total_myopic[run_idx]     = R_myop
        total_roundrobin[run_idx] = R_rr
        total_mpindex[run_idx]    = R_mp
        return nothing
    end

    if Threads.nthreads() > 1
        Threads.@threads for run_idx in 1:nruns
            simulate_one_run!(run_idx)
        end
    else
        for run_idx in 1:nruns
            simulate_one_run!(run_idx)
        end
    end

    norm_factor = (1.0 - beta) / N

    function summarize(raw_samples::Vector{Float64})
        samples = norm_factor .* raw_samples
        μ  = mean(samples)
        s  = std(samples)
        se = s / sqrt(length(samples))
        z  = quantile(Normal(), 0.5 + conf_level / 2)
        ci_low  = μ - z * se
        ci_high = μ + z * se
        return (mean = μ,
                stderr = se,
                ci_low = ci_low,
                ci_high = ci_high,
                nruns = length(samples))
    end

    results = Dict{String,NamedTuple}()
    results["random"]          = summarize(total_random)
    results["myopic"]          = summarize(total_myopic)
    results["round_robin"]     = summarize(total_roundrobin)
    results["mp_index"]        = summarize(total_mpindex)

    lagrangian_bound = norm_factor * Lmin
    results["lagrangian_bound"] = (
        lambda_star = λstar,
        raw_Lmin = Lmin,
        mean = lagrangian_bound,
        stderr = 0.0,
        ci_low = lagrangian_bound,
        ci_high = lagrangian_bound,
        nruns = 1
    )

    return results
end

In [ ]:
# -----------------------------------------------------------
# Driver for scaling experiments on a grid of parameter instances
# -----------------------------------------------------------

"""
    run_typed_scaling_experiment(; scenario_grid, prop_grid, alphaM_values, N_values, ...)

Runs the two-type / K-type benchmark experiment over a grid of:
- parameter scenarios,
- type proportions,
- capacity ratios alphaM = M/N,
- system sizes N.

Each scenario should be a NamedTuple with fields:
- name
- q_types
- rho_types
- kappa_types
- r_types
"""
function run_typed_scaling_experiment(;
    scenario_grid,
    prop_grid,
    alphaM_values,
    N_values,
    beta = 0.99,
    T_policy = 200,
    nruns_policy = 1000,
    conf_level = 0.95,
    base_seed = 1234,
    ε = 1e-10,
    mp_lookup_ngrid = 1001,
    mp_lookup_mode = :linear,
    x_init_scheme = :linspace,
    x0 = 0.5,
    xlo = 0.0,
    xhi = 1.0,
    shuffle_types = true,
    shuffle_x = true
)

    rows = NamedTuple[]
    counter = 0

    for (s_idx, scen) in enumerate(scenario_grid)
        name = hasproperty(scen, :name) ? String(scen.name) : "scenario_$s_idx"

        q_types     = Float64.(scen.q_types)
        rho_types   = Float64.(scen.rho_types)
        kappa_types = Float64.(scen.kappa_types)
        r_types     = Float64.(scen.r_types)

        mp_lookups = build_type_mp_lookups(
            q_types, rho_types, kappa_types, r_types, beta;
            ε = ε,
            ngrid = mp_lookup_ngrid,
            mode = mp_lookup_mode
        )

        for prop_type in prop_grid
            prop_type_f = Float64.(prop_type)
            K = length(prop_type_f)

            for alphaM in alphaM_values
                for N in N_values
                    counter += 1
                    M = max(1, min(N, round(Int, alphaM * N)))

                    instance_seed = base_seed + counter

                    inst = build_typed_instance(
                        N;
                        prop_type = prop_type_f,
                        q_types = q_types,
                        rho_types = rho_types,
                        kappa_types = kappa_types,
                        r_types = r_types,
                        x_init_scheme = x_init_scheme,
                        x0 = x0,
                        xlo = xlo,
                        xhi = xhi,
                        shuffle_types = shuffle_types,
                        shuffle_x = shuffle_x,
                        seed = instance_seed
                    )

                    println("Running ",
                            name,
                            " | prop = ", prop_type_f,
                            " | αM = ", alphaM,
                            " | N = ", N,
                            " | M = ", M)

                    results = simulate_multi_project_policies_lookup(
                        inst.x_vec,
                        inst.type_id_vec,
                        q_types,
                        rho_types,
                        kappa_types,
                        r_types,
                        beta,
                        M;
                        T = T_policy,
                        nruns = nruns_policy,
                        conf_level = conf_level,
                        seed = instance_seed,
                        ε = ε,
                        mp_lookups = mp_lookups
                    )

                    rand_stats = results["random"]
                    myop_stats = results["myopic"]
                    rr_stats   = results["round_robin"]
                    mp_stats   = results["mp_index"]
                    ub_stats   = results["lagrangian_bound"]

                    ub = ub_stats.mean
                    rand_mean = rand_stats.mean
                    myop_mean = myop_stats.mean
                    rr_mean   = rr_stats.mean
                    mp_mean   = mp_stats.mean

                    abs_gap_mp = ub - mp_mean
                    rel_gap_mp = ub != 0.0 ? abs_gap_mp / ub : NaN

                    abs_gap_myop = ub - myop_mean
                    abs_gap_rr   = ub - rr_mean
                    abs_gap_rand = ub - rand_mean

                    sqrtN = sqrt(Float64(N))

                    row = (
                        scenario = name,
                        N = N,
                        M = M,
                        capacity_ratio = M / N,
                        beta = beta,

                        K = K,
                        prop_type = join(string.(round.(prop_type_f; digits=4)), ","),

                        q_types = join(string.(q_types), ","),
                        rho_types = join(string.(rho_types), ","),
                        kappa_types = join(string.(kappa_types), ","),
                        r_types = join(string.(r_types), ","),

                        lambda_star = ub_stats.lambda_star,
                        raw_Lmin = ub_stats.raw_Lmin,
                        lagrangian_bound = ub,

                        random_mean = rand_mean,
                        random_ci_low = rand_stats.ci_low,
                        random_ci_high = rand_stats.ci_high,

                        myopic_mean = myop_mean,
                        myopic_ci_low = myop_stats.ci_low,
                        myopic_ci_high = myop_stats.ci_high,

                        round_robin_mean = rr_mean,
                        round_robin_ci_low = rr_stats.ci_low,
                        round_robin_ci_high = rr_stats.ci_high,

                        mp_index_mean = mp_mean,
                        mp_index_ci_low = mp_stats.ci_low,
                        mp_index_ci_high = mp_stats.ci_high,

                        abs_gap_mp_to_bound = abs_gap_mp,
                        rel_gap_mp_to_bound = rel_gap_mp,
                        N_abs_gap_mp_to_bound = N * abs_gap_mp,
                        sqrtN_abs_gap_mp_to_bound = sqrtN * abs_gap_mp,

                        abs_gap_myopic_to_bound = abs_gap_myop,
                        abs_gap_roundrobin_to_bound = abs_gap_rr,
                        abs_gap_random_to_bound = abs_gap_rand,

                        mp_lookup_ngrid = mp_lookup_ngrid,
                        mp_lookup_mode = String(mp_lookup_mode),
                        x_init_scheme = String(x_init_scheme),
                        seed = instance_seed
                    )

                    # add actual counts by type as separate columns when K is small
                    if K >= 1
                        row = merge(row, (n_type1 = inst.n_types[1],))
                    end
                    if K >= 2
                        row = merge(row, (n_type2 = inst.n_types[2],))
                    end

                    push!(rows, row)
                end
            end
        end
    end

    return rows
end

In [ ]:
function typed_scaling_rows_to_dataframe(rows)
    df = DataFrame(rows)
    sort!(df, [:scenario, :prop_type, :capacity_ratio, :N])
    return df
end

In [ ]:
# -----------------------------------------------------------
# Example scenario grid
# -----------------------------------------------------------

scenario_grid = [
    (
        name = "A_selfhealing_B_fragile",
        q_types = [0.01, 0.10],
        rho_types = [0.90, 0.10],
        kappa_types = [0.70, 0.95],
        r_types = [1.0, 1.0]
    ),

    (
        name = "A_mild_B_fragile",
        q_types = [0.03, 0.10],
        rho_types = [0.80, 0.10],
        kappa_types = [0.70, 0.95],
        r_types = [1.0, 1.0]
    )
]

prop_grid = [
    [0.20, 0.80],
    [0.40, 0.60],
    [0.60, 0.40],
    [0.80, 0.20]
]

alphaM_values = [0.05, 0.10, 0.20]
N_values = [50, 100, 200, 400]

rows = run_typed_scaling_experiment(
    scenario_grid = scenario_grid,
    prop_grid = prop_grid,
    alphaM_values = alphaM_values,
    N_values = N_values,
    beta = 0.99,
    T_policy = 200,
    nruns_policy = 1000,
    conf_level = 0.95,
    base_seed = 1234,
    ε = 1e-10,
    mp_lookup_ngrid = 1001,
    mp_lookup_mode = :linear,
    x_init_scheme = :linspace,
    xlo = 0.0,
    xhi = 1.0,
    shuffle_types = true,
    shuffle_x = true
)

df = typed_scaling_rows_to_dataframe(rows)

CSV.write("typed_scaling_experiment.csv", df)

first(df, 10)

In [ ]:
using CSV
using DataFrames
using Statistics
using Printf
using PGFPlotsX

# -----------------------------------------------------------
# 1) Load and basic checks
# -----------------------------------------------------------

csvfile = "typed_scaling_experiment.csv"
df = CSV.read(csvfile, DataFrame)

function check_typed_experiment_completeness(df::DataFrame)
    ns = length(unique(df.scenario))
    np = length(unique(df.prop_type))
    na = length(unique(df.capacity_ratio))
    nN = length(unique(df.N))
    expected = ns * np * na * nN
    actual = nrow(df)

    println("Rows in CSV          = ", actual)
    println("Unique scenarios     = ", ns)
    println("Unique prop_type     = ", np)
    println("Unique capacity_ratio= ", na)
    println("Unique N             = ", nN)
    println("Expected rows        = ", expected)
    println("Complete?            = ", actual == expected)

    return (actual=actual, expected=expected, complete=(actual == expected))
end

chk = check_typed_experiment_completeness(df)

# -----------------------------------------------------------
# 2) Add derived columns
# -----------------------------------------------------------

function enrich_typed_results!(df::DataFrame)
    # CI half-widths
    df.mp_ci_halfwidth      = (df.mp_index_ci_high .- df.mp_index_ci_low) ./ 2
    df.myopic_ci_halfwidth  = (df.myopic_ci_high .- df.myopic_ci_low) ./ 2
    df.rr_ci_halfwidth      = (df.round_robin_ci_high .- df.round_robin_ci_low) ./ 2
    df.random_ci_halfwidth  = (df.random_ci_high .- df.random_ci_low) ./ 2

    # Scaled gaps for other benchmarks too
    df.rel_gap_myopic_to_bound      = df.abs_gap_myopic_to_bound ./ df.lagrangian_bound
    df.rel_gap_roundrobin_to_bound  = df.abs_gap_roundrobin_to_bound ./ df.lagrangian_bound
    df.rel_gap_random_to_bound      = df.abs_gap_random_to_bound ./ df.lagrangian_bound

    df.N_abs_gap_myopic_to_bound     = df.N .* df.abs_gap_myopic_to_bound
    df.N_abs_gap_roundrobin_to_bound = df.N .* df.abs_gap_roundrobin_to_bound
    df.N_abs_gap_random_to_bound     = df.N .* df.abs_gap_random_to_bound

    df.sqrtN_abs_gap_myopic_to_bound     = sqrt.(Float64.(df.N)) .* df.abs_gap_myopic_to_bound
    df.sqrtN_abs_gap_roundrobin_to_bound = sqrt.(Float64.(df.N)) .* df.abs_gap_roundrobin_to_bound
    df.sqrtN_abs_gap_random_to_bound     = sqrt.(Float64.(df.N)) .* df.abs_gap_random_to_bound

    # Best simulated benchmark in each row
    best_name = Vector{String}(undef, nrow(df))
    best_mean = Vector{Float64}(undef, nrow(df))

    for i in 1:nrow(df)
        vals = [
            ("random",      df.random_mean[i]),
            ("myopic",      df.myopic_mean[i]),
            ("round_robin", df.round_robin_mean[i]),
            ("mp_index",    df.mp_index_mean[i]),
        ]
        j = argmax(last.(vals))
        best_name[i] = vals[j][1]
        best_mean[i] = vals[j][2]
    end

    df.best_policy = best_name
    df.best_policy_mean = best_mean

    sort!(df, [:scenario, :prop_type, :capacity_ratio, :N])
    return df
end

enrich_typed_results!(df)

# -----------------------------------------------------------
# 3) Quick textual summaries
# -----------------------------------------------------------

function quick_summary_tables(df::DataFrame)
    println("\n=== Overall averages by scenario ===")
    scen = combine(groupby(df, :scenario),
        :abs_gap_mp_to_bound => mean => :mean_abs_gap_mp,
        :rel_gap_mp_to_bound => mean => :mean_rel_gap_mp,
        :mp_index_mean       => mean => :mean_mp,
        :myopic_mean         => mean => :mean_myopic,
        :round_robin_mean    => mean => :mean_rr,
        :random_mean         => mean => :mean_random
    )
    display(scen)

    println("\n=== How often each policy is best ===")
    besttab = combine(groupby(df, [:scenario, :best_policy]), nrow => :count)
    display(besttab)

    println("\n=== Worst MP gaps (largest abs_gap_mp_to_bound) ===")
    display(first(sort(df, :abs_gap_mp_to_bound, rev=true), 12))

    println("\n=== Best MP cases (smallest abs_gap_mp_to_bound) ===")
    display(first(sort(df, :abs_gap_mp_to_bound), 12))

    return nothing
end

quick_summary_tables(df)

# -----------------------------------------------------------
# 4) Slice-specific summary table
# -----------------------------------------------------------

function slice_table(df::DataFrame; scenario, prop_type, capacity_ratio)
    d = filter(row ->
        row.scenario == scenario &&
        row.prop_type == prop_type &&
        row.capacity_ratio == capacity_ratio,
        df
    )
    sort!(d, :N)
    return select(d,
        :scenario, :prop_type, :capacity_ratio, :N, :M,
        :lagrangian_bound,
        :mp_index_mean, :mp_index_ci_low, :mp_index_ci_high,
        :myopic_mean, :round_robin_mean, :random_mean,
        :abs_gap_mp_to_bound, :rel_gap_mp_to_bound,
        :N_abs_gap_mp_to_bound, :sqrtN_abs_gap_mp_to_bound,
        :best_policy
    )
end

# Example:
# display(slice_table(df; scenario="A_selfhealing_B_fragile", prop_type="0.4,0.6", capacity_ratio=0.1))

# -----------------------------------------------------------
# 5) PGFPlotsX helpers
# -----------------------------------------------------------

_pgf_xmode(xscale) = xscale == :log10 ? "log" : "linear"

# NOTE:
# Do not use helper tuples expanded inside @pgf option braces.
# Inline the options directly; that is more robust with PGFPlotsX.

function performance_axis_typed(d::DataFrame;
                                xscale=:identity,
                                width="11cm",
                                height="7cm",
                                legend_pos="south east")

    N = Float64.(d.N)

    ax = @pgf Axis(
        {
            width = width,
            height = height,
            xlabel = raw"$N$",
            ylabel = raw"normalized reward per project",
            xmode = _pgf_xmode(xscale),
            ymajorgrids,
            xmajorgrids,
            "legend pos" = legend_pos,
            "legend cell align" = "left",
            "font" = raw"\small",
            "mark size" = "2.2pt"
        },

        Plot({color="black", mark="square*", "thick"},
             Table(x=N, y=d.lagrangian_bound)),
        LegendEntry("Lagrangian bound"),

        Plot({color="blue", mark="*", "thick"},
             Table(x=N, y=d.mp_index_mean)),
        LegendEntry("MP-index"),

        Plot({color="blue", style="dashed", "thin"},
             Table(x=N, y=d.mp_index_ci_low)),
        LegendEntry("MP-index CI bounds"),

        Plot({color="blue", style="dashed", "thin", "forget plot"},
             Table(x=N, y=d.mp_index_ci_high)),

        Plot({color="red", mark="triangle*", "thick"},
             Table(x=N, y=d.myopic_mean)),
        LegendEntry("Myopic"),

        Plot({color="gray", mark="diamond*", "thick"},
             Table(x=N, y=d.round_robin_mean)),
        LegendEntry("Round-robin"),

        Plot({color="black!50", mark="o", "thick"},
             Table(x=N, y=d.random_mean)),
        LegendEntry("Random")
    )

    return ax
end

function abs_gap_axis_typed(d::DataFrame;
                            xscale=:identity,
                            width="11cm",
                            height="7cm",
                            legend_pos="north east")

    N = Float64.(d.N)

    ax = @pgf Axis(
        {
            width = width,
            height = height,
            xlabel = raw"$N$",
            ylabel = raw"absolute gap to bound",
            xmode = _pgf_xmode(xscale),
            ymajorgrids,
            xmajorgrids,
            "legend pos" = legend_pos,
            "legend cell align" = "left",
            "font" = raw"\small",
            "mark size" = "2.2pt"
        },

        Plot({color="blue", mark="*", "thick"},
             Table(x=N, y=d.abs_gap_mp_to_bound)),
        LegendEntry("MP-index"),

        Plot({color="red", mark="triangle*", "thick"},
             Table(x=N, y=d.abs_gap_myopic_to_bound)),
        LegendEntry("Myopic"),

        Plot({color="gray", mark="diamond*", "thick"},
             Table(x=N, y=d.abs_gap_roundrobin_to_bound)),
        LegendEntry("Round-robin"),

        Plot({color="black!50", mark="o", "thick"},
             Table(x=N, y=d.abs_gap_random_to_bound)),
        LegendEntry("Random")
    )

    return ax
end

function scaled_gap_axis_typed(d::DataFrame;
                               scale=:N,
                               xscale=:identity,
                               width="11cm",
                               height="7cm",
                               legend_pos="north east")

    N = Float64.(d.N)

    if scale == :N
        y_mp = d.N_abs_gap_mp_to_bound
        y_my = d.N_abs_gap_myopic_to_bound
        y_rr = d.N_abs_gap_roundrobin_to_bound
        y_ra = d.N_abs_gap_random_to_bound
        ylabel = raw"$N\,(\mathrm{bound}-\mathrm{policy})$"
    elseif scale == :sqrtN
        y_mp = d.sqrtN_abs_gap_mp_to_bound
        y_my = d.sqrtN_abs_gap_myopic_to_bound
        y_rr = d.sqrtN_abs_gap_roundrobin_to_bound
        y_ra = d.sqrtN_abs_gap_random_to_bound
        ylabel = raw"$\sqrt{N}\,(\mathrm{bound}-\mathrm{policy})$"
    else
        error("scale must be :N or :sqrtN")
    end

    ax = @pgf Axis(
        {
            width = width,
            height = height,
            xlabel = raw"$N$",
            ylabel = ylabel,
            xmode = _pgf_xmode(xscale),
            ymajorgrids,
            xmajorgrids,
            "legend pos" = legend_pos,
            "legend cell align" = "left",
            "font" = raw"\small",
            "mark size" = "2.2pt"
        },

        Plot({color="blue", mark="*", "thick"},
             Table(x=N, y=y_mp)),
        LegendEntry("MP-index"),

        Plot({color="red", mark="triangle*", "thick"},
             Table(x=N, y=y_my)),
        LegendEntry("Myopic"),

        Plot({color="gray", mark="diamond*", "thick"},
             Table(x=N, y=y_rr)),
        LegendEntry("Round-robin"),

        Plot({color="black!50", mark="o", "thick"},
             Table(x=N, y=y_ra)),
        LegendEntry("Random")
    )

    return ax
end

# -----------------------------------------------------------
# 6) Build and save plots for one slice
# -----------------------------------------------------------

function save_typed_slice_plots(df::DataFrame;
                                scenario,
                                prop_type,
                                capacity_ratio,
                                prefix="typed_slice",
                                xscale=:log10,
                                width="12cm",
                                height="7cm")

    d = filter(row ->
        row.scenario == scenario &&
        row.prop_type == prop_type &&
        row.capacity_ratio == capacity_ratio,
        df
    )
    sort!(d, :N)

    @assert nrow(d) > 0 "No rows found for the requested slice."

    ax_perf  = performance_axis_typed(d; xscale=xscale, width=width, height=height)
    ax_abs   = abs_gap_axis_typed(d; xscale=xscale, width=width, height=height)
    ax_Ngap  = scaled_gap_axis_typed(d; scale=:N, xscale=xscale, width=width, height=height)
    ax_sNgap = scaled_gap_axis_typed(d; scale=:sqrtN, xscale=xscale, width=width, height=height)

    base = string(prefix,
                  "_", replace(scenario, " " => "_"),
                  "_prop_", replace(prop_type, "," => "-"),
                  "_cap_", capacity_ratio)

    PGFPlotsX.pgfsave(base * "_performance.tex", ax_perf)
    PGFPlotsX.pgfsave(base * "_abs_gap.tex", ax_abs)
    PGFPlotsX.pgfsave(base * "_N_gap.tex", ax_Ngap)
    PGFPlotsX.pgfsave(base * "_sqrtN_gap.tex", ax_sNgap)

    return (performance=ax_perf, abs_gap=ax_abs, N_gap=ax_Ngap, sqrtN_gap=ax_sNgap, slice=d)
end

# Example:
# axs = save_typed_slice_plots(df;
#     scenario="A_selfhealing_B_fragile",
#     prop_type="0.4,0.6",
#     capacity_ratio=0.1,
#     prefix="typed_scaling")
# axs.performance
# axs.abs_gap
# axs.N_gap
# axs.sqrtN_gap

# -----------------------------------------------------------
# 7) Save a summary CSV aggregated over N
# -----------------------------------------------------------

function aggregated_summary_by_slice(df::DataFrame)
    agg = combine(groupby(df, [:scenario, :prop_type, :capacity_ratio]),
        :abs_gap_mp_to_bound => mean => :mean_abs_gap_mp,
        :rel_gap_mp_to_bound => mean => :mean_rel_gap_mp,
        :N_abs_gap_mp_to_bound => mean => :mean_N_gap_mp,
        :sqrtN_abs_gap_mp_to_bound => mean => :mean_sqrtN_gap_mp,
        :mp_index_mean => mean => :mean_mp,
        :myopic_mean => mean => :mean_myopic,
        :round_robin_mean => mean => :mean_round_robin,
        :random_mean => mean => :mean_random
    )
    sort!(agg, [:scenario, :prop_type, :capacity_ratio])
    return agg
end

agg = aggregated_summary_by_slice(df)
CSV.write("typed_scaling_experiment_aggregated.csv", agg)

# -----------------------------------------------------------
# 8) Handy commands to inspect the results
# -----------------------------------------------------------

println("\nRows in df = ", nrow(df))
println("Scenarios = ", unique(df.scenario))
println("Proportions = ", unique(df.prop_type))
println("Capacity ratios = ", unique(df.capacity_ratio))
println("N values = ", unique(df.N))

println("\nTop 12 smallest MP gaps:")
display(first(sort(df, :abs_gap_mp_to_bound), 12))

println("\nTop 12 largest MP gaps:")
display(first(sort(df, :abs_gap_mp_to_bound, rev=true), 12))

select(df,
    :scenario, :prop_type, :capacity_ratio, :N, :M,
    :random_mean, :random_ci_low, :random_ci_high,
    :myopic_mean, :myopic_ci_low, :myopic_ci_high,
    :round_robin_mean, :round_robin_ci_low, :round_robin_ci_high,
    :mp_index_mean, :mp_index_ci_low, :mp_index_ci_high
)

In [ ]:
using DataFrames
using Statistics
using Printf

# ----------------------------------------------------------
# 0) Ensure the needed derived columns exist
# ----------------------------------------------------------

"""
    ensure_gap_and_improvement_columns!(df)

Add, if missing:

- relative gaps to Lagrangian bound for each policy
- absolute MP-minus-myopic improvement
- relative MP-minus-myopic improvement:
      (J_mp - J_myopic) / J_myopic
"""
function ensure_gap_and_improvement_columns!(df::DataFrame)
    # Relative gaps to bound
    if :rel_gap_mp_to_bound ∉ names(df)
        df.rel_gap_mp_to_bound = df.abs_gap_mp_to_bound ./ df.lagrangian_bound
    end
    if :rel_gap_myopic_to_bound ∉ names(df)
        df.rel_gap_myopic_to_bound = df.abs_gap_myopic_to_bound ./ df.lagrangian_bound
    end
    if :rel_gap_roundrobin_to_bound ∉ names(df)
        df.rel_gap_roundrobin_to_bound = df.abs_gap_roundrobin_to_bound ./ df.lagrangian_bound
    end
    if :rel_gap_random_to_bound ∉ names(df)
        df.rel_gap_random_to_bound = df.abs_gap_random_to_bound ./ df.lagrangian_bound
    end

    # MP vs myopic improvement
    if :abs_improvement_mp_vs_myopic ∉ names(df)
        df.abs_improvement_mp_vs_myopic = df.mp_index_mean .- df.myopic_mean
    end
    if :rel_improvement_mp_vs_myopic ∉ names(df)
        df.rel_improvement_mp_vs_myopic =
            ifelse.(df.myopic_mean .!= 0.0,
                    (df.mp_index_mean .- df.myopic_mean) ./ df.myopic_mean,
                    NaN)
    end

    return df
end

ensure_gap_and_improvement_columns!(df)

# ----------------------------------------------------------
# 1) Range of relative gaps w.r.t. Lagrangian bound for each policy
# ----------------------------------------------------------

"""
    relative_gap_ranges(df)

Return a DataFrame with min/max relative gap to the Lagrangian bound
for each policy.
"""
function relative_gap_ranges(df::DataFrame)
    return DataFrame(
        policy = ["mp_index", "myopic", "round_robin", "random"],
        min_rel_gap = [
            minimum(skipmissing(df.rel_gap_mp_to_bound)),
            minimum(skipmissing(df.rel_gap_myopic_to_bound)),
            minimum(skipmissing(df.rel_gap_roundrobin_to_bound)),
            minimum(skipmissing(df.rel_gap_random_to_bound))
        ],
        max_rel_gap = [
            maximum(skipmissing(df.rel_gap_mp_to_bound)),
            maximum(skipmissing(df.rel_gap_myopic_to_bound)),
            maximum(skipmissing(df.rel_gap_roundrobin_to_bound)),
            maximum(skipmissing(df.rel_gap_random_to_bound))
        ]
    )
end

gap_range_df = relative_gap_ranges(df)
println("\n=== Range of relative gaps to Lagrangian bound for each policy ===")
display(gap_range_df)

# ----------------------------------------------------------
# 2) Best/worst instances for MP and myopic relative gaps
#    ("best" = smallest relative gap to bound; "worst" = largest)
# ----------------------------------------------------------

"""
    best_worst_gap_instances(df, col)

Return NamedTuple(best=..., worst=...) where:
- best  = row with smallest value in column `col`
- worst = row with largest  value in column `col`
"""
function best_worst_gap_instances(df::DataFrame, col::Symbol)
    d = filter(row -> isfinite(row[col]), df)
    d_best  = first(sort(d, col), 1)
    d_worst = first(sort(d, col, rev=true), 1)

    keepcols = [
        :scenario, :prop_type, :capacity_ratio, :N, :M,
        :lagrangian_bound,
        :mp_index_mean, :myopic_mean, :round_robin_mean, :random_mean,
        col
    ]

    return (
        best = select(d_best, keepcols...),
        worst = select(d_worst, keepcols...)
    )
end

mp_gap_extremes = best_worst_gap_instances(df, :rel_gap_mp_to_bound)
my_gap_extremes = best_worst_gap_instances(df, :rel_gap_myopic_to_bound)

println("\n=== Best instance for MP relative gap (smallest gap) ===")
display(mp_gap_extremes.best)

println("\n=== Worst instance for MP relative gap (largest gap) ===")
display(mp_gap_extremes.worst)

println("\n=== Best instance for myopic relative gap (smallest gap) ===")
display(my_gap_extremes.best)

println("\n=== Worst instance for myopic relative gap (largest gap) ===")
display(my_gap_extremes.worst)

# ----------------------------------------------------------
# 3) Range of relative performance improvement of MP vs myopic
#    relative improvement = (J_mp - J_myopic)/J_myopic
# ----------------------------------------------------------

"""
    relative_improvement_range(df)

Return min/max of relative MP-vs-myopic improvement.
"""
function relative_improvement_range(df::DataFrame)
    vals = filter(isfinite, df.rel_improvement_mp_vs_myopic)
    return (
        min_rel_improvement = minimum(vals),
        max_rel_improvement = maximum(vals)
    )
end

imp_range = relative_improvement_range(df)

println("\n=== Range of relative performance improvement of MP vs myopic ===")
println("min relative improvement = ", imp_range.min_rel_improvement,
        "  (", 100 * imp_range.min_rel_improvement, "%)")
println("max relative improvement = ", imp_range.max_rel_improvement,
        "  (", 100 * imp_range.max_rel_improvement, "%)")

# ----------------------------------------------------------
# 4) Instances where MP-vs-myopic relative improvement is smallest/largest
# ----------------------------------------------------------

"""
    best_worst_improvement_instances(df)

Return NamedTuple(best=..., worst=...) where:
- best  = row with largest relative MP-vs-myopic improvement
- worst = row with smallest relative MP-vs-myopic improvement
"""
function best_worst_improvement_instances(df::DataFrame)
    d = filter(row -> isfinite(row.rel_improvement_mp_vs_myopic), df)

    d_best  = first(sort(d, :rel_improvement_mp_vs_myopic, rev=true), 1)
    d_worst = first(sort(d, :rel_improvement_mp_vs_myopic), 1)

    keepcols = [
        :scenario, :prop_type, :capacity_ratio, :N, :M,
        :lagrangian_bound,
        :mp_index_mean, :myopic_mean,
        :abs_improvement_mp_vs_myopic,
        :rel_improvement_mp_vs_myopic,
        :rel_gap_mp_to_bound,
        :rel_gap_myopic_to_bound
    ]

    return (
        largest = select(d_best, keepcols...),
        smallest = select(d_worst, keepcols...)
    )
end

imp_extremes = best_worst_improvement_instances(df)

println("\n=== Instance with largest relative performance improvement of MP over myopic ===")
display(imp_extremes.largest)

println("\n=== Instance with smallest relative performance improvement of MP over myopic ===")
display(imp_extremes.smallest)

# ----------------------------------------------------------
# 5) Optional: concise printable summary
# ----------------------------------------------------------

function print_gap_and_improvement_summary(df::DataFrame)
    ensure_gap_and_improvement_columns!(df)

    gr = relative_gap_ranges(df)
    mp_ext = best_worst_gap_instances(df, :rel_gap_mp_to_bound)
    my_ext = best_worst_gap_instances(df, :rel_gap_myopic_to_bound)
    ir = relative_improvement_range(df)
    ie = best_worst_improvement_instances(df)

    println("\n==================================================")
    println("Relative gaps to Lagrangian bound: ranges")
    println("==================================================")
    display(gr)

    println("\n==================================================")
    println("MP relative gap: best (smallest)")
    println("==================================================")
    display(mp_ext.best)

    println("\n==================================================")
    println("MP relative gap: worst (largest)")
    println("==================================================")
    display(mp_ext.worst)

    println("\n==================================================")
    println("Myopic relative gap: best (smallest)")
    println("==================================================")
    display(my_ext.best)

    println("\n==================================================")
    println("Myopic relative gap: worst (largest)")
    println("==================================================")
    display(my_ext.worst)

    println("\n==================================================")
    println("Relative performance improvement of MP over myopic")
    println("==================================================")
    println("Min relative improvement = ",
            @sprintf("%.6f", ir.min_rel_improvement),
            " = ", @sprintf("%.2f", 100 * ir.min_rel_improvement), "%")
    println("Max relative improvement = ",
            @sprintf("%.6f", ir.max_rel_improvement),
            " = ", @sprintf("%.2f", 100 * ir.max_rel_improvement), "%")

    println("\n==================================================")
    println("Largest relative MP-vs-myopic improvement")
    println("==================================================")
    display(ie.largest)

    println("\n==================================================")
    println("Smallest relative MP-vs-myopic improvement")
    println("==================================================")
    display(ie.smallest)

    return nothing
end

# Run the full summary if desired:
# print_gap_and_improvement_summary(df)

gap_range_df = relative_gap_ranges(df)
mp_gap_extremes = best_worst_gap_instances(df, :rel_gap_mp_to_bound)
my_gap_extremes = best_worst_gap_instances(df, :rel_gap_myopic_to_bound)
imp_range = relative_improvement_range(df)
imp_extremes = best_worst_improvement_instances(df)

display(gap_range_df)
display(mp_gap_extremes.best)
display(mp_gap_extremes.worst)
display(my_gap_extremes.best)
display(my_gap_extremes.worst)
println(imp_range)
display(imp_extremes.largest)
display(imp_extremes.smallest)

In [ ]:
using DataFrames
using Statistics

"""
    add_ci_dominance_flags!(df)

Add Boolean columns indicating whether the MP-index CI lies entirely above
the CI of each benchmark policy, row by row.
"""
function add_ci_dominance_flags!(df::DataFrame)
    df.mp_ci_above_myopic =
        df.mp_index_ci_low .> df.myopic_ci_high

    df.mp_ci_above_round_robin =
        df.mp_index_ci_low .> df.round_robin_ci_high

    df.mp_ci_above_random =
        df.mp_index_ci_low .> df.random_ci_high

    # stricter: MP CI above all three benchmark CIs simultaneously
    df.mp_ci_above_all =
        df.mp_ci_above_myopic .&
        df.mp_ci_above_round_robin .&
        df.mp_ci_above_random

    return df
end

add_ci_dominance_flags!(df)

ci_summary_scenario = combine(groupby(df, :scenario),
    :mp_ci_above_myopic      => sum => :count_mp_above_myopic,
    :mp_ci_above_round_robin => sum => :count_mp_above_round_robin,
    :mp_ci_above_random      => sum => :count_mp_above_random,
    :mp_ci_above_all         => sum => :count_mp_above_all,
    nrow => :total_cases
)

display(ci_summary_scenario)

ci_summary_slice = combine(groupby(df, [:scenario, :prop_type, :capacity_ratio]),
    :mp_ci_above_myopic      => sum => :count_mp_above_myopic,
    :mp_ci_above_round_robin => sum => :count_mp_above_round_robin,
    :mp_ci_above_random      => sum => :count_mp_above_random,
    :mp_ci_above_all         => sum => :count_mp_above_all,
    nrow => :total_cases
)

sort!(ci_summary_slice, [:scenario, :prop_type, :capacity_ratio])
display(ci_summary_slice)

rows_unclear_vs_myopic = filter(row ->
    row.best_policy == "mp_index" && !row.mp_ci_above_myopic,
    df
)

display(select(rows_unclear_vs_myopic,
    :scenario, :prop_type, :capacity_ratio, :N, :M,
    :mp_index_mean, :mp_index_ci_low, :mp_index_ci_high,
    :myopic_mean, :myopic_ci_low, :myopic_ci_high
))

rows_clear_mp = filter(:mp_ci_above_all => identity, df)

display(select(rows_clear_mp,
    :scenario, :prop_type, :capacity_ratio, :N, :M,
    :mp_index_mean, :mp_index_ci_low, :mp_index_ci_high,
    :myopic_mean, :myopic_ci_low, :myopic_ci_high,
    :round_robin_mean, :round_robin_ci_low, :round_robin_ci_high,
    :random_mean, :random_ci_low, :random_ci_high
))




In [ ]:
using Statistics
using Printf
using DataFrames
using CSV
using PGFPlotsX

# -----------------------------------------------------------
# 1) Helpers
# -----------------------------------------------------------

function _sanitize_filename_piece(s)
    s = string(s)
    s = replace(s, " " => "_")
    s = replace(s, "," => "-")
    s = replace(s, "." => "p")
    s = replace(s, "[" => "")
    s = replace(s, "]" => "")
    s = replace(s, "(" => "")
    s = replace(s, ")" => "")
    return s
end

function _filter_slice(df::DataFrame; scenario, prop_type, capacity_ratio, atol=1e-12)
    d = filter(row ->
        row.scenario == scenario &&
        row.prop_type == prop_type &&
        isapprox(row.capacity_ratio, capacity_ratio; atol=atol, rtol=0.0),
        df
    )
    sort!(d, :N)
    return d
end

"""
    loglog_slope(x, y)

Estimate the slope in the regression
    log(y) ≈ a + b log(x)
using ordinary least squares on the positive finite entries.
If y ≈ C N^{-p}, then slope ≈ -p.
"""
function loglog_slope(x::AbstractVector, y::AbstractVector)
    keep = [isfinite(xi) && isfinite(yi) && xi > 0 && yi > 0 for (xi, yi) in zip(x, y)]
    xx = log.(Float64.(x[keep]))
    yy = log.(Float64.(y[keep]))

    if length(xx) < 2
        return NaN
    end

    xbar = mean(xx)
    ybar = mean(yy)
    num = sum((xx .- xbar) .* (yy .- ybar))
    den = sum((xx .- xbar).^2)

    return den > 0 ? num / den : NaN
end

# -----------------------------------------------------------
# 2) Slice table for one fixed (scenario, prop_type, capacity_ratio)
# -----------------------------------------------------------

function typed_slice_table(df::DataFrame; scenario, prop_type, capacity_ratio)
    d = _filter_slice(df; scenario=scenario, prop_type=prop_type, capacity_ratio=capacity_ratio)

    @assert nrow(d) > 0 "No rows found for the requested slice."

    return select(d,
        :scenario, :prop_type, :capacity_ratio,
        :N, :M,
        :lagrangian_bound,
        :mp_index_mean, :mp_index_ci_low, :mp_index_ci_high,
        :myopic_mean, :round_robin_mean, :random_mean,
        :abs_gap_mp_to_bound, :rel_gap_mp_to_bound,
        :N_abs_gap_mp_to_bound, :sqrtN_abs_gap_mp_to_bound,
        :abs_gap_myopic_to_bound,
        :abs_gap_roundrobin_to_bound,
        :abs_gap_random_to_bound,
        :best_policy
    )
end

# Example:
# display(typed_slice_table(df;
#     scenario="A_selfhealing_B_fragile",
#     prop_type="0.4,0.6",
#     capacity_ratio=0.1))

# -----------------------------------------------------------
# 3) One-line summary for one slice
# -----------------------------------------------------------

function typed_slice_summary(df::DataFrame; scenario, prop_type, capacity_ratio)
    d = _filter_slice(df; scenario=scenario, prop_type=prop_type, capacity_ratio=capacity_ratio)
    @assert nrow(d) > 0 "No rows found for the requested slice."

    Nvals = Float64.(d.N)
    gap   = Float64.(d.abs_gap_mp_to_bound)
    Ngap  = Float64.(d.N_abs_gap_mp_to_bound)
    sNgap = Float64.(d.sqrtN_abs_gap_mp_to_bound)

    return (
        scenario = scenario,
        prop_type = prop_type,
        capacity_ratio = capacity_ratio,
        npoints = nrow(d),
        N_min = minimum(d.N),
        N_max = maximum(d.N),

        gap_first = gap[1],
        gap_last = gap[end],
        gap_last_over_first = gap[end] / gap[1],

        Ngap_first = Ngap[1],
        Ngap_last = Ngap[end],
        Ngap_last_over_first = Ngap[end] / Ngap[1],

        sqrtNgap_first = sNgap[1],
        sqrtNgap_last = sNgap[end],
        sqrtNgap_last_over_first = sNgap[end] / sNgap[1],

        loglog_slope_gap = loglog_slope(Nvals, gap),
        loglog_slope_Ngap = loglog_slope(Nvals, Ngap),
        loglog_slope_sqrtNgap = loglog_slope(Nvals, sNgap),

        mp_last = d.mp_index_mean[end],
        bound_last = d.lagrangian_bound[end],
        myopic_last = d.myopic_mean[end],
        rr_last = d.round_robin_mean[end],
        random_last = d.random_mean[end],
    )
end

# -----------------------------------------------------------
# 4) Summary table over all slices
# -----------------------------------------------------------

function build_all_slice_summaries(df::DataFrame)
    rows = NamedTuple[]

    scenarios = unique(df.scenario)
    props = unique(df.prop_type)
    caps = unique(df.capacity_ratio)

    for scen in scenarios, prop in props, cap in caps
        d = _filter_slice(df; scenario=scen, prop_type=prop, capacity_ratio=cap)
        if nrow(d) > 0
            push!(rows, typed_slice_summary(df;
                scenario=scen,
                prop_type=prop,
                capacity_ratio=cap
            ))
        end
    end

    out = DataFrame(rows)
    sort!(out, [:scenario, :prop_type, :capacity_ratio])
    return out
end

slice_summary_df = build_all_slice_summaries(df)
CSV.write("typed_scaling_slice_summary.csv", slice_summary_df)

println("\n=== Slice-by-slice summary ===")
display(slice_summary_df)

println("\n=== Slices with smallest final MP gap ===")
display(first(sort(slice_summary_df, :gap_last), 12))

println("\n=== Slices with largest final MP gap ===")
display(first(sort(slice_summary_df, :gap_last, rev=true), 12))

println("\n=== Slices whose raw MP gap seems to decay fastest (most negative log-log slope) ===")
display(first(sort(slice_summary_df, :loglog_slope_gap), 12))

# -----------------------------------------------------------
# 5) Print a readable summary for one slice
# -----------------------------------------------------------

function print_typed_slice_summary(df::DataFrame; scenario, prop_type, capacity_ratio)
    s = typed_slice_summary(df; scenario=scenario, prop_type=prop_type, capacity_ratio=capacity_ratio)

    println("--------------------------------------------------")
    println("scenario       = ", s.scenario)
    println("prop_type      = ", s.prop_type)
    println("capacity_ratio = ", s.capacity_ratio)
    println("N range        = [", s.N_min, ", ", s.N_max, "]")
    println("gap first/last = ", s.gap_first, " / ", s.gap_last,
            "   (last/first = ", s.gap_last_over_first, ")")
    println("N*gap first/last = ", s.Ngap_first, " / ", s.Ngap_last,
            "   (last/first = ", s.Ngap_last_over_first, ")")
    println("sqrt(N)*gap first/last = ", s.sqrtNgap_first, " / ", s.sqrtNgap_last,
            "   (last/first = ", s.sqrtNgap_last_over_first, ")")
    println("log-log slope of gap          = ", s.loglog_slope_gap)
    println("log-log slope of N*gap        = ", s.loglog_slope_Ngap)
    println("log-log slope of sqrt(N)*gap  = ", s.loglog_slope_sqrtNgap)
    println("last-N performances:")
    println("  bound      = ", s.bound_last)
    println("  MP-index   = ", s.mp_last)
    println("  myopic     = ", s.myopic_last)
    println("  roundrobin = ", s.rr_last)
    println("  random     = ", s.random_last)
    println("--------------------------------------------------")
    return nothing
end

# Example:
# print_typed_slice_summary(df;
#     scenario="A_selfhealing_B_fragile",
#     prop_type="0.4,0.6",
#     capacity_ratio=0.1)

# -----------------------------------------------------------
# 6) Save plots for all slices
#    (uses your existing save_typed_slice_plots function)
# -----------------------------------------------------------

function save_all_typed_slice_plots(df::DataFrame;
                                    outdir="typed_slice_plots",
                                    xscale=:log10,
                                    width="12cm",
                                    height="7cm")

    mkpath(outdir)

    scenarios = unique(df.scenario)
    props = unique(df.prop_type)
    caps = unique(df.capacity_ratio)

    saved = NamedTuple[]

    for scen in scenarios, prop in props, cap in caps
        d = _filter_slice(df; scenario=scen, prop_type=prop, capacity_ratio=cap)
        if nrow(d) == 0
            continue
        end

        prefix = joinpath(
            outdir,
            "slice_" *
            _sanitize_filename_piece(scen) *
            "_prop_" * _sanitize_filename_piece(prop) *
            "_cap_" * _sanitize_filename_piece(cap)
        )

        axs = save_typed_slice_plots(df;
            scenario = scen,
            prop_type = prop,
            capacity_ratio = cap,
            prefix = prefix,
            xscale = xscale,
            width = width,
            height = height
        )

        push!(saved, (
            scenario = scen,
            prop_type = prop,
            capacity_ratio = cap,
            prefix = prefix
        ))
    end

    saved_df = DataFrame(saved)
    CSV.write(joinpath(outdir, "saved_plot_prefixes.csv"), saved_df)
    return saved_df
end

# Uncomment to save all slice plots:
# saved_plot_df = save_all_typed_slice_plots(df;
#     outdir="typed_slice_plots",
#     xscale=:log10,
#     width="12cm",
#     height="7cm")

In [ ]:
@inline function choose_topM_mask!(mask::BitVector,
                                   vals::AbstractVector{<:Real},
                                   M::Int;
                                   positive::Bool = false,
                                   nonnegative::Bool = false)
    fill!(mask, false)
    m = min(M, length(vals))
    ord = partialsortperm(vals, 1:m; rev = true)

    for idx in ord
        v = vals[idx]
        if positive
            v > 0.0 || break
        elseif nonnegative
            v >= 0.0 || break
        end
        mask[idx] = true
    end
    return nothing
end
    
function simulate_multi_project_policies_lookup_newbound(
    x_vec::AbstractVector{<:Real},
    type_id_vec::AbstractVector{<:Integer},
    q_types::AbstractVector{<:Real},
    rho_types::AbstractVector{<:Real},
    kappa_types::AbstractVector{<:Real},
    r_types::AbstractVector{<:Real},
    beta::Float64,
    M::Integer;
    T::Int = 100,
    nruns::Int = 1_000,
    conf_level::Float64 = 0.95,
    seed::Union{Int,Nothing} = nothing,
    ε::Float64 = 1e-10,
    mp_lookups::Union{Nothing,AbstractVector{<:MPIndexLookupUniform}} = nothing,
    bound_λ0::Float64 = 0.0,
    bound_step0::Float64 = 1.0,
    bound_λtol::Float64 = 1e-6,
    bound_max_bisect_iter::Int = 80,
    bound_xatol::Float64 = 1e-10,
    bound_λatol::Float64 = 1e-10,
    bound_maxiter::Int = 200,
    verbose_bound::Bool = false
)

    N = length(x_vec)
    @assert length(type_id_vec) == N
    K = length(q_types)
    @assert length(rho_types)   == K
    @assert length(kappa_types) == K
    @assert length(r_types)     == K
    @assert 0.0 < beta < 1.0
    @assert 1 <= M <= N
    @assert T > 0
    @assert nruns > 0

    if mp_lookups !== nothing
        @assert length(mp_lookups) == K
    end

    x_vec_f = Float64.(x_vec)

    q_types_f     = Float64.(q_types)
    rho_types_f   = Float64.(rho_types)
    kappa_types_f = Float64.(kappa_types)
    r_types_f     = Float64.(r_types)

    type_id = Int.(type_id_vec)

    for i in 1:N
        @assert 1 <= type_id[i] <= K
        @assert 0.0 <= x_vec_f[i] <= 1.0
    end

    for k in 1:K
        @assert 0.0 <= q_types_f[k] < 1.0
        @assert 0.0 <= rho_types_f[k] < 1.0 - q_types_f[k]
        @assert 0.0 <= kappa_types_f[k] <= 1.0
        @assert r_types_f[k] >= 0.0
    end

    q_vec     = [q_types_f[type_id[i]]     for i in 1:N]
    rho_vec   = [rho_types_f[type_id[i]]   for i in 1:N]
    kappa_vec = [kappa_types_f[type_id[i]] for i in 1:N]
    r_vec     = [r_types_f[type_id[i]]     for i in 1:N]

    λstar, Lmin, bound_info = lambda_star_vector_grouped_bisect(
        x_vec_f,
        q_vec,
        rho_vec,
        kappa_vec,
        r_vec,
        beta,
        M;
        λ0 = bound_λ0,
        step0 = bound_step0,
        λtol = bound_λtol,
        max_bisect_iter = bound_max_bisect_iter,
        ε = ε,
        xatol = bound_xatol,
        λatol = bound_λatol,
        maxiter = bound_maxiter,
        verbose = verbose_bound
    )

    β_powers = [beta^t for t in 0:(T-1)]

    total_random     = zeros(Float64, nruns)
    total_myopic     = zeros(Float64, nruns)
    total_roundrobin = zeros(Float64, nruns)
    total_mpindex    = zeros(Float64, nruns)

    seedgen = seed === nothing ? Random.default_rng() : MersenneTwister(seed)
    run_seeds = rand(seedgen, UInt64, nruns)

    @inline function phi0(x, q, rho)
        q + rho * x
    end

    @inline function phi1(x, q, rho, kappa)
        denom = 1.0 - kappa * x
        if denom <= 0.0
            return q + rho
        end
        post_nack = (1.0 - kappa) * x / denom
        return q + rho * post_nack
    end

    @inline function mp_index_current_exact(x, q, rho, kappa, r, beta, ε)
        fxx = float(f_metric2(x, x; q=q, ρ=rho, κ=kappa, r=r, β=beta, ε=ε))
        gxx = float(g_metric2(x, x; q=q, ρ=rho, κ=kappa,      β=beta, ε=ε))
        return (isfinite(fxx) && isfinite(gxx) && gxx != 0.0) ? (fxx / gxx) : -Inf
    end

    function simulate_one_run!(run_idx::Int)
        rng = MersenneTwister(run_seeds[run_idx])

        X_rand = copy(x_vec_f)
        X_myop = copy(x_vec_f)
        X_rr   = copy(x_vec_f)
        X_mp   = copy(x_vec_f)

        rr_start = 1

        R_rand = 0.0
        R_myop = 0.0
        R_rr   = 0.0
        R_mp   = 0.0

        myopic_value = zeros(Float64, N)
        mpvals       = zeros(Float64, N)

        active_rand_mask = falses(N)
        active_myop_mask = falses(N)
        active_rr_mask   = falses(N)
        active_mp_mask   = falses(N)

        perm = collect(1:N)

        for t in 1:T
            disc = β_powers[t]

            fill!(active_rand_mask, false)
            shuffle!(rng, perm)
            active_rand_mask[perm[1:M]] .= true

            reward_t = 0.0
            for i in 1:N
                q  = q_vec[i]
                rho = rho_vec[i]
                kappa = kappa_vec[i]
                r  = r_vec[i]
                xi = X_rand[i]

                if active_rand_mask[i]
                    p_ack = kappa * xi
                    if rand(rng) < p_ack
                        reward_t += r
                        X_rand[i] = q + rho
                    else
                        X_rand[i] = phi1(xi, q, rho, kappa)
                    end
                else
                    X_rand[i] = phi0(xi, q, rho)
                end
            end
            R_rand += disc * reward_t

            for i in 1:N
                myopic_value[i] = r_vec[i] * kappa_vec[i] * X_myop[i]
            end
            choose_topM_mask!(active_myop_mask, myopic_value, M; positive=true)

            reward_t = 0.0
            for i in 1:N
                q  = q_vec[i]
                rho = rho_vec[i]
                kappa = kappa_vec[i]
                r  = r_vec[i]
                xi = X_myop[i]

                if active_myop_mask[i]
                    p_ack = kappa * xi
                    if rand(rng) < p_ack
                        reward_t += r
                        X_myop[i] = q + rho
                    else
                        X_myop[i] = phi1(xi, q, rho, kappa)
                    end
                else
                    X_myop[i] = phi0(xi, q, rho)
                end
            end
            R_myop += disc * reward_t

            fill!(active_rr_mask, false)
            for k in 0:(M-1)
                i = ((rr_start - 1 + k) % N) + 1
                active_rr_mask[i] = true
            end
            rr_start = ((rr_start - 1 + M) % N) + 1

            reward_t = 0.0
            for i in 1:N
                q  = q_vec[i]
                rho = rho_vec[i]
                kappa = kappa_vec[i]
                r  = r_vec[i]
                xi = X_rr[i]

                if active_rr_mask[i]
                    p_ack = kappa * xi
                    if rand(rng) < p_ack
                        reward_t += r
                        X_rr[i] = q + rho
                    else
                        X_rr[i] = phi1(xi, q, rho, kappa)
                    end
                else
                    X_rr[i] = phi0(xi, q, rho)
                end
            end
            R_rr += disc * reward_t

            for i in 1:N
                k = type_id[i]
                xi = X_mp[i]
                mpvals[i] = mp_lookups === nothing ?
                    mp_index_current_exact(xi,
                                           q_types_f[k], rho_types_f[k], kappa_types_f[k],
                                           r_types_f[k], beta, ε) :
                    mp_index_lookup(mp_lookups[k], xi)
            end
            choose_topM_mask!(active_mp_mask, mpvals, M; nonnegative=true)

            reward_t = 0.0
            for i in 1:N
                q  = q_vec[i]
                rho = rho_vec[i]
                kappa = kappa_vec[i]
                r  = r_vec[i]
                xi = X_mp[i]

                if active_mp_mask[i]
                    p_ack = kappa * xi
                    if rand(rng) < p_ack
                        reward_t += r
                        X_mp[i] = q + rho
                    else
                        X_mp[i] = phi1(xi, q, rho, kappa)
                    end
                else
                    X_mp[i] = phi0(xi, q, rho)
                end
            end
            R_mp += disc * reward_t
        end

        total_random[run_idx]     = R_rand
        total_myopic[run_idx]     = R_myop
        total_roundrobin[run_idx] = R_rr
        total_mpindex[run_idx]    = R_mp
        return nothing
    end

    if Threads.nthreads() > 1
        Threads.@threads for run_idx in 1:nruns
            simulate_one_run!(run_idx)
        end
    else
        for run_idx in 1:nruns
            simulate_one_run!(run_idx)
        end
    end

    norm_factor = (1.0 - beta) / N

    function summarize(raw_samples::Vector{Float64})
        samples = norm_factor .* raw_samples
        μ  = mean(samples)
        s  = std(samples)
        se = s / sqrt(length(samples))
        z  = quantile(Normal(), 0.5 + conf_level / 2)
        ci_low  = μ - z * se
        ci_high = μ + z * se
        return (mean = μ,
                stderr = se,
                ci_low = ci_low,
                ci_high = ci_high,
                nruns = length(samples))
    end

    results = Dict{String,NamedTuple}()
    results["random"]          = summarize(total_random)
    results["myopic"]          = summarize(total_myopic)
    results["round_robin"]     = summarize(total_roundrobin)
    results["mp_index"]        = summarize(total_mpindex)

    lagrangian_bound = norm_factor * Lmin
    results["lagrangian_bound"] = (
        lambda_star = λstar,
        raw_Lmin = Lmin,
        mean = lagrangian_bound,
        stderr = 0.0,
        ci_low = lagrangian_bound,
        ci_high = lagrangian_bound,
        nruns = 1
    )

    results["bound_info"] = bound_info

    return results
end

In [ ]:
# ----------------------------------------------------------
# Exact typed scaling experiment with new bound solver
# ----------------------------------------------------------

function run_typed_scaling_experiment_newbound(;
    scenario_grid,
    prop_grid,
    alphaM_values,
    N_values,
    beta = 0.99,
    T_policy = 200,
    nruns_policy = 1000,
    conf_level = 0.95,
    base_seed = 1234,
    ε = 1e-10,
    mp_lookup_ngrid = 1001,
    mp_lookup_mode = :linear,
    x_init_scheme = :linspace,
    x0 = 0.5,
    xlo = 0.0,
    xhi = 1.0,
    shuffle_types = true,
    shuffle_x = true,
    show_progress = true,
    # new bound-solver controls
    bound_λ0::Float64 = 0.0,
    bound_step0::Float64 = 1.0,
    bound_λtol::Float64 = 1e-6,
    bound_max_bisect_iter::Int = 80,
    bound_xatol::Float64 = 1e-10,
    bound_λatol::Float64 = 1e-10,
    bound_maxiter::Int = 200,
    verbose_bound::Bool = false
)

    rows = NamedTuple[]
    counter = 0
    total_cases = length(scenario_grid) * length(prop_grid) * length(alphaM_values) * length(N_values)

    t0 = time()

    for (s_idx, scen) in enumerate(scenario_grid)
        name = hasproperty(scen, :name) ? String(scen.name) : "scenario_$s_idx"

        q_types     = Float64.(scen.q_types)
        rho_types   = Float64.(scen.rho_types)
        kappa_types = Float64.(scen.kappa_types)
        r_types     = Float64.(scen.r_types)

        # Build MP-index lookup tables once per scenario
        mp_lookups = build_type_mp_lookups(
            q_types, rho_types, kappa_types, r_types, beta;
            ε = ε,
            ngrid = mp_lookup_ngrid,
            mode = mp_lookup_mode
        )

        for prop_type in prop_grid
            prop_type_f = Float64.(prop_type)
            K = length(prop_type_f)

            for alphaM in alphaM_values
                for N in N_values
                    counter += 1
                    M = max(1, min(N, round(Int, alphaM * N)))
                    capacity_ratio = M / N

                    instance_seed = base_seed + counter

                    inst = build_typed_instance(
                        N;
                        prop_type = prop_type_f,
                        q_types = q_types,
                        rho_types = rho_types,
                        kappa_types = kappa_types,
                        r_types = r_types,
                        x_init_scheme = x_init_scheme,
                        x0 = x0,
                        xlo = xlo,
                        xhi = xhi,
                        shuffle_types = shuffle_types,
                        shuffle_x = shuffle_x,
                        seed = instance_seed
                    )

                    if show_progress
                        elapsed = time() - t0
                        avg_time = elapsed / max(counter - 1, 1)
                        eta = counter == 1 ? NaN : avg_time * (total_cases - counter)

                        println("[$counter/$total_cases] ",
                                name,
                                " | prop = ", prop_type_f,
                                " | αM_target = ", alphaM,
                                " | realized M/N = ", @sprintf("%.4f", capacity_ratio),
                                " | N = ", N,
                                " | M = ", M,
                                " | elapsed = ", _fmt_seconds(elapsed),
                                " | ETA = ", counter == 1 ? "?" : _fmt_seconds(eta))
                    end

                    results = simulate_multi_project_policies_lookup_newbound(
                        inst.x_vec,
                        inst.type_id_vec,
                        q_types,
                        rho_types,
                        kappa_types,
                        r_types,
                        beta,
                        M;
                        T = T_policy,
                        nruns = nruns_policy,
                        conf_level = conf_level,
                        seed = instance_seed,
                        ε = ε,
                        mp_lookups = mp_lookups,
                        bound_λ0 = bound_λ0,
                        bound_step0 = bound_step0,
                        bound_λtol = bound_λtol,
                        bound_max_bisect_iter = bound_max_bisect_iter,
                        bound_xatol = bound_xatol,
                        bound_λatol = bound_λatol,
                        bound_maxiter = bound_maxiter,
                        verbose_bound = verbose_bound
                    )

                    rand_stats = results["random"]
                    myop_stats = results["myopic"]
                    rr_stats   = results["round_robin"]
                    mp_stats   = results["mp_index"]
                    ub_stats   = results["lagrangian_bound"]
                    binfo      = results["bound_info"]

                    ub = ub_stats.mean
                    rand_mean = rand_stats.mean
                    myop_mean = myop_stats.mean
                    rr_mean   = rr_stats.mean
                    mp_mean   = mp_stats.mean

                    abs_gap_mp = ub - mp_mean
                    rel_gap_mp = ub != 0.0 ? abs_gap_mp / ub : NaN

                    abs_gap_myop = ub - myop_mean
                    abs_gap_rr   = ub - rr_mean
                    abs_gap_rand = ub - rand_mean

                    sqrtN = sqrt(Float64(N))

                    row = (
                        scenario = name,
                        N = N,
                        M = M,
                        alphaM_target = Float64(alphaM),
                        capacity_ratio = M / N,
                        beta = beta,

                        K = K,
                        prop_type = join(string.(round.(prop_type_f; digits=4)), ","),

                        q_types = join(string.(q_types), ","),
                        rho_types = join(string.(rho_types), ","),
                        kappa_types = join(string.(kappa_types), ","),
                        r_types = join(string.(r_types), ","),

                        lambda_star = ub_stats.lambda_star,
                        raw_Lmin = ub_stats.raw_Lmin,
                        lagrangian_bound = ub,

                        random_mean = rand_mean,
                        random_ci_low = rand_stats.ci_low,
                        random_ci_high = rand_stats.ci_high,

                        myopic_mean = myop_mean,
                        myopic_ci_low = myop_stats.ci_low,
                        myopic_ci_high = myop_stats.ci_high,

                        round_robin_mean = rr_mean,
                        round_robin_ci_low = rr_stats.ci_low,
                        round_robin_ci_high = rr_stats.ci_high,

                        mp_index_mean = mp_mean,
                        mp_index_ci_low = mp_stats.ci_low,
                        mp_index_ci_high = mp_stats.ci_high,

                        abs_gap_mp_to_bound = abs_gap_mp,
                        rel_gap_mp_to_bound = rel_gap_mp,
                        N_abs_gap_mp_to_bound = N * abs_gap_mp,
                        sqrtN_abs_gap_mp_to_bound = sqrtN * abs_gap_mp,

                        abs_gap_myopic_to_bound = abs_gap_myop,
                        abs_gap_roundrobin_to_bound = abs_gap_rr,
                        abs_gap_random_to_bound = abs_gap_rand,

                        mp_lookup_ngrid = mp_lookup_ngrid,
                        mp_lookup_mode = String(mp_lookup_mode),
                        x_init_scheme = String(x_init_scheme),
                        seed = instance_seed,

                        bound_nit = binfo.nit,
                        bound_cache_size = binfo.cache_size,
                        bound_ngroups = binfo.ngroups,
                        bound_bracket_subgrad_lo = binfo.bracket_subgrad_lo,
                        bound_bracket_subgrad_hi = binfo.bracket_subgrad_hi
                    )

                    if K >= 1
                        row = merge(row, (n_type1 = inst.n_types[1],))
                    end
                    if K >= 2
                        row = merge(row, (n_type2 = inst.n_types[2],))
                    end

                    push!(rows, row)
                end
            end
        end
    end

    total_time = time() - t0
    println("\nCompleted ", total_cases, " cases in ", _fmt_seconds(total_time), ".")

    return rows
end

In [ ]:
rows = run_typed_scaling_experiment_newbound(
    scenario_grid = scenario_grid,
    prop_grid = prop_grid,
    alphaM_values = [0.05, 0.10, 0.20],
    N_values = [50, 100, 200, 400],
    beta = 0.99,
    T_policy = 200,
    nruns_policy = 1000,
    conf_level = 0.95,
    base_seed = 1234,
    ε = 1e-10,
    mp_lookup_ngrid = 1001,
    mp_lookup_mode = :linear,
    x_init_scheme = :linspace,
    xlo = 0.0,
    xhi = 1.0,
    shuffle_types = true,
    shuffle_x = true,
    show_progress = true,
    bound_λtol = 1e-6,
    bound_max_bisect_iter = 80,
    verbose_bound = false
)

df = DataFrame(rows)
sort!(df, [:scenario, :prop_type, :alphaM_target, :N])

CSV.write("typed_scaling_experiment_newbound.csv", df)

In [ ]:
# -----------------------------------------------------------
# Serious experimental grid for the paper
# All projects start at x0 = 0.5
# -----------------------------------------------------------

# Four type-A profiles ("more self-healing")
A_types = [
    (name = "A_very_selfhealing", q = 0.01, rho = 0.90, kappa = 0.70, r = 1.0),
    (name = "A_mild_selfhealing", q = 0.03, rho = 0.80, kappa = 0.70, r = 1.0),
    (name = "A_balanced",         q = 0.05, rho = 0.70, kappa = 0.70, r = 1.0),
    (name = "A_harder_obs",       q = 0.02, rho = 0.85, kappa = 0.55, r = 1.0)
]

# Three type-B profiles ("more fragile")
B_types = [
    (name = "B_fragile",          q = 0.10, rho = 0.10, kappa = 0.95, r = 1.0),
    (name = "B_very_fragile",     q = 0.15, rho = 0.05, kappa = 0.95, r = 1.0),
    (name = "B_persistent_bad",   q = 0.08, rho = 0.20, kappa = 0.95, r = 1.0)
]

# Cross A-types and B-types to form 12 scenarios
scenario_grid = vec([
    (
        name = "$(A.name)_$(B.name)",
        q_types = [A.q, B.q],
        rho_types = [A.rho, B.rho],
        kappa_types = [A.kappa, B.kappa],
        r_types = [A.r, B.r]
    )
    for A in A_types, B in B_types
])

# Type-A proportions
prop_grid = [[p, 1 - p] for p in 0.1:0.1:0.9]

# Capacity ratios: all multiples of 0.05
alphaM_values = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]

# N values: multiples of 20 so alphaM * N is always an integer
N_values = [100, 200, 400, 800, 1600]

ncases = length(scenario_grid) * length(prop_grid) * length(alphaM_values) * length(N_values)
println("Total number of cases = ", ncases)

rows = run_typed_scaling_experiment_newbound(
    scenario_grid = scenario_grid,
    prop_grid = prop_grid,
    alphaM_values = alphaM_values,
    N_values = N_values,
    beta = 0.99,
    T_policy = 300,
    nruns_policy = 1000,
    conf_level = 0.95,
    base_seed = 1234,
    ε = 1e-10,
    mp_lookup_ngrid = 1001,   # keep the exact-style lookup resolution you already validated
    mp_lookup_mode = :linear,
    x_init_scheme = :constant,
    x0 = 0.5,
    xlo = 0.0,
    xhi = 1.0,
    shuffle_types = true,
    shuffle_x = false,        # irrelevant when x is constant, but explicit
    show_progress = true,
    bound_λtol = 1e-6,
    bound_max_bisect_iter = 80,
    verbose_bound = false
)

df = DataFrame(rows)
sort!(df, [:scenario, :prop_type, :alphaM_target, :N])

CSV.write("typed_scaling_experiment_paper_run.csv", df)

In [ ]:
combine(groupby(df, :scenario),
    :t_bound_case => mean => :mean_t_bound,
    :t_init_counts_case => mean => :mean_t_init,
    :t_mc_case => mean => :mean_t_mc,
    :t_case_total => mean => :mean_t_case_total
)

select(df,
    :scenario, :prop_type, :alphaM_target, :capacity_ratio, :N, :M,
    :t_bound_case, :t_init_counts_case, :t_mc_case, :t_case_total
)

In [ ]:
scenario_grid = [
    (
        name = "A_selfhealing_B_fragile",
        q_types = [0.01, 0.10],
        rho_types = [0.90, 0.10],
        kappa_types = [0.70, 0.95],
        r_types = [1.0, 1.0]
    ),
    (
        name = "A_mild_B_fragile",
        q_types = [0.03, 0.10],
        rho_types = [0.80, 0.10],
        kappa_types = [0.70, 0.95],
        r_types = [1.0, 1.0]
    )
]

prop_grid = [
    [0.20, 0.80],
    [0.40, 0.60],
    [0.60, 0.40],
    [0.80, 0.20]
]

alphaM_values = [0.05, 0.10, 0.20]
N_values = [50, 100, 200, 400]

rows = run_typed_scaling_experiment_binned(
    scenario_grid = scenario_grid,
    prop_grid = prop_grid,
    alphaM_values = alphaM_values,
    N_values = N_values,
    beta = 0.99,
    T_policy = 300,
    nruns_policy = 1000,
    conf_level = 0.95,
    base_seed = 1234,
    ε = 1e-10,
    state_ngrid = 101,
    mp_lookup_ngrid = 201,
    mp_lookup_mode = :linear,
    x_init_scheme = :linspace,
    xlo = 0.0,
    xhi = 1.0,
    shuffle_types = true,
    shuffle_x = true,
    show_progress = true
)

df = typed_binned_rows_to_dataframe(rows)
CSV.write("typed_scaling_experiment_binned.csv", df)

first(df, 10)

In [ ]:
using PGFPlotsX

"""
    plot_indices_two_types(p_types, q_types, r_types, beta; filename="")

For K = 2 project types with parameters (p_k, q_k, r_k), k = 1,2, and discount
factor `beta`, plot their health-based MP indices m_k(x) and the common myopic
index r x as functions of the belief state x ∈ [0,1], without adjusting
for intervention costs.

Assumes a function

    mpindex(x::Float64, p::Float64, q::Float64, r::Float64, beta::Float64) -> Float64

is defined and returns the MP index m(x) for a single project.
"""
function plot_indices_two_types(p_types::AbstractVector{<:Real},
                                q_types::AbstractVector{<:Real},
                                r_types::AbstractVector{<:Real},
                                beta::Float64;
                                filename::AbstractString = "")

    @assert length(p_types) == 2 "p_types must have length 2"
    @assert length(q_types) == 2 "q_types must have length 2"
    @assert length(r_types) == 2 "r_types must have length 2"
    @assert 0.0 < beta < 1.0 "beta must be in (0,1)"
    @assert r_types[1] == r_types[2] "this plot assumes r_1 = r_2 (common myopic index)"

    # x-grid on [0,1]
    xs = collect(range(0.0, 1.0; length = 501))

    # Preallocate arrays for indices
    mA_mp    = similar(xs)  # MP index for type A (type 1)
    mB_mp    = similar(xs)  # MP index for type B (type 2)
    m_myopic = similar(xs)  # common myopic index r x

    pA, pB = float.(p_types)
    qA, qB = float.(q_types)
    r      = float(r_types[1])

    # Compute indices for each x
    for (i, x) in enumerate(xs)
        m_myopic[i] = r * x
        mA_mp[i]    = mpindex(x, pA, qA, r, beta)
        mB_mp[i]    = mpindex(x, pB, qB, r, beta)
    end

    gp = @pgf GroupPlot(
        {
            group_style = {
                group_size = "2 by 1",
                horizontal_sep = "2.5cm",
            },
        },

        # Left panel: x in [0,1]
        Axis(
            {
                xmin = 0,
                xmax = 1,
                ymin = 0,
                axis_lines = "left",
                xlabel = "x",
                ylabel = "Index value",
                legend_pos = "north west",
            },
            Plot(
                {thin, color = "black"},
                Table(x = xs, y = mA_mp),
            ),
            Plot(
                {thin, "dotted", color = "black"},
                Table(x = xs, y = mB_mp),
            ),
            Plot(
                {thin, "dashed", color = "black"},
                Table(x = xs, y = m_myopic),
            ),
            Legend([
                "Type A MP",
                "Type B MP",
                "Myopic",
            ]),
        ),

        # Right panel: zoom x in [0,0.5]
        Axis(
            {
                xmin = 0,
                xmax = 0.5,
                ymin = 0,
                axis_lines = "left",
                xlabel = "x",
                ylabel = "Index value",
            },
            Plot(
                {thin, color = "black"},
                Table(x = xs, y = mA_mp),
            ),
            Plot(
                {thin, "dotted", color = "black"},
                Table(x = xs, y = mB_mp),
            ),
            Plot(
                {thin, "dashed", color = "black"},
                Table(x = xs, y = m_myopic),
            ),
        ),
    )

    if filename != ""
        PGFPlotsX.save(filename, gp)  # e.g. "indices_two_types.pdf"
    end

    return gp
end

In [ ]:
# Type 1 (self-healing-ish), Type 2 (fragile-ish)
p_types = [0.10, 0.35]
q_types = [0.05, 0.01]
r_types = [1.0, 1.0]
beta    = 0.99

ax = plot_indices_two_types(p_types, q_types, r_types, beta;
                            filename = "indices_two_types.pdf")

In [ ]:
using Random
using CSV
using DataFrames
using Dates

# helper: format seconds nicely
function _fmt_seconds(s::Real)
    s = max(0.0, s)
    si = round(Int, s)
    h, r = divrem(si, 3600)
    m, sec = divrem(r, 60)
    h > 0 ? "$(h)h $(m)m $(sec)s" :
    m > 0 ? "$(m)m $(sec)s" : "$(sec)s"
end

function run_two_type_grid_to_csv(filename::AbstractString;
                                  p_grid::AbstractVector,
                                  q_grid::AbstractVector,
                                  beta::Float64,
                                  N::Int,
                                  prop_type_A_list::AbstractVector,
                                  M_ratios::AbstractVector,
                                  T_policy::Int,
                                  nruns_policy::Int,
                                  conf_level::Float64 = 0.95,
                                  delta_rho::Float64 = 0.05,
                                  base_seed::Int = 1234,
                                  # NEW:
                                  show_progress::Bool = true,
                                  update_every::Int = 25,                 # print every N completed cases
                                  min_seconds_between_updates::Real = 2.0 # and/or at least every this many seconds
                                  )

    @assert 0.0 < beta < 1.0 "beta must be in (0,1)"

    # Filter feasible (p,q) pairs (no p+q too close to 1)
    feasible_pairs = [(p,q) for p in p_grid, q in q_grid
                      if p > 0 && q > 0 && p + q <= 1 - delta_rho]

    if isempty(feasible_pairs)
        error("No feasible (p,q) pairs satisfy p>0, q>0, p+q ≤ 1-delta_rho. Check p_grid, q_grid, delta_rho.")
    end

    # Precompute valid ordered type-pairs so we can compute an accurate total for ETA
    type_pairs = [(pA,qA,pB,qB) for (pA,qA) in feasible_pairs, (pB,qB) in feasible_pairs
                  if (pA < pB && qA > qB)]

    if isempty(type_pairs)
        error("No (A,B) type pairs satisfy pA < pB and qA > qB under the feasible_pairs set.")
    end

    total_cases = length(type_pairs) * length(prop_type_A_list) * length(M_ratios)
    done_cases  = 0
    t0 = time()
    t_last = t0

    rows = Vector{NamedTuple}()
    case_idx = 0

    for (pA, qA, pB, qB) in type_pairs
        # Type-level parameters (we keep r_types = 1.0)
        p_types = [pA, pB]
        q_types = [qA, qB]
        r_types = [1.0, 1.0]

        for prop_type_A in prop_type_A_list
            @assert 0.0 < prop_type_A < 1.0
            nA = round(Int, prop_type_A * N)
            nA = max(1, min(nA, N-1))  # ensure at least 1 of each type
            nB = N - nA

            # Build project-level vectors
            p_vec = vcat(fill(pA, nA), fill(pB, nB))
            q_vec = vcat(fill(qA, nA), fill(qB, nB))
            r_vec = ones(Float64, N)

            case_idx += 1
            case_seed = base_seed + case_idx

            for ratio in M_ratios
                M = max(1, round(Int, ratio * N))
                M = min(M, N)

                # 1) Lagrangian bound (cost-free case: c_vec ≡ 0)
                λ_star, L_star = lambda_star_vector(p_vec, q_vec, r_vec, beta, M;
                                                    tol = 1e-6,
                                                    max_iters = 60,
                                                    return_diagnostics = false)

                # Normalize by (1-β)/N
                L_norm = (1 - beta) * L_star / N

                # 2) Simulate policies (MP, myopic, random, round-robin),
                # using same seed for common randomness
                res = simulate_multi_project_policies(p_vec, q_vec, r_vec, beta, M;
                                                      T = T_policy,
                                                      nruns = nruns_policy,
                                                      conf_level = conf_level,
                                                      seed = case_seed)

                J_mp     = res["mp_index"].mean
                J_myopic = res["myopic"].mean
                J_rand   = haskey(res, "random")      ? res["random"].mean      : NaN
                J_rr     = haskey(res, "round_robin") ? res["round_robin"].mean : NaN

                push!(rows, (
                    pA = pA,
                    qA = qA,
                    pB = pB,
                    qB = qB,
                    beta = beta,
                    N = N,
                    prop_type_A = prop_type_A,
                    nA = nA,
                    nB = nB,
                    M = M,
                    M_ratio = ratio,
                    T = T_policy,
                    nruns = nruns_policy,
                    L_norm = L_norm,
                    J_mp = J_mp,
                    J_myopic = J_myopic,
                    J_random = J_rand,
                    J_roundrobin = J_rr,
                ))

                # ---- NEW: progress / ETA ----
                done_cases += 1
                if show_progress
                    nowt = time()
                    if done_cases == 1 || (done_cases % update_every == 0) || (nowt - t_last ≥ min_seconds_between_updates)
                        elapsed = nowt - t0
                        rate = done_cases / max(elapsed, 1e-9)          # cases/sec
                        eta  = (total_cases - done_cases) / max(rate, 1e-9)

                        pct = 100 * done_cases / total_cases
                        finish_time = Dates.now() + Dates.Second(round(Int, eta))

                        println("Progress: $(round(pct; digits=1))%  ($done_cases/$total_cases)  ",
                                "elapsed=", _fmt_seconds(elapsed), "  ",
                                "ETA≈", _fmt_seconds(eta), "  ",
                                "finish≈", Dates.format(finish_time, "yyyy-mm-dd HH:MM:SS"))
                        flush(stdout)
                        t_last = nowt
                    end
                end
            end
        end
    end

    df = DataFrame(rows)
    CSV.write(filename, df)
    return df
end

In [ ]:
using CSV
using DataFrames
using Statistics

"""
    analyze_two_type_grid(filename; print_summary=true)

Read the CSV produced by `run_two_type_grid_to_csv`, compute:

  - gaps to the Lagrangian bound for each policy: gap_mp, gap_myopic, gap_random, gap_rr
  - gap ratio gap_ratio_mp_over_myopic = gap_mp / gap_myopic
  - reward ratio reward_ratio_mp_over_myopic = J_mp / J_myopic

Return:

  - df::DataFrame with added columns gap_*, gap_ratio_*, reward_ratio_*
  - stats::Dict{String,Any} with summary statistics for each gap and ratio.

If `print_summary=true`, print a tidy multi-line summary.
"""
function analyze_two_type_grid(filename::AbstractString; print_summary::Bool = true)
    df = CSV.read(filename, DataFrame)

    # Avoid division by zero in gaps
    valid_L = df.L_norm .> 0
    if !all(valid_L)
        @warn "Some rows have non-positive L_norm; gaps will be NaN there."
    end

    # Compute gaps to the bound (L_norm - J)/L_norm
    df.gap_mp     = similar(df.L_norm)
    df.gap_myopic = similar(df.L_norm)
    df.gap_random = similar(df.L_norm)
    df.gap_rr     = similar(df.L_norm)

    df.gap_mp[valid_L]     .= (df.L_norm[valid_L] .- df.J_mp[valid_L])       ./ df.L_norm[valid_L]
    df.gap_myopic[valid_L] .= (df.L_norm[valid_L] .- df.J_myopic[valid_L])   ./ df.L_norm[valid_L]
    df.gap_random[valid_L] .= (df.L_norm[valid_L] .- df.J_random[valid_L])   ./ df.L_norm[valid_L]
    df.gap_rr[valid_L]     .= (df.L_norm[valid_L] .- df.J_roundrobin[valid_L])./ df.L_norm[valid_L]

    # Initialize gaps for invalid rows to NaN
    df.gap_mp[.!valid_L]     .= NaN
    df.gap_myopic[.!valid_L] .= NaN
    df.gap_random[.!valid_L] .= NaN
    df.gap_rr[.!valid_L]     .= NaN

    # ------------------------------------------------------------------
    # Gap ratio: gap_ratio_mp_over_myopic = gap_mp / gap_myopic
    # Reward ratio: reward_ratio_mp_over_myopic = J_mp / J_myopic
    # ------------------------------------------------------------------
    n = nrow(df)
    df.gap_ratio_mp_over_myopic    = fill(NaN, n)
    df.reward_ratio_mp_over_myopic = fill(NaN, n)

    eps = 1e-8

    for i in 1:n
        gm = df.gap_mp[i]
        gy = df.gap_myopic[i]
        Jm = df.J_mp[i]
        Jy = df.J_myopic[i]

        # gap ratio: only if both gaps finite and |gap_myopic| not tiny
        if isfinite(gm) && isfinite(gy) && abs(gy) > eps
            df.gap_ratio_mp_over_myopic[i] = gm / gy
        end

        # reward ratio: only if J_myopic not tiny and both finite
        if isfinite(Jm) && isfinite(Jy) && abs(Jy) > eps
            df.reward_ratio_mp_over_myopic[i] = Jm / Jy
        end
    end

    # ------------------------------------------------------------------
    # Summary statistics
    # ------------------------------------------------------------------
    stats = Dict{String,Any}()

    # Helper to compute stats for a given column name
    function col_stats(colname::String)
        v = df[!, colname]
        v = v[.!isnan.(v) .& .!isinf.(v)]
        if isempty(v)
            return "no data"
        else
            return (
                mean   = mean(v),
                std    = std(v),
                min    = minimum(v),
                q25    = quantile(v, 0.25),
                median = quantile(v, 0.50),
                q75    = quantile(v, 0.75),
                max    = maximum(v),
            )
        end
    end

    # Gaps
    for col in ("gap_mp", "gap_myopic", "gap_random", "gap_rr")
        stats[col] = col_stats(col)
    end

    # Ratios
    stats["gap_ratio_mp_over_myopic"]    = col_stats("gap_ratio_mp_over_myopic")
    stats["reward_ratio_mp_over_myopic"] = col_stats("reward_ratio_mp_over_myopic")

    # ------------------------------------------------------------------
    # Pretty printing
    # ------------------------------------------------------------------
    if print_summary
        fmt(x) = lpad(string(round(x; digits = 4)), 8)

        println("Summary of gaps to Lagrangian bound (per-policy):")
        println(rpad("metric", 18), "  mean     std      min      q25      med      q75      max")
        println(repeat("-", 90))
        for name in ("gap_mp", "gap_myopic", "gap_random", "gap_rr")
            st = stats[name]
            if st === "no data"
                println(rpad(name, 18), "  (no data)")
            else
                line = string(
                    rpad(name, 18), "  ",
                    fmt(st.mean),   " ",
                    fmt(st.std),    " ",
                    fmt(st.min),    " ",
                    fmt(st.q25),    " ",
                    fmt(st.median), " ",
                    fmt(st.q75),    " ",
                    fmt(st.max)
                )
                println(line)
            end
        end

        println()
        println("Summary of ratios (MP vs Myopic):")
        println(rpad("metric", 28), "  mean     std      min      q25      med      q75      max")
        println(repeat("-", 100))

        for name in ("gap_ratio_mp_over_myopic", "reward_ratio_mp_over_myopic")
            st = stats[name]
            if st === "no data"
                println(rpad(name, 28), "  (no data)")
            else
                line = string(
                    rpad(name, 28), "  ",
                    fmt(st.mean),   " ",
                    fmt(st.std),    " ",
                    fmt(st.min),    " ",
                    fmt(st.q25),    " ",
                    fmt(st.median), " ",
                    fmt(st.q75),    " ",
                    fmt(st.max)
                )
                println(line)
            end
        end
    end

    return df, stats
end

"""
    mp_vs_myopic_cases(df;
                       gap_ratio_thresh=0.7,
                       reward_ratio_thresh=1.03,
                       print_summary=true)

Given a DataFrame `df` (as returned by `analyze_two_type_grid`) with columns

  - gap_ratio_mp_over_myopic = gap_mp / gap_myopic
  - reward_ratio_mp_over_myopic = J_mp / J_myopic,

identify:

  1. all cases where gap_ratio_mp_over_myopic < gap_ratio_thresh
     (MP closer to the bound than myopic, by at least that factor);
  2. all cases where reward_ratio_mp_over_myopic > reward_ratio_thresh
     (MP reward at least that multiplicative factor above myopic).

Returns `(df_gap_better, df_reward_better)` as DataFrames.

If `print_summary=true`, prints counts and a few representative rows for each set,
with constant columns and some “uninteresting” ones dropped (beta, M, N, nA).
"""
function mp_vs_myopic_cases(df::DataFrame;
                            gap_ratio_thresh::Float64 = 0.7,
                            reward_ratio_thresh::Float64 = 1.03,
                            print_summary::Bool = true)

    gap_ratio    = df.gap_ratio_mp_over_myopic
    reward_ratio = df.reward_ratio_mp_over_myopic

    valid_gap    = .!isnan.(gap_ratio) .& .!isinf.(gap_ratio)
    valid_reward = .!isnan.(reward_ratio) .& .!isinf.(reward_ratio)

    # 1. MP clearly closer to bound than myopic
    gap_better_mask = valid_gap .& (gap_ratio .< gap_ratio_thresh)

    # 2. MP clearly better in reward than myopic
    reward_better_mask = valid_reward .& (reward_ratio .> reward_ratio_thresh)

    df_gap_better     = df[gap_better_mask, :]
    df_reward_better  = df[reward_better_mask, :]

    # Helper: drop non-varying columns and some explicitly unwanted ones
    function reduce_columns(df_sub::DataFrame)
        if nrow(df_sub) == 0
            return df_sub
        end
        # Always hide these if present
        skip = Set(Symbol.(["beta", "M", "N", "nA"]))
        keep = Symbol[]
        for name in names(df_sub)
            col_sym = Symbol(name)
            if col_sym in skip
                continue
            end
            # keep column only if it varies within this subset
            if length(unique(df_sub[!, name])) > 1
                push!(keep, col_sym)
            end
        end
        if isempty(keep)
            return df_sub
        else
            return select(df_sub, keep; copycols=false)
        end
    end

    if print_summary
        println("\nMP vs Myopic comparison:")
        println("  MP closer to bound than myopic (gap_ratio < $(gap_ratio_thresh)): ",
                nrow(df_gap_better))
        println("  MP higher reward than myopic (reward_ratio > $(reward_ratio_thresh)): ",
                nrow(df_reward_better))

        if nrow(df_gap_better) > 0
            println("\nExamples where MP has much smaller gap than myopic:")
            df_g_red = reduce_columns(df_gap_better)
            show(first(df_g_red, min(5, nrow(df_g_red))))
            println()
        end

        if nrow(df_reward_better) > 0
            println("\nExamples where MP has significantly higher reward than myopic:")
            df_r_red = reduce_columns(df_reward_better)
            show(first(df_r_red, min(5, nrow(df_r_red))))
            println()
        end
    end

    # df_mp_gap_better and df_mp_reward_better returned by mp_vs_myopic_cases
    show(first(df_mp_gap_better, 5); allcols=true)
    show(first(df_mp_reward_better, 5); allcols=true)

    return df_gap_better, df_reward_better
end

In [ ]:
p_grid = [0.01, 0.05, 0.10, 0.20, 0.30, 0.35]
q_grid = [0.01, 0.05, 0.10, 0.20, 0.30, 0.35]

df = run_two_type_grid_to_csv("two_type_grid_results.csv";
                              p_grid = collect(p_grid),
                              q_grid = collect(q_grid),
                              beta = 0.99,
                              N = 100,
                              prop_type_A_list = [0.1, 0.3, 0.5, 0.7, 0.9],      # 10%, 30%, 50% A
                              M_ratios = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5],        # capacities
                              T_policy = 300,
                              nruns_policy = 1000,
                              delta_rho = 0.05,
                              base_seed = 1234)

In [ ]:
df_res, stats = analyze_two_type_grid("two_type_grid_results.csv"; print_summary=true)

df_mp_gap_better, df_mp_reward_better =
    mp_vs_myopic_cases(df_res;
                       gap_ratio_thresh=0.7,
                       reward_ratio_thresh=1.03,
                       print_summary=true)

In [ ]:
using CSV
using DataFrames
using Statistics

# ----------------------------------------------------------
# 1. Load the CSV produced by run_two_type_grid_to_csv
# ----------------------------------------------------------
df = CSV.read("two_type_grid_1000.csv", DataFrame)

println("Loaded two_type_grid_1000.csv with ", nrow(df), " rows.")
println("Columns: ", names(df))

# Expected relevant columns from the function you gave:
# :pA, :qA, :pB, :qB,
# :beta, :N,
# :prop_type_A, :nA, :nB,
# :M, :M_ratio,
# :T, :nruns,
# :L_norm,
# :J_mp, :J_myopic, :J_random, :J_roundrobin

# ----------------------------------------------------------
# 2. Compute relative Lagrangian gaps (in percent)
#    gap_pi = 100 * (L_norm - J_pi) / L_norm
# ----------------------------------------------------------

# Protect against L_norm == 0 just in case (should not happen here)
function rel_lagrangian_gap(L_norm, J)
    if isnan(J) || L_norm ≤ 0
        return NaN
    else
        return 100 * (L_norm - J) / L_norm
    end
end

df.gap_mp        = map(rel_lagrangian_gap, df.L_norm, df.J_mp)
df.gap_myopic    = map(rel_lagrangian_gap, df.L_norm, df.J_myopic)
df.gap_random    = map(rel_lagrangian_gap, df.L_norm, df.J_random)
df.gap_roundrobin = map(rel_lagrangian_gap, df.L_norm, df.J_roundrobin)

# ----------------------------------------------------------
# 3. Ratio r = gap_myopic / gap_mp, and bad-myopic flag r ≥ 1.3
# ----------------------------------------------------------

function safe_ratio(num, den)
    if isnan(num) || isnan(den) || den == 0
        return NaN
    else
        return num / den
    end
end

df.ratio_myopic_mp = map(safe_ratio, df.gap_myopic, df.gap_mp)

threshold_ratio = 1.3  # myopic gap at least 30% larger than MP gap
df.is_bad_myopic = df.ratio_myopic_mp .>= threshold_ratio

n_total = nrow(df)
n_bad   = count(df.is_bad_myopic)

println("\nTotal grid points: ", n_total)
println("Bad myopic points (ratio ≥ ", threshold_ratio, "): ",
        n_bad, " (", round(100 * n_bad / n_total, digits = 1), "%)")

# ----------------------------------------------------------
# 4. Look at the worst myopic cases
# ----------------------------------------------------------
bad_df = filter(:is_bad_myopic => ==(true), df)
bad_df_sorted = sort(bad_df, :ratio_myopic_mp, rev = true)

println("\nTop 10 worst myopic / MP ratios:")
first(bad_df_sorted, 10) |> println

# This prints rows with:
# pA, qA, pB, qB, prop_type_A, M_ratio, gap_mp, gap_myopic, ratio_myopic_mp, ...

# ----------------------------------------------------------
# 5. Qualitative summaries: where do bad cases concentrate?
#    (a) by capacity ratio M_ratio
# ----------------------------------------------------------

println("\nShare of bad myopic cases by M_ratio:")
by_M = combine(groupby(df, :M_ratio)) do sdf
    n = nrow(sdf)
    n_bad = count(sdf.is_bad_myopic)
    (; M_ratio = first(sdf.M_ratio),
       n_total = n,
       n_bad = n_bad,
       frac_bad = n_bad / n)
end
println(by_M)

# (b) by prop_type_A
println("\nShare of bad myopic cases by prop_type_A:")
by_rhoA = combine(groupby(df, :prop_type_A)) do sdf
    n = nrow(sdf)
    n_bad = count(sdf.is_bad_myopic)
    (; prop_type_A = first(sdf.prop_type_A),
       n_total = n,
       n_bad = n_bad,
       frac_bad = n_bad / n)
end
println(by_rhoA)

# (c) by pA and qA (type A parameters)
println("\nShare of bad myopic cases by pA:")
by_pA = combine(groupby(df, :pA)) do sdf
    n = nrow(sdf)
    n_bad = count(sdf.is_bad_myopic)
    (; pA = first(sdf.pA),
       n_total = n,
       n_bad = n_bad,
       frac_bad = n_bad / n)
end
println(by_pA)

println("\nShare of bad myopic cases by qA:")
by_qA = combine(groupby(df, :qA)) do sdf
    n = nrow(sdf)
    n_bad = count(sdf.is_bad_myopic)
    (; qA = first(sdf.qA),
       n_total = n,
       n_bad = n_bad,
       frac_bad = n_bad / n)
end
println(by_qA)

# (d) by pB and qB (type B parameters)
println("\nShare of bad myopic cases by pB:")
by_pB = combine(groupby(df, :pB)) do sdf
    n = nrow(sdf)
    n_bad = count(sdf.is_bad_myopic)
    (; pB = first(sdf.pB),
       n_total = n,
       n_bad = n_bad,
       frac_bad = n_bad / n)
end
println(by_pB)

println("\nShare of bad myopic cases by qB:")
by_qB = combine(groupby(df, :qB)) do sdf
    n = nrow(sdf)
    n_bad = count(sdf.is_bad_myopic)
    (; qB = first(sdf.qB),
       n_total = n,
       n_bad = n_bad,
       frac_bad = n_bad / n)
end
println(by_qB)

In [ ]:
using CSV
using DataFrames
using Statistics
using Printf

# ----------------------------------------------------------
# 1. Load results (if not already loaded)
# ----------------------------------------------------------
# If you already have `df` from before, you can skip this line.
# df = CSV.read("two_type_grid_results.csv", DataFrame)
df = CSV.read("two_type_grid_1000.csv", DataFrame)

println("Loaded two_type_grid_results.csv with ", nrow(df), " rows.")

# Expected columns: :pA, :qA, :pB, :qB, :prop_type_A, :M_ratio, :J_mp, :J_myopic, ...

# ----------------------------------------------------------
# 2. Compute reward ratio R = J_mp / J_myopic
# ----------------------------------------------------------

function safe_ratio(num, den)
    if isnan(num) || isnan(den) || den == 0
        return NaN
    else
        return num / den
    end
end

df.ratio_mp_myopic = map(safe_ratio, df.J_mp, df.J_myopic)

# Drop NaNs if any (should not happen in your current setup, but just in case)
ratio_vals = filter(!isnan, df.ratio_mp_myopic)

# ----------------------------------------------------------
# 3. Summary statistics for J_mp / J_myopic
# ----------------------------------------------------------

ratio_mean = mean(ratio_vals)
ratio_std  = std(ratio_vals)
ratio_min  = minimum(ratio_vals)
q25, q50, q75 = quantile(ratio_vals, (0.25, 0.50, 0.75))
ratio_max  = maximum(ratio_vals)

println("\nSummary of reward ratio J_mp / J_myopic:")
@printf("mean   = %.4f\n", ratio_mean)
@printf("std    = %.4f\n", ratio_std)
@printf("min    = %.4f\n", ratio_min)
@printf("q25    = %.4f\n", q25)
@printf("median = %.4f\n", q50)
@printf("q75    = %.4f\n", q75)
@printf("max    = %.4f\n", ratio_max)

# If you want a small DataFrame that you can use to format a LaTeX table:
summary_df = DataFrame(
    metric = ["J_mp / J_myopic"],
    mean   = [ratio_mean],
    std    = [ratio_std],
    min    = [ratio_min],
    q25    = [q25],
    median = [q50],
    q75    = [q75],
    max    = [ratio_max],
)

println("\nSummary DataFrame:")
println(summary_df)

# ----------------------------------------------------------
# 4. Top 10 worst cases (largest J_mp / J_myopic)
# ----------------------------------------------------------

df_nonan = filter(:ratio_mp_myopic => x -> !isnan(x), df)
df_sorted = sort(df_nonan, :ratio_mp_myopic, rev = true)

worst10 = first(df_sorted, 10)

# Keep only the requested columns
worst10_small = select(
    worst10,
    :pA, :qA, :pB, :qB, :prop_type_A, :M_ratio, :ratio_mp_myopic
)

println("\nTop 10 instances by reward ratio J_mp / J_myopic:")
println(worst10_small)

# ----------------------------------------------------------
# 5. (Optional) Print worst 10 in LaTeX-friendly format
# ----------------------------------------------------------

println("\nLaTeX rows for worst 10 (pA, qA, pB, qB, rho_A, M/N, ratio):")
for row in eachrow(worst10_small)
    @printf("%.2f & %.2f & %.2f & %.2f & %.2f & %.2f & %.3f \\\\\n",
            row.pA, row.qA, row.pB, row.qB,
            row.prop_type_A, row.M_ratio, row.ratio_mp_myopic)
end

In [ ]:
using CSV
using DataFrames
using Statistics
using Printf

# -----------------------------
# Load
# -----------------------------
df = CSV.read("two_type_grid_1000.csv", DataFrame)
@info "Loaded" nrow(df) names(df)

# -----------------------------
# Helpers
# -----------------------------
# Relative Lagrangian gap in percent: 100*(L - J)/L
rel_gap(L, J) = (isnan(J) || L <= 0) ? NaN : 100 * (L - J) / L

safe_ratio(num, den) = (isnan(num) || isnan(den) || den == 0) ? NaN : num / den

# Summary statistics function (matches your table columns)
function summarize_vec(v::AbstractVector{<:Real})
    vv = collect(skipmissing(v))
    vv = filter(!isnan, vv)
    return (
        mean   = mean(vv),
        std    = std(vv),
        min    = minimum(vv),
        q25    = quantile(vv, 0.25),
        median = quantile(vv, 0.50),
        q75    = quantile(vv, 0.75),
        max    = maximum(vv),
    )
end

# Nice LaTeX row printer
function latex_row(name::AbstractString, s; digits=2)
    @printf "%s & %.*f & %.*f & %.*f & %.*f & %.*f & %.*f & %.*f \\\\\n" name digits s.mean digits s.std digits s.min digits s.q25 digits s.median digits s.q75 digits s.max
end

# -----------------------------
# Compute gaps + ratio
# -----------------------------
df.gap_mp         = map(rel_gap, df.L_norm, df.J_mp)
df.gap_myopic     = map(rel_gap, df.L_norm, df.J_myopic)
df.gap_random     = map(rel_gap, df.L_norm, df.J_random)
df.gap_roundrobin = map(rel_gap, df.L_norm, df.J_roundrobin)

df.ratio_myopic_mp = map(safe_ratio, df.gap_myopic, df.gap_mp)

# -----------------------------
# TABLE A (Main paper): gap summary for 4 policies
# -----------------------------
summ_mp     = summarize_vec(df.gap_mp)
summ_myopic = summarize_vec(df.gap_myopic)
summ_rand   = summarize_vec(df.gap_random)
summ_rr     = summarize_vec(df.gap_roundrobin)

gap_summary = DataFrame(
    Policy = ["MP index", "Myopic index", "Random", "Round-robin"],
    mean   = [summ_mp.mean, summ_myopic.mean, summ_rand.mean, summ_rr.mean],
    std    = [summ_mp.std,  summ_myopic.std,  summ_rand.std,  summ_rr.std],
    min    = [summ_mp.min,  summ_myopic.min,  summ_rand.min,  summ_rr.min],
    q25    = [summ_mp.q25,  summ_myopic.q25,  summ_rand.q25,  summ_rr.q25],
    median = [summ_mp.median, summ_myopic.median, summ_rand.median, summ_rr.median],
    q75    = [summ_mp.q75,  summ_myopic.q75,  summ_rand.q75,  summ_rr.q75],
    max    = [summ_mp.max,  summ_myopic.max,  summ_rand.max,  summ_rr.max],
)

println("\n=== TABLE A: Summary of relative Lagrangian gaps (percent) ===")
println(gap_summary)

println("\nLaTeX rows for tab:gaps-summary (mean/std/min/q25/median/q75/max):")
latex_row("MP index",      summ_mp;     digits=2)
latex_row("Myopic index",  summ_myopic; digits=2)
latex_row("Random",        summ_rand;   digits=2)
latex_row("Round-robin",   summ_rr;     digits=2)

# -----------------------------
# TABLE C (Main paper, optional): reward ratio summary J_mp / J_myopic
# -----------------------------
df.ratio_J_mp_myopic = df.J_mp ./ df.J_myopic
summ_ratio = summarize_vec(df.ratio_J_mp_myopic)

ratio_summary = DataFrame(
    metric = ["J_mp / J_myopic"],
    mean   = [summ_ratio.mean],
    std    = [summ_ratio.std],
    min    = [summ_ratio.min],
    q25    = [summ_ratio.q25],
    median = [summ_ratio.median],
    q75    = [summ_ratio.q75],
    max    = [summ_ratio.max],
)

println("\n=== TABLE C: Summary of reward ratio J_mp / J_myopic ===")
println(ratio_summary)

# -----------------------------
# TABLE B (Supplement): top-10 worst myopic/MP gap ratios
# -----------------------------
worst10 = sort(df, :ratio_myopic_mp, rev=true)[1:10, [
    :pA, :qA, :pB, :qB, :prop_type_A, :M_ratio,
    :gap_mp, :gap_myopic, :gap_random, :gap_roundrobin,
    :ratio_myopic_mp
]]

println("\n=== TABLE B: Top 10 by ratio gap_myopic / gap_mp (supplement) ===")
println(worst10)

println("\nLaTeX rows for Table S1 (pA,qA,pB,qB,rhoA,M/N,gap_MP,gap_myopic,gap_random,gap_rr,ratio):")
for r in eachrow(worst10)
    @printf("%.2f & %.2f & %.2f & %.2f & %.2f & %.2f & %.3f & %.3f & %.3f & %.3f & %.3f \\\\\n",
            r.pA, r.qA, r.pB, r.qB, r.prop_type_A, r.M_ratio,
            r.gap_mp, r.gap_myopic, r.gap_random, r.gap_roundrobin,
            r.ratio_myopic_mp)
end
